In [ ]:
import os
import DataManipulation as DM
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import scipy
import scipy.stats as stats 
import importlib
import warnings
import StepFunctionModelWithNoise as sfmConNoise
import NonGenSSMWithNoise as NonGenSSM
import HMM as hmm
import QLearning as qLearn
import NonGenDualProcessSSMWithNoise as SSMImp
import HMMImplicit as hmmImp
import QLearningImplicit as qLearnImp
import DualProcessKnownImplicit as SSMImpKI
import HMMKnownImplicit as hmmKI
import QLearningKnownImplicit as qLearnKI
import math
from pathlib import Path
from statsmodels.stats.multitest import multipletests
import colorsys
from scipy.stats import bootstrap
from matplotlib.container import BarContainer
from matplotlib.ticker import MaxNLocator, AutoMinorLocator, FixedLocator
from scipy.stats import linregress
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from scipy.stats import ttest_1samp, ttest_rel                           
from scipy import ndimage
from scipy.stats import sem
from matplotlib.patheffects import withStroke
from collections import defaultdict
import matplotlib.colors as mcolors
import matplotlib.patheffects as patheffects
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
from scipy import signal
from scipy.stats import norm

from matplotlib import cm                      
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from collections import defaultdict
from IPython.display import display                                

from matplotlib.colors import to_rgb

from scipy.stats import ks_2samp                         
from scipy.stats import ttest_rel, t, norm
from scipy.stats import norm as norm_dist
from scipy.signal import convolve
from matplotlib.colors import Normalize, PowerNorm
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from scipy.stats import pearsonr
import gc
from IPython import get_ipython

folderPath = Path("paperFigures")
folderPath.mkdir(exist_ok=True)
warnings.filterwarnings('ignore')
!pip freeze > requirements.txt
targetFolder = 'paperFigures/'
%matplotlib inline
np.random.seed(42)

In [ ]:

def auto_gc(*args, **kwargs):
    plt.close('all')
    gc.collect()

get_ipython().events.register('post_run_cell', auto_gc)


In [ ]:
_original_savefig = plt.savefig

                                                                                     
def _despined_savefig(*args, **kwargs):
    fig = plt.gcf()
    sns.despine(fig=fig)
   
    for ax in fig.axes:
        ax.tick_params(
            bottom=True, 
            left=True,     
        )
        ax.grid(False)                                                         
   
    return _original_savefig(*args, **kwargs)

                                                     
plt.savefig = _despined_savefig

In [ ]:

                               
BrudnerEightDelay = pd.read_csv('BrudnerData.csv')
rDelays = [5000]          
for rd in rDelays:
    df = BrudnerEightDelay[np.round(BrudnerEightDelay['reward_delay'])== rd]
                                          

DM.addCycleColumn(df,cLen=4,blLen=64,ppIndicator='participant')
rDelay = 5000
df['blockNum'] = 0
                                                      
df['participantNum'] = df['participant']
df['aim'] = df['relativeHandAngle']
df['rotation'] = np.round(-df['rotation'])
isRotation = (df['rotation'] != 0).astype(int)
prevCumsum = isRotation.cumsum().shift(1).fillna(0)
hasPrevRotation = prevCumsum > 0
df['trialNum'] = df['trial']
df['phase'] = np.where(df['rotation'] != 0, 'rotation',
                       np.where(~hasPrevRotation, 'baseline', 'washout'))
df['targetPosition'] = df['target_angle']

In [ ]:
CGDat = pd.read_csv('CGData_.csv')
                                                      
                                               
exp1 = CGDat[(CGDat["targetCount"] == 1) &
        (CGDat["hasAutocorrection"] == 0)]
exp2 = CGDat[(CGDat["targetCount"] == 8) &
        (CGDat["hasAutocorrection"] == 0)]
exp3 = CGDat[(CGDat["targetCount"] == 1) &
        (CGDat["hasAutocorrection"] == 1)]
exp4 = CGDat[(CGDat["targetCount"] == 8) &
        (CGDat["hasAutocorrection"] == 1)]
dfs = [exp1,exp2,exp3,exp4]
BTDat = pd.read_csv('BTData_.csv')
targetFolder = 'paperFigures/'

In [ ]:
           
            
importlib.reload(DM)
markSize = 400 
                                        
oneTarDfs = [dfs[0],dfs[2]]
for i in range(len(oneTarDfs)):
    oneTarDfs[i] = oneTarDfs[i][(oneTarDfs[i]['blockTrial'] >= -5) & (oneTarDfs[i]['blockTrial'] < 35)]
eightTarFigSize = (12,10)
otfsMod = 1
oneTarFigSize =  (eightTarFigSize[0]*otfsMod,eightTarFigSize[1])
yLox = [0,15,30,45,60,90]
yTix = [0,15,30,45,60,90]
%matplotlib inline
DM.plotTrialSeries(oneTarDfs[0],markSize=markSize,figSize=oneTarFigSize,save=True,findStartAsymp=False,printN=True,fileName=targetFolder+'basicTrialSeriesAim1Tar.svg',xLim=(-6,35),killLabels=True,customY=True,yLox=yLox,yTix=yTix,yLim=(-5,98))
                                                                                                                                                                                                                                         
DM.plotTrialSeries(dfs[1],markSize=markSize,figSize=eightTarFigSize,save=True,findStartAsymp=False,printN=True,x='cycle',xNumP=0.07,fileName=targetFolder+'basicTrialSeriesAim8Tar.svg',xLim=(-6,45),killLabels=True,customY=True,yLox=yLox,yTix=yTix,yLim=(-5,98))
                                                                                                                                                                                                                                                          

In [ ]:
               
importlib.reload(DM)
ylim=(-7,110)
figSize=(10,10)
fileName = 'paperFigures/barPlot1Tar.svg'
yLox = [0,15,30,45,60,90]
yTix = [0,15,30,45,60,90]
DM.allRotBarPlot(dfs[0],xIdcs=[np.arange(25,30)],fileName=fileName,ylim=ylim,figSize=figSize,customY=True,yLox=yLox,yTix=yTix)

In [ ]:
                
importlib.reload(DM)
fileName = 'paperFigures/barPlot8Tar.svg'
DM.allRotBarPlot(dfs[1],xIdcs=[np.arange(280,320)],fileName=fileName,ylim=ylim,figSize=figSize,customY=True,yLox=yLox,yTix=yTix)

In [ ]:
                                                                                                                        
importlib.reload(DM)

xTix = [[np.arange(-5,45,5)[it] for it in range(len(np.arange(-5,45,5))) if it%2==1] for p in range(6)]
xLox = xTix
xLabs= ['Rotation Cycle','Rotation Cycle']
yLabs=[np.arange(200,-181,-40)[it] for it in range(10) if it%2==1]

oneTarParams = []
eightTarParams = []
onePeak = []
oneLow = []
oneDecrease = []
oneTimeToPeak = []
oneMnAfterPeak = []
oneStdAfterPeak = []
eightPeak = []
eightLow = []
eightDecrease = []
eightTimeToPeak = []
eightMnAfterPeak = []
eightStdAfterPeak = []
btTarParams = []
btPeak = []
btLow = []
btDecrease = []
btTimeToPeak = []
btMnAfterPeak = []
btStdAfterPeak = []
orps=[]
orls=[]
erps=[]
erls=[]
brps=[]
brls=[]
oPkTs = []
ePkTs = []
bPkTs = []
BTRotDat = BTDat[(BTDat['cycle'] >= 0) & (BTDat['cycle'] < 40) ]

rots = [-15,-30,-45,-60,-90]
for r in rots:
    dat = dfs[2][dfs[2]['blockRot'] == r]
    dat = dat[(dat['blockTrial'] >= 0) & (dat['blockTrial'] <30)] 
    pars,peak,low,aimDec,peakTrial,mnAfterPeak,stdAfterPeak,rawPeaks,rawLows,peakTs= DM.componentTrialSeries([dat],x='blockTrial',supTitle='',expTitles=[r],y0='aim',y="imp",y2='totalAngle',
                            markSize=markSize,idlLines=True,figSize=(12,10),ylim=(-2,-(r-7)),caption='',xLabs=xLabs,yLabs=yLabs,yTickStepSize=1,
                            legTitle='',legLabs=[''],xTix=xTix,xLox=xLox,capOffset=0.7,fileName=targetFolder+'Component1TarBasic'+str(r)+'.svg',save=True,singlePane=True)
    
    oneTarParams.append(pars)
    onePeak.append(peak)
    oneLow.append(low)
    oneDecrease.append(aimDec)
    oneTimeToPeak.append(peakTrial)
    oneMnAfterPeak.append(mnAfterPeak)
    oneStdAfterPeak.append(stdAfterPeak)
    orps.append(rawPeaks)
    orls.append(rawLows)
    oPkTs.append([peakTs])
    
    dat = dfs[3][dfs[3]['blockRot'] == r]
    dat = dat[(dat['cycle'] >= 0) & (dat['cycle'] < 40)] 
    pars,peak,low,aimDec,peakTrial,mnAfterPeak,stdAfterPeak,rawPeaks,rawLows,peakTs = DM.componentTrialSeries([dat],x='cycle',supTitle='',expTitles=[r],y0='aim',y="imp",y2='totalAngle',
                            markSize=markSize,idlLines=True,figSize=(12,10),ylim=(-2,-(r-7)),caption='',xLabs=xLabs,yLabs=yLabs,yTickStepSize=1,
                            legTitle='',legLabs=[''],xTix=xTix,xLox=xLox,capOffset=0.7,fileName=targetFolder+'Component8TarBasic'+str(r)+'.svg',save=True,singlePane=True)
  
    eightTarParams.append(pars)
    eightPeak.append(peak)
    eightLow.append(low)
    eightDecrease.append(aimDec)
    eightTimeToPeak.append(peakTrial)
    eightMnAfterPeak.append(mnAfterPeak)
    eightStdAfterPeak.append(stdAfterPeak)
    erps.append(rawPeaks)
    erls.append(rawLows)
    ePkTs.append([peakTs])
    
    dat = BTRotDat[BTRotDat['blockRot'] == r]
    pars,peak,low,aimDec,peakTrial,mnAfterPeak,stdAfterPeak,rawPeaks,rawLows,peakTs = DM.componentTrialSeries([dat],x='cycle',supTitle='',expTitles=[r],y0='aim',y="imp",y2='totalAngle',
                            markSize=markSize,idlLines=True,figSize=(12,10),ylim=(-2,-(r-7)),caption='',xLabs=xLabs,yLabs=yLabs,
                            legTitle='',legLabs=[''],xTix=xTix,xLox=xLox,capOffset=0.7,fileName=targetFolder+'BTBasic'+str(r)+'.svg',save=True,rotationOnly=True,singlePane=True)
    
    btTarParams.append(pars)
    btPeak.append(peak)
    btLow.append(low)
    btDecrease.append(aimDec)
    btTimeToPeak.append(peakTrial)
    btMnAfterPeak.append(mnAfterPeak)
    btStdAfterPeak.append(stdAfterPeak)
    brps.append(rawPeaks)
    brls.append(rawLows)
    bPkTs.append([peakTs])

In [ ]:
                                                             
           
importlib.reload(DM)
rots=[-90,-60,-45,-30,-15]
for r in rots:
    xTix = [[np.arange(-5,45,5)[it] for it in range(len(np.arange(-5,45,5))) if it%2==1] for p in range(6)]
    xLox = xTix
    xLabs= ['Rotation Cycle','Rotation Cycle']
    yLabs=[np.arange(200,-181,-40)[it] for it in range(10) if it%2==1]
    caption = ''
    dDat=dfs[3][dfs[3]['blockRot'] == r]
    bDat=BTDat[BTDat['blockRot'] == r]
    dDat = dDat[dDat['cycle']>=0]
    bDat = bDat[bDat['cycle']>=0]
    DM.muiltiExpTrialSeries([dDat,bDat],x='cycle',supTitle='',expTitles=[r],y0='aim',y="imp",y2='totalAngle',
                            markSize=markSize,idlLines=True,figSize=(12,10),ylim=(-2,-(r-7)),caption=caption,xLabs=xLabs,yLabs=yLabs,
                            xTix=xTix,xLox=xLox,capOffset=0,fileName=targetFolder+str(r)+'.svg',save=True,yTickStepSize=1)

In [ ]:
          
importlib.reload(DM)
erds = [np.subtract(np.array(i),np.array(j)) for i,j in zip(erps,erls)]
ords = [np.subtract(np.array(i),np.array(j)) for i,j in zip(orps,orls)]
brds = [np.subtract(np.array(i),np.array(j)) for i,j in zip(brps,brls)]
tPeakVal = DM.boxPlotDifference(erps,brps,yLab='peakVal',filename=targetFolder+'peakVal.svg',ylim=(-15,115),yLox=[-15,0,15,30,45,60,90],yTix=[-15,0,15,30,45,60,90],squeeze=False)
tFinalVal = DM.boxPlotDifference(erls,brls,yLab='finalVal',filename=targetFolder+'finalVal.svg',ylim=(-15,115),yLox=[-15,0,15,30,45,60,90],yTix=[-15,0,15,30,45,60,90],squeeze=False)
tDecreas = DM.boxPlotDifference(erds,brds,yLab='decrease',filename=targetFolder+'decrease.svg',ylim=(-1,60),yLox=[-30,-15,0,15,30,45,60],yTix=[-30,-15,0,15,30,45,60],squeeze=False,idlLines=False)
tPeakCyc = DM.boxPlotDifference(ePkTs,bPkTs,yLab='peakCyc',filename=targetFolder+'peakCyc.svg',ylim=(0,40),idlLines=False)

In [ ]:

dataframes = {
    'tPeakVal': tPeakVal,
    'tFinalVal': tFinalVal,
    'tDecreas': tDecreas,
    'tPeakCyc': tPeakCyc
}

def test_exp_difference(df, metric_name):
    results = []
    unique_rotations = sorted(df['rotation'].unique())
    
    for rotation in unique_rotations:
        subset = df[df['rotation'] == rotation]
        exp0 = subset[subset['exp'] == 0]['val'].values
        exp1 = subset[subset['exp'] == 1]['val'].values
        
        if len(exp0) > 1 and len(exp1) > 1: 
            t_stat, p_value = stats.ttest_ind(exp0, exp1, equal_var=True)

            mean_exp0 = np.mean(exp0)
            mean_exp1 = np.mean(exp1)
            
            results.append({
                'Metric': metric_name,
                'Rotation': rotation,
                'N_exp0': len(exp0),
                'N_exp1': len(exp1),
                'Mean_exp0': mean_exp0,
                'Mean_exp1': mean_exp1,
                't_stat': t_stat,
                'p_value': p_value
            })
        else:
            print(f"Warning: Insufficient data for rotation {rotation} in {metric_name}")
    
    return pd.DataFrame(results)


all_results = []
for name, df in dataframes.items():
    result_df = test_exp_difference(df, name)
    all_results.append(result_df)


combined_results = pd.concat(all_results, ignore_index=True)


m = 5                                                                               

combined_results['bonferroni_p'] = np.minimum(1, combined_results['p_value'] * m)

_, holm_p, _, _ = multipletests(combined_results['p_value'], alpha=0.05, method='holm', is_sorted=False)
combined_results['holm_p'] = holm_p

combined_results['raw_significant (p<0.05)'] = combined_results['p_value'] < 0.05
combined_results['bonferroni_significant (adj_p<0.05)'] = combined_results['bonferroni_p'] < 0.05
combined_results['holm_significant (adj_p<0.05)'] = combined_results['holm_p'] < 0.05

print(combined_results)


#FIGURE 3D
#Heatmaps
%matplotlib agg
importlib.reload(DM)
#rots=[-90,-60,-45,-30,-15]
rots = [-90]
#binSizes=[8,5,5,3,3]
binSizes=[8,8,8,8,8]
figSize=(10*1.05,10)
i=0
for r in rots:
    #1tar
    tmp = dfs[0][dfs[0]['blockRot'] == r]
    tmp = tmp[tmp['blockTrial'] <30]
    tmp = tmp[tmp['blockNum']==0]
    numBins = int(360 / binSizes[i])
    xLox = [np.arange(0,60,5)[it] for it in range(len(np.arange(0,60,5))) if it%2==1]
    xTix =[np.arange(-15,45,5)[it] for it in range(len(np.arange(-15,45,5))) if it%2==1]
    yTix = [np.arange(-2.5,numBins+1,5)[it] for it in range(10) if it%2==1]
    minYTix = np.arange(0.5,numBins,1)
    yLabs=[np.arange(200,-181,-40)[it] for it in range(10) if it%2==1]
    caption = ''
    norm = mpl.colors.LogNorm(vmin=4e-2,vmax=1)
    #norm = None
    t = DM.getHeatDF(tmp,expLen=60,numBlocks=1,var='aim',binWidth=binSizes[i],x='blockTrial',cond='blockRot')
    DM.heatMap(df=t,kdeDf=dfs[0],norm=norm,deleteTitle=True,rotStart=15,rotLen=30,xTickStepSize=2,numBlocks=1,panelLabs=[r],kde=False,kdeBW=0.04,xLox=xLox,xTix=xTix,yTix=yTix,minYTix=minYTix,yLabs=yLabs,caption=caption,capOffset=-0.17,overlayTrialSeries=False,figSize=figSize,cbar=True,showPlot=False,save=True,fileName=targetFolder+'HeatMap1Tar'+str(r)+'.svg')
    tmp = dfs[2][dfs[2]['blockRot'] == r]
    tmp = tmp[tmp['blockNum']==0]
    i+=1

#FIGURE 3E
#Truncated heatmaps
importlib.reload(DM)
rots=[-90,-60,-45,-30,-15]
binSizes = [9,6,4.5,3,1.5]
binSizes = [i*2 for i in binSizes]
figSize=(10*1.05,10)
yHighs=[180,120,90,60,30]
yLows=[-i*1.5 for i in binSizes]
i=0
for r in rots:#1tar
    tmp = dfs[0][dfs[0]['blockRot'] == r]
    tmp = tmp[tmp['blockTrial'] <30]
    tmp = tmp[tmp['blockNum']==0]
    xLox = [np.arange(0,60,5)[it] for it in range(len(np.arange(0,60,5))) if it%2==1]
    xTix =[np.arange(-15,45,5)[it] for it in range(len(np.arange(-15,45,5))) if it%2==1]
    yTix,minYTix,yLabs=[],[],[]
    yTix = np.arange(1,10,1)+0.5
    caption = ''
    norm = mpl.colors.LogNorm(vmin=4e-2,vmax=6e-1)
    t = DM.getHeatDF(tmp,expLen=45,numBlocks=1,var='aim',binWidth=binSizes[i],x='blockTrial',cond='blockRot',yLow=yLows[i],yHigh=yHighs[i])
    
    DM.heatMap(df=t,xlim=(15,45),ySize=360,kdeDf=dfs[0],norm=norm,deleteTitle=True,rotStart=-1,rotLen=30,xTickStepSize=2,numBlocks=1,
               panelLabs=[r],kde=False,kdeBW=0.04,xLox=xLox,xTix=xTix,yTix=yTix,minYTix=minYTix,yTickStepSize=1,yLabs=yLabs,caption=caption,capOffset=-0.17,
               overlayTrialSeries=False,figSize=figSize,cbar=True,showPlot=False,save=True,fileName=targetFolder+'truncatedHeatMap1Tar'+str(r)+'.svg')
    i+=1

#SIMULATE AIMING DAT FOR HEATMAP
tmp = dfs[0][dfs[0]['blockRot'] == -90]
tmp = tmp[tmp['blockTrial'] >= -5]
trialwiseNoise = tmp.groupby('blockTrial').std()['aim'].tolist()
fittedParams = False
it = 0
for it in range(len(trialwiseNoise)):
    if it >= 0:
        trialwiseNoise[it] *= 0
    it+=1
print(trialwiseNoise)
importlib.reload(NonGenSSM)
#sample params
numP = 100
rots = [0]*5
[rots.append(i) for i in [-90]*30]
[rots.append(i) for i in [0]*5]
rots=DM.flattenJagged(rots)
if fittedParams:
    lrs = paramDF['lr'].tolist()#
    rets = paramDF['retention'].tolist()#
else:
    lrs = [np.random.normal(0.26,0.015) for i in range(numP)]#
    rets = [min(np.random.normal(0.995,0.02),1) for i in range(numP)]#

ssm = NonGenSSM.fitShell()
states = []
for p in range(numP):
    states.append(ssm.genDat(params=(lrs[p],rets[p]),rots=rots))
for p in states:
    for t in range(len(p)):
        if t < 35:
            p[t]+=np.random.normal(0,trialwiseNoise[t])

#Make df from simulations above
importlib.reload(DM)
importlib.reload(NonGenSSM)
pp = DM.flattenJagged([[i]*40 for i in np.arange(numP)])
blockTrial = DM.flattenJagged([np.arange(-5,35,1) for i in range(numP)])
blockRot =DM.flattenJagged([[-90]*40 for i in np.arange(numP)])
aim = DM.flattenJagged(states)
print([len(i) for i in [pp,blockTrial,blockRot,aim]])
data = {'participantNum':pp,'blockTrial':blockTrial,'blockRot':blockRot,'aim':aim}
simDF = pd.DataFrame(data)

#FIGURE 3A
#heatmap for simDat
simDF = simDF[simDF['blockTrial']<30]
t = DM.getHeatDF(simDF,expLen=60,numBlocks=1,binWidth=8)
xLox = [np.arange(0.5,40.5,5)[it] for it in range(len(np.arange(0,40,5))) if True]
xTix =[np.arange(-5,35,5)[it] for it in range(len(np.arange(-5,35,5))) if True]
yTix = [np.arange(-2.5,46,5)[it] for it in range(10) if it%2==1]
minYTix = np.arange(0.5,45,1)
yLabs=[np.arange(200,-181,-40)[it] for it in range(10) if it%2==1]
caption = ''
r=''
if fittedParams:
    fileName= 'fittedSSM901TarHeatmap.svg'
else:
    fileName = 'test901TarSSMHeatmap.svg'
fileName = targetFolder + fileName
DM.heatMap(df=t,norm=mpl.colors.LogNorm(),kdeDf=dfs[0],rotStart=15,rotLen=30,xTickStepSize=2,numBlocks=1,panelLabs=[r],
           kde=False,kdeBW=0.04,xLox=xLox,xTix=xTix,yTix=yTix,minYTix=minYTix,yLabs=yLabs,caption=caption,
           capOffset=-0.17,overlayTrialSeries=False,figSize=(10*1.05,10),cbar=True,showPlot=False,save=True,
           fileName=fileName)
    

#SIMULATE AIMING DAT FOR HEATMAP
importlib.reload(hmm)
#sample params
fittedParams = False
rots = [0]*5
[rots.append(i) for i in [-90]*30]
[rots.append(i) for i in [0]*5]
rots=DM.flattenJagged(rots)
if fittedParams:
    ss = paramDF['stepStart'].tolist()
    sh = paramDF['stepHeight'].tolist()
else:
    alpha = 2.5
    beta = 3########
    ss = beta * (np.random.pareto(alpha,size=numP))
    sh = [np.random.normal(90,3) for i in range(numP)]
stepper = hmm.fitShell(df='none')
states = []
for p in range(numP):
    states.append(stepper.genDat(params=(ss[p],sh[p]),rots=rots))
for p in states:
    for t in range(len(p)):
        if True:#t < 6:
            p[t]+=np.random.normal(0,trialwiseNoise[t])
        else:
            pass

#Make df from simulations above
importlib.reload(DM)
importlib.reload(hmm)
pp = DM.flattenJagged([[i]*40 for i in np.arange(numP)])
blockTrial = DM.flattenJagged([np.arange(-5,35,1) for i in range(numP)])
blockRot =DM.flattenJagged([[-90]*40 for i in np.arange(numP)])
aim = DM.flattenJagged(states)
print([len(i) for i in [pp,blockTrial,blockRot,aim]])
data = {'participantNum':pp,'blockTrial':blockTrial,'blockRot':blockRot,'aim':aim}
simStepDF = pd.DataFrame(data)

#FIGURE 3B
#heatmap for simDat
t = DM.getHeatDF(simStepDF,expLen=60,numBlocks=1,binWidth=8)
xLox = [np.arange(0.5,40.5,5)[it] for it in range(len(np.arange(0,40,5))) if True]
xTix =[np.arange(-5,35,5)[it] for it in range(len(np.arange(-5,35,5))) if True]
yTix = [np.arange(-2.5,46,5)[it] for it in range(10) if it%2==1]
minYTix = np.arange(0.5,45,1)
yLabs=[np.arange(200,-181,-40)[it] for it in range(10) if it%2==1]
caption = ''
fileName = 'fittedStepper901TarHeatmap.svg' if fittedParams else 'roughParamsStepper901TarHeatmpat.svg'
fileName = targetFolder + fileName
DM.heatMap(df=t,norm=mpl.colors.LogNorm(),kdeDf=dfs[0],rotStart=15,rotLen=30,xTickStepSize=2,numBlocks=1,panelLabs=[r],
           kde=False,kdeBW=0.04,xLox=xLox,xTix=xTix,yTix=yTix,minYTix=minYTix,yLabs=yLabs,caption=caption,
           capOffset=-0.17,overlayTrialSeries=False,figSize=(10*1.05,10),cbar=True,showPlot=False,save=True,
           fileName=fileName)
    

#SIMULATE AIMING DAT FOR HEATMAP # QLEARN model
importlib.reload(qLearn)
#sample params
fittedParams = False
numP = 100
rot = 90
sigmaM = [0] * numP # execution noise
sigmaE = [min(max(np.random.normal(7000,1000),0),100000) for i in range(numP)] # exploratory noise
alpha = [1] * numP
qLearn = qLearn.fitShell(df='none')
states = []
for p in range(numP):
    states.append(qLearn.genDat(params=(alpha[p],sigmaE[p],sigmaM[p]),rot=rot,prepend=5))


#Make df from simulations above
importlib.reload(DM)
importlib.reload(qLearn)
pp = DM.flattenJagged([[i]*35 for i in np.arange(numP)])
blockTrial = DM.flattenJagged([np.arange(-5,30,1) for i in range(numP)])
blockRot =DM.flattenJagged([[-90]*35 for i in np.arange(numP)])
aim = DM.flattenJagged(states)
print([len(i) for i in [pp,blockTrial,blockRot,aim]])
data = {'participantNum':pp,'blockTrial':blockTrial,'blockRot':blockRot,'aim':aim}
simqLearnDF = pd.DataFrame(data)

#FIGURE 3C
#heatmap for simDat
importlib.reload(DM)
t = DM.getHeatDF(simqLearnDF,expLen=60,numBlocks=1,binWidth=8)
xLox = [np.arange(0.5,35.5,5)[it] for it in range(len(np.arange(0,35,5))) if True]
xTix =[np.arange(-5,30,5)[it] for it in range(len(np.arange(-5,30,5))) if True]
yTix = [np.arange(-2.5,46,5)[it] for it in range(10) if it%2==1]
minYTix = np.arange(0.5,45,1)
yLabs=[np.arange(200,-181,-40)[it] for it in range(10) if it%2==1]
caption = ''
r=''
fileName = 'fittedQLEARN901TarHeatmap.svg' if fittedParams else 'roughParamsQLEARN901TarHeatmpat.svg'
fileName = targetFolder + fileName
DM.heatMap(df=t,norm=mpl.colors.LogNorm(),kdeDf=dfs[0],rotStart=15,rotLen=30,xTickStepSize=2,numBlocks=1,panelLabs=[r],
           kde=False,kdeBW=0.04,xLox=xLox,xTix=xTix,yTix=yTix,minYTix=minYTix,yLabs=yLabs,caption=caption,
           capOffset=-0.17,overlayTrialSeries=False,figSize=(10*1.05,10),cbar=True,showPlot=True,save=True,
           fileName=fileName,plotTitle = "")
    

In [ ]:
targetFolder = 'paperFigures/'
%matplotlib inline
ssmsAll = np.load("ssmsVanilla1AllPPs.npy",allow_pickle=True)
qLearnsAll = np.load("QLearningVanilla1AllPPs.npy",allow_pickle=True)
hmmsAll = np.load("HMMVanilla1AllPPs.npy",allow_pickle=True)
                                 
                                                                                              
ssmBIC = [i.bics for i in ssmsAll]
qLearnBIC = [i.bics for i in qLearnsAll]
hmmBIC = [i.bics for i in hmmsAll]

In [ ]:
          
                 
importlib.reload(DM)
DM.plotThreeWayRelativeBIC(ssmBIC,hmmBIC,qLearnBIC,markerSize=100,targetFolder=targetFolder,fileName="ThreeWayModelComparisonCannonGame1Tar.svg",
                           xlim=(-40,350),ylim=(-55,360),averageXlim=(-10,250),xlab='Delta-BIC "Aha!" - SSM',ylab='Delta-BIC "Aha!" - QLEARN',figSize=(8, 8))

In [ ]:
allRotParams = []
for r in range(5):
    numP = len(ssmsAll[r].mStates)
    ssmParams = []
    hmmParams = []
    qLearnParams = []
    for p in range(numP):
        params = ssmsAll[r].xs[p]
        ssmParams.append(params)
        params = hmmsAll[r].xs[p]
                                            
        params = [params[3],params[4],params[0],params[5],params[1],params[2]]
        hmmParams.append(params)
        params = qLearnsAll[r].xs[p]
        qLearnParams.append(params)
    BICs = [ssmBIC,hmmBIC,qLearnBIC]
    allParams = [ssmParams,hmmParams,qLearnParams]
    allRotParams.append(allParams)

In [ ]:
              
                                       
importlib.reload(DM)
allPNames = [['Learning Rate','Retention Rate','Execution noise (STD)'],['Learning Rate','Inverse Temperature (mag)','Execution noise (STD)','Inverse Temp (sign)'],
             ['Learning Rate', 'Inverse Temperature (mag)', 'Execution noise (STD)','Inverse Temp (sign)', 'Kappa','Perceptual noise (STD)']]
for rot in range(5):
    ssmParams = [i[0] for i in allRotParams]
    hmmParams = [i[1] for i in allRotParams]
    qLearnParams = [i[2] for i in allRotParams]
DM.plotParams(ssmParams,qLearnParams,hmmParams,sqrtVariance=False,fileName=targetFolder+'CannonGameModelComparisonFits.svg',allPNames=allPNames)

In [ ]:
                                                     
ssmsAllBrudner = np.load("BrudnerSsmsDelay5000_1Tar.npy",allow_pickle=True)
hmmsAllBrudner = np.load("BrudnerHMMDelay5000_1Tar.npy",allow_pickle=True)
qLearnsAllBrudner = np.load("BrudnerQLearningDelay5000_1Tar.npy",allow_pickle=True)
ssmBICBrudner = [i.bics for i in ssmsAllBrudner]
hmmBICBrudner = [i.bics for i in hmmsAllBrudner]
qLearnBICBrudner = [i.bics for i in qLearnsAllBrudner]

In [ ]:
          
             
importlib.reload(DM)
DM.plotThreeWayRelativeBIC(ssmBICBrudner,hmmBICBrudner,qLearnBICBrudner,markerSize=100,targetFolder=targetFolder,fileName="ThreeWayModelComparisonBrudner5000Delay_1Tar.svg",
                           xlim=(-30,450),ylim=(-30,800),averageXlim=(-10,250),xlab='Delta-BIC "Aha!" - SSM',ylab='Delta-BIC "Aha!" - QLEARN',figSize=(8, 8))

In [ ]:
r=0
numP = len(ssmsAll[r].mStates)
allRotParams=[]

num_rows = 5
num_cols = 4
num_panels = num_rows * num_cols

num_panels = min(num_panels, numP)

ssmParamsBrudner = []
hmmParamsBrudner = []
qLearnParamsBrudner = []

for p in range(num_panels):
    params = ssmsAllBrudner[r].xs[p]
    ssmParamsBrudner.append(params)
    params = hmmsAllBrudner[r].xs[p]
                                        
    params = [params[3],params[4],params[0],params[5],params[1],params[2]]
    hmmParamsBrudner.append(params)
    params = qLearnsAllBrudner[r].xs[p]
    qLearnParamsBrudner.append(params)

In [ ]:
              
                       
importlib.reload(DM)
DM.plotParams([ssmParamsBrudner],[qLearnParamsBrudner],[hmmParamsBrudner],rots=[30],sqrtVariance=False,squeezeX=True,fileName=targetFolder+'BrudnerModelComparisonFits.svg',allPNames=allPNames)

In [ ]:
           
                                                                                                                        
importlib.reload(DM)
rots = list(reversed([15,30,45,60,90]))

        
ssmsAllBrudner = np.load("BrudnerSsmsDelay5000_1Tar.npy",allow_pickle=True)
hmmsAllBrudner = np.load("BrudnerHMMDelay5000_1Tar.npy",allow_pickle=True)
qLearnsAllBrudner = np.load("BrudnerQLearningDelay5000_1Tar.npy",allow_pickle=True)
ssmBICBrudner = [i.bics for i in ssmsAllBrudner]
hmmBICBrudner = [i.bics for i in hmmsAllBrudner]
qLearnBICBrudner = [i.bics for i in qLearnsAllBrudner]
BrudnerDict = {
    'xBIC':[ssmBICBrudner],
    'yBIC':[hmmBICBrudner],
    'zBIC':[qLearnBICBrudner],
    'rotSizes':[30],
    'name':'Brudner'}


    
ssmsAll1CG = np.load("ssmsVanilla1AllPPs.npy",allow_pickle=True)
qLearnsAll1CG = np.load("QLearningVanilla1AllPPs.npy",allow_pickle=True)
hmmsAll1CG = np.load("HMMVanilla1AllPPs.npy",allow_pickle=True)
ssmBIC1CG = [i.bics for i in ssmsAll1CG]
qLearnBIC1CG = [i.bics for i in qLearnsAll1CG]
hmmBIC1CG = [i.bics for i in hmmsAll1CG]
CG1VanDict = {
    'xBIC':ssmBIC1CG,
    'yBIC':hmmBIC1CG,
    'zBIC':qLearnBIC1CG,
    'rotSizes':rots,
    'name':'CG1Van'}
       
ssmsAll1ImpCG = np.load("ssmsAutocorrect1AllPPs.npy",allow_pickle=True)
qLearnsAll1ImpCG = np.load("QLearningImplicitCG1TarAllPPs.npy",allow_pickle=True)
hmmsAll1ImpCG = np.load("HMMImplicitCG1TarAllPPs.npy",allow_pickle=True)
ssmBIC1ImpCG = [i.bics for i in ssmsAll1ImpCG]
qLearnBIC1ImpCG = [i.bics for i in qLearnsAll1ImpCG]
hmmBIC1ImpCG = [i.bics for i in hmmsAll1ImpCG]
CG1ImpDict = {
    'xBIC':ssmBIC1ImpCG,
    'yBIC':hmmBIC1ImpCG,
    'zBIC':qLearnBIC1ImpCG,
    'rotSizes':rots,
    'name':'CG1Imp'}

    
ssmsAll8CG = np.load("ssmsVanilla8AllPPs.npy",allow_pickle=True)
qLearnsAll8CG = np.load("QLearningVanilla8AllPPs.npy",allow_pickle=True)
hmmsAll8CG = np.load("HMMVanilla8AllPPs.npy",allow_pickle=True)
ssmBIC8CG = [i.bics for i in ssmsAll8CG]
qLearnBIC8CG = [i.bics for i in qLearnsAll8CG]
hmmBIC8CG = [i.bics for i in hmmsAll8CG]
CG8VanDict = {
    'xBIC':ssmBIC8CG,
    'yBIC':hmmBIC8CG,
    'zBIC':qLearnBIC8CG,
    'rotSizes':rots,
    'name':'CG8Van'}

              
ssmsAll8CGIMP = np.load("ssmsAutocorrect8AllPPs.npy",allow_pickle=True)
qLearnsAll8CGIMP = np.load("QLearningImpCGAllPPs.npy",allow_pickle=True)
hmmsAll8CGIMP = np.load("HMMImpCG8AllPPs.npy",allow_pickle=True) 
ssmBIC8CGIMP = [i.bics for i in ssmsAll8CGIMP]
qLearnBIC8CGIMP = [i.bics for i in qLearnsAll8CGIMP]
hmmBIC8CGIMP = [i.bics for i in hmmsAll8CGIMP]
CG8ImpDict = {
    'xBIC':ssmBIC8CGIMP,
    'yBIC':hmmBIC8CGIMP,
    'zBIC':qLearnBIC8CGIMP,
    'rotSizes':rots,
    'name':'CG8Imp'}

             
ssmsAll8BT = np.load("BondTaylorssmsAutocorrect8AllPPs.npy",allow_pickle=True)
qLearnsAll8BT = np.load("QLearningImplicitBTAllPPs.npy",allow_pickle=True)
hmmsAll8BT = np.load("HMMImplicitBTAllPPs.npy",allow_pickle=True) 
ssmBIC8BTIMP = [i.bics for i in ssmsAll8BT]
qLearnBIC8BTIMP = [i.bics for i in qLearnsAll8BT]
hmmBIC8BTIMP = [i.bics for i in hmmsAll8BT]
BT8ImpDict = {
    'xBIC':ssmBIC8BTIMP,
    'yBIC':hmmBIC8BTIMP,
    'zBIC':qLearnBIC8BTIMP,
    'rotSizes':rots,
    'name':'BT8Imp'}

                                                             


datasets = [BrudnerDict, CG1VanDict, CG1ImpDict, CG8VanDict, BT8ImpDict, CG8ImpDict]
DM.plotMultipleDatasetsRelativeBIC(datasets=datasets, targetFolder=targetFolder, markerSize=20, alpha=0.5,
                                  xlab='Delta-BIC: SSM - "Aha!"', ylab='Delta-BIC: QLEARN - "Aha!"')

In [ ]:

def plotDeltaBICHists(datasets, rotSizes=[15, 30, 45, 60, 90], num_bins=10, figSize=(12, 5),
                      targetFolder='', fileName="deltaBIC_hists.svg"):
    """
    Plots two histograms: one for Delta-BIC (SSM - Aha) and one for Delta-BIC (QLEARN - Aha).
    Each divided into 10 bins symmetric around 0, with bars colored cyan if Aha is better (delta > 0) or magenta otherwise.
 
    Parameters:
    - datasets: list of dicts, as in plotMultipleDatasetsRelativeBIC
    - Other parameters for customization.
    """
    def get_participants_bic(bic_data, rotIdx):
        if len(bic_data) == 1:
            bic = bic_data[0]
        else:
            bic = bic_data[rotIdx]
        if isinstance(bic, list):
            bic = bic[0]
        return np.asarray(bic)
 
    all_rel_x = []
    all_rel_y = []
    for ds in datasets:
        for rotIdx, rot in enumerate(ds['rotSizes']):
            if rot not in rotSizes:
                continue
            participantsX = get_participants_bic(ds['xBIC'], rotIdx)
            participantsY = get_participants_bic(ds['yBIC'], rotIdx)
            participantsZ = get_participants_bic(ds['zBIC'], rotIdx)
            relXY = participantsX - participantsY            
            relZY = participantsZ - participantsY               
            all_rel_x.extend(relXY)
            all_rel_y.extend(relZY)
 
    all_rel_x = np.array([val for val in all_rel_x if not np.isnan(val)])
    all_rel_y = np.array([val for val in all_rel_y if not np.isnan(val)])
 
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figSize)
 
                             
    if len(all_rel_x) > 0:
        min_x, max_x = np.min(all_rel_x), np.max(all_rel_x)
        max_abs_x = max(abs(min_x), abs(max_x))
                                 
        bins_x = np.linspace(-max_abs_x, max_abs_x, num_bins + 1)
        centers_x = (bins_x[:-1] + bins_x[1:]) / 2
        heights_x, _ = np.histogram(all_rel_x, bins=bins_x)
      
                                           
        populated_mask = heights_x > 0
        if np.any(populated_mask):
            left_pop_idx = np.min(np.where(populated_mask)[0])
            right_pop_idx = np.max(np.where(populated_mask)[0])
                                                                                            
            data_range = max_x - min_x if max_x != min_x else 1
            padding = 0.05 * data_range
            xlim_left = centers_x[left_pop_idx] - 0.5 * np.diff(bins_x)[0] - padding
            xlim_right = centers_x[right_pop_idx] + 0.5 * np.diff(bins_x)[0] + padding
                                                                                         
                                                                                 
                                                               
                                                               
        else:
                                           
            xlim_left, xlim_right = -max_abs_x, max_abs_x
      
        colors_x = ['green' if c > 0 else 'magenta' for c in centers_x]
        width_x = np.diff(bins_x)
        ax1.bar(centers_x, heights_x, width=width_x, color=colors_x, edgecolor='black', alpha=0.7)
        ax1.axvline(0, color='k', linestyle='--', linewidth=1)
        ax1.set_xlabel('Delta-BIC: SSM - Aha')
        ax1.set_ylabel('Frequency')
        ax1.set_title('SSM vs Aha')
        ax1.set_xlim(xlim_left, xlim_right)
        ax1.set_ylim(0,320)
        ax1.xaxis.set_major_locator(MaxNLocator(nbins=5))
                                                          
        ax1.yaxis.set_minor_locator(AutoMinorLocator(4))
 
                                
    if len(all_rel_y) > 0:
        min_y, max_y = np.min(all_rel_y), np.max(all_rel_y)
        max_abs_y = max(abs(min_y), abs(max_y))
                                 
        bins_y = np.linspace(-max_abs_y, max_abs_y, num_bins + 1)
        centers_y = (bins_y[:-1] + bins_y[1:]) / 2
        heights_y, _ = np.histogram(all_rel_y, bins=bins_y)
      
                                           
        populated_mask = heights_y > 0
        if np.any(populated_mask):
            left_pop_idx = np.min(np.where(populated_mask)[0])
            right_pop_idx = np.max(np.where(populated_mask)[0])
                                                                                            
            data_range = max_y - min_y if max_y != min_y else 1
            padding = 0.05 * data_range
            xlim_left = centers_y[left_pop_idx] - 0.5 * np.diff(bins_y)[0] - padding
            xlim_right = centers_y[right_pop_idx] + 0.5 * np.diff(bins_y)[0] + padding
                                                                                         
                                                                                 
                                                               
                                                               
        else:
                                           
            xlim_left, xlim_right = -max_abs_y, max_abs_y
      
        colors_y = ['green' if c > 0 else 'magenta' for c in centers_y]
        width_y = np.diff(bins_y)
        ax2.bar(centers_y, heights_y, width=width_y, color=colors_y, edgecolor='black', alpha=0.7)
        ax2.axvline(0, color='k', linestyle='--', linewidth=1)
        ax2.set_xlabel('Delta-BIC: QLEARN - Aha')
        ax2.set_ylabel('Frequency')
        ax2.set_title('QLEARN vs Aha')
        ax2.set_xlim(xlim_left, xlim_right)
        ax2.set_ylim(0,320)
        ax2.xaxis.set_major_locator(MaxNLocator(nbins=5))
                                                          
        ax2.yaxis.set_minor_locator(AutoMinorLocator(4))
 
    plt.tight_layout()
    plt.savefig(targetFolder + fileName)
               
               
plotDeltaBICHists(datasets=datasets, targetFolder=targetFolder, num_bins=40)

In [ ]:
                                                                                                                      
                                                                   
ssmBIC8CG = [i.bics for i in ssmsAll8CG]
ssmBICBrudner = [i.bics for i in ssmsAllBrudner]
dualssmsAllBrudner = np.load("BrudnerDUALSsmsDelay5000_1Tar.npy",allow_pickle=True)
dualssmsAll8CG = np.load("dualssmsVanilla8AllPPs.npy",allow_pickle=True)
dualssmBIC8CG = [i.bics for i in dualssmsAll8CG]
dualssmBICBrudner = [i.bics for i in dualssmsAllBrudner]

In [ ]:


def plotDeltaBICHists_SSM_DualSSM(ssmBIC8CG, dualssmBIC8CG, ssmBICBrudner, dualssmBICBrudner,
                                 num_bins=10, figSize=(12, 5), targetFolder='', fileName="deltaBIC_SSM_DualSSM_hists.svg"):
    """
    Plots two histograms: one for Delta-BIC (SSM - DualSSM) for 8CG dataset and one for Brudner dataset.
    Each divided into num_bins bins symmetric around 0, with bars colored cyan if DualSSM is better (delta > 0) or magenta otherwise.
    Adds a vertical dashed red line at the mean delta BIC for each histogram.
    Parameters:
    - ssmBIC8CG: list of lists or arrays of BIC values for SSM on 8CG, per rotation group
    - dualssmBIC8CG: list of lists or arrays of BIC values for DualSSM on 8CG, per rotation group
    - ssmBICBrudner: list of lists or arrays of BIC values for SSM on Brudner, per rotation group
    - dualssmBICBrudner: list of lists or arrays of BIC values for DualSSM on Brudner, per rotation group
    - Other parameters for customization.
    """
                                              
    all_rel_8CG = []
    for ssm_bics, dual_bics in zip(ssmBIC8CG, dualssmBIC8CG):
        rel = np.asarray(ssm_bics) - np.asarray(dual_bics)
        all_rel_8CG.extend(rel)
 
                                                  
    all_rel_Brudner = []
    for ssm_bics, dual_bics in zip(ssmBICBrudner, dualssmBICBrudner):
        rel = np.asarray(ssm_bics) - np.asarray(dual_bics)
        all_rel_Brudner.extend(rel)
 
    rel_8CG = np.array([val for val in all_rel_8CG if not np.isnan(val)])
    rel_Brudner = np.array([val for val in all_rel_Brudner if not np.isnan(val)])
 
                     
    mean_8CG = np.mean(rel_8CG) if len(rel_8CG) > 0 else 0
    mean_Brudner = np.mean(rel_Brudner) if len(rel_Brudner) > 0 else 0
 
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figSize)
 
                                      
    if len(rel_8CG) > 0:
        min_val, max_val = np.min(rel_8CG), np.max(rel_8CG)
        max_abs = max(abs(min_val), abs(max_val))
        bins = np.linspace(-max_abs, max_abs, num_bins + 1)
        centers = (bins[:-1] + bins[1:]) / 2
        heights, _ = np.histogram(rel_8CG, bins=bins)
     
                                           
        populated_mask = heights > 0
        if np.any(populated_mask):
            left_pop_idx = np.min(np.where(populated_mask)[0])
            right_pop_idx = np.max(np.where(populated_mask)[0])
            data_range = max_val - min_val if max_val != min_val else 1
            padding = 0.05 * data_range
            xlim_left = centers[left_pop_idx] - 0.5 * np.diff(bins)[0] - padding
            xlim_right = centers[right_pop_idx] + 0.5 * np.diff(bins)[0] + padding
        else:
            xlim_left, xlim_right = -max_abs, max_abs
     
        colors = ['cyan' if c > 0 else 'magenta' for c in centers]
        width = np.diff(bins)
        ax1.bar(centers, heights, width=width, color=colors, edgecolor='black', alpha=0.7)
        ax1.axvline(0, color='k', linestyle='--', linewidth=1)
        ax1.axvline(mean_8CG, color='red', linestyle='dotted', linewidth=1)
        ax1.set_xlabel('Delta-BIC: SSM - DualSSM')
        ax1.set_ylabel('Frequency')
        ax1.set_title('SSM vs DualSSM (8CG)')
        ax1.set_xlim(xlim_left, xlim_right)
        ax1.set_ylim(bottom=0)
        ax1.xaxis.set_major_locator(MaxNLocator(nbins=5))
        ax1.yaxis.set_major_locator(MaxNLocator(integer=True))
        ax1.yaxis.set_minor_locator(AutoMinorLocator(4))
       
                       
        data_range = xlim_right - xlim_left
        offset = 0.02 * data_range
        text_x = mean_8CG + offset
        text_y = ax1.get_ylim()[1] * 0.95
        ax1.text(text_x, text_y, f'Mean = {mean_8CG:.2f}', color='red', fontsize=10, ha='left', va='top')
 
                                          
    if len(rel_Brudner) > 0:
        min_val, max_val = np.min(rel_Brudner), np.max(rel_Brudner)
        max_abs = max(abs(min_val), abs(max_val))
        bins = np.linspace(-max_abs, max_abs, num_bins + 1)
        centers = (bins[:-1] + bins[1:]) / 2
        heights, _ = np.histogram(rel_Brudner, bins=bins)
     
                                           
        populated_mask = heights > 0
        if np.any(populated_mask):
            left_pop_idx = np.min(np.where(populated_mask)[0])
            right_pop_idx = np.max(np.where(populated_mask)[0])
            data_range = max_val - min_val if max_val != min_val else 1
            padding = 0.05 * data_range
            xlim_left = centers[left_pop_idx] - 0.5 * np.diff(bins)[0] - padding
            xlim_right = centers[right_pop_idx] + 0.5 * np.diff(bins)[0] + padding
        else:
            xlim_left, xlim_right = -max_abs, max_abs
     
        colors = ['cyan' if c > 0 else 'magenta' for c in centers]
        width = np.diff(bins)
        ax2.bar(centers, heights, width=width, color=colors, edgecolor='black', alpha=0.7)
        ax2.axvline(0, color='k', linestyle='--', linewidth=1)
        ax2.axvline(mean_Brudner, color='red', linestyle='dotted', linewidth=1)
        ax2.set_xlabel('Delta-BIC: SSM - DualSSM')
        ax2.set_ylabel('Frequency')
        ax2.set_title('SSM vs DualSSM (Brudner)')
        ax2.set_xlim(xlim_left, xlim_right)
        ax2.set_ylim(bottom=0)
        ax2.xaxis.set_major_locator(MaxNLocator(nbins=5))
        ax2.yaxis.set_major_locator(MaxNLocator(integer=True))
        ax2.yaxis.set_minor_locator(AutoMinorLocator(4))
       
                       
        data_range = xlim_right - xlim_left
        offset = 0.02 * data_range
        text_x = mean_Brudner + offset
        text_y = ax2.get_ylim()[1] * 0.95
        ax2.text(text_x, text_y, f'Mean = {mean_Brudner:.2f}', color='red', fontsize=10, ha='left', va='top')
 
    plt.tight_layout()
    plt.savefig(targetFolder + fileName)
                
    
plotDeltaBICHists_SSM_DualSSM(ssmBIC8CG=ssmBIC8CG, dualssmBIC8CG=dualssmBIC8CG,
                              ssmBICBrudner=ssmBICBrudner, dualssmBICBrudner=dualssmBICBrudner,
                              targetFolder=targetFolder, num_bins=50)

In [ ]:

rots = list(reversed([15,30,45,60,90]))
datasets = [[ssmsAll8CG,qLearnsAll8CG,hmmsAll8CG],[ssmsAll8CGIMP,qLearnsAll8CGIMP,hmmsAll8CGIMP],[ssmsAllBrudner,qLearnsAllBrudner,hmmsAllBrudner],
            [ssmsAll8BT,qLearnsAll8BT,hmmsAll8BT]]
modelLabels = ['ssm','qlearn','hmm']
datasetLabels = ['CGVanilla','CGImp','Brudner','BondTaylor']
hasImp = [0,1,0,1]

rotationMap = [rots, rots, [30], rots]                      
                                                                                   
                                        
                                                                                
                                        
                                            
                                                                           

colours = ['#44AA99', '#88CCEE', '#FF9825', '#CC6677', '#AA4499']
rot_palette = dict(zip(rots, colours))

                                               
markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h'][:len(datasetLabels)]
marker_dict = dict(zip(datasetLabels, markers))

                       
model_idx = 2      
original_param_idx = 2                                                                    
core_max_p = 6          
reorder_idx = [3, 4, 0, 5, 1, 2]                                                       

                                      
all_percep = []
for ds_idx, ds_name in enumerate(datasetLabels):
    rotations = rotationMap[ds_idx]
    model_lists = datasets[ds_idx][model_idx]           
    for rot_idx, rot in enumerate(rotations):
        if rot_idx >= len(model_lists):
            continue
        fit = model_lists[rot_idx]
        if fit is None:
            continue
        xs = fit.xs
                                                                           
        is_per_person = False
        numP_model = None
        raw_person_params = None
        if isinstance(xs, list) and len(xs) > 0:
            first_item = xs[0]
            if hasattr(first_item, '__len__'):
                n_items = len(xs)
                len_first = len(first_item)
                if len_first > 0 and n_items > 0:
                    if len_first == n_items:
                                             
                        numP_model = n_items
                    elif len_first in [3, 6, 9] or len_first > core_max_p:
                        is_per_person = True
                        numP_model = len_first
                        raw_person_params = xs
                    else:
                                                              
                        if len_first > core_max_p:
                            is_per_person = True
                            numP_model = len_first
                            raw_person_params = xs
                        else:
                            print(f"Warning: Unexpected structure for {ds_name}, rot {rot}: n_items={n_items}, len_first={len_first}")
                            continue
                else:
                    continue
            else:
                continue
        elif isinstance(xs, np.ndarray) and xs.ndim == 2:
            n_persons, n_params = xs.shape
            if n_params in [3, 6, 9] or n_params > core_max_p:
                is_per_person = True
                numP_model = n_params
                raw_person_params = [xs[i, :] for i in range(n_persons)]
            else:
                print(f"Warning: Unexpected 2D shape for {ds_name}, rot {rot}: {xs.shape}")
                continue
        else:
            print(f"Warning: Unsupported xs type/shape for {ds_name}, rot {rot}: {type(xs)}")
            continue
        if numP_model is None or original_param_idx >= numP_model:
            print(f"Warning: Insufficient params ({numP_model}) for perceptual noise (idx {original_param_idx}) in {ds_name}, rot {rot}")
            continue
                                                                          
        if is_per_person:
            person_params = [np.array(person) for person in raw_person_params]
            percep_values = [person[original_param_idx] for person in person_params]
        else:
                                                               
            percep_values = np.array(xs[original_param_idx])
                           
        for val in percep_values:
            if np.isfinite(val):
                all_percep.append({
                    'dataset': ds_name,
                    'rot': rot,
                    'val': val
                })

                  
df_percep = pd.DataFrame(all_percep)
if df_percep.empty:
    print("No perceptual noise data found.")
else:
    print(f"Extracted {len(df_percep)} perceptual noise values across {len(df_percep['dataset'].unique())} datasets.")
    
                                                  
    num_ds = len(datasetLabels)
    dodge_width = 0.4
    offsets = np.linspace(-dodge_width / 2, dodge_width / 2, num_ds)
    dodge_dict = dict(zip(datasetLabels, offsets))
    
                 
    fig, ax = plt.subplots(figsize=(12, 8))
    
                                                                                      
    np.random.seed(42)                          
    for ds_name in datasetLabels:
        ds_data = df_percep[df_percep['dataset'] == ds_name]
        if len(ds_data) == 0:
            continue
        offset = dodge_dict[ds_name]
        marker = marker_dict[ds_name]
        for rot in rots:
            rot_data = ds_data[ds_data['rot'] == rot]
            if len(rot_data) == 0:
                continue
            color = rot_palette.get(rot, 'gray')
            x_base = rot + offset
            x_pos = np.full(len(rot_data), x_base) + np.random.uniform(-1.5, 1.5, len(rot_data))               
            ax.scatter(x_pos, rot_data['val'], c=[color] * len(rot_data), marker=marker, s=30,
                       edgecolor='black', linewidth=0.5, alpha=0.8, zorder=3,
                       label=ds_name if rot == rots[0] else None)                         
    
                                                                       
    all_data = df_percep.dropna(subset=['rot', 'val'])
    if len(all_data) >= 2:
        slope, intercept, r_value, p_value, std_err = linregress(all_data['rot'], all_data['val'])
        x_fit = np.array(rots)
        y_fit = intercept + slope * x_fit
                                                         
        line_label = f"Overall fit (r={r_value:.2f})"
        ax.plot(x_fit, y_fit, 'k--', linewidth=4, alpha=0.8, label=line_label)
        
                                                  
        text_str = f'Slope: {slope:.4f}\nIntercept: {intercept:.4f}'
        ax.text(0.02, 0.98, text_str, transform=ax.transAxes, fontsize=14,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    else:
        print("Insufficient data for overall fit.")
    
             
    ax.set_xlabel('Rotation (deg)', fontsize=12)
    ax.set_ylabel('Perceptual noise (STD)', fontsize=12)
    ax.set_xticks(rots)
    ax.set_xticklabels([str(r) for r in rots])
    ax.legend(loc='upper right', frameon=True, fancybox=False, fontsize=8)
    sns.despine(ax=ax)
    plt.tight_layout()
    
               
    fileName = 'perceptual_noise_hmm_fits.svg'                                           
    plt.savefig(targetFolder + fileName, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Plot saved to {fileName}")

In [ ]:
                                               
            
allPNames = [['Learning Rate','Retention Rate','Execution noise (STD)'],
             ['Learning Rate','Inverse Temperature (mag)','Execution noise (STD)','Inverse Temp (sign)'],
             ['Learning Rate', 'Inverse Temperature (mag)', 'Execution noise (STD)','Inverse Temp (sign)', 'Kappa','Perceptual noise (STD)']]
datasets = [
    [ssmsAll8CG, qLearnsAll8CG, hmmsAll8CG],                    
    [ssmsAll8CGIMP, qLearnsAll8CGIMP, hmmsAll8CGIMP],        
    [ssmsAllBrudner, qLearnsAllBrudner, hmmsAllBrudner],                  
    [ssmsAll8BT, qLearnsAll8BT, hmmsAll8BT]                       
]
           
importlib.reload(DM)
DM.plotMultiDatasetParams(datasets, datasetLabels, rotationMap, hasImp, allPNames,
                          rots=rots, markerSize=6, figsize=(20, 25),
                          sqrtVariance=False, squeezeX=False, debug=False, plot_type='core',
                          fileName=targetFolder + 'MultiDatasetModelComparisonFits_Core.svg')

                        
DM.plotMultiDatasetParams([datasets[3]], [datasetLabels[3]], [rotationMap[3]], [hasImp[3]], allPNames,
                          rots=rots, markerSize=6, figsize=(15, 15),
                          sqrtVariance=False, squeezeX=False, debug=False, plot_type='extras',
                          fileName=targetFolder + 'MultiDatasetModelComparisonFits_Extras.svg')

In [ ]:
                                                        
               
                                                                                                                                                   
rots = list(reversed([15,30,45,60,90]))
datasets = [[ssmsAll8CG,qLearnsAll8CG,hmmsAll8CG],[ssmsAll8CGIMP,qLearnsAll8CGIMP,hmmsAll8CGIMP],[ssmsAllBrudner,qLearnsAllBrudner,hmmsAllBrudner],
            [ssmsAll8BT,qLearnsAll8BT,hmmsAll8BT]]
modelLabels = ['ssm','qlearn','hmm']
datasetLabels = ['CGVanilla','CGImp','Brudner','BondTaylor']
hasImp = [0,1,0,1]

for models in datasets:
    for m in models:
        pass


In [ ]:

def angular_diff(a, b, period=360.0):
    """Compute the signed angular difference between a and b, in [-period/2, period/2]."""
    diff = (a - b) % period
    return np.where(diff > period / 2, diff - period, diff)

def circular_mean(angles, period=360.0):
    """Compute the circular mean of angles in degrees."""
    angles_rad = np.deg2rad(angles)
    mean_sin = np.nanmean(np.sin(angles_rad))
    mean_cos = np.nanmean(np.cos(angles_rad))
    mean_rad = np.arctan2(mean_sin, mean_cos)
    return np.rad2deg(mean_rad)

def compute_circular_rmse(y_true, y_pred, period=360.0):
    """Circular RMSE using angular differences."""
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    if mask.sum() < 2:
        return np.nan
    yt, yp = y_true[mask], y_pred[mask]
    diffs = angular_diff(yt, yp, period)
    return np.sqrt(np.nanmean(diffs ** 2))

def compute_circular_r2(y_true, y_pred, period=360.0):
    """Circular R²: 1 - (SS_res / SS_tot), using angular deviations."""
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    if mask.sum() < 2:
        return np.nan
    yt, yp = y_true[mask], y_pred[mask]
   
                                       
    res_diffs = angular_diff(yt, yp, period)
    ss_res = np.nansum(res_diffs ** 2)
   
                                                         
    mu = circular_mean(yt, period)
    tot_diffs = angular_diff(yt, mu, period)
    ss_tot = np.nansum(tot_diffs ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 1 else np.nan

rots = list(reversed([15, 30, 45, 60, 90]))                       
datasets = [
    [ssmsAll8CG, qLearnsAll8CG, hmmsAll8CG],            
    [ssmsAll8CGIMP, qLearnsAll8CGIMP, hmmsAll8CGIMP],        
    [ssmsAllBrudner, qLearnsAllBrudner, hmmsAllBrudner],                  
    [ssmsAll8BT, qLearnsAll8BT, hmmsAll8BT]             
]
modelLabels = ['ssm', 'qlearn', 'hmm']
datasetLabels = ['CGVanilla', 'CGImp', 'Brudner', 'BondTaylor']
hasImp = [False, True, False, True]
                              
rotationMap = [rots, rots, [30], rots]                      
rows = []
for dsIdx, datasetName in enumerate(datasetLabels):
    modelLists = datasets[dsIdx]                                    
    rotations = rotationMap[dsIdx]
    impFlag = hasImp[dsIdx]
   
    for rotIdx, rotation in enumerate(rotations):
                                                                         
        fitShellsThisRot = []
        for modelIdx in range(3):
            modelFitList = modelLists[modelIdx]
            if rotIdx < len(modelFitList):
                fitShellsThisRot.append(modelFitList[rotIdx])
            else:
                fitShellsThisRot.append(None)
       
                                                                     
        nParticipants = None
        for fs in fitShellsThisRot:
            if fs is not None:
                                                                              
                if hasattr(fs, 'allAims'):
                    nParticipants = len(fs.allAims)
                elif hasattr(fs, 'human_explicits'):
                    nParticipants = len(fs.human_explicits)
                elif hasattr(fs, 'rmses'):                                  
                    nParticipants = len(fs.rmses)
                break
        if nParticipants is None:
            continue
       
                                                                              
        for pId in range(nParticipants):
            for modelIdx in range(3):
                modelName = modelLabels[modelIdx]
                entry = fitShellsThisRot[modelIdx]
                                                                        
                if entry is None:
                    continue
                
                                                                                             
                y_true = np.array([])
                y_pred = np.array([])
                
                if modelName == 'ssm':
                                           
                    h_exp = np.array([])
                    try:
                        if hasattr(entry, 'allAims'):
                            h_exp = np.array(entry.allAims[pId]).ravel()
                    except (IndexError, AttributeError, TypeError):
                        h_exp = np.array([])
                    
                                           
                    m_exp = np.array([])
                    try:
                        if hasattr(entry, 'mOut1'):
                            m_exp = np.array(entry.mOut1[pId]).ravel()
                        elif hasattr(entry, 'mStates'):
                            m_exp = np.array(entry.mStates[pId]).ravel()
                    except (IndexError, AttributeError, TypeError):
                        m_exp = np.array([])
                    
                    if impFlag:
                        h_imp = np.array([])
                        m_imp = np.array([])
                        try:
                            if hasattr(entry, 'allImps'):
                                h_imp = np.array(entry.allImps[pId]).ravel()
                            if hasattr(entry, 'mOut2'):
                                m_imp = np.array(entry.mOut2[pId]).ravel()
                        except (IndexError, AttributeError, TypeError):
                            pass
                        y_true = np.nansum(np.stack([h_exp, h_imp]), axis=0)
                        y_pred = np.nansum(np.stack([m_exp, m_imp]), axis=0)
                        y_true = y_true % 360.0
                        y_pred = y_pred % 360.0
                    else:
                        y_true = h_exp
                        y_pred = m_exp
                        
                elif modelName in ['qlearn', 'hmm']:
                                                    
                    h_exp = np.array([])
                    try:
                        if hasattr(entry, 'human_explicits'):
                            h_exp = np.array(entry.human_explicits[pId]).ravel()
                        elif hasattr(entry, 'allAims'):
                            h_exp = np.array(entry.allAims[pId]).ravel()
                    except (IndexError, AttributeError, TypeError):
                        h_exp = np.array([])
                    
                                           
                    m_exp = np.array([])
                    try:
                        if impFlag:
                            if hasattr(entry, 'model_explicits'):
                                m_exp = np.array(entry.model_explicits[pId]).ravel()
                        else:
                            if hasattr(entry, 'mStates'):
                                m_exp = np.array(entry.mStates[pId]).ravel()
                    except (IndexError, AttributeError, TypeError):
                        m_exp = np.array([])
                    
                    if impFlag:
                        h_imp = np.array([])
                        m_imp = np.array([])
                        try:
                            if hasattr(entry, 'human_implicits'):
                                h_imp = np.array(entry.human_implicits[pId]).ravel()
                            if hasattr(entry, 'model_implicits'):
                                m_imp = np.array(entry.model_implicits[pId]).ravel()
                        except (IndexError, AttributeError, TypeError):
                            pass
                        y_true = np.nansum(np.stack([h_exp, h_imp]), axis=0)
                        y_pred = np.nansum(np.stack([m_exp, m_imp]), axis=0)
                        y_true = y_true % 360.0
                        y_pred = y_pred % 360.0
                    else:
                        y_true = h_exp
                        y_pred = m_exp
                
                rmseVal = compute_circular_rmse(y_true, y_pred)
                rSqVal = compute_circular_r2(y_true, y_pred)
                
                rows.append({
                    'dataset' : datasetName,
                    'rotation' : rotation,
                    'participantId': pId,
                    'model' : modelName,
                    'hasImp' : impFlag,
                    'rmse' : rmseVal,
                    'rSquared' : rSqVal
                })
                  
df = pd.DataFrame(rows)
df = df[['dataset', 'rotation', 'participantId', 'model', 'hasImp', 'rmse', 'rSquared']]
df = df.sort_values(['dataset', 'rotation', 'participantId', 'model']).reset_index(drop=True)
                                                                  
                                                                                 
                                                                  
print("\n" + "="*80)
print("SUMMARY STATISTICS (mean ± std) for RMSE and R²")
print("="*80)
summary = (df.groupby(['dataset', 'rotation', 'model'])
             .agg(
                 rmse_mean = ('rmse', 'mean'),
                 rmse_std = ('rmse', 'std'),
                 rSquared_mean = ('rSquared', 'mean'),
                 rSquared_std = ('rSquared', 'std'),
                 n_participants = ('participantId', 'nunique')
             )
             .round(4))
                 
def format_mean_std(row, col_mean, col_std):
    return f"{row[col_mean]:.4f} ± {row[col_std]:.4f}"
summary['RMSE'] = summary.apply(lambda row: format_mean_std(row, 'rmse_mean', 'rmse_std'), axis=1)
summary['R²'] = summary.apply(lambda row: format_mean_std(row, 'rSquared_mean', 'rSquared_std'), axis=1)
summary['N'] = summary['n_participants']
                                                
pretty_summary = summary[['RMSE', 'R²', 'N']].reset_index()
print(pretty_summary.to_string(index=False))
                                        
                                                                     
                                                                  
                                   
                                                                  
print("\nFirst 18 rows of the full dataframe:")
print(df.head(18))
print(f"\nTotal rows: {len(df)}")
print(f"Shape: {df.shape}")
print("\nExample — Participant 0 in CGVanilla, rotation 90:")
print(df.query("dataset == 'CGVanilla' and rotation == 90 and participantId == 0"))

In [ ]:


                                                                    
                                 
                                                                    
df_clean = df[df['rSquared'].between(-1, 1) & (df['rmse'] >= 0)]
df_clean = df_clean[df_clean['rmse'] < df_clean['rmse'].quantile(0.995)].copy()

                                                                    
                    
                                                                    
sns.set_style("white")
sns.set_context("paper", font_scale=1.15)
plt.rcParams['font.family'] = 'Arial'

palette = {'ssm': '#2E86AB', 'qlearn': '#A23B72', 'hmm': '#F18F01'}

                                                                    
                                                                
                                                                    
g = sns.FacetGrid(
    data=df_clean,
    col="dataset",
    row="rotation",
    col_order=datasetLabels,
    row_order=[90, 60, 45, 30, 15],                              
    sharey=True,
    height=1.7,                                     
    aspect=1.2,
    despine=False
)

                                                
g.map_dataframe(
    sns.boxplot,
    x="model",
    y="rSquared",
    order=modelLabels,
    palette=palette,
    width=0.65,
    linewidth=1.3,
    fliersize=0,                                                   
    boxprops={'edgecolor': 'black'},
    medianprops={'color': 'white', 'linewidth': 2},
    whiskerprops={'linewidth': 1.2},
    capprops={'linewidth': 1.2}
)

                                      
g.map_dataframe(
    sns.stripplot,
    x="model",
    y="rSquared",
    order=modelLabels,
    palette=palette,
    size=3.5,
    jitter=0.20,
    alpha=0.7,
    linewidth=0.5,
    edgecolor="black"
)

                                                                    
                 
                                                                    
g.set_titles(col_template="{col_name}", row_template="{row_name}°")
g.set_xlabels("")
g.set_ylabels("R²")
                      
g.set(yticks=[0, 0.5, 1])

                                                            
for ax in g.axes.flat:
    if ax.get_legend():
        ax.get_legend().remove()

                                                       
g.fig.subplots_adjust(top=0.90, bottom=0.07, hspace=0.25, wspace=0.12)

                                      
handles, labels = g.axes[0,0].get_legend_handles_labels()
g.fig.legend(
    handles[0:3], labels[0:3],
    title="Model",
    loc="upper center",
    ncol=3,
    bbox_to_anchor=(0.5, 0.965),
    frameon=False,
    fontsize=12
)

            
g.fig.suptitle("Model Fit (R²) Across Datasets and Rotations",
                fontsize=16, y=0.995)

           

In [ ]:


                                                                    
            
                                                                    
df_clean = df[df['rSquared'].between(-0.5, 1) & (df['rmse'] >= 0)].copy()
df_clean = df_clean[df_clean['rmse'] < df_clean['rmse'].quantile(0.995)]

                                                                    
                    
                                                                    
sns.set_style("white")
sns.set_context("paper", font_scale=1.25)
palette = {'ssm': '#2E86AB', 'qlearn': '#A23B72', 'hmm': '#F18F01'}

                                      
rotation_order = [90, 60, 45, 30, 15]
dataset_order = ['CGVanilla', 'CGImp', 'Brudner', 'BondTaylor']
model_order    = ['ssm', 'qlearn', 'hmm']

                                                                    
                             
                                                                    
g = sns.FacetGrid(
    data=df_clean,
    row="rotation",
    row_order=rotation_order,
    sharey=True,
    height=2.4,
    aspect=3.2,                                        
    despine=False
)

         
g.map_dataframe(
    sns.boxplot,
    x="dataset",
    y="rSquared",
    hue="model",
    order=dataset_order,
    hue_order=model_order,
    palette=palette,
    width=0.75,
    linewidth=1.4,
    fliersize=0,
    boxprops={'edgecolor': 'black', 'linewidth': 1},
    medianprops={'color': 'white', 'linewidth': 2.2},
    whiskerprops={'linewidth': 1.3},
    capprops={'linewidth': 1.3}
)

                        
g.map_dataframe(
    sns.stripplot,
    x="dataset",
    y="rSquared",
    hue="model",
    order=dataset_order,
    hue_order=model_order,
    palette=palette,
    dodge=True,
    jitter=0.18,
    size=3.8,
    alpha=0.68,
    linewidth=0.6,
    edgecolor="black"
)

                                                                    
                 
                                                                    
g.set_titles("{row_name}° Rotation")
g.set_xlabels("")
g.set_ylabels("Model Fit (R²)")
                          
g.set(yticks=[0, 0.5, 1])

                            
for ax in g.axes.flat:
    if ax.get_legend():
        ax.get_legend().remove()

                              
handles, labels = g.axes[0,0].get_legend_handles_labels()
g.fig.legend(
    handles[:3], labels[:3],
    title="Model",
    loc="upper center",
    ncol=3,
    bbox_to_anchor=(0.5, 0.96),
    frameon=False,
    fontsize=13
)

                           
g.fig.subplots_adjust(top=0.88, hspace=0.35)
g.fig.suptitle("Model Performance by Rotation (All Datasets)", 
                fontsize=18, y=0.98, weight='bold')

           

In [ ]:
                                                                
                                                       
                                                                
                       
                                                                
summary_stats = []
detailed_stats = []
for rot in [90, 60, 45, 30, 15]:
    subset = df_clean[df_clean['rotation'] == rot]
    if len(subset) == 0:
        continue
  
               
    for model in ['ssm', 'qlearn', 'hmm']:
        model_subset = subset[subset['model'] == model]
        if len(model_subset) == 0:
            continue
        r2_mean = model_subset['rSquared'].mean()
        r2_std = model_subset['rSquared'].std()
        rmse_mean = model_subset['rmse'].mean()
        rmse_std = model_subset['rmse'].std()
        n = len(model_subset)
      
        summary_stats.append({
            'Rotation': rot,
            'Model': model.upper(),
            'N': n,
            'R²': f"{r2_mean:.3f} ± {r2_std:.3f}",
            'RMSE': f"{rmse_mean:.3f} ± {rmse_std:.3f}"
        })
  
                    
    pivot = subset.pivot_table(index='participantId', columns='model', values=['rSquared','rmse']).dropna()
    if len(pivot) < 3:
        continue
  
    for metric, values_col, better_higher in [('R²', 'rSquared', True), ('RMSE', 'rmse', False)]:
        for a, b in [('ssm','qlearn'), ('ssm','hmm'), ('qlearn','hmm')]:
            if (values_col, a) not in pivot.columns or (values_col, b) not in pivot.columns:
                continue
            x = pivot[(values_col, a)]
            y = pivot[(values_col, b)]
            t, p = stats.ttest_rel(x, y)
            diff = x - y if better_higher else y - x
            d = diff.mean() / diff.std() if diff.std() != 0 else 0
            winner = a.upper() if d > 0 else b.upper()
          
            detailed_stats.append({
                'Rotation': rot,
                'Metric': metric,
                'Comparison': f"{a.upper()} vs {b.upper()}",
                't': round(t, 3),
                'df': len(x)-1,
                'p': p,
                'd': round(d, 3),
                'Winner': winner
            })
            
summary_df = pd.DataFrame(summary_stats)
stats_df = pd.DataFrame(detailed_stats)
                 
def holm_correct(g):
    if len(g) <= 1:
        g['p_corr'] = g['p']
        return g
    reject, p_corr, _, _ = multipletests(g['p'], method='holm')
    g = g.copy()
    g['p_corr'] = p_corr
    return g
if not stats_df.empty:
    stats_df = stats_df.groupby(['Rotation','Metric']).apply(holm_correct).reset_index(drop=True)
    stats_df['sig'] = stats_df['p_corr'].apply(lambda p: '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '')
else:
    stats_df['p_corr'] = np.nan
    stats_df['sig'] = ''
                                                                
                  
                                                                
print("="*100)
print("MEAN ± SD (ALL DATASETS POOLED)")
print("="*100)
if not summary_df.empty:
    print(summary_df.sort_values(['Rotation','Model']).to_string(index=False))
else:
    print("No data available.")
print("\n" + "="*100)
print("PAIRED t-TESTS (Holm-corrected) — Positive d = first model better (R²) or lower error (RMSE)")
print("="*100)
if not stats_df.empty:
    print(stats_df[['Rotation','Metric','Comparison','t','df','p_corr','d','Winner','sig']].round(4))
else:
    print("No paired t-tests could be performed due to insufficient complete data (need >=3 participants with all models fitted).")
print("="*100)
                                                                
                          
                                                                
expected_order = ['ssm', 'qlearn', 'hmm']
models_present = [m for m in expected_order if m in df_clean['model'].unique()]
palette = {'ssm': '#2E86AB', 'qlearn': '#A23B72', 'hmm': '#F18F01'}
                                                                
                                         
                                                                
if len(models_present) == 3:
    comparisons = [
        ('SSM vs QLEARN', -0.15),
        ('QLEARN vs HMM', 0.15),
        ('SSM vs HMM', 0)
    ]
elif len(models_present) == 2:
    if set(models_present) == {'ssm', 'hmm'}:
        comparisons = [('SSM vs HMM', 0)]
    elif set(models_present) == {'ssm', 'qlearn'}:
        comparisons = [('SSM vs QLEARN', 0)]
    elif set(models_present) == {'qlearn', 'hmm'}:
        comparisons = [('QLEARN vs HMM', 0)]
    else:
        comparisons = []
elif len(models_present) == 1:
    comparisons = []
else:
    comparisons = []
                                                                
                               
                                                                
sns.set_style("white")
sns.set_context("paper", font_scale=1.6)
rot_order = list(reversed([90, 60, 45, 30, 15]))
fig_r2, ax_r2 = plt.subplots(figsize=(14, 10))
sns.boxplot(data=df_clean, x="rotation", y="rSquared", hue="model",
            order=rot_order, hue_order=models_present,
            palette=palette, width=0.6, linewidth=1.8, fliersize=0,
            boxprops=dict(edgecolor='black', alpha=0.3), medianprops=dict(color='white', linewidth=3),
            legend=False, ax=ax_r2)
sns.stripplot(data=df_clean, x="rotation", y="rSquared", hue="model",
              order=rot_order, hue_order=models_present,
              palette=palette, size=6, jitter=0.25, alpha=0.75, linewidth=0.8, edgecolor="black",
              dodge=True, legend=False, ax=ax_r2)
           
for i, rot in enumerate(rot_order):
    subset_rot = df_clean[df_clean['rotation'] == rot]
    if len(subset_rot) == 0:
        continue
    y = subset_rot['rSquared'].max() + 0.08
    offset = 0
    for comp, x_pos in comparisons:
        if stats_df.empty:
            continue
        star_row = stats_df[(stats_df['Rotation'] == rot) &
                           (stats_df['Metric'] == 'R²') &
                           (stats_df['Comparison'] == comp)]
        if not star_row.empty and star_row['sig'].iloc[0]:
            ax_r2.text(i + x_pos, y + offset, star_row['sig'].iloc[0],
                       ha='center', va='bottom', fontsize=17, fontweight='bold', color='black')
            offset += 0.08
ax_r2.set_xticklabels([f"{r}°" for r in rot_order])
ax_r2.set_xlabel("")
ax_r2.set_ylabel("Model Fit (R²)")
                             
               
handles = [mpatches.Patch(color=palette[m], label=m.upper()) for m in models_present]
ncol = len(models_present)
ax_r2.legend(handles=handles, title="Model", loc="upper center", ncol=ncol,
             bbox_to_anchor=(0.5, 0.94), frameon=False)
fig_r2.subplots_adjust(top=0.86)
fig_r2.suptitle("Best-Fitting Model by Rotation Size (R²)\n(All Datasets Pooled)", fontsize=21, y=0.98, fontweight='bold')
fig_r2.savefig(targetFolder+"FINAL_R2_with_stars_and_stats.svg", dpi=600, bbox_inches='tight')
           
                                                                
                                 
                                                                
fig_rmse, ax_rmse = plt.subplots(figsize=(14, 10))
sns.boxplot(data=df_clean, x="rotation", y="rmse", hue="model",
            order=rot_order, hue_order=models_present,
            palette=palette, width=0.6, linewidth=1.8, fliersize=0,
            boxprops=dict(edgecolor='black', alpha=0.3), medianprops=dict(color='white', linewidth=3),
            legend=False, ax=ax_rmse)
sns.stripplot(data=df_clean, x="rotation", y="rmse", hue="model",
              order=rot_order, hue_order=models_present,
              palette=palette, size=6, jitter=0.25, alpha=0.75, linewidth=0.8, edgecolor="black",
              dodge=True, legend=False, ax=ax_rmse)
                       
max_ylim_rmse = 0
for rot in rot_order:
    subset_rot = df_clean[df_clean['rotation'] == rot]
    if len(subset_rot) == 0:
        continue
    top = subset_rot['rmse'].quantile(0.98)
    max_ylim_rmse = max(max_ylim_rmse, top * 1.65)
                                   
           
for i, rot in enumerate(rot_order):
    subset_rot = df_clean[df_clean['rotation'] == rot]
    if len(subset_rot) == 0:
        continue
    top = subset_rot['rmse'].quantile(0.98)
    y = top * 1.15
    offset = 0
    for comp, x_pos in comparisons:
        if stats_df.empty:
            continue
        star_row = stats_df[(stats_df['Rotation'] == rot) &
                           (stats_df['Metric'] == 'RMSE') &
                           (stats_df['Comparison'] == comp)]
        if not star_row.empty and star_row['sig'].iloc[0]:
            ax_rmse.text(i + x_pos, y + offset, star_row['sig'].iloc[0],
                         ha='center', va='bottom', fontsize=17, fontweight='bold', color='black')
            offset += top * 0.08
ax_rmse.set_xticklabels([f"{r}°" for r in rot_order])
ax_rmse.set_xlabel("")
ax_rmse.set_ylabel("Prediction Error (RMSE)\n(lower = better)")
               
ax_rmse.legend(handles=handles, title="Model", loc="upper center", ncol=ncol,
               bbox_to_anchor=(0.5, 0.94), frameon=False)
fig_rmse.subplots_adjust(top=0.86)
fig_rmse.suptitle("Model Prediction Error by Rotation Size (RMSE)\n(All Datasets Pooled)", fontsize=21, y=0.98, fontweight='bold')
fig_rmse.savefig(targetFolder+"FINAL_RMSE_with_stars_and_stats.svg", dpi=600, bbox_inches='tight')
           
print("\nFigures saved with CORRECT STARS:")
print("→ FINAL_R2_with_stars_and_stats.pdf")
print("→ FINAL_RMSE_with_stars_and_stats.pdf")
print("\nYou are 100% ready for submission.")

In [ ]:


def angular_diff(a, b, period=360.0):
    """Compute the signed angular difference between a and b, in [-period/2, period/2]."""
    diff = (a - b) % period
    return np.where(diff > period / 2, diff - period, diff)

def circular_mean(angles, period=360.0):
    """Compute the circular mean of angles in degrees."""
    angles_rad = np.deg2rad(angles)
    mean_sin = np.mean(np.sin(angles_rad))
    mean_cos = np.mean(np.cos(angles_rad))
    mean_rad = np.arctan2(mean_sin, mean_cos)
    return np.rad2deg(mean_rad)

def compute_rmse(y_true, y_pred, period=360.0):
    """Circular RMSE using angular differences."""
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    if mask.sum() < 2:
        return np.nan
    yt, yp = y_true[mask], y_pred[mask]
    diffs = angular_diff(yt, yp, period)
    return np.sqrt(np.mean(diffs ** 2))

def compute_r2(y_true, y_pred, period=360.0):
    """Circular R²: 1 - (SS_res / SS_tot), using angular deviations."""
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    if mask.sum() < 2:
        return np.nan
    yt, yp = y_true[mask], y_pred[mask]
    
                                       
    res_diffs = angular_diff(yt, yp, period)
    ss_res = np.sum(res_diffs ** 2)
    
                                                         
    mu = circular_mean(yt, period)
    tot_diffs = angular_diff(yt, mu, period)
    ss_tot = np.sum(tot_diffs ** 2)
    
    return 1 - ss_res / ss_tot if ss_tot > 1 else np.nan

                                                                    
                                                        
                                                                    
rows = []
rots = list(reversed([15, 30, 45, 60, 90]))                        

for rot_idx, rotation in enumerate(rots):
    ssm_fit   = ssmsAll8BT[rot_idx]   if rot_idx < len(ssmsAll8BT)   else None
    qlearn_fit = qLearnsAll8BT[rot_idx] if rot_idx < len(qLearnsAll8BT) else None
    hmm_fit   = hmmsAll8BT[rot_idx]   if rot_idx < len(hmmsAll8BT)   else None

                                                         
    n_parts = None
    for f in (ssm_fit, qlearn_fit, hmm_fit):
        if f is not None:
            if hasattr(f, 'allAims'):
                n_parts = len(f.allAims)
            elif hasattr(f, 'human_explicits'):
                n_parts = len(f.human_explicits)
            break
    if n_parts is None:
        continue

    for p in range(n_parts):
                                                               
        def get_seq(attr_list):
            if attr_list is None or p >= len(attr_list):
                return np.array([])
            seq = np.array(attr_list[p])
            return seq.ravel() if seq.ndim > 1 else seq

                                                                         
        if ssm_fit is not None:
            h_exp = get_seq(ssm_fit.allAims)
            m_exp = get_seq(ssm_fit.mOut1)
            h_imp = get_seq(ssm_fit.allImps)
            m_imp = get_seq(ssm_fit.mOut2)

            rows.append({
                'dataset'       : 'BondTaylor',
                'rotation'      : rotation,
                'participantId' : p,
                'model'         : 'ssm',
                'rmse_explicit' : compute_rmse(h_exp, m_exp),
                'r2_explicit'   : compute_r2(h_exp, m_exp),
                'rmse_implicit' : compute_rmse(h_imp, m_imp),
                'r2_implicit'   : compute_r2(h_imp, m_imp)
            })

                                                                          
        for model_name, fit in [('qlearn', qlearn_fit), ('hmm', hmm_fit)]:
            if fit is None or not hasattr(fit, 'human_explicits') or p >= len(fit.human_explicits):
                continue

            h_exp = np.array(fit.human_explicits[p]).ravel()
            m_exp = np.array(fit.model_explicits[p]).ravel()
            h_imp = np.array(fit.human_implicits[p]).ravel()
            m_imp = np.array(fit.model_implicits[p]).ravel()

            rows.append({
                'dataset'       : 'BondTaylor',
                'rotation'      : rotation,
                'participantId' : p,
                'model'         : model_name,
                'rmse_explicit' : compute_rmse(h_exp, m_exp),
                'r2_explicit'   : compute_r2(h_exp, m_exp),
                'rmse_implicit' : compute_rmse(h_imp, m_imp),
                'r2_implicit'   : compute_r2(h_imp, m_imp)
            })

                                                                    
                                     
                                                                    
df_bt = pd.DataFrame(rows)

             
df_bt = df_bt.sort_values(['rotation', 'participantId', 'model']).reset_index(drop=True)

                                                                    
                                                     
                                                                    
print("\nBondTaylor – Explicit vs Implicit Trial-wise Fit (per participant)")
print("="*100)
print(df_bt.head(24))
print(f"Total rows: {len(df_bt)}")

print("\n" + "="*100)
print("SUMMARY STATISTICS (mean ± std) – NaNs are excluded automatically")
print("="*100)

summary = (df_bt
           .groupby(['rotation', 'model'])
           .agg(
               rmse_explicit_mean   = ('rmse_explicit', 'mean'),
               rmse_explicit_std    = ('rmse_explicit', 'std'),
               r2_explicit_mean     = ('r2_explicit',   'mean'),
               r2_explicit_std      = ('r2_explicit',   'std'),
               rmse_implicit_mean   = ('rmse_implicit', 'mean'),
               rmse_implicit_std    = ('rmse_implicit', 'std'),
               r2_implicit_mean     = ('r2_implicit',   'mean'),
               r2_implicit_std      = ('r2_implicit',   'std'),
               n_valid              = ('participantId', 'count')                                         
           )
           .round(4))

                   
summary['Explicit RMSE'] = summary['rmse_explicit_mean'].astype(str) + " ± " + summary['rmse_explicit_std'].astype(str)
summary['Explicit R²']   = summary['r2_explicit_mean'].astype(str) + " ± " + summary['r2_explicit_std'].astype(str)
summary['Implicit RMSE'] = summary['rmse_implicit_mean'].astype(str) + " ± " + summary['rmse_implicit_std'].astype(str)
summary['Implicit R²']   = summary['r2_implicit_mean'].astype(str) + " ± " + summary['r2_implicit_std'].astype(str)

pretty = summary[['Explicit RMSE', 'Explicit R²', 'Implicit RMSE', 'Implicit R²', 'n_valid']].reset_index()
print(pretty.to_string(index=False))

In [ ]:
                                                                
                                                                        
                                                                

                                                             
                                                                      
df_explicit = df_bt[['rotation', 'model', 'participantId', 'r2_explicit', 'rmse_explicit']].copy()
df_explicit.rename(columns={'r2_explicit': 'rSquared', 'rmse_explicit': 'rmse'}, inplace=True)
df_implicit = df_bt[['rotation', 'model', 'participantId', 'r2_implicit', 'rmse_implicit']].copy()
df_implicit.rename(columns={'r2_implicit': 'rSquared', 'rmse_implicit': 'rmse'}, inplace=True)
                                                                    
def run_analysis(df, title_suffix,ylimR2=(0,1)):
    summary_stats = []
    detailed_stats = []
    for rot in [90, 60, 45, 30, 15]:
        subset = df[df['rotation'] == rot]
        if len(subset) == 0:
            continue
     
                   
        for model in ['ssm', 'qlearn', 'hmm']:
            model_subset = subset[subset['model'] == model]
            if len(model_subset) == 0:
                continue
            r2_mean = model_subset['rSquared'].mean()
            r2_std = model_subset['rSquared'].std()
            rmse_mean = model_subset['rmse'].mean()
            rmse_std = model_subset['rmse'].std()
            n = len(model_subset)
         
            summary_stats.append({
                'Rotation': rot,
                'Model': model.upper(),
                'N': n,
                'R²': f"{r2_mean:.3f} ± {r2_std:.3f}",
                'RMSE': f"{rmse_mean:.3f} ± {rmse_std:.3f}"
            })
     
                        
        pivot = subset.pivot_table(index='participantId', columns='model', values=['rSquared','rmse']).dropna()
        if len(pivot) < 3:
            continue
     
        for metric, values_col, better_higher in [('R²', 'rSquared', True), ('RMSE', 'rmse', False)]:
            for a, b in [('ssm','qlearn'), ('ssm','hmm'), ('qlearn','hmm')]:
                if (values_col, a) not in pivot.columns or (values_col, b) not in pivot.columns:
                    continue
                x = pivot[(values_col, a)]
                y = pivot[(values_col, b)]
                t, p = stats.ttest_rel(x, y)
                diff = x - y if better_higher else y - x
                d = diff.mean() / diff.std() if diff.std() != 0 else 0
                winner = a.upper() if d > 0 else b.upper()
             
                detailed_stats.append({
                    'Rotation': rot,
                    'Metric': metric,
                    'Comparison': f"{a.upper()} vs {b.upper()}",
                    't': round(t, 3),
                    'df': len(x)-1,
                    'p': p,
                    'd': round(d, 3),
                    'Winner': winner
                })
  
                
    summary_df = pd.DataFrame(summary_stats)
    stats_df = pd.DataFrame(detailed_stats)
  
                                       
    if not stats_df.empty:
        def holm_correct(g):
            if len(g) <= 1:
                g['p_corr'] = g['p']
                return g
            reject, p_corr, _, _ = multipletests(g['p'], method='holm')
            g = g.copy()
            g['p_corr'] = p_corr
            return g
        stats_df = stats_df.groupby(['Rotation','Metric']).apply(holm_correct).reset_index(drop=True)
        stats_df['sig'] = stats_df['p_corr'].apply(lambda p: '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '')
    else:
                                                            
        stats_df['p_corr'] = np.nan
        stats_df['sig'] = ''
  
                      
    print("="*100)
    print(f"MEAN ± SD (BOND TAYLOR {title_suffix.upper()} — ALL DATASETS POOLED)")
    print("="*100)
    if not summary_df.empty:
        print(summary_df.sort_values(['Rotation','Model']).to_string(index=False))
    else:
        print("No data available.")
    print("\n" + "="*100)
    print(f"PAIRED t-TESTS (Holm-corrected) — Positive d = first model better (R²) or lower error (RMSE) ({title_suffix.upper()})")
    print("="*100)
    if not stats_df.empty:
        print(stats_df[['Rotation','Metric','Comparison','t','df','p_corr','d','Winner','sig']].round(4))
    else:
        print("No paired t-tests could be performed due to insufficient complete data (need >=3 participants with all models fitted).")
    print("="*100)
  
               
    sns.set_style("white")
    sns.set_context("paper", font_scale=1.6)
    palette = {'ssm': '#2E86AB', 'qlearn': '#A23B72', 'hmm': '#F18F01'}
    rot_order = list(reversed([90, 60, 45, 30, 15]))
    fig_r2, ax_r2 = plt.subplots(figsize=(14, 6))
    sns.boxplot(data=df, x="rotation", y="rSquared", hue="model",
                order=rot_order, hue_order=['ssm', 'qlearn', 'hmm'],
                palette=palette, width=0.6, linewidth=1.8, fliersize=0,
                boxprops=dict(edgecolor='black', alpha=0.3), medianprops=dict(color='white', linewidth=3),
                legend=False, ax=ax_r2)
    sns.stripplot(data=df, x="rotation", y="rSquared", hue="model",
                  order=rot_order, hue_order=['ssm', 'qlearn', 'hmm'],
                  palette=palette, size=12, jitter=0.25, alpha=0.75, linewidth=0.8, edgecolor="black",
                  dodge=True, legend=False, ax=ax_r2)
               
    comparisons = [('SSM vs QLEARN', -0.1), ('QLEARN vs HMM', 0.1), ('SSM vs HMM', 0)]
    for i, rot in enumerate(rot_order):
        subset_rot = df[df['rotation'] == rot]
        if len(subset_rot) == 0:
            continue
        y_start = subset_rot['rSquared'].max() + 0.08
        offset = 0
        for comp, x_pos in comparisons:
            if stats_df.empty:
                continue
            star_row = stats_df[(stats_df['Rotation'] == rot) &
                               (stats_df['Metric'] == 'R²') &
                               (stats_df['Comparison'] == comp)]
            if not star_row.empty and star_row['sig'].iloc[0]:
                ax_r2.text(i + x_pos, y_start + offset, star_row['sig'].iloc[0],
                           ha='center', va='bottom', fontsize=17, fontweight='bold', color='black')
                offset += 0.08
    ax_r2.set_xticklabels([f"{r}°" for r in rot_order])
    ax_r2.set_xlabel("")
    ax_r2.set_ylabel("Model Fit (R²)")
    ax_r2.set_ylim(ylimR2)
                   
    hue_order = ['ssm', 'qlearn', 'hmm']
    handles = [mpatches.Patch(color=palette[m], label=m.upper()) for m in hue_order]
                                                                             
                                                            
    fig_r2.subplots_adjust(top=0.86)
    fig_r2.suptitle(f"Best-Fitting Model by Rotation Size (R²)\n(BondTaylor {title_suffix.title()})", fontsize=21, y=0.98, fontweight='bold')
    fig_r2.savefig(targetFolder+f"BT_{title_suffix}_R2_with_stars_and_stats.svg", dpi=600, bbox_inches='tight')
               
  
                 
    fig_rmse, ax_rmse = plt.subplots(figsize=(14, 6))
    sns.boxplot(data=df, x="rotation", y="rmse", hue="model",
                order=rot_order, hue_order=['ssm', 'qlearn', 'hmm'],
                palette=palette, width=0.6, linewidth=1.8, fliersize=0,
                boxprops=dict(edgecolor='black', alpha=0.3), medianprops=dict(color='white', linewidth=3),
                legend=False, ax=ax_rmse)
    sns.stripplot(data=df, x="rotation", y="rmse", hue="model",
                  order=rot_order, hue_order=['ssm', 'qlearn', 'hmm'],
                  palette=palette, size=12, jitter=0.25, alpha=0.75, linewidth=0.8, edgecolor="black",
                  dodge=True, legend=False, ax=ax_rmse)
               
    max_rmse_global = df['rmse'].max()
    for i, rot in enumerate(rot_order):
        subset_rot = df[df['rotation'] == rot]
        if len(subset_rot) == 0:
            continue
        top = subset_rot['rmse'].quantile(0.98)
        y_start = top * 1.15
        offset = 0
        for comp, x_pos in comparisons:
            if stats_df.empty:
                continue
            star_row = stats_df[(stats_df['Rotation'] == rot) &
                               (stats_df['Metric'] == 'RMSE') &
                               (stats_df['Comparison'] == comp)]
            if not star_row.empty and star_row['sig'].iloc[0]:
                ax_rmse.text(i + x_pos, y_start + offset, star_row['sig'].iloc[0],
                             ha='center', va='bottom', fontsize=17, fontweight='bold', color='black')
                offset += top * 0.08
    ax_rmse.set_xticklabels([f"{r}°" for r in rot_order])
    ax_rmse.set_xlabel("")
    ax_rmse.set_ylabel("Prediction Error (RMSE)\n(lower = better)")
                                                  
                   
    handles = [mpatches.Patch(color=palette[m], label=m.upper()) for m in hue_order]
    ax_rmse.legend(handles=handles, title="Model", loc="upper center", ncol=3,
                   bbox_to_anchor=(0.5, 0.94), frameon=False)
    fig_rmse.subplots_adjust(top=0.86)
    fig_rmse.suptitle(f"Model Prediction Error by Rotation Size (RMSE)\n(BondTaylor {title_suffix.title()})", fontsize=21, y=0.98, fontweight='bold')
    fig_rmse.savefig(targetFolder+f"BT_{title_suffix}_RMSE_with_stars_and_stats.svg", dpi=600, bbox_inches='tight')
               
  
    print(f"\nFigures saved for {title_suffix.upper()}:")
    print(f"→ BT_{title_suffix}_R2_with_stars_and_stats.svg")
    print(f"→ BT_{title_suffix}_RMSE_with_stars_and_stats.svg")
    print(f"\nYou are 100% ready for submission ({title_suffix.upper()}).")
    return summary_df, stats_df
                  
print("EXPLICIT ANALYSIS:")
summary_exp, stats_exp = run_analysis(df_explicit, "explicit",ylimR2=(-4,1.3))
                  
print("\n\nIMPLICIT ANALYSIS:")
summary_imp, stats_imp = run_analysis(df_implicit, "implicit",ylimR2=(-35,1.5))

In [ ]:
                                                                                                                                                     
                                                                 
                                                                                                                      
                                                                                                                                                        
                                                                                                   
                                                                                                                                                                 
                                                                                       


rots = list(reversed([15, 30, 45, 60, 90]))                       
datasets = [
    [ssmsAll8CG, qLearnsAll8CG, hmmsAll8CG],            
    [ssmsAll8CGIMP, qLearnsAll8CGIMP, hmmsAll8CGIMP],        
    [ssmsAllBrudner, qLearnsAllBrudner, hmmsAllBrudner],                  
    [ssmsAll8BT, qLearnsAll8BT, hmmsAll8BT]             
]

In [ ]:

baselineLength = 40
washoutLength = 40
def plot_human_aim_baseline_aha_next(ylim=None, save_filename=None, explicit_only=False):
    """
    Generate plots of mean absolute human aim (with 95% CI error bands across humans) for:
    - Baseline as a horizontal dashed line.
    - The trial series spanning t-10 to t+9 relative to the detected Aha! trial t (averaged across participants).
    Separate subplots for each dataset, with lines for each rotation.
    Only includes participants/groups where the Aha! trial t >= baselineLength
    (with valid t-10 to t+9 after baseline where possible, handling out-of-bounds as NaN), and t+9 before the washout phase (last 40 trials).
    The Aha! trial is detected separately for each participant: the post-baseline t (<=200) that minimizes
    the individual cost of |aim[part,t-1]| + max(0, |rot[part,t]| - |aim[part,t]|),
    preferring larger |aim[part,t]| in case of average cost ties.
    Requires sufficient valid data around t for cost computation (up to t-5 to t+4).
    Uses np.nanmean, np.nanstd, etc., to handle NaNs by omitting them.
    Parameters:
    - ylim: Tuple (bottom, top) for y-axis limits; if None, auto-scales to data.
    - save_filename: String for saving the plot; if None, just shows it.
    - explicit_only: If True, use only human_explicits (no +implicits) for aims.
    Returns:
    - df: The DataFrame of collected summary data (means, CIs, n per group).
    """
                                                                       
    summary_points = []
    datasets = [
        [ssmsAll8CG, qLearnsAll8CG, hmmsAll8CG],            
        [ssmsAll8CGIMP, qLearnsAll8CGIMP, hmmsAll8CGIMP],        
        [ssmsAllBrudner, qLearnsAllBrudner, hmmsAllBrudner],                  
        [ssmsAll8BT, qLearnsAll8BT, hmmsAll8BT]             
    ]
    datasetLabels = ['CGVanilla8', 'CGImp8', 'Brudner', 'BondTaylor']
                                                                                             
    rotation_colors = list(reversed(['#44AA99', '#88CCEE', '#FF9825', '#CC6677', '#AA4499']))
    rel_positions = list(range(-20, 20))
    for dsIdx, datasetName in enumerate(datasetLabels):
        ssm_list = datasets[dsIdx][0]
        hmm_list = datasets[dsIdx][2]
        rotations = []
        rotation_sizes = []
        if datasetName == 'Brudner':
            bl = 64
            washoutLength = 64
        elif '1' in datasetName:
            bl = 15
            washoutLength = 15
        else:
            bl = 40
            washoutLength = 40
                                                    
                                                       
                                                       
        for ssm in ssm_list:
            if hasattr(ssm, 'rotations'):
                rot_array = np.asarray(ssm.rotations)
                rotations.append(rot_array)
                non_zero_rots = rot_array[rot_array != 0]
                if len(non_zero_rots) > 0:
                    rot_size = float(np.unique(non_zero_rots)[0])
                else:
                    rot_size = 0.0
                rotation_sizes.append(rot_size)
            else:
                rotations.append(None)
                rotation_sizes.append(0.0)
                                                                                                                      
        for rotIdx, (rot_array, rotation_size) in enumerate(zip(rotations, rotation_sizes)):
            if rot_array is None or rotIdx >= len(hmm_list) or rotIdx >= len(ssm_list):
                continue
            ssm_fit = ssm_list[rotIdx]
            hmm_fit = hmm_list[rotIdx]
            if not hasattr(hmm_fit, 'human_explicits'):
                                                                                        
                continue
            human_explicits_full = np.array(hmm_fit.human_explicits)
            if explicit_only:
                human_abs_full = np.abs(human_explicits_full)
            else:
                if hasattr(hmm_fit, 'human_implicits'):
                    human_implicits_full = np.array(hmm_fit.human_implicits)
                    human_full = human_explicits_full + human_implicits_full
                    human_abs_full = np.abs(human_full)
                else:
                    human_abs_full = np.abs(human_explicits_full)
            n_parts, n_trials = human_abs_full.shape
                                                                                            
 
                                   
            baseline_human_abs_list = []
            series_list = []
            valid_parts = 0
 
            baseline_trial = bl - 1
            if baseline_trial < 0 or baseline_trial >= n_trials:
                                                                                           
                continue
 
            min_t = bl+1 
            max_t = n_trials - (washoutLength + min(40,bl))
            if min_t > max_t:
                                                                                                                
                continue
                                                           
            for part in range(n_parts):
                aim = human_abs_full[part, :]
          
                                                    
                if rot_array.ndim == 1:
                    part_rot_array = rot_array[part]
                else:
                    part_rot_array = rot_array[part, :]
          
                                                                                                 
              
                min_cost = np.inf
                best_t_part = None
                best_aim_t = -np.inf
                for t in range(min_t, max_t + 1):
                    pre_aha_trial = t - 1
                    pre_prev_trial = t - 2
                    next_trial = t + 1
                    pre_5 = t - 5
                    post_4 = t + 4
              
                                                                                         
              
                                            
                    if len(part_rot_array) > 1 and t < len(part_rot_array):
                        rot_val = part_rot_array[t]
                    else:
                        rot_val = part_rot_array[0] if len(part_rot_array) > 0 else 0.0
                    if np.isnan(rot_val):
                        valid_rots = [r for r in part_rot_array if not np.isnan(r)]
                        if len(valid_rots) == 0:
                            continue
                        rot_val = valid_rots[t+1]
                    target_mag = np.abs(rot_val)
                                                                                                                                        
                    cost = np.nansum(np.maximum(np.abs(human_abs_full[part, bl:t]) - np.nanmean(human_abs_full[part, :bl]), 0)) + (20) * (np.nanmean(np.abs(np.abs(rot_val) - human_abs_full[part, t:min(max_t, t + min(40, bl))])))
                                                           
                    if cost < min_cost:
                        min_cost = cost
                        best_t_part = t
                        best_aim_t = np.abs(aim[t])
          
                if best_t_part is None:
                                                                                                             
                    continue
          
                                                                           
                series_part = np.full(len(rel_positions), np.nan, dtype=float)
                for ii, rel in enumerate(rel_positions):
                    trial_idx = best_t_part + rel
                    if 0 <= trial_idx < n_trials:
                        series_part[ii] = aim[trial_idx]
                baseline_human_abs = aim[baseline_trial]
          
                baseline_human_abs_list.append(baseline_human_abs)
                series_list.append(series_part)
                valid_parts += 1
          
                                                                                                                    
 
                                                                                                                 
 
                                                                       
            n_series = len(series_list)
            if n_series > 0:
                series_array = np.stack(series_list)
                mean_series = np.nanmean(series_array, axis=0)
                std_series = np.nanstd(series_array, axis=0, ddof=1)
                n_per_pos = np.sum(~np.isnan(series_array), axis=0)
                sem_series = np.divide(std_series, np.sqrt(n_per_pos), out=np.full_like(std_series, np.nan), where=(n_per_pos > 1))
                ci_series = 1.96 * sem_series
           
                baseline_array = np.array(baseline_human_abs_list)
                n_base = np.sum(~np.isnan(baseline_array))
                if n_base > 0:
                    mean_base = np.nanmean(baseline_array)
                    std_base = np.nanstd(baseline_array, ddof=1)
                    sem_base = std_base / np.sqrt(n_base)
                    ci_base = 1.96 * sem_base
                else:
                    mean_base = np.nan
                    ci_base = np.nan
           
                summary_points.append({
                    'rotation': rotation_size,
                    'dataset': datasetName,
                    'mean_baseline': mean_base,
                    'ci_baseline': ci_base,
                    'n_base': n_base,
                    'mean_series': mean_series.tolist(),
                    'ci_series': ci_series.tolist(),
                    'n_series': n_series,
                    'individual_series': series_list,
                    'rel_pos': rel_positions
                })
            else:
                pass                                                                                  
                          
    df = pd.DataFrame(summary_points)
    if df.empty:
        pass                                                             
        return df
                                                        
                                                  
    unique_rots = sorted(df['rotation'].unique(), reverse=True)
    rot_to_color_idx = {rot: i for i, rot in enumerate(unique_rots)}
                                           
    fig, axs = plt.subplots(2, 2, figsize=(15, 16))
    axs = axs.flatten()
    for dsIdx, datasetName in enumerate(datasetLabels):
        ax = axs[dsIdx]
        subset = df[df['dataset'] == datasetName]
        if subset.empty:
            ax.set_visible(False)
            continue
                                               
        subset_rots = sorted(subset['rotation'].unique())
                            
        ax.axhline(y=0, color='grey', linestyle='--', linewidth=1, alpha=0.3)
                                                  
        for rot in subset_rots:
            color_idx = rot_to_color_idx[rot]
            rot_color = rotation_colors[color_idx]
            ax.axhline(y=-rot, color=rot_color, alpha=0.5, linestyle='--', linewidth=1)
                                              
        yticks = [-i for i in sorted([0] + list(subset_rots))]
        yticklabels = [f'{int(t)}°' for t in yticks]
        ax.set_yticks(yticks)
        ax.set_yticklabels(yticklabels)
        for _, row in subset.iterrows():
            rot = row['rotation']
            color_idx = rot_to_color_idx[rot]
            rot_color = rotation_colors[color_idx]
            mean_ser = np.array(row['mean_series'])
            ci_ser = np.array(row['ci_series'])
            rel_pos = np.array(row['rel_pos'])                                              
                                                                    
            mean_ser = np.maximum(mean_ser, 0)
                                          
            lower_ci = np.maximum(mean_ser - ci_ser, 0)
            upper_ci = mean_ser + ci_ser
            
                                                                      
            rgb = mcolors.to_rgb(rot_color)
            edge_rgb = tuple(c * 0.65 for c in rgb)                                              
            
                                                                  
            ax.scatter(rel_pos, mean_ser, 
                       c=rot_color,                      
                       edgecolor=edge_rgb,                           
                       linewidth=2.5,                        
                       s=140, 
                       zorder=15, 
                       label=f'{rot}°')
                                                  
            ax.fill_between(rel_pos, lower_ci, upper_ci, color=rot_color, alpha=0.2)
                                                       
            for ind_series in row['individual_series']:
                valid_mask = ~np.isnan(ind_series)
                if np.any(valid_mask):
                    valid_x = rel_pos[valid_mask]
                    valid_y = ind_series[valid_mask]
                    jitter = np.random.uniform(-0.3, 0.3, size=len(valid_x))
                    x_jittered = valid_x + jitter
                    ax.scatter(x_jittered, valid_y, color=rot_color, alpha=0.3, s=15, zorder=6)
         
                           
        ax.set_title(datasetName)
        ax.set_xlabel('Relative Trial (0 = Aha!)')
        ax.set_ylabel('Mean |Human Aim| (°)')
        ax.axvline(x=0, color='black', linestyle='-', alpha=0.3, linewidth=1)                  
                    
        ax.set_ylim(ylim)
        ax.set_xlim(-16.5,16.45)
        ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    sns.despine()
    fig.suptitle('Mean |Aim| Around Aha! Trial with 95% CI Bands (Separate Detection per Participant)', fontsize=14)
    plt.tight_layout()
    if save_filename:
        plt.savefig(save_filename, dpi=300, bbox_inches='tight')
               
                   
                                                                                             
                                                  
                                 
                                                                                     
    return df
df_human = plot_human_aim_baseline_aha_next(ylim=(-5,130), explicit_only=True,save_filename=targetFolder+'ahaAlignedAimMagnitude.svg')


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import bootstrap
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.container import BarContainer
import os
baselineLength = 40
washoutLength = 40
def compute_ci95(data):
    series = pd.Series(data).dropna()
    n = len(series)
    if n == 0:
        return np.nan, np.nan, np.nan
    mean = series.mean()
    if n < 2:
        return mean, np.nan, np.nan
    data_tuple = (series.values,)
    def mean_stat(x):
        return np.mean(x)
    try:
        res = bootstrap(data_tuple, mean_stat, confidence_level=0.95,
                        n_resamples=20000, method='BCa', random_state=42)
        low = res.confidence_interval.low
        up = res.confidence_interval.high
    except Exception:
        res = bootstrap(data_tuple, mean_stat, confidence_level=0.95,
                        n_resamples=20000, method='percentile', random_state=42)
        low = res.confidence_interval.low
        up = res.confidence_interval.high
    return mean, low, up
def get_mean_ci_str(data):
    mean, low, up = compute_ci95(data)
    if np.isnan(mean):
        return "no data"
    if np.isnan(low) or np.isnan(up):
        return f"{mean:.3f} (insuff. data for CI)"
    return f"{mean:.3f} [{low:.3f}, {up:.3f}]"
def analyze_pre_aha_behavior(explicit_only=False, min_pre_trials=3):
    var_points = []
    reg_points = []
    ac_points = []
    summary_var = []
    summary_reg = []
    summary_ac = []
    datasets = [
        [ssmsAll8CG, qLearnsAll8CG, hmmsAll8CG],
        [ssmsAll8CGIMP, qLearnsAll8CGIMP, hmmsAll8CGIMP],
        [ssmsAllBrudner, qLearnsAllBrudner, hmmsAllBrudner],
        [ssmsAll8BT, qLearnsAll8BT, hmmsAll8BT]
    ]
    datasetLabels = ['CGVanilla8', 'CGImp8', 'Brudner', 'BondTaylor']
    for dsIdx, datasetName in enumerate(datasetLabels):
        ssm_list = datasets[dsIdx][0]
        hmm_list = datasets[dsIdx][2]
        rotations = []
        rotation_sizes = []
        if datasetName == 'Brudner':
            bl = 64
            washoutLength_local = 64
        elif '1' in datasetName:
            bl = 15
            washoutLength_local = 15
        else:
            bl = 40
            washoutLength_local = 40
        for ssm in ssm_list:
            if hasattr(ssm, 'rotations'):
                rot_array = np.asarray(ssm.rotations)
                rotations.append(rot_array)
                non_zero_rots = rot_array[rot_array != 0]
                if len(non_zero_rots) > 0:
                    rot_size = float(np.unique(non_zero_rots)[0])
                else:
                    rot_size = 0.0
                rotation_sizes.append(rot_size)
            else:
                rotations.append(None)
                rotation_sizes.append(0.0)
        for rotIdx, (rot_array, rotation_size) in enumerate(zip(rotations, rotation_sizes)):
            if rot_array is None or rotIdx >= len(hmm_list) or rotIdx >= len(ssm_list):
                continue
            ssm_fit = ssm_list[rotIdx]
            hmm_fit = hmm_list[rotIdx]
            if not hasattr(hmm_fit, 'human_explicits'):
                continue
            human_explicits_full = np.array(hmm_fit.human_explicits)
            if explicit_only:
                human_abs_full = np.abs(human_explicits_full)
                human_signed_full = human_explicits_full.copy()
            else:
                if hasattr(hmm_fit, 'human_implicits'):
                    human_implicits_full = np.array(hmm_fit.human_implicits)
                    human_full = human_explicits_full + human_implicits_full
                    human_abs_full = np.abs(human_full)
                    human_signed_full = human_full
                else:
                    human_abs_full = np.abs(human_explicits_full)
                    human_signed_full = human_explicits_full.copy()
                                                                 
            if 'CG' in datasetName or 'BT' in datasetName:
                human_abs_full = np.round(human_abs_full)
                human_signed_full = np.round(human_signed_full)
            n_parts, n_trials = human_abs_full.shape
            min_t = bl + 1
            max_t = n_trials - (washoutLength_local + min(40, bl))
            if min_t > max_t:
                continue
            rot_signed_constant = rotation_size
     
            for part in range(n_parts):
                aim_abs = human_abs_full[part, :]
                aim_signed = human_signed_full[part, :]
                if rot_array.ndim == 1:
                    part_rot_array = rot_array[part]
                else:
                    part_rot_array = rot_array[part, :]
                min_cost = np.inf
                best_t_part = None
                best_aim_t = -np.inf
                for t in range(min_t, max_t + 1):
                    if len(part_rot_array) > 1 and t < len(part_rot_array):
                        rot_val = part_rot_array[t]
                    else:
                        rot_val = part_rot_array[0] if len(part_rot_array) > 0 else 0.0
                    if np.isnan(rot_val):
                        valid_rots = [r for r in part_rot_array if not np.isnan(r)]
                        rot_val = 0.0 if len(valid_rots) == 0 else valid_rots[0]
                    aim = aim_abs
                    cost = np.nansum(np.maximum(np.abs(human_abs_full[part, bl:t]) - np.nanmean(human_abs_full[part, :bl]), 0)) + (20) * (np.nanmean(np.abs(np.abs(rot_val) - human_abs_full[part, t:min(max_t, t + min(40, bl))])))
                    abs_aim_t = np.abs(aim_abs[t]) if t < n_trials else np.nan
                    if np.isnan(abs_aim_t):
                        continue
                    if cost < min_cost or (cost == min_cost and abs_aim_t > best_aim_t):
                        min_cost = cost
                        best_t_part = t
                        best_aim_t = abs_aim_t
   
                if best_t_part is None:
                    continue
                pre_aha_start = bl
                pre_aha_end = best_t_part
                n_pre_aha = pre_aha_end - pre_aha_start
                if n_pre_aha < min_pre_trials:
                    continue
                window_size = min(n_pre_aha, n_trials - best_t_part)
                aim_pre_rot = aim_signed[pre_aha_start:pre_aha_start + window_size]
                aim_post_rot = aim_signed[best_t_part:best_t_part + window_size]
                baseline_size = min(bl,window_size)
                late_start = bl - baseline_size
                aim_late_baseline = aim_signed[late_start:bl]
             
                def std_of_aims(seq):
                    valid = seq[~np.isnan(seq)]
                    if len(valid) < 2:
                        return np.nan
                    return np.std(valid, ddof=1)
                def lag1_autocorr(seq):
                    valid = seq[~np.isnan(seq)]
                    if len(valid) < 2:
                        return np.nan
                    corr = np.corrcoef(valid[:-1], valid[1:])[0, 1]
                    if np.isnan(corr) or abs(corr) >= 1 - 1e-10:
                        return np.nan
                    return corr
         
                def compute_rate(seq, rot_constant, is_baseline=False):
                    if len(seq) < 2:
                        return np.nan
                    deltas = np.diff(seq)
                    if is_baseline:
                        errors = seq[:-1]
                    else:
                        errors = rot_constant + seq[:-1]
                    valid_mask = ~np.isnan(deltas) & ~np.isnan(errors)
                    valid_x = errors[valid_mask]
                    valid_y = deltas[valid_mask]
                    if len(valid_x) == 0:
                        return np.nan
                    denom = np.sum(valid_x ** 2)
                    if denom <= 1e-6:
                        return np.nan
                    slope = np.sum(valid_x * valid_y) / denom
                    return -slope
         
                std_late_baseline = std_of_aims(aim_late_baseline)
                std_pre_aha = std_of_aims(aim_pre_rot)
                std_post_aha = std_of_aims(aim_post_rot)
                var_points.append({
                    'dataset': datasetName,
                    'rotation': abs(rotation_size),
                    'participant': part,
                    'n_pre_aha': n_pre_aha,
                    'std_late_baseline': std_late_baseline,
                    'std_pre_aha': std_pre_aha,
                    'std_post_aha': std_post_aha
                })
                rate_late_baseline = compute_rate(aim_late_baseline, rot_signed_constant, is_baseline=True)
                rate_pre_aha = compute_rate(aim_pre_rot, rot_signed_constant, is_baseline=False)
                rate_post_aha = compute_rate(aim_post_rot, rot_signed_constant, is_baseline=False)
                reg_points.append({
                    'dataset': datasetName,
                    'rotation': abs(rotation_size),
                    'participant': part,
                    'n_pre_aha': n_pre_aha,
                    'rate_late_baseline': rate_late_baseline,
                    'rate_pre_aha': rate_pre_aha,
                    'rate_post_aha': rate_post_aha
                })
                autocorr_late_baseline = lag1_autocorr(aim_late_baseline)
                autocorr_pre_aha = lag1_autocorr(aim_pre_rot)
                autocorr_post_aha = lag1_autocorr(aim_post_rot)
                ac_points.append({
                    'dataset': datasetName,
                    'rotation': abs(rotation_size),
                    'participant': part,
                    'n_pre_aha': n_pre_aha,
                    'autocorr_late_baseline': autocorr_late_baseline,
                    'autocorr_pre_aha': autocorr_pre_aha,
                    'autocorr_post_aha': autocorr_post_aha
                })
    df_var = pd.DataFrame(var_points).sort_values(['dataset', 'rotation', 'participant']).reset_index(drop=True)
    df_reg = pd.DataFrame(reg_points).sort_values(['dataset', 'rotation', 'participant']).reset_index(drop=True)
    df_ac = pd.DataFrame(ac_points).sort_values(['dataset', 'rotation', 'participant']).reset_index(drop=True)
 
    def get_stars(p, corrected=False, corr_factor=1):
        if np.isnan(p):
            return ''
        if corrected:
            corrected_p = min(p * corr_factor, 1.0)
        else:
            corrected_p = p
        if corrected_p < 0.001:
            return '***'
        elif corrected_p < 0.01:
            return '**'
        elif corrected_p < 0.05:
            return '*'
        return ''
     
    print("Descriptive statistics are reported as mean [95% CI lower, upper].")
    print("Significance stars: paired comparisons and one-sample tests (per-group Bonferroni-corrected across multiple tests, x5 for non-Brudner datasets, x1 for Brudner; pooled uncorrected): * p<0.05, ** p<0.01, *** p<0.001")
    print("\n=== Analysis (i): Trial-to-trial variability (std of aims) in matched windows ===")
    if not df_var.empty:
        for (ds, rot), group in df_var.groupby(['dataset', 'rotation']):
            n = len(group)
            corr_factor = 1 if ds == 'Brudner' else 5
            base_str = get_mean_ci_str(group['std_late_baseline'])
            pre_str = get_mean_ci_str(group['std_pre_aha'])
            post_str = get_mean_ci_str(group['std_post_aha'])
            pairs = [
                ('Late baseline vs Pre-Aha!', group.dropna(subset=['std_late_baseline', 'std_pre_aha']), 'std_late_baseline', 'std_pre_aha'),
                ('Late baseline vs Post-Aha!', group.dropna(subset=['std_late_baseline', 'std_post_aha']), 'std_late_baseline', 'std_post_aha'),
                ('Pre-Aha! vs Post-Aha!', group.dropna(subset=['std_pre_aha', 'std_post_aha']), 'std_pre_aha', 'std_post_aha')
            ]
            paired_strs = []
            for label, paired_df, col1, col2 in pairs:
                if len(paired_df) > 3:
                    t, raw_p = stats.ttest_rel(paired_df[col2], paired_df[col1])
                    corrected_p = min(raw_p * corr_factor, 1.0)
                    stars = get_stars(raw_p, corrected=True, corr_factor=corr_factor)
                    paired_strs.append(f"{label}: t({len(paired_df)-1})={t:.2f}, p={corrected_p:.3f}{stars}")
                else:
                    paired_strs.append(f"{label}: insufficient data")
            print(f"{ds} {rot}° (n={n}): late baseline = {base_str}, pre-Aha! = {pre_str}, post-Aha! = {post_str}")
            print(" " + "; ".join(paired_strs))
            summary_var.append({
                'Dataset': ds,
                'Rotation (°)': f"{rot:.0f}",
                'N': n,
                'Late baseline (mean [95% CI])': base_str,
                'Pre-Aha! (mean [95% CI])': pre_str,
                'Post-Aha! (mean [95% CI])': post_str,
                'Comparisons': "; ".join(paired_strs)
            })
    print("\n=== Analysis (ii): Error correction rate in matched windows ===")
    if not df_reg.empty:
        for (ds, rot), group in df_reg.groupby(['dataset', 'rotation']):
            n = len(group)
            corr_factor = 1 if ds == 'Brudner' else 5
            base_str = get_mean_ci_str(group['rate_late_baseline'])
            pre_str = get_mean_ci_str(group['rate_pre_aha'])
            post_str = get_mean_ci_str(group['rate_post_aha'])
            onesample_strs = []
            for label, col in [('Late baseline', 'rate_late_baseline'), ('Pre-Aha!', 'rate_pre_aha'), ('Post-Aha!', 'rate_post_aha')]:
                valid = group[col].dropna()
                if len(valid) > 3:
                    t, raw_p = stats.ttest_1samp(valid, 0).statistic, stats.ttest_1samp(valid, 0).pvalue
                    corrected_p = min(raw_p * corr_factor, 1.0)
                    stars = get_stars(raw_p, corrected=True, corr_factor=corr_factor)
                    onesample_strs.append(f"{label} vs 0: t({len(valid)-1})={t:.2f}, p={corrected_p:.3f}{stars}")
                else:
                    onesample_strs.append(f"{label} vs 0: insufficient data")
            pairs = [
                ('Late baseline vs Pre-Aha!', group.dropna(subset=['rate_late_baseline', 'rate_pre_aha']), 'rate_late_baseline', 'rate_pre_aha'),
                ('Late baseline vs Post-Aha!', group.dropna(subset=['rate_late_baseline', 'rate_post_aha']), 'rate_late_baseline', 'rate_post_aha'),
                ('Pre-Aha! vs Post-Aha!', group.dropna(subset=['rate_pre_aha', 'rate_post_aha']), 'rate_pre_aha', 'rate_post_aha')
            ]
            paired_strs = []
            for label, paired_df, col1, col2 in pairs:
                if len(paired_df) > 3:
                    t, raw_p = stats.ttest_rel(paired_df[col2], paired_df[col1])
                    corrected_p = min(raw_p * corr_factor, 1.0)
                    stars = get_stars(raw_p, corrected=True, corr_factor=corr_factor)
                    paired_strs.append(f"{label}: t({len(paired_df)-1})={t:.2f}, p={corrected_p:.3f}{stars}")
                else:
                    paired_strs.append(f"{label}: insufficient data")
            print(f"{ds} {rot}° (n={n}): late baseline = {base_str}, pre-Aha! = {pre_str}, post-Aha! = {post_str}")
            print(" One-sample: " + "; ".join(onesample_strs))
            print(" Paired: " + "; ".join(paired_strs))
            summary_reg.append({
                'Dataset': ds,
                'Rotation (°)': f"{rot:.0f}",
                'N': n,
                'Late baseline (mean [95% CI])': base_str,
                'Pre-Aha! (mean [95% CI])': pre_str,
                'Post-Aha! (mean [95% CI])': post_str,
                'One-sample tests': "; ".join(onesample_strs),
                'Paired comparisons': "; ".join(paired_strs)
            })
    print("\n=== Analysis (iii): Lag-1 autocorrelation of signed aim in matched windows ===")
    if not df_ac.empty:
        for (ds, rot), group in df_ac.groupby(['dataset', 'rotation']):
            n = len(group)
            corr_factor = 1 if ds == 'Brudner' else 5
            base_str = get_mean_ci_str(group['autocorr_late_baseline'])
            pre_str = get_mean_ci_str(group['autocorr_pre_aha'])
            post_str = get_mean_ci_str(group['autocorr_post_aha'])
            onesample_strs = []
            for label, col in [('Late baseline', 'autocorr_late_baseline'), ('Pre-Aha!', 'autocorr_pre_aha'), ('Post-Aha!', 'autocorr_post_aha')]:
                valid = group[col].dropna()
                if len(valid) > 3:
                    t, raw_p = stats.ttest_1samp(valid, 0).statistic, stats.ttest_1samp(valid, 0).pvalue
                    corrected_p = min(raw_p * corr_factor, 1.0)
                    stars = get_stars(raw_p, corrected=True, corr_factor=corr_factor)
                    onesample_strs.append(f"{label} vs 0: t({len(valid)-1})={t:.2f}, p={corrected_p:.3f}{stars}")
                else:
                    onesample_strs.append(f"{label} vs 0: insufficient data")
            pairs = [
                ('Late baseline vs Pre-Aha!', group.dropna(subset=['autocorr_late_baseline', 'autocorr_pre_aha']), 'autocorr_late_baseline', 'autocorr_pre_aha'),
                ('Late baseline vs Post-Aha!', group.dropna(subset=['autocorr_late_baseline', 'autocorr_post_aha']), 'autocorr_late_baseline', 'autocorr_post_aha'),
                ('Pre-Aha! vs Post-Aha!', group.dropna(subset=['autocorr_pre_aha', 'autocorr_post_aha']), 'autocorr_pre_aha', 'autocorr_post_aha')
            ]
            paired_strs = []
            for label, paired_df, col1, col2 in pairs:
                if len(paired_df) > 3:
                    t, raw_p = stats.ttest_rel(paired_df[col2], paired_df[col1])
                    corrected_p = min(raw_p * corr_factor, 1.0)
                    stars = get_stars(raw_p, corrected=True, corr_factor=corr_factor)
                    paired_strs.append(f"{label}: t({len(paired_df)-1})={t:.2f}, p={corrected_p:.3f}{stars}")
                else:
                    paired_strs.append(f"{label}: insufficient data")
            print(f"{ds} {rot}° (n={n}): late baseline = {base_str}, pre-Aha! = {pre_str}, post-Aha! = {post_str}")
            print(" One-sample: " + "; ".join(onesample_strs))
            print(" Paired: " + "; ".join(paired_strs))
            summary_ac.append({
                'Dataset': ds,
                'Rotation (°)': f"{rot:.0f}",
                'N': n,
                'Late baseline (mean [95% CI])': base_str,
                'Pre-Aha! (mean [95% CI])': pre_str,
                'Post-Aha! (mean [95% CI])': post_str,
                'One-sample tests': "; ".join(onesample_strs),
                'Paired comparisons': "; ".join(paired_strs)
            })
    print("\n=== Manuscript-ready summary tables (per dataset/rotation) ===")
    if summary_var:
        df_summary_var = pd.DataFrame(summary_var).sort_values(['Dataset', 'Rotation (°)']).reset_index(drop=True)
        print("\nAnalysis (i): Trial-to-trial variability (matched windows)")
        print(df_summary_var.to_string(index=False))
    if summary_reg:
        df_summary_reg = pd.DataFrame(summary_reg).sort_values(['Dataset', 'Rotation (°)']).reset_index(drop=True)
        print("\nAnalysis (ii): Error correction rate (matched windows)")
        print(df_summary_reg.to_string(index=False))
    if summary_ac:
        df_summary_ac = pd.DataFrame(summary_ac).sort_values(['Dataset', 'Rotation (°)']).reset_index(drop=True)
        print("\nAnalysis (iii): Lag-1 autocorrelation of signed aim (matched windows)")
        print(df_summary_ac.to_string(index=False))
    if 'targetFolder' in globals() and targetFolder:
        os.makedirs(targetFolder, exist_ok=True)
        sns.set(style="whitegrid", context="paper", font_scale=1.2, palette="muted")
        order = ['Late Baseline', 'Pre-Aha! Rotation', 'Post-Aha! Rotation']
        datasets_present = sorted(df_var['dataset'].unique())
        for ds in datasets_present:
            df_var_ds = df_var[df_var['dataset'] == ds].copy()
            df_reg_ds = df_reg[df_reg['dataset'] == ds].copy()
            df_ac_ds = df_ac[df_ac['dataset'] == ds].copy()
            if df_var_ds.empty:
                continue
            rotations = sorted(df_var_ds['rotation'].unique())
            if not rotations:
                continue
            group_order = [f"{r:.0f}°" for r in rotations]
            n_rots = len(rotations)
            width = max(8, 4 + n_rots * 2)
            corr_factor = 1 if ds == 'Brudner' else 5
                              
            if not df_var_ds.empty:
                df_long = pd.melt(df_var_ds, id_vars=['rotation'], value_vars=['std_late_baseline', 'std_pre_aha', 'std_post_aha'],
                                  var_name='phase', value_name='variability')
                df_long['phase'] = df_long['phase'].map({
                    'std_late_baseline': 'Late Baseline',
                    'std_pre_aha': 'Pre-Aha! Rotation',
                    'std_post_aha': 'Post-Aha! Rotation'
                })
                df_long['group'] = df_long['rotation'].apply(lambda x: f"{x:.0f}°")
                fig, ax = plt.subplots(figsize=(width/3, 2))
                sns.barplot(data=df_long, x='group', y='variability', hue='phase', order=group_order, hue_order=order,
                            errorbar=None, ax=ax)
                sns.stripplot(data=df_long, x='group', y='variability', hue='phase', order=group_order, hue_order=order,
                              dodge=True, jitter=0.33, size=3, alpha=0.7, edgecolor='k', linewidth=0.5, legend=False, ax=ax)
                ax.set_title(f"Trial-to-trial variability (matched windows) — {ds}")
                ax.set_ylabel("Standard deviation of aiming directions (°)")
                ax.set_xlabel("Rotation size")
                                                            
                bar_containers = [c for c in ax.containers if isinstance(c, BarContainer)]
                if len(bar_containers) == 3 and all(len(c) == n_rots for c in bar_containers):
                    heights = np.array([[rect.get_height() for rect in cont] for cont in bar_containers])              
                    centers = np.array([[rect.get_x() + rect.get_width()/2 for rect in cont] for cont in bar_containers])
                                           
                    col_map = {
                        'Late Baseline': 'std_late_baseline',
                        'Pre-Aha! Rotation': 'std_pre_aha',
                        'Post-Aha! Rotation': 'std_post_aha'
                    }
                    for rot_i, rot in enumerate(rotations):
                        group_df = df_var_ds[df_var_ds['rotation'] == rot]
                        for phase_i, phase in enumerate(order):
                            col = col_map[phase]
                            data = group_df[col].dropna()
                            if len(data) < 2:
                                continue
                            mean, low, up = compute_ci95(data)
                            if np.isnan(low) or np.isnan(up):
                                continue
                            x = centers[phase_i, rot_i]
                            y = heights[phase_i, rot_i]
                            lower_err = y - low
                            upper_err = up - y
                            ax.errorbar(x, y, yerr=[[lower_err], [upper_err]],
                                        fmt='none', capsize=5, elinewidth=1.5, capthick=1.5, color='k')
                    max_h = np.nanmax(heights)
                    if np.isnan(max_h):
                        max_h = 0
                                                                       
                    line_ys = [max_h + 20, max_h + 45, max_h + 70]
                    tick_length = 10
                    pair_info = [
                        (0, 1, line_ys[0], 'std_late_baseline', 'std_pre_aha'),
                        (1, 2, line_ys[1], 'std_pre_aha', 'std_post_aha'),
                        (0, 2, line_ys[2], 'std_late_baseline', 'std_post_aha')
                    ]
                    for rot_i, rot in enumerate(rotations):
                        group_df = df_var_ds[df_var_ds['rotation'] == rot]
                        for i1, i2, line_y, col1, col2 in pair_info:
                            paired_df = group_df.dropna(subset=[col1, col2])
                            if len(paired_df) <= 3:
                                continue
                            raw_p = stats.ttest_rel(paired_df[col2], paired_df[col1]).pvalue
                            stars = get_stars(raw_p, corrected=True, corr_factor=corr_factor)
                            text = stars if stars else 'n.s.'
                            x1 = centers[i1, rot_i]
                            x2 = centers[i2, rot_i]
                            ax.plot([x1, x2], [line_y, line_y], 'k-', lw=1.2)
                            ax.plot([x1, x1], [line_y, line_y - tick_length], 'k-', lw=1.2)
                            ax.plot([x2, x2], [line_y, line_y - tick_length], 'k-', lw=1.2)
                            ax.text((x1 + x2)/2, line_y + 8, text, ha='center', va='bottom', fontsize=10, fontweight='bold')
                    ax.set_ylim(0, 200)
                    ax.yaxis.set_major_locator(MultipleLocator(50))
                    ax.yaxis.set_minor_locator(AutoMinorLocator(3))
                plt.tight_layout()
                plt.savefig(os.path.join(targetFolder, f'aha_variability_{ds}.svg'), dpi=300)
                plt.close()
                                        
            if not df_reg_ds.empty:
                df_long = pd.melt(df_reg_ds, id_vars=['rotation'], value_vars=['rate_late_baseline', 'rate_pre_aha', 'rate_post_aha'],
                                  var_name='phase', value_name='rate')
                df_long['phase'] = df_long['phase'].map({
                    'rate_late_baseline': 'Late Baseline',
                    'rate_pre_aha': 'Pre-Aha! Rotation',
                    'rate_post_aha': 'Post-Aha! Rotation'
                })
                df_long['group'] = df_long['rotation'].apply(lambda x: f"{x:.0f}°")
                fig, ax = plt.subplots(figsize=(width/3, 2))
                sns.barplot(data=df_long, x='group', y='rate', hue='phase', order=group_order, hue_order=order,
                            errorbar=None, ax=ax)
                sns.stripplot(data=df_long, x='group', y='rate', hue='phase', order=group_order, hue_order=order,
                              dodge=True, jitter=0.33, size=3, alpha=0.7, edgecolor='k', linewidth=0.5, legend=False, ax=ax)
                ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
                ax.set_title(f"Error correction rate (matched windows) — {ds}")
                ax.set_ylabel("Correction rate")
                ax.set_xlabel("Rotation size")
                                                            
                bar_containers = [c for c in ax.containers if isinstance(c, BarContainer)]
                if len(bar_containers) == 3 and all(len(c) == n_rots for c in bar_containers):
                    heights = np.array([[rect.get_height() for rect in cont] for cont in bar_containers])
                    centers = np.array([[rect.get_x() + rect.get_width()/2 for rect in cont] for cont in bar_containers])
                                           
                    col_map = {
                        'Late Baseline': 'rate_late_baseline',
                        'Pre-Aha! Rotation': 'rate_pre_aha',
                        'Post-Aha! Rotation': 'rate_post_aha'
                    }
                    for rot_i, rot in enumerate(rotations):
                        group_df = df_reg_ds[df_reg_ds['rotation'] == rot]
                        for phase_i, phase in enumerate(order):
                            col = col_map[phase]
                            data = group_df[col].dropna()
                            if len(data) < 2:
                                continue
                            mean, low, up = compute_ci95(data)
                            if np.isnan(low) or np.isnan(up):
                                continue
                            x = centers[phase_i, rot_i]
                            y = heights[phase_i, rot_i]
                            lower_err = y - low
                            upper_err = up - y
                            ax.errorbar(x, y, yerr=[[lower_err], [upper_err]],
                                        fmt='none', capsize=5, elinewidth=1.5, capthick=1.5, color='k')
                    max_h = np.nanmax(heights)
                    min_h = np.nanmin(heights)
                    y_range = max_h - min_h + 1e-6
                    offset = y_range * 0.08
                    cols = ['rate_late_baseline', 'rate_pre_aha', 'rate_post_aha']
                    for rot_i, rot in enumerate(rotations):
                        group_df = df_reg_ds[df_reg_ds['rotation'] == rot]
                        for j in range(3):
                            valid = group_df[cols[j]].dropna()
                            if len(valid) < 2:
                                continue
                            raw_p = stats.ttest_1samp(valid, 0).pvalue
                            stars = get_stars(raw_p, corrected=True, corr_factor=corr_factor)
                            text = stars if stars else 'n.s.'
                            h = heights[j, rot_i]
                            if np.isnan(h):
                                continue
                            if h >= 0:
                                y = h + offset
                                va = 'bottom'
                            else:
                                y = h - offset
                                va = 'top'
                            ax.text(centers[j, rot_i], y, text, ha='center', va=va, fontsize=10, fontweight='bold')
                    margin = y_range * 0.3                                                 
                    ax.set_ylim(-3.2,3.82)
                    ax.yaxis.set_major_locator(MultipleLocator(2))
                    ax.yaxis.set_minor_locator(AutoMinorLocator(3))
                plt.tight_layout()
                plt.savefig(os.path.join(targetFolder, f'aha_rate_{ds}.svg'), dpi=300)
                plt.close()
                                  
            if not df_ac_ds.empty:
                df_long = pd.melt(df_ac_ds, id_vars=['rotation'], value_vars=['autocorr_late_baseline', 'autocorr_pre_aha', 'autocorr_post_aha'],
                                  var_name='phase', value_name='autocorr')
                df_long['phase'] = df_long['phase'].map({
                    'autocorr_late_baseline': 'Late Baseline',
                    'autocorr_pre_aha': 'Pre-Aha! Rotation',
                    'autocorr_post_aha': 'Post-Aha! Rotation'
                })
                df_long['group'] = df_long['rotation'].apply(lambda x: f"{x:.0f}°")
                fig, ax = plt.subplots(figsize=(width/3, 2))
                sns.barplot(data=df_long, x='group', y='autocorr', hue='phase', order=group_order, hue_order=order,
                            errorbar=None, ax=ax)
                sns.stripplot(data=df_long, x='group', y='autocorr', hue='phase', order=group_order, hue_order=order,
                              dodge=True, jitter=0.33, size=3, alpha=0.7, edgecolor='k', linewidth=0.5, legend=False, ax=ax)
                ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
                ax.set_title(f"Lag-1 autocorrelation of signed aim (matched windows) — {ds}")
                ax.set_ylabel("Lag-1 autocorrelation")
                ax.set_xlabel("Rotation size")
                                                            
                bar_containers = [c for c in ax.containers if isinstance(c, BarContainer)]
                if len(bar_containers) == 3 and all(len(c) == n_rots for c in bar_containers):
                    heights = np.array([[rect.get_height() for rect in cont] for cont in bar_containers])
                    centers = np.array([[rect.get_x() + rect.get_width()/2 for rect in cont] for cont in bar_containers])
                                           
                    col_map = {
                        'Late Baseline': 'autocorr_late_baseline',
                        'Pre-Aha! Rotation': 'autocorr_pre_aha',
                        'Post-Aha! Rotation': 'autocorr_post_aha'
                    }
                    for rot_i, rot in enumerate(rotations):
                        group_df = df_ac_ds[df_ac_ds['rotation'] == rot]
                        for phase_i, phase in enumerate(order):
                            col = col_map[phase]
                            data = group_df[col].dropna()
                            if len(data) < 2:
                                continue
                            mean, low, up = compute_ci95(data)
                            if np.isnan(low) or np.isnan(up):
                                continue
                            x = centers[phase_i, rot_i]
                            y = heights[phase_i, rot_i]
                            lower_err = y - low
                            upper_err = up - y
                            ax.errorbar(x, y, yerr=[[lower_err], [upper_err]],
                                        fmt='none', capsize=5, elinewidth=1.5, capthick=1.5, color='k')
                    max_h = np.nanmax(heights)
                    min_h = np.nanmin(heights)
                    y_range = max_h - min_h + 1e-6
                    offset = y_range * 0.08
                    cols = ['autocorr_late_baseline', 'autocorr_pre_aha', 'autocorr_post_aha']
                    for rot_i, rot in enumerate(rotations):
                        group_df = df_ac_ds[df_ac_ds['rotation'] == rot]
                        for j in range(3):
                            valid = group_df[cols[j]].dropna()
                            if len(valid) < 2:
                                continue
                            raw_p = stats.ttest_1samp(valid, 0).pvalue
                            stars = get_stars(raw_p, corrected=True, corr_factor=corr_factor)
                            if not stars:
                                continue
                            h = heights[j, rot_i]
                            if np.isnan(h):
                                continue
                            if h >= 0:
                                y = h + offset
                                va = 'bottom'
                            else:
                                y = h - offset
                                va = 'top'
                            ax.text(centers[j, rot_i], y, stars, ha='center', va=va, fontsize=10, fontweight='bold')
                    margin = y_range * 0.3                                                 
                    ax.set_ylim(min_h - margin, max_h + margin * 1.5)
                plt.tight_layout()
                plt.savefig(os.path.join(targetFolder, f'aha_autocorr_{ds}.svg'), dpi=300)
                plt.close()
        print("\nSaving participant-level DataFrames and summary tables to CSV in targetFolder...")
        df_var.to_csv(os.path.join(targetFolder, 'aha_participant_var.csv'), index=False, float_format='%.3f')
        df_reg.to_csv(os.path.join(targetFolder, 'aha_participant_reg.csv'), index=False, float_format='%.3f')
        df_ac.to_csv(os.path.join(targetFolder, 'aha_participant_ac.csv'), index=False, float_format='%.3f')
        if summary_var:
            pd.DataFrame(summary_var).to_csv(os.path.join(targetFolder, 'aha_summary_var.csv'), index=False)
        if summary_reg:
            pd.DataFrame(summary_reg).to_csv(os.path.join(targetFolder, 'aha_summary_reg.csv'), index=False)
        if summary_ac:
            pd.DataFrame(summary_ac).to_csv(os.path.join(targetFolder, 'aha_summary_ac.csv'), index=False)
    return df_var, df_reg, df_ac
df_var, df_reg, df_ac = analyze_pre_aha_behavior(explicit_only=True, min_pre_trials=2)

In [ ]:
baselineLength = 40
washoutLength = 40

from matplotlib.ticker import MultipleLocator, AutoMinorLocator

def plot_human_aim_hist_aha_by_rot(ylim=None, save_filename=None, explicit_only=False, bin_width=5, max_bin=150, font_scale=1.0):
    """
    Updated version (fixed panel sizing):
    - All datasets now produce figures of identical overall size.
    - Individual Pre-Aha!/Aha! panel pairs (per rotation magnitude) are now identical physical size across ALL datasets.
    - Datasets with fewer rotation magnitudes have their panels placed at the top (higher magnitudes first), with empty/white space at the bottom.
    - Other features unchanged (independent y-scales, N labels, large rotation titles, per-dataset legends, separate saves, etc.).
    - Stats:
      - Pre-Aha! (t-1): Paired one-sample t-test (pre > individual baseline mean |aim|), one-sided greater.
      - Aha! (t): One-sample t-test vs. ideal magnitude (|rotation|), two-sided.
      - Bonferroni-corrected p-values (x5 for all datasets except x1 for Brudner).
      - Printed test results + significance stars (* p_corr<0.05, ** p_corr<0.01, *** p_corr<0.001) in top-left of significant panels.
    """
                   
    base_size =10 * font_scale * 3
    title_size = 10 * font_scale * 2
    label_size = 10 * font_scale * 3
    tick_size = 10 * font_scale * 3
    legend_size = 8 * font_scale
    plt.rcParams.update({'font.size': base_size})
    plt.rcParams.update({'axes.titlesize': title_size})
    plt.rcParams.update({'axes.labelsize': label_size})
    plt.rcParams.update({'xtick.labelsize': tick_size})
    plt.rcParams.update({'ytick.labelsize': tick_size})
    plt.rcParams.update({'legend.fontsize': legend_size})
                            
    color_map = {90: '#44AA99', 60: '#88CCEE', 45: '#FF9825', 30: '#CC6677', 15: '#AA4499'}
                                                                           
    aims_by_dataset_rot = {}
    datasets = [
        [ssmsAll8CG, qLearnsAll8CG, hmmsAll8CG],
        [ssmsAll8CGIMP, qLearnsAll8CGIMP, hmmsAll8CGIMP],
        [ssmsAllBrudner, qLearnsAllBrudner, hmmsAllBrudner],
        [ssmsAll8BT, qLearnsAll8BT, hmmsAll8BT]
    ]
    datasetLabels = ['CGVanilla8', 'CGImp8', 'Brudner', 'BondTaylor']
    for dsIdx, datasetName in enumerate(datasetLabels):
        aims_by_dataset_rot[datasetName] = {}
        ssm_list = datasets[dsIdx][0]
        hmm_list = datasets[dsIdx][2]
        rotations = []
        if datasetName == 'Brudner':
            bl = 64
            washoutLength_ds = 64
        else:
            bl = baselineLength
            washoutLength_ds = washoutLength
        for ssm in ssm_list:
            if hasattr(ssm, 'rotations'):
                rot_array = np.asarray(ssm.rotations)
                rotations.append(rot_array)
            else:
                rotations.append(None)
        for rotIdx, rot_array in enumerate(rotations):
            if rot_array is None or rot_array.size == 0 or rotIdx >= len(hmm_list) or rotIdx >= len(ssm_list):
                continue
            non_zero_rots = rot_array[rot_array != 0]
            if len(non_zero_rots) == 0:
                continue
            mags = np.abs(non_zero_rots)
            unique_mags = np.unique(mags)
            if len(unique_mags) != 1:
                print(f"Warning: Multiple magnitudes {unique_mags} in {datasetName} participant {rotIdx}, skipping.")
                continue
            rotation_magnitude = float(unique_mags[0])
            ssm_fit = ssm_list[rotIdx]
            hmm_fit = hmm_list[rotIdx]
            if not hasattr(hmm_fit, 'human_explicits'):
                continue
            human_explicits_full = np.array(hmm_fit.human_explicits)
            if explicit_only:
                human_abs_full = np.abs(human_explicits_full)
            else:
                if hasattr(hmm_fit, 'human_implicits'):
                    human_implicits_full = np.array(hmm_fit.human_implicits)
                    human_full = human_explicits_full + human_implicits_full
                    human_abs_full = np.abs(human_full)
                else:
                    human_abs_full = np.abs(human_explicits_full)
            n_parts, n_trials = human_abs_full.shape
            min_t = bl + 1
            max_t = n_trials - (washoutLength_ds + min(40, bl))
            if min_t > max_t:
                continue
            if rotation_magnitude not in aims_by_dataset_rot[datasetName]:
                aims_by_dataset_rot[datasetName][rotation_magnitude] = {'tm1': [], 't0': [], 'baseline_means': []}
            for part in range(n_parts):
                aim = human_abs_full[part, :]
                if rot_array.ndim == 1:
                    part_rot_array = rot_array
                else:
                    part_rot_array = rot_array[part, :]
                min_cost = np.inf
                best_t_part = None
                for t in range(min_t, max_t + 1):
                    if t >= n_trials:
                        continue
                    idx = min(t, len(part_rot_array) - 1)
                    rot_val = part_rot_array[idx]
                    if np.isnan(rot_val):
                        continue
                    target_mag = np.abs(rot_val)
                    pre_start = max(0, t - min(40, bl))
                    cost = np.nansum(np.maximum(np.abs(human_abs_full[part, bl:t]) - np.nanmean(human_abs_full[part, :bl]), 0)) + (20) * (np.nanmean(np.abs(np.abs(rot_val) - human_abs_full[part, t:min(max_t, t + min(40, bl))])))
                    if cost < min_cost:
                        min_cost = cost
                        best_t_part = t
                if best_t_part is None:
                    continue
                t_minus1 = best_t_part - 1
                t0 = best_t_part
                if (0 <= t_minus1 < n_trials and 0 <= t0 < n_trials and
                    not np.isnan(aim[t_minus1]) and not np.isnan(aim[t0])):
                    baseline_mean = np.nanmean(human_abs_full[part, :bl])
                    aims_by_dataset_rot[datasetName][rotation_magnitude]['tm1'].append(aim[t_minus1])
                    aims_by_dataset_rot[datasetName][rotation_magnitude]['t0'].append(aim[t0])
                    aims_by_dataset_rot[datasetName][rotation_magnitude]['baseline_means'].append(baseline_mean)
                     
    active_datasets = [name for name in datasetLabels if name in aims_by_dataset_rot and aims_by_dataset_rot[name]]
    if not active_datasets:
        print("No valid Aha! data collected.")
        return pd.DataFrame()
          
    bins = np.arange(0, max_bin + bin_width, bin_width)
                                 
    summary_data = []
    for datasetName in active_datasets:
        dataset_dict = aims_by_dataset_rot[datasetName]
        for mag in dataset_dict:
            aims_tm1 = np.array(dataset_dict[mag]['tm1'])
            aims_t0 = np.array(dataset_dict[mag]['t0'])
            hist_tm1, _ = np.histogram(aims_tm1, bins=bins)
            hist_t0, _ = np.histogram(aims_t0, bins=bins)
            for i in range(len(bins) - 1):
                summary_data.append({
                    'dataset': datasetName,
                    'rotation_magnitude': mag,
                    'bin_start': bins[i],
                    'bin_end': bins[i + 1],
                    'count_t-1': hist_tm1[i],
                    'count_t': hist_t0[i]
                })
              
    for datasetName in active_datasets:
        print(f"\n{datasetName}:")
        for mag in sorted(aims_by_dataset_rot[datasetName], reverse=True):
            n_tm1 = len(aims_by_dataset_rot[datasetName][mag]['tm1'])
            n_t0 = len(aims_by_dataset_rot[datasetName][mag]['t0'])
            print(f" {int(mag)}° magnitude: t-1 N = {n_tm1}, t N = {n_t0}")
                                                      
    def get_significance_stars(p_corr):
        if np.isnan(p_corr):
            return ''
        if p_corr < 0.001:
            return '***'
        elif p_corr < 0.01:
            return '**'
        elif p_corr < 0.05:
            return '*'
        else:
            return ''
    for datasetName in active_datasets:
        dataset_dict = aims_by_dataset_rot[datasetName]
        its_mags = sorted([mag for mag in dataset_dict if len(dataset_dict[mag]['tm1']) > 0 or len(dataset_dict[mag]['t0']) > 0], reverse=True)
        correction_factor = 1 if datasetName == 'Brudner' else 5
        print(f"\nStatistical tests for {datasetName} (Bonferroni correction factor: x{correction_factor}):")
        for mag in its_mags:
            data = dataset_dict[mag]
            aims_tm1 = np.array(data['tm1'])
            aims_t0 = np.array(data['t0'])
            baseline_means = np.array(data.get('baseline_means', []))
            n_tm1 = len(aims_tm1)
            n_t0 = len(aims_t0)
            mean_tm1 = np.nanmean(aims_tm1)
            mean_t0 = np.nanmean(aims_t0)
            mean_baseline = np.nanmean(baseline_means) if len(baseline_means) > 0 else np.nan
                                              
            if n_tm1 >= 2 and len(baseline_means) == n_tm1:
                stat_pre, p_pre = ttest_rel(aims_tm1, baseline_means, alternative='greater', nan_policy='omit')
                p_pre_corr = min(p_pre * correction_factor, 1.0)
            else:
                stat_pre = p_pre = p_pre_corr = np.nan
                                          
            if n_t0 >= 2:
                stat_aha, p_aha = ttest_1samp(aims_t0, popmean=mag, nan_policy='omit')
                p_aha_corr = min(p_aha * correction_factor, 1.0)
            else:
                stat_aha = p_aha = p_aha_corr = np.nan
                                       
            mag_int = int(mag)
            mean_tm1_str = f"{mean_tm1:.2f}" if np.isfinite(mean_tm1) else "NaN"
            mean_baseline_str = f"{mean_baseline:.2f}" if np.isfinite(mean_baseline) else "NaN"
            mean_t0_str = f"{mean_t0:.2f}" if np.isfinite(mean_t0) else "NaN"
            t_pre_str = f"{stat_pre:.2f}" if np.isfinite(stat_pre) else "NaN"
            p_pre_str = f"{p_pre:.5f}" if np.isfinite(p_pre) else "NaN"
            p_pre_corr_str = f"{p_pre_corr:.5f}" if np.isfinite(p_pre_corr) else "NaN"
            t_aha_str = f"{stat_aha:.2f}" if np.isfinite(stat_aha) else "NaN"
            p_aha_str = f"{p_aha:.5f}" if np.isfinite(p_aha) else "NaN"
            p_aha_corr_str = f"{p_aha_corr:.5f}" if np.isfinite(p_aha_corr) else "NaN"
            print(f" {mag_int}° rotation magnitude:")
            print(f" Pre-Aha! (t-1): mean |aim| = {mean_tm1_str}°, mean baseline |aim| = {mean_baseline_str}°, N = {n_tm1}, paired one-sided t-test (pre > baseline): t = {t_pre_str}, p = {p_pre_str}, p_corr = {p_pre_corr_str}")
            print(f" Aha! (t): mean |aim| = {mean_t0_str}°, N = {n_t0}, two-sided t-test vs ideal {mag_int}°: t = {t_aha_str}, p = {p_aha_str}, p_corr = {p_aha_corr_str}")
            data['sig_pre'] = get_significance_stars(p_pre_corr)
            data['sig_aha'] = get_significance_stars(p_aha_corr)
                                                                            
    row_height = 1.8
    max_n_mags = 0
    for datasetName in active_datasets:
        dataset_dict = aims_by_dataset_rot[datasetName]
        its_mags_temp = [mag for mag in dataset_dict if len(dataset_dict[mag]['tm1']) > 0 or len(dataset_dict[mag]['t0']) > 0]
        max_n_mags = max(max_n_mags, len(its_mags_temp))
    if max_n_mags == 0:
        return pd.DataFrame()
    fixed_fig_height = max_n_mags * row_height + 6                                             
                                          
    for datasetName in active_datasets:
        dataset_dict = aims_by_dataset_rot[datasetName]
        its_mags = sorted([mag for mag in dataset_dict.keys()
                           if len(dataset_dict[mag]['tm1']) > 0 or len(dataset_dict[mag]['t0']) > 0], reverse=True)
        n_mags_ds = len(its_mags)
        if n_mags_ds == 0:
            continue
        fig = plt.figure(figsize=(18, fixed_fig_height))
        gs = fig.add_gridspec(max_n_mags, 2, height_ratios=[1]*max_n_mags,
                              width_ratios=[1.2, 1.2], wspace=0.3, hspace=0.6)
        pre_axs = [fig.add_subplot(gs[j, 0]) for j in range(max_n_mags)]
        aha_axs = [fig.add_subplot(gs[j, 1]) for j in range(max_n_mags)]
                                                        
        for j in range(n_mags_ds, max_n_mags):
            pre_axs[j].axis('off')
            aha_axs[j].axis('off')
            pre_axs[j].patch.set_visible(False)
            aha_axs[j].patch.set_visible(False)
                        
        fig.text(0.33, 0.98, 'Pre-Aha! (t-1)', ha='center', va='top', fontsize=15 * font_scale, fontweight='bold')
        fig.text(0.74, 0.98, 'Aha! (t)', ha='center', va='top', fontsize=15 * font_scale, fontweight='bold')
                                                              
        legend_handles = []
        legend_labels = []
        for mag in its_mags:
            color = color_map.get(int(mag), 'gray')
            legend_handles.append(plt.Rectangle((0, 0), 1, 1, facecolor=color, alpha=0.7, edgecolor='black'))
            legend_labels.append(f'{int(mag)}°')
        if legend_handles:
            fig.legend(legend_handles, legend_labels, loc='upper center', ncol=len(its_mags),
                       bbox_to_anchor=(0.5, 0.94), title='Rotation Magnitude')
                  
        fig.suptitle(f'{datasetName}\nHistograms of |Aim| Around Detected Aha! Trial\n(Faceted by Rotation Magnitude · Independent y-scaling per panel)',
                     fontsize=15 * font_scale, y=0.96)
                                           
        ticks = np.arange(0, max_bin + 1, 30)
        tick_labels = [str(int(tick)) if i % 2 == 0 else '' for i, tick in enumerate(ticks)]
        for j, mag in enumerate(its_mags):
            pre_ax = pre_axs[j]
            aha_ax = aha_axs[j]
            color = color_map.get(int(mag), 'gray')
            pre_ax.set_xticks(ticks)
            pre_ax.set_xticklabels(tick_labels)
            aha_ax.set_xticks(ticks)
            aha_ax.set_xticklabels(tick_labels)
                                    
            pre_ax.set_title(f'{int(mag)}°', fontsize=title_size * 1.5, pad=10, color=color)
            aha_ax.set_title(f'{int(mag)}°', fontsize=title_size * 1.5, pad=10, color=color)
                                                  
            if False:                   
                pre_ax.tick_params(labelbottom=False)
                aha_ax.tick_params(labelbottom=False)
                      
            aims_tm1 = np.array(dataset_dict[mag]['tm1'])
            n_tm1 = len(aims_tm1)
            if n_tm1 > 0:
                hist_tm1, _ = np.histogram(aims_tm1, bins=bins)
                pre_ax.stairs(hist_tm1, bins, color=color, linewidth=3.5, fill=False, alpha=1)
                pre_ax.axvline(0, color='gray', linestyle='--', linewidth=2.5, alpha=0.7)
                pre_ax.text(0.98, 0.95, f'N={n_tm1}', ha='right', va='top', transform=pre_ax.transAxes,
                            fontsize=label_size, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'), color=color)
                                               
                pre_ax.text(0.05, 0.95, dataset_dict[mag]['sig_pre'], ha='left', va='top', transform=pre_ax.transAxes,
                            fontsize=label_size * 1.5, fontweight='bold')
                max_pre = hist_tm1.max()
                ylim_pre = (0,60) if (max_pre < 61 and max_pre > 30) else (0, max(max_pre * 1.1, 11.5))
            else:
                pre_ax.text(0.5, 0.5, 'No data', transform=pre_ax.transAxes, ha='center', va='center', fontsize=10)
                ylim_pre = (0, 11.5)
            pre_ax.set_ylim(ylim_pre)
            pre_ax.set_ylabel('Count')
            
            if datasetName == 'Brudner':
                pre_ax.yaxis.set_major_locator(MultipleLocator(5))
            else:
                pre_ax.yaxis.set_major_locator(MultipleLocator(30))
            pre_ax.yaxis.set_minor_locator(AutoMinorLocator(2))
            sns.despine(ax=pre_ax)
                  
            aims_t0 = np.array(dataset_dict[mag]['t0'])
            n_t0 = len(aims_t0)
            if n_t0 > 0:
                hist_t0, _ = np.histogram(aims_t0, bins=bins)
                aha_ax.stairs(hist_t0, bins, color=color, linewidth=3.5, fill=False, alpha=1)
                aha_ax.axvline(0, color='gray', linestyle='--', linewidth=2.5, alpha=0.7)
                aha_ax.axvline(mag, color=color, linestyle='--', linewidth=4, alpha=0.5)
                aha_ax.text(0.98, 0.95, f'N={n_t0}', ha='right', va='top', transform=aha_ax.transAxes,
                            fontsize=label_size, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'), color=color)
                                               
                aha_ax.text(0.05, 0.95, dataset_dict[mag]['sig_aha'], ha='left', va='top', transform=aha_ax.transAxes,
                            fontsize=label_size * 1.5, fontweight='bold')
                max_aha = hist_t0.max()
                ylim_aha = (0, 11.5)                              
            else:
                aha_ax.text(0.5, 0.5, 'No data', transform=aha_ax.transAxes, ha='center', va='center', fontsize=10)
                ylim_aha = (0, 11.5)
          
            aha_ax.set_ylim(ylim_aha)
            aha_ax.set_ylabel('')
            aha_ax.yaxis.set_major_locator(MultipleLocator(5))
            aha_ax.yaxis.set_minor_locator(AutoMinorLocator(2))
            sns.despine(ax=aha_ax)
                                          
                                                              
                                                            
        plt.tight_layout(rect=[0, 0, 1, 0.92])
                                     
        if save_filename:
            dir_name = os.path.dirname(save_filename)
            base_name = os.path.basename(save_filename)
            ds_save = os.path.join(dir_name, f'{datasetName}_{base_name}') if dir_name else f'{datasetName}_{base_name}'
            plt.savefig(ds_save, dpi=300, bbox_inches='tight')
            print(f"Saved: {ds_save}")
                   
    df_hist = pd.DataFrame(summary_data)
    return df_hist
df_hist_by_rot = plot_human_aim_hist_aha_by_rot(explicit_only=True,
                                                bin_width=3,
                                                max_bin=180,
                                                save_filename=targetFolder+'ahaAimHistogramsByDatasetAndRotation.svg')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import signal
from scipy.stats import norm
from matplotlib.ticker import AutoMinorLocator
from collections import defaultdict
import seaborn as sns

baselineLength = 40
washoutLength = 40

def darken_color(hex_color, factor=0.6):
    rgb = mcolors.hex2color(hex_color)
    darkened = tuple(c * factor for c in rgb)
    return mcolors.to_hex(darkened)

def plot_human_direction_post_aha(ylim=(0, 1.05), save_filename=None, explicit_only=False, max_rel_pos=60):
    """
    Modified version: now plots a SEPARATE figure for EACH dataset (no pooling).
    Additionally fixed a likely inconsistency/bug: human proportions are now only counted
    when |aim| > 5° (decisive trials), to match the axis labels and the model's conditional probabilities.
    Datasets with only one rotation magnitude (e.g., Brudner) get a single-panel figure.
    """
    threshold = 10.0
                                                                  
    fine_angles = np.arange(-180, 181, dtype=float)
    zero_idx = np.searchsorted(fine_angles, 0)
    delta_probs = np.zeros(len(fine_angles))
    delta_probs[zero_idx] = 1.0
    orig_idx = np.mod(fine_angles + 180, 360).astype(int)
    dataset_names = ['CGVanilla', 'CGImp', 'Brudner', 'BondTaylor']
    datasets = [
        [ssmsAll8CG, qLearnsAll8CG, hmmsAll8CG],            
        [ssmsAll8CGIMP, qLearnsAll8CGIMP, hmmsAll8CGIMP],        
        [ssmsAllBrudner, qLearnsAllBrudner, hmmsAllBrudner],          
        [ssmsAll8BT, qLearnsAll8BT, hmmsAll8BT]             
    ]
    rel_positions = list(range(0, max_rel_pos + 1))
    rel_pos_arr = np.arange(max_rel_pos + 1)
    dfs = {}
    for ds_idx, dataset_name in enumerate(dataset_names):
        dataset = datasets[ds_idx]
        ssm_list = dataset[0]
        hmm_list = dataset[2]
        if len(ssm_list) == 0:
            continue
        bl = 64 if dataset_name == 'Brudner' else baselineLength
        washoutLength_ds = 64 if dataset_name == 'Brudner' else washoutLength
        positive_count = defaultdict(int)
        negative_count = defaultdict(int)
        model_cond_pos = defaultdict(list)
        all_rotation_magnitudes = set()
        rotations = []
        for ssm in ssm_list:
            if hasattr(ssm, 'rotations'):
                rotations.append(np.asarray(ssm.rotations))
            else:
                rotations.append(None)
        for rotIdx in range(len(rotations)):
            rot_array = rotations[rotIdx]
            if rot_array is None or rotIdx >= len(hmm_list):
                continue
            non_zero = rot_array[rot_array != 0]
            if len(non_zero) == 0:
                rotation_magnitude = 0.0
            else:
                rotation_magnitude = float(np.unique(np.abs(non_zero))[0])
            if rotation_magnitude == 0:
                continue
            all_rotation_magnitudes.add(rotation_magnitude)
            hmm_fit = hmm_list[rotIdx]
                                                      
            policies_full = None
            pi_preds_full = None
            try:
                policies_full = np.array(hmm_fit.model_predictive_policies)
                pi_preds_full = np.array(hmm_fit.pi_preds)
            except AttributeError:
                pass
            human_explicits_full = np.array(hmm_fit.human_explicits)
            if explicit_only:
                human_signed_full = human_explicits_full
            else:
                human_signed_full = human_explicits_full
                if hasattr(hmm_fit, 'human_implicits'):
                    human_signed_full += np.array(hmm_fit.human_implicits)
            n_parts, n_trials = human_signed_full.shape
            min_t = bl + 1
            max_t = n_trials - (washoutLength_ds + min(40, bl))
            if min_t > max_t:
                continue
            human_abs_full = np.abs(human_signed_full)
            for part in range(n_parts):
                aim_signed = human_signed_full[part, :]
                if rot_array.ndim == 1:
                    part_rot_array = rot_array
                else:
                    part_rot_array = rot_array[part, :]
                                            
                min_cost = np.inf
                best_t_part = None
                best_aim_t = -np.inf
                for t in range(min_t, max_t + 1):
                    rot_val = part_rot_array[min(t, len(part_rot_array)-1)] if len(part_rot_array) > t else (part_rot_array[0] if len(part_rot_array) > 0 else 0.0)
                    if np.isnan(rot_val):
                        valid_rots = [r for r in part_rot_array if not np.isnan(r)]
                        rot_val = valid_rots[0] if valid_rots else 0.0
                    if part >= len(human_abs_full):
                        print(part,len(human_abs_full))
                    cost = np.nansum(np.maximum(np.abs(human_abs_full[part, bl:t]) - np.nanmean(human_abs_full[part, :bl]), 0)) + (20) * (np.nanmean(np.abs(np.abs(rot_val) - human_abs_full[part, t:min(max_t, t + min(40, bl))])))
                    current_aim_mag = np.abs(aim_signed[t])
                    update = False
                    if cost < min_cost:
                        update = True
                    elif np.abs(cost - min_cost) < 1e-8 and current_aim_mag > best_aim_t:
                        update = True
                    if update:
                        min_cost = cost
                        best_t_part = t
                        best_aim_t = current_aim_mag
                if best_t_part is None:
                    continue
                for k in rel_positions:
                    trial_idx = best_t_part + k
                    if trial_idx >= n_trials or trial_idx >= n_trials - washoutLength_ds:
                        break
                    aim_val = aim_signed[trial_idx]
                    if np.isnan(aim_val):
                        continue
                    rot_val = part_rot_array[min(trial_idx, len(part_rot_array)-1)] if len(part_rot_array) > trial_idx else (part_rot_array[0] if len(part_rot_array) > 0 else 0.0)
                    if np.isnan(rot_val) or rot_val == 0:
                        continue
                    abs_aim = np.abs(aim_val)
                    if abs_aim > threshold:                                           
                        key = (rotation_magnitude, k)
                        if aim_val > 0:
                            positive_count[key] += 1
                        else:
                            negative_count[key] += 1
                                                                       
                        if policies_full is not None:
                            try:
                                state0_prob = pi_preds_full[part, trial_idx, 0]
                                policy_this = policies_full[part, trial_idx, :]
                                recentered_probs = policy_this[orig_idx]
                                sigma = 0.0
                                if dataset_name == 'BondTaylor':
                                    sigma = 0.0
                                elif hasattr(hmm_fit, 'xs') and part < len(hmm_fit.xs):
                                    try:
                                        sigma = float(hmm_fit.xs[part][0])
                                        if np.isnan(sigma):
                                            sigma = 0.0
                                    except:
                                        sigma = 0.0
                                if sigma > 0.0001:
                                    support_size = max(int(6 * sigma) + 1, 3)
                                    if support_size % 2 == 0:
                                        support_size += 1
                                    half_support = support_size // 2
                                    support = np.arange(-half_support, half_support + 1)
                                    kernel = norm.pdf(support, 0, sigma)
                                    kernel /= kernel.sum()
                                    convolved0 = ndimage.convolve1d(delta_probs, kernel, mode='wrap')
                                    convolved1 = ndimage.convolve1d(recentered_probs, kernel, mode='wrap')
                                else:
                                    convolved0 = delta_probs.copy()
                                    convolved1 = recentered_probs.copy()
                               
                                convolved1 = np.maximum(convolved1, 0)
                                convolved0 = np.maximum(convolved0, 0)
                                convolved0_norm = convolved0 / convolved0.sum() if convolved0.sum() > 0 else convolved0
                                convolved1_norm = convolved1 / convolved1.sum() if convolved1.sum() > 0 else convolved1
                                marginal = state0_prob * convolved0_norm + (1 - state0_prob) * convolved1_norm
                                if marginal.sum() > 0:
                                    marginal /= marginal.sum()
                                p_pos = np.sum(marginal[fine_angles > threshold])
                                p_neg = np.sum(marginal[fine_angles < -threshold])
                                decisive_p = p_pos + p_neg
                                cond_pos = p_pos / decisive_p if decisive_p > 0.05 else 0.5
                                model_cond_pos[key].append(cond_pos)
                            except (IndexError, ValueError, TypeError):
                                pass
        unique_rots = sorted([r for r in all_rotation_magnitudes if r > 0], reverse=True)
        if len(unique_rots) == 0:
            print(f"No rotation data collected for {dataset_name}.")
            continue
                                                       
        summary_points = []
        for rot in unique_rots:
            pos_series = []
            neg_series = []
            n_dec_series = []
            model_pos_series = []
            model_neg_series = []
            model_pos_low_series = []
            model_pos_high_series = []
            model_neg_low_series = []
            model_neg_high_series = []
            for k in rel_positions:
                key = (rot, k)
                p = positive_count.get(key, 0)
                n_c = negative_count.get(key, 0)
                dec = p + n_c
                pos_prop = p / dec if dec > 0 else 0.0
                neg_prop = n_c / dec if dec > 0 else 0.0
                pos_series.append(pos_prop)
                neg_series.append(neg_prop)
                n_dec_series.append(dec)
                cond_pos_list = model_cond_pos.get(key, [])
                n_model = len(cond_pos_list)
                if dec > 0 and n_model > 0:
                    mean_pos_m = np.mean(cond_pos_list)
                    mean_neg_m = 1 - mean_pos_m
                    if n_model > 1:
                        sem = np.std(cond_pos_list, ddof=1) / np.sqrt(n_model)
                        low_pos = max(0, mean_pos_m - 1.96 * sem)
                        high_pos = min(1, mean_pos_m + 1.96 * sem)
                        low_neg = 1 - high_pos
                        high_neg = 1 - low_pos
                    else:
                        low_pos = high_pos = mean_pos_m
                        low_neg = high_neg = mean_neg_m
                else:
                    mean_pos_m = np.nan
                    mean_neg_m = np.nan
                    low_pos = high_pos = np.nan
                    low_neg = high_neg = np.nan
                model_pos_series.append(mean_pos_m)
                model_neg_series.append(mean_neg_m)
                model_pos_low_series.append(low_pos)
                model_pos_high_series.append(high_pos)
                model_neg_low_series.append(low_neg)
                model_neg_high_series.append(high_neg)
            summary_points.append({
                'rotation': rot,
                'rel_pos': rel_positions,
                'pos_prop': pos_series,
                'neg_prop': neg_series,
                'n_decisive': n_dec_series,
                'model_pos_prop': model_pos_series,
                'model_neg_prop': model_neg_series,
                'model_pos_low': model_pos_low_series,
                'model_pos_high': model_pos_high_series,
                'model_neg_low': model_neg_low_series,
                'model_neg_high': model_neg_high_series
            })
        df = pd.DataFrame(summary_points)
        dfs[dataset_name] = df
                                                        
        n_rots = len(unique_rots)
        
                                                                
        ncols = 1 if n_rots == 1 else 2
        nrows = (n_rots + ncols - 1) // ncols
        panel_width = 6.0/2
        panel_height = 5.0/2.5
        figsize = (panel_width * ncols, panel_height * nrows)
        
        fig, axs = plt.subplots(nrows, ncols, figsize=figsize,
                                sharex=False, sharey=False,
                                constrained_layout=True)
        axs = np.ravel(axs)
        
        for ax in axs:
            ax.grid(False)
            
        pos_color_human = '#3399ff'
        neg_color_human = '#ff3333'
        model_pos_color = '#66ccff'
        model_neg_color = '#ff8080'
        
        for idx, rot in enumerate(unique_rots):
            ax = axs[idx]
            row = df[df['rotation'] == rot].iloc[0]
            pos_arr = np.array(row['pos_prop'])
            neg_arr = np.array(row['neg_prop'])
            ax.bar(rel_pos_arr, pos_arr, width=1.015, color=pos_color_human, alpha=0.5,
                   edgecolor='none', label='Positive Aims (>thresh°)' if idx == 0 else None)
            ax.bar(rel_pos_arr, neg_arr, width=1.015, color=neg_color_human, alpha=0.8,
                   edgecolor='none', label='Negative Aims (<-thresh°)' if idx == 0 else None)
            model_pos_arr = np.array(row['model_pos_prop'])
            model_neg_arr = np.array(row['model_neg_prop'])
            if np.any(np.isfinite(model_pos_arr)):
                dark_pos = darken_color(model_pos_color)
                ax.plot(rel_pos_arr, model_pos_arr, color=dark_pos, linewidth=1.8, zorder=10,alpha=0.7,
                        label='Model Positive' if idx == 0 else None)
            if np.any(np.isfinite(model_neg_arr)):
                dark_neg = darken_color(model_neg_color)
                ax.plot(rel_pos_arr, model_neg_arr, color=dark_neg, linewidth=1.8, zorder=10,alpha=0.7,
                        label='Model Negative' if idx == 0 else None)
            ax.set_title(f'{int(rot)}°',fontsize=16)
            ax.set_ylim(ylim)
            ax.axvline(x=0, color='black', linestyle='-', linewidth=1, alpha=0.4)
            
                                      
        for j in range(n_rots, len(axs)):
            axs[j].axis('off')
            
                          
        for ax in axs[::ncols]:
            ax.set_ylabel('')
        for ax in axs[-ncols:]:
            ax.set_xlabel('')
        for ax in axs.flat:
            ax.yaxis.set_major_locator(MultipleLocator(0.5))
            ax.xaxis.set_major_locator(MultipleLocator(50))
            ax.yaxis.set_minor_locator(AutoMinorLocator(2))
            ax.xaxis.set_minor_locator(AutoMinorLocator(2))
            for label in (ax.get_xticklabels() + ax.get_yticklabels()):
                label.set_fontsize(16)
                                                                           
        
        sns.set_style('ticks')
        sns.despine()
        fig.suptitle(f'({dataset_name})', fontsize=16)
        
        if save_filename:
            ds_save = save_filename.replace('.svg', f'_{dataset_name}.svg')
            plt.savefig(ds_save, dpi=300)
                    
        
    return dfs

              
dfs_direction = plot_human_direction_post_aha(ylim=(0, 1.05), explicit_only=True,
                                             save_filename=targetFolder + 'ahaAlignedAimDirectionWithModel.svg',
                                             max_rel_pos=150)

In [ ]:


baselineLength = 40
washoutLength = 40

def perform_direction_phase_anova(explicit_only=False):
    """
    Collects per-participant proportion of positive decisive aims (among decisive only)
    for Early (0-5), Mid (90-95), Late (200-205) post-Aha! phases.
    EXCLUDES Brudner dataset entirely to avoid multicollinearity issues.
    Applies arcsin-square-root transformation to stabilize variance.
    Fits ordinary least squares (fixed effects) three-way ANOVA:
        transformed_prop ~ C(dataset) * C(rotation) * C(phase)
    Prints descriptives and the Type II ANOVA table (appropriate for unbalanced designs).
    Returns the DataFrame, transformed data, model, and ANOVA table.
    """
    rows = []
    participant_counter = 0
    
    datasets = [
        [ssmsAll8CG, qLearnsAll8CG, hmmsAll8CG],                 
        [ssmsAll8CGIMP, qLearnsAll8CGIMP, hmmsAll8CGIMP],         
        [ssmsAll8BT, qLearnsAll8BT, hmmsAll8BT]                   
    ]
    dataset_labels = ['CGVanilla8', 'CGImp8', 'BondTaylor']
    
    phases = {
        'Early': list(range(0, 6)),
        'Mid':   list(range(90, 96)),
        'Late':  list(range(200, 206))
    }
    
    bl = 40
    washout_ds = 40
    
    for ds_idx, dataset_name in enumerate(dataset_labels):
        dataset = datasets[ds_idx]
        ssm_list = dataset[0]
        hmm_list = dataset[2]
        
        rotations = []
        for ssm in ssm_list:
            if hasattr(ssm, 'rotations'):
                rotations.append(np.asarray(ssm.rotations))
            else:
                rotations.append(None)
        
        for rot_idx, rot_array in enumerate(rotations):
            if rot_array is None or rot_idx >= len(hmm_list) or rot_idx >= len(ssm_list):
                continue
            
            non_zero = rot_array[rot_array != 0]
            if len(non_zero) == 0:
                continue
            rot_mag = float(np.unique(np.abs(non_zero))[0])
            
            hmm_fit = hmm_list[rot_idx]
            if not hasattr(hmm_fit, 'human_explicits'):
                continue
            
            human_explicits_full = np.array(hmm_fit.human_explicits)
            if explicit_only:
                human_signed_full = human_explicits_full
            else:
                human_implicits_full = np.array(getattr(hmm_fit, 'human_implicits', np.zeros_like(human_explicits_full)))
                human_signed_full = human_explicits_full + human_implicits_full
            
            n_parts, n_trials = human_signed_full.shape
            
            min_t = bl + 1
            max_t = n_trials - (washout_ds + min(40,bl))
            if min_t > max_t:
                continue
            human_abs_full = np.abs(human_signed_full)
            
            for part in range(n_parts):
                participant_id = participant_counter
                participant_counter += 1
                
                aim_signed = human_signed_full[part, :]
                part_rot_array = rot_array if rot_array.ndim == 1 else rot_array[part, :]
                
                                                
                min_cost = np.inf
                best_t_part = None
                best_aim_t = -np.inf
                for t in range(min_t, max_t + 1):
                    rot_val = part_rot_array[min(t, len(part_rot_array)-1)] if len(part_rot_array) > 0 else 0.0
                    if np.isnan(rot_val):
                        valid_rots = [r for r in part_rot_array if not np.isnan(r)]
                        rot_val = valid_rots[0] if valid_rots else 0.0
                    aim = np.abs(aim_signed)
                    cost = np.nansum(np.maximum(np.abs(human_abs_full[part, bl:t]) - np.nanmean(human_abs_full[part, :bl]), 0)) + (20) * (np.nanmean(np.abs(np.abs(rot_val) - human_abs_full[part, t:min(max_t, t + min(40, bl))])))
                    current_aim_mag = np.abs(aim_signed[t])
                    
                    update = cost < min_cost or (np.abs(cost - min_cost) < 1e-8 and current_aim_mag > best_aim_t)
                    if update:
                        min_cost = cost
                        best_t_part = t
                        best_aim_t = current_aim_mag
                
                if best_t_part is None:
                    continue
                
                for phase_name, rel_ks in phases.items():
                    pos_count = 0
                    dec_count = 0
                    for k in rel_ks:
                        trial_idx = best_t_part + k
                        if trial_idx >= n_trials or trial_idx >= n_trials - washout_ds:
                            continue
                        
                        aim_val = aim_signed[trial_idx]
                        if np.isnan(aim_val):
                            continue
                        
                        rot_val = part_rot_array[min(trial_idx, len(part_rot_array)-1)] if len(part_rot_array) > 0 else 0.0
                        if np.isnan(rot_val) or rot_val == 0:
                            continue
                        
                        if np.abs(aim_val) > 5:
                            dec_count += 1
                            if aim_val > 0:
                                pos_count += 1
                    
                    if dec_count > 0:
                        rows.append({
                            'participant': participant_id,
                            'dataset': dataset_name,
                            'rotation': rot_mag,
                            'phase': phase_name,
                            'prop_positive': pos_count / dec_count,
                            'n_decisive': dec_count
                        })
    
    if not rows:
        print("No decisive aims found in any phase.")
        return None, None, None, None
    
    df = pd.DataFrame(rows)
    
                  
    print("\nNumber of participant-phase observations:", len(df))
    print("\nObservations per phase:")
    print(df['phase'].value_counts().sort_index())
    
    print("\nMean proportion positive per cell (with N and SD):")
    desc = df.groupby(['dataset', 'rotation', 'phase'])['prop_positive'].agg(['mean', 'std', 'count'])
    print(desc.round(3))
    
                                       
    df['transformed_prop'] = np.arcsin(np.sqrt(df['prop_positive']))
    
                       
    df['dataset'] = df['dataset'].astype('category')
    df['rotation'] = df['rotation'].astype('category')
    df['phase'] = pd.Categorical(df['phase'], categories=['Early', 'Mid', 'Late'], ordered=True)
    
                                              
    formula = 'transformed_prop ~ C(dataset) * C(rotation) * C(phase)'
    model = ols(formula, data=df).fit()
    
    print("\nOLS Model Summary (on arcsin-sqrt transformed proportions):")
    print(model.summary())
    
                         
    anova_table = anova_lm(model, typ=2)
    print("\nThree-way Type II ANOVA Table:")
    print(anova_table.round(4))
    
    return df, model, anova_table

                                                                
df_anova, model, anova_results = perform_direction_phase_anova(explicit_only=True)

In [ ]:

ding_baseline_length = 24                                         
ding_washout_length = 12                                            

def angular_difference(h, t):
    diff = (h - t + 180) % 360 - 180
    return diff

                              
df_ding = pd.read_csv('2025-12_Ding_Data.csv')
df_ding['aim_signed'] = df_ding.apply(lambda row: angular_difference(row['Hand Angle'], row['Target Angle']), axis=1)
df_ding['aim_abs'] = np.abs(df_ding['aim_signed'])

                                                  
ding_conditions = sorted(df_ding['condition'].unique())
ding_n_trials = 132                                          

                                                                                                        
ding_datasets = []
ding_dataset_labels = []
for cond in ding_conditions:
    sub_df = df_ding[df_ding['condition'] == cond]
    participant_ids = sorted(sub_df['participant_idx'].unique())
    n_parts = len(participant_ids)
    
                       
    human_abs_full = np.full((n_parts, ding_n_trials), np.nan)
    rot_per_trial = np.full(ding_n_trials, np.nan)
    
                                
    for i, part_id in enumerate(participant_ids):
        part_df = sub_df[sub_df['participant_idx'] == part_id].sort_values('trial_number')
        trial_indices = part_df['trial_number'].values - 1           
        abs_aims = part_df['aim_abs'].values
        human_abs_full[i, trial_indices] = abs_aims
    
                                                     
    trial_rots = sub_df.groupby('trial_number')['Rotation'].first().sort_index()
    rot_per_trial[:len(trial_rots)] = trial_rots.values
    
    ding_datasets.append([[human_abs_full], [rot_per_trial]])
    ding_dataset_labels.append(cond)

def plot_ding_human_aim_baseline_aha_next(ding_datasets, ding_dataset_labels, ylim=None, save_filename=None, bl=ding_baseline_length, washout_len=ding_washout_length):
    """
    Modified version for this dataset.
    - Uses provided datasets and labels: each dataset is [human_abs_lists, rot_lists], lists of arrays (one per "rotation group", here one per condition).
    - Fixed bl and washout_len.
    - Computes rot_size as abs of unique non-zero rotation.
    - Fixes reference lines and yticks to positive |rot| values.
    - Adds tie-breaking for larger |aim[t]| in cost.
    - Changes cost slicing to n_trials - washout_len.
    """
                                                                       
    summary_points = []
    
                                                                                             
    rotation_colors = list(reversed(['#44AA99', '#88CCEE', '#FF9825', '#CC6677', '#AA4499']))
    rel_positions = list(range(-20, 20))
    
    for dsIdx, datasetName in enumerate(ding_dataset_labels):
        human_abs_lists = ding_datasets[dsIdx][0]
        rot_lists = ding_datasets[dsIdx][1]
        
                                                    
        
        for rotIdx in range(len(human_abs_lists)):
            human_abs_full = human_abs_lists[rotIdx]
            rot_array = rot_lists[rotIdx]
            
                                   
            non_zero_rots = rot_array[rot_array != 0]
            if len(non_zero_rots) > 0:
                rotation_size = abs(float(np.unique(non_zero_rots)[0]))
            else:
                rotation_size = 0.0
            
            n_parts, n_trials_local = human_abs_full.shape
                                                                                                  

                                   
            baseline_human_abs_list = []
            series_list = []
            valid_parts = 0

            baseline_trial = bl - 1
            if baseline_trial < 0 or baseline_trial >= n_trials_local:
                                                                                                 
                continue

            min_t = bl                                       
            max_t = n_trials_local - (washout_len + min(40,bl))
            if min_t > max_t:
                                                                                                                
                continue
                                                           
            for part in range(n_parts):
                aim = human_abs_full[part, :]

                                                                        
                part_rot_array = rot_array.copy()

                                                            
                min_cost = np.inf
                best_t_part = None
                best_aim_t = -np.inf
                for t in range(min_t, max_t + 1):
                                            
                    if t < len(part_rot_array):
                        rot_val = part_rot_array[t]
                    else:
                        rot_val = part_rot_array[-1] if len(part_rot_array) > 0 else 0.0
                    if np.isnan(rot_val):
                        continue                    
                    target_mag = np.abs(rot_val)
                    after_slice = slice(t, t+bl)
                    if after_slice.stop <= after_slice.start:
                        continue                  
                    cost = np.nansum(np.maximum(np.abs(human_abs_full[part, bl:t]) - np.nanmean(human_abs_full[part, :bl]), 0)) + (20) * (np.nanmean(np.abs(np.abs(rot_val) - human_abs_full[part, t:min(max_t, t + min(40, bl))])))
                                                                             
                    if cost < min_cost or (np.isclose(cost, min_cost) and np.abs(aim[t]) > best_aim_t):
                        min_cost = cost
                        best_t_part = t
                        best_aim_t = np.abs(aim[t])

                if best_t_part is None:
                                                                                                             
                    continue

                                                                           
                series_part = np.full(len(rel_positions), np.nan, dtype=float)
                for ii, rel in enumerate(rel_positions):
                    trial_idx = best_t_part + rel
                    if 0 <= trial_idx < n_trials_local:
                        series_part[ii] = aim[trial_idx]
                baseline_human_abs = aim[baseline_trial]

                baseline_human_abs_list.append(baseline_human_abs)
                series_list.append(series_part)
                valid_parts += 1

                                                                                                                    

                                                                                                                 

                                                                       
            n_series = len(series_list)
            if n_series > 0:
                series_array = np.stack(series_list)
                mean_series = np.nanmean(series_array, axis=0)
                std_series = np.nanstd(series_array, axis=0, ddof=1)
                n_per_pos = np.sum(~np.isnan(series_array), axis=0)
                sem_series = np.divide(std_series, np.sqrt(n_per_pos), out=np.full_like(std_series, np.nan), where=(n_per_pos > 1))
                ci_series = 1.96 * sem_series
               
                baseline_array = np.array(baseline_human_abs_list)
                n_base = np.sum(~np.isnan(baseline_array))
                if n_base > 0:
                    mean_base = np.nanmean(baseline_array)
                    std_base = np.nanstd(baseline_array, ddof=1)
                    sem_base = std_base / np.sqrt(n_base)
                    ci_base = 1.96 * sem_base
                else:
                    mean_base = np.nan
                    ci_base = np.nan
               
                summary_points.append({
                    'rotation': rotation_size,
                    'dataset': datasetName,
                    'mean_baseline': mean_base,
                    'ci_baseline': ci_base,
                    'n_base': n_base,
                    'mean_series': mean_series.tolist(),
                    'ci_series': ci_series.tolist(),
                    'n_series': n_series,
                    'individual_series': series_list,
                    'rel_pos': rel_positions
                })
            else:
                pass                                                                                     
                          
    df_summary = pd.DataFrame(summary_points)
    if df_summary.empty:
        print("No data points collected; check attributes and data.")
        return df_summary
                                                                
                                                  
    unique_rots = sorted(df_summary['rotation'].unique(), reverse=True)
    rot_to_color_idx = {rot: i for i, rot in enumerate(unique_rots)}
                                                                           
    num_ds = len(ding_dataset_labels)
    nrows = (num_ds + 1) // 2
    ncols = 2
    fig, axs = plt.subplots(nrows, ncols, figsize=(15, 8 * nrows))
    if nrows == 1 and ncols == 1:
        axs = [axs]
    else:
        axs = axs.flatten()
    for dsIdx, datasetName in enumerate(ding_dataset_labels):
        if dsIdx >= len(axs):
            break
        ax = axs[dsIdx]
        subset = df_summary[df_summary['dataset'] == datasetName]
        if subset.empty:
            ax.set_visible(False)
            continue
                                                          
        subset_rots = sorted(subset['rotation'].unique())
                            
        ax.axhline(y=0, color='grey', linestyle='--', linewidth=1, alpha=0.3)
                                                    
        for rot in subset_rots:
            color_idx = rot_to_color_idx[rot]
            rot_color = rotation_colors[color_idx]
            ax.axhline(y=rot, color=rot_color, alpha=0.5, linestyle='--', linewidth=1)
                                                         
        yticks = sorted([0] + list(subset_rots))
        yticklabels = [f'{int(t)}°' for t in yticks]
        ax.set_yticks(yticks)
        ax.set_yticklabels(yticklabels)
        for _, row in subset.iterrows():
            rot = row['rotation']
            color_idx = rot_to_color_idx[rot]
            rot_color = rotation_colors[color_idx]
            mean_ser = np.array(row['mean_series'])
            ci_ser = np.array(row['ci_series'])
            rel_pos = np.array(row['rel_pos'])
                                                                    
            mean_ser = np.maximum(mean_ser, 0)
                                          
            lower_ci = np.maximum(mean_ser - ci_ser, 0)
            upper_ci = mean_ser + ci_ser
                                                       
            ax.scatter(rel_pos, mean_ser, color=rot_color, s=50, zorder=5, label=f'{rot}°')
                                                  
            ax.fill_between(rel_pos, lower_ci, upper_ci, color=rot_color, alpha=0.2)
                                                       
            for ind_series in row['individual_series']:
                valid_mask = ~np.isnan(ind_series)
                if np.any(valid_mask):
                    valid_x = rel_pos[valid_mask]
                    valid_y = ind_series[valid_mask]
                    jitter = np.random.uniform(-0.3, 0.3, size=len(valid_x))
                    x_jittered = valid_x + jitter
                    ax.scatter(x_jittered, valid_y, color=rot_color, alpha=0.3, s=2, zorder=6)
         
                           
        ax.set_title(datasetName)
        ax.set_xlabel('Relative Trial (0 = Aha!)')
        ax.set_ylabel('Mean |Human Aim| (°)')
        ax.axvline(x=0, color='black', linestyle='-', alpha=0.3, linewidth=1)                  
                    
        if ylim is not None:
            ax.set_ylim(ylim)
                          
    for idx in range(num_ds, len(axs)):
        axs[idx].set_visible(False)
    sns.despine()
    fig.suptitle('Mean |Aim| Around Aha! Trial with 95% CI Bands (Separate Detection per Participant)', fontsize=14)
    plt.tight_layout()
    if save_filename:
        plt.savefig(save_filename, dpi=300, bbox_inches='tight')
               
                   
                                                                                                     
    return df_summary

df_ding_human = plot_ding_human_aim_baseline_aha_next(ding_datasets, ding_dataset_labels, ylim=(0, 100), save_filename=targetFolder+'dingTest.svg')

In [ ]:


baselineLength = 40
washoutLength = 40
BONFERRONI_FACTOR = 5                                                   

def analyze_human_aim_aha_by_dataset_rot(explicit_only=True):
    """
    Uses the exact same Aha! trial detection technique as the original functions.
    No plotting.
    For each dataset separately and each rotation size separately:
    - Computes mean ± std of baseline |aim|, |aim| on t-1, and |aim| on t
    - Paired t-test: t-1 vs participant-specific baseline mean |aim|
    - One-sample t-test: t vs |rotation size|
    - Reports both raw and Bonferroni-corrected p-values (corrected = min(1, raw × 5))
    """
    aims_by_dataset_rot = defaultdict(lambda: defaultdict(lambda: {'tm1': [], 't0': [], 'baseline': []}))
    
    datasets = [
        [ssmsAll8CG, qLearnsAll8CG, hmmsAll8CG],                    
        [ssmsAll8CGIMP, qLearnsAll8CGIMP, hmmsAll8CGIMP],        
        [ssmsAllBrudner, qLearnsAllBrudner, hmmsAllBrudner],          
        [ssmsAll8BT, qLearnsAll8BT, hmmsAll8BT]                      
    ]
    datasetLabels = ['CGVanilla8', 'CGImp8', 'Brudner', 'BondTaylor']
    
    for dsIdx, datasetName in enumerate(datasetLabels):
        ssm_list = datasets[dsIdx][0]
        hmm_list = datasets[dsIdx][2]
        
                                                   
        bl = baselineLength
        this_washout = washoutLength
        if datasetName == 'Brudner':
            bl = 64
            this_washout = 64
        
        rotations = []
        rotation_sizes = []
        
        for ssm in ssm_list:
            if hasattr(ssm, 'rotations'):
                rot_array = np.asarray(ssm.rotations)
                rotations.append(rot_array)
                non_zero_rots = rot_array[rot_array != 0]
                if len(non_zero_rots) > 0:
                    rot_size = float(np.unique(non_zero_rots)[0])
                else:
                    rot_size = 0.0
                rotation_sizes.append(rot_size)
            else:
                rotations.append(None)
                rotation_sizes.append(0.0)
        
        for rotIdx, (rot_array, rotation_size) in enumerate(zip(rotations, rotation_sizes)):
            if rot_array is None or rot_array.size == 0 or rotIdx >= len(hmm_list) or rotIdx >= len(ssm_list):
                continue
            hmm_fit = hmm_list[rotIdx]
            if not hasattr(hmm_fit, 'human_explicits'):
                continue
                
            human_explicits_full = np.array(hmm_fit.human_explicits)
            if explicit_only:
                human_abs_full = np.abs(human_explicits_full)
            else:
                if hasattr(hmm_fit, 'human_implicits'):
                    human_implicits_full = np.array(hmm_fit.human_implicits)
                    human_full = human_explicits_full + human_implicits_full
                    human_abs_full = np.abs(human_full)
                else:
                    human_abs_full = np.abs(human_explicits_full)
                    
            n_parts, n_trials = human_abs_full.shape
            min_t = bl + 1
            max_t = n_trials - (this_washout + min(40,bl))
            if min_t > max_t:
                continue
            
            for part in range(n_parts):
                aim = human_abs_full[part, :]
                baseline_mean = np.nanmean(aim[0:bl])                                            
                
                                                     
                if rot_array.ndim == 1:
                    part_rot_array = rot_array[part]
                else:
                    part_rot_array = rot_array[part, :]
                
                                          
                min_cost = np.inf
                best_t_part = None
                for t in range(min_t, max_t + 1):
                    if t >= n_trials:
                        continue
                                               
                    if len(part_rot_array) > 1 and t < len(part_rot_array):
                        rot_val = part_rot_array[t]
                    else:
                        rot_val = part_rot_array[0] if len(part_rot_array) > 0 else 0.0
                    if np.isnan(rot_val):
                        valid_rots = [r for r in part_rot_array if not np.isnan(r)]
                        if len(valid_rots) == 0:
                            continue
                        rot_val = valid_rots[min(t % len(valid_rots), len(valid_rots)-1)]
                    target_mag = np.abs(rot_val)
                    
                                                  
                    post_start = t
                    post_end = t + bl
                    cost = np.nansum(np.maximum(np.abs(human_abs_full[part, bl:t]) - np.nanmean(human_abs_full[part, :bl]), 0)) + (20) * (np.nanmean(np.abs(np.abs(rot_val) - human_abs_full[part, t:min(max_t, t + min(40, bl))])))
                    
                    if cost < min_cost:
                        min_cost = cost
                        best_t_part = t
                
                if best_t_part is None:
                    continue
                
                                    
                t_minus1 = best_t_part - 1
                t0 = best_t_part
                if (0 <= t_minus1 < n_trials and 0 <= t0 < n_trials and
                    not np.isnan(aim[t_minus1]) and not np.isnan(aim[t0]) and not np.isnan(baseline_mean)):
                    aims_by_dataset_rot[datasetName][rotation_size]['tm1'].append(aim[t_minus1])
                    aims_by_dataset_rot[datasetName][rotation_size]['t0'].append(aim[t0])
                    aims_by_dataset_rot[datasetName][rotation_size]['baseline'].append(baseline_mean)
    
                           
    print("Statistical Analysis of |Human Aim| Around Detected Aha! Trial")
    print("(Explicit only" + ("" if explicit_only else " + implicit") + ")")
    print("Bonferroni correction: raw p × 5 (capped at 1)\n")
    
    def stars(p):
        if p < 0.001: return '***'
        if p < 0.01:  return '**'
        if p < 0.05:  return '*'
        return ''
    
    for datasetName in datasetLabels:
        if datasetName not in aims_by_dataset_rot or not aims_by_dataset_rot[datasetName]:
            print(f"{datasetName}: No valid Aha! trials detected.")
            continue
            
        print(f"=== {datasetName} ===")
        rots = sorted(aims_by_dataset_rot[datasetName].keys(), reverse=True)
        
        for rot in rots:
            if rot == 0:
                continue
            data = aims_by_dataset_rot[datasetName][rot]
            aims_tm1 = np.array(data['tm1'])
            aims_t0 = np.array(data['t0'])
            aims_baseline = np.array(data['baseline'])
            n = len(aims_tm1)
            
            if n < 2:
                print(f"  Rotation {int(rot)}°: N = {n} (insufficient for t-tests)")
                continue
                
            mean_baseline = np.mean(aims_baseline)
            std_baseline = np.std(aims_baseline, ddof=1)
            mean_tm1 = np.mean(aims_tm1)
            std_tm1 = np.std(aims_tm1, ddof=1)
            mean_t0 = np.mean(aims_t0)
            std_t0 = np.std(aims_t0, ddof=1)
            target = abs(rot)
            
                                            
            t_pre, p_pre = stats.ttest_rel(aims_tm1, aims_baseline)
            p_pre_corr = min(1.0, p_pre * BONFERRONI_FACTOR)
            
                                            
            t_t0, p_t0 = stats.ttest_1samp(aims_t0, target)
            p_t0_corr = min(1.0, p_t0 * BONFERRONI_FACTOR)
            
            print(f"  Rotation {int(rot)}° (N = {n} participants)")
            print(f"    Baseline (|aim|): M = {mean_baseline:.2f}° (±{std_baseline:.2f})")
            print(f"    t-1      (|aim|): M = {mean_tm1:.2f}° (±{std_tm1:.2f})")
            print(f"      → Paired t-test (t-1 vs Baseline): t({n-1}) = {t_pre:+.2f}, raw p = {p_pre:.4f}, corr. p = {p_pre_corr:.4f}{stars(p_pre_corr)}")
            print(f"    t        (|aim|): M = {mean_t0:.2f}° (±{std_t0:.2f})   vs {int(target)}°")
            print(f"      → One-sample t-test (t vs target): t({n-1}) = {t_t0:+.2f}, raw p = {p_t0:.4f}, corr. p = {p_t0_corr:.4f}{stars(p_t0_corr)}")
        print("")

              
analyze_human_aim_aha_by_dataset_rot(explicit_only=True)

In [ ]:

sns.set(style="whitegrid", context="talk", font_scale=1.1)
baselineLength = 40
washoutLength = 40
BONFERRONI_FACTOR = 5
def stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return ''
def analyze_human_aim_aha_by_dataset_rot(explicit_only=True):
    aims_by_dataset_rot = defaultdict(lambda: defaultdict(lambda: {'tm1': [], 't0': [], 'baseline': []}))
  
    datasets = [
        [ssmsAll8CG, qLearnsAll8CG, hmmsAll8CG],
        [ssmsAll8CGIMP, qLearnsAll8CGIMP, hmmsAll8CGIMP],
        [ssmsAllBrudner, qLearnsAllBrudner, hmmsAllBrudner],
        [ssmsAll8BT, qLearnsAll8BT, hmmsAll8BT]
    ]
    datasetLabels = ['CGVanilla8', 'CGImp8', 'Brudner', 'BondTaylor']
  
                                         
    for dsIdx, datasetName in enumerate(datasetLabels):
        ssm_list = datasets[dsIdx][0]
        hmm_list = datasets[dsIdx][2]
      
        bl = baselineLength
        this_washout = washoutLength
        if datasetName == 'Brudner':
            bl = 64
            this_washout = 64
      
        rotations = []
        rotation_sizes = []
      
        for ssm in ssm_list:
            if hasattr(ssm, 'rotations'):
                rot_array = np.asarray(ssm.rotations)
                rotations.append(rot_array)
                non_zero_rots = rot_array[rot_array != 0]
                if len(non_zero_rots) > 0:
                    rot_size = float(np.unique(non_zero_rots)[0])
                else:
                    rot_size = 0.0
                rotation_sizes.append(rot_size)
            else:
                rotations.append(None)
                rotation_sizes.append(0.0)
      
        for rotIdx, (rot_array, rotation_size) in enumerate(zip(rotations, rotation_sizes)):
            if rot_array is None or rot_array.size == 0 or rotIdx >= len(hmm_list) or rotIdx >= len(ssm_list):
                continue
            hmm_fit = hmm_list[rotIdx]
            if not hasattr(hmm_fit, 'human_explicits'):
                continue
              
            human_explicits_full = np.array(hmm_fit.human_explicits)
            if explicit_only:
                human_abs_full = np.abs(human_explicits_full)
            else:
                if hasattr(hmm_fit, 'human_implicits'):
                    human_implicits_full = np.array(hmm_fit.human_implicits)
                    human_full = human_explicits_full + human_implicits_full
                    human_abs_full = np.abs(human_full)
                else:
                    human_abs_full = np.abs(human_explicits_full)
                  
            n_parts, n_trials = human_abs_full.shape
            min_t = bl + 1
            max_t = n_trials - (this_washout + min(40,bl))
            if min_t > max_t:
                continue
          
            for part in range(n_parts):
                aim = human_abs_full[part, :]
                baseline_mean = np.nanmean(aim[0:bl])
              
                if rot_array.ndim == 1:
                    part_rot_array = rot_array[part]
                else:
                    part_rot_array = rot_array[part, :]
              
                min_cost = np.inf
                best_t_part = None
                for t in range(min_t, max_t + 1):
                    if t >= n_trials:
                        continue
                    if len(part_rot_array) > 1 and t < len(part_rot_array):
                        rot_val = part_rot_array[t]
                    else:
                        rot_val = part_rot_array[0] if len(part_rot_array) > 0 else 0.0
                    if np.isnan(rot_val):
                        valid_rots = [r for r in part_rot_array if not np.isnan(r)]
                        if len(valid_rots) == 0:
                            continue
                        rot_val = valid_rots[min(t % len(valid_rots), len(valid_rots)-1)]
                    target_mag = np.abs(rot_val)
                  
                    cost = np.nansum(np.maximum(np.abs(human_abs_full[part, bl:t]) - np.nanmean(human_abs_full[part, :bl]), 0)) + (20) * (np.nanmean(np.abs(np.abs(rot_val) - human_abs_full[part, t:min(max_t, t + min(40, bl))])))
                  
                    if cost < min_cost:
                        min_cost = cost
                        best_t_part = t
              
                if best_t_part is None:
                    continue
              
                t_minus1 = best_t_part - 1
                t0 = best_t_part
                if (0 <= t_minus1 < n_trials and 0 <= t0 < n_trials and
                    not np.isnan(aim[t_minus1]) and not np.isnan(aim[t0]) and not np.isnan(baseline_mean)):
                    aims_by_dataset_rot[datasetName][rotation_size]['tm1'].append(aim[t_minus1])
                    aims_by_dataset_rot[datasetName][rotation_size]['t0'].append(aim[t0])
                    aims_by_dataset_rot[datasetName][rotation_size]['baseline'].append(baseline_mean)
  
                                   
    results = []
   
    print("Statistical Analysis of |Human Aim| Around Detected Aha! Trial")
    print("(Explicit only" + ("" if explicit_only else " + implicit") + ")\n")

                       
    def calc_mean_ci(arr):
        n = len(arr)
        mean = np.mean(arr)
        if n < 2:
            return mean, np.nan, np.nan
        sem = np.std(arr, ddof=1) / np.sqrt(n)
        t_crit = stats.t.ppf(0.975, n - 1)
        delta = t_crit * sem
        return mean, mean - delta, mean + delta
   
    for datasetName in datasetLabels:
        if datasetName not in aims_by_dataset_rot or not aims_by_dataset_rot[datasetName]:
            print(f"{datasetName}: No valid Aha! trials detected.\n")
            continue
          
        print(f"=== {datasetName} ===")
        rots = sorted(aims_by_dataset_rot[datasetName].keys(), reverse=True)
      
        for rot in rots:
            if rot == 0:
                continue
            data = aims_by_dataset_rot[datasetName][rot]
            aims_tm1 = np.array(data['tm1'])
            aims_t0 = np.array(data['t0'])
            aims_baseline = np.array(data['baseline'])
            n = len(aims_tm1)
          
            if n < 2:
                print(f" Rotation {int(rot)}°: N = {n} (insufficient for t-tests)\n")
                continue
              
            mean_baseline, low_baseline, high_baseline = calc_mean_ci(aims_baseline)
            mean_tm1, low_tm1, high_tm1 = calc_mean_ci(aims_tm1)
            mean_t0, low_t0, high_t0 = calc_mean_ci(aims_t0)
            target = abs(rot)
          
            t_pre, p_pre = stats.ttest_rel(aims_tm1, aims_baseline)
            t_t0, p_t0 = stats.ttest_1samp(aims_t0, target)
           
            bonf_factor = 1 if datasetName == 'Brudner' else BONFERRONI_FACTOR
            p_pre_corr = min(1.0, p_pre * bonf_factor)
            p_t0_corr = min(1.0, p_t0 * bonf_factor)
          
            print(f" Rotation {int(rot)}° (N = {n} participants)")
            print(f" Baseline (|aim|): M (95% CI) = {mean_baseline:.2f} ({low_baseline:.2f}–{high_baseline:.2f})°")
            print(f" t-1 (|aim|): M (95% CI) = {mean_tm1:.2f} ({low_tm1:.2f}–{high_tm1:.2f})°")
            print(f" → Paired t-test (t-1 vs Baseline): t({n-1}) = {t_pre:+.2f}, raw p = {p_pre:.4f}, corr. p = {p_pre_corr:.4f}{stars(p_pre_corr)}")
            print(f" t (|aim|): M (95% CI) = {mean_t0:.2f} ({low_t0:.2f}–{high_t0:.2f})° vs {int(target)}°")
            print(f" → One-sample t-test (t vs target): t({n-1}) = {t_t0:+.2f}, raw p = {p_t0:.4f}, corr. p = {p_t0_corr:.4f}{stars(p_t0_corr)}\n")
           
            results.append({
                'dataset': datasetName,
                'rotation': int(rot),
                'n': n,
                'mean_baseline': mean_baseline,
                'low_baseline': low_baseline,
                'high_baseline': high_baseline,
                'mean_tm1': mean_tm1,
                'low_tm1': low_tm1,
                'high_tm1': high_tm1,
                'mean_t0': mean_t0,
                'low_t0': low_t0,
                'high_t0': high_t0,
                'target': target,
                'stars_pre': stars(p_pre_corr),
                'stars_post': stars(p_t0_corr),
            })
                        
    if results:
        df = pd.DataFrame(results)
       
                                      
        dataset_order = ['CGVanilla8', 'CGImp8', 'Brudner', 'BondTaylor']
        df['dataset'] = pd.Categorical(df['dataset'], categories=dataset_order, ordered=True)
        df = df.sort_values(['dataset', 'rotation'], ascending=[True, False])
       
                           
        df['Baseline'] = df.apply(lambda row: f"{row.mean_baseline:.2f} ({row.low_baseline:.2f}–{row.high_baseline:.2f})", axis=1)
        df['Pre-Aha!'] = df.apply(lambda row: f"{row.mean_tm1:.2f} ({row.low_tm1:.2f}–{row.high_tm1:.2f}){row.stars_pre}", axis=1)
        df['Aha!'] = df.apply(lambda row: f"{row.mean_t0:.2f} ({row.low_t0:.2f}–{row.high_t0:.2f}){row.stars_post}\n(vs {row.target:.0f}°)", axis=1)
       
        display_df = df[['dataset', 'rotation', 'n', 'Baseline', 'Pre-Aha!', 'Aha!']].copy()
        display_df.columns = ['Dataset', 'Rotation (°)', 'N', 'Baseline (|aim|) M (95% CI)', 'Pre-Aha! (|aim|) M (95% CI)', 'Aha! (|aim|) M (95% CI)']
        display_df = display_df.set_index(['Dataset', 'Rotation (°)'])
       
                                           
        def highlight_sig(val):
            if any(c in val for c in ['*', '**', '***']):
                return 'font-weight: bold; color: #d00000;'
            return ''
       
        caption_title = "Human |Aim| Around Detected Aha! Trial " + ("(Explicit Only)" if explicit_only else "(Explicit + Implicit)")
        styler = display_df.style\
            .set_caption(f"<b>{caption_title}</b><br>"
                         "M (95% CI lower–upper) in degrees. Significance markers on Pre-Aha! (vs Baseline) and Aha! (vs target): "
                         "* p<sub>corr</sub> < 0.05, ** < 0.01, *** < 0.001<br>"
                         "<i>Bonferroni ×5 applied except for Brudner (single rotation group)</i>")\
            .set_table_styles([
                {'selector': 'caption', 'props': 'caption-side: top; font-size: 1.3em; text-align: center;'},
                {'selector': 'th', 'props': 'text-align: center; background-color: #f0f0f0;'},
                {'selector': 'td', 'props': 'text-align: center;'},
            ])\
            .applymap(highlight_sig, subset=['Pre-Aha! (|aim|) M (95% CI)', 'Aha! (|aim|) M (95% CI)'])\
            .format({'N': '{:.0f}'})
       
        display(styler)
    else:
        print("No results to display in table.")
        
analyze_human_aim_aha_by_dataset_rot(explicit_only=True)

In [ ]:
                        
color_map = {90: '#44AA99', 60: '#88CCEE', 45: '#FF9825', 30: '#CC6677', 15: '#AA4499'}
baselineLength = 40
washoutLength = 40
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
def plot_aha_indices_stairs(ylim=None, save_filename=None, explicit_only=False, bin_width=1, max_bin=80, font_scale=1.0):
    """
    Generate SEPARATE stairs plots (one figure per dataset) of the detected Aha! trial indices (t - baseline),
    with step lines colored by rotation magnitude (linewidth=3 for clear visibility).
    """
                   
    base_size =10 * font_scale * 3
    title_size = 10 * font_scale * 2
    label_size = 10 * font_scale * 3
    tick_size = 10 * font_scale * 3
    legend_size = 8 * font_scale
    plt.rcParams.update({'font.size': base_size})
    plt.rcParams.update({'axes.titlesize': title_size})
    plt.rcParams.update({'axes.labelsize': label_size})
    plt.rcParams.update({'xtick.labelsize': tick_size})
    plt.rcParams.update({'ytick.labelsize': tick_size})
    plt.rcParams.update({'legend.fontsize': legend_size})
    datasets = [
        [ssmsAll8CG, qLearnsAll8CG, hmmsAll8CG],            
        [ssmsAll8CGIMP, qLearnsAll8CGIMP, hmmsAll8CGIMP],        
        [ssmsAllBrudner, qLearnsAllBrudner, hmmsAllBrudner],          
        [ssmsAll8BT, qLearnsAll8BT, hmmsAll8BT]             
    ]
    datasetLabels = ['CGVanilla8', 'CGImp8', 'Brudner', 'BondTaylor']
    hist_dfs = {}
    for dsIdx, datasetName in enumerate(datasetLabels):
        ssm_list = datasets[dsIdx][0]
        hmm_list = datasets[dsIdx][2]
        if datasetName == 'Brudner':
            bl = 64
            washoutLength_ds = 64
        else:
            bl = baselineLength
            washoutLength_ds = washoutLength
        aha_per_rot = {}                                       
        rotations = []
        for ssm in ssm_list:
            if hasattr(ssm, 'rotations'):
                rotations.append(np.asarray(ssm.rotations))
            else:
                rotations.append(None)
        for rotIdx, rot_array in enumerate(rotations):
            if rot_array is None or rot_array.size == 0 or rotIdx >= len(hmm_list):
                continue
                                                                                               
            rot_abs = np.abs(rot_array)
            if rot_abs.ndim == 1:
                rot_abs = rot_abs[np.newaxis, :]
            flat = rot_abs.flatten()
            flat = flat[~np.isnan(flat)]
            non_zero = flat[flat > 1]                                       
            if len(non_zero) == 0:
                continue                            
            rounded = np.round(non_zero).astype(int)
            unique, counts = np.unique(rounded, return_counts=True)
            rot_mag = int(unique[np.argmax(counts)])
            if len(unique) > 1:
                print(f"Warning: multiple non-zero rotation magnitudes {unique} in {datasetName} group {rotIdx}. Using most common {rot_mag}°.")
            hmm_fit = hmm_list[rotIdx]
            if not hasattr(hmm_fit, 'human_explicits'):
                continue
            human_explicits_full = np.array(hmm_fit.human_explicits)
            if explicit_only:
                human_abs_full = np.abs(human_explicits_full)
            else:
                if hasattr(hmm_fit, 'human_implicits'):
                    human_implicits_full = np.array(hmm_fit.human_implicits)
                    human_full = human_explicits_full + human_implicits_full
                    human_abs_full = np.abs(human_full)
                else:
                    human_abs_full = np.abs(human_explicits_full)
            n_parts, n_trials = human_abs_full.shape
            min_t = bl + 1
            max_t = n_trials - (washoutLength_ds + min(40, bl))
            if min_t > max_t:
                continue
                                                    
            for part in range(n_parts):
                aim = human_abs_full[part, :]
                                                        
                if rot_array.ndim == 1:
                    part_rot_array = rot_array[part] if part < len(rot_array) else rot_array[0]
                else:
                    part_rot_array = rot_array[part, :]
                min_cost = np.inf
                best_t_part = None
                for t in range(min_t, max_t + 1):
                    if t >= n_trials:
                        continue
                                                                   
                    if len(part_rot_array) > 1 and t < len(part_rot_array):
                        rot_val = part_rot_array[t]
                    else:
                        rot_val = part_rot_array[0] if len(part_rot_array) > 0 else 0.0
                    if np.isnan(rot_val):
                        valid_rots = [r for r in part_rot_array if not np.isnan(r)]
                        if len(valid_rots) == 0:
                            continue
                        rot_val = valid_rots[min(t % len(valid_rots), len(valid_rots)-1)]
                    target_mag = np.abs(rot_val)
                                                  
                    pre_len = min(40, bl)
                    post_len = min(max_t, pre_len)
                    cost = np.nansum(np.maximum(np.abs(human_abs_full[part, bl:t]) - np.nanmean(human_abs_full[part, :bl]), 0)) + (20) * (np.nanmean(np.abs(np.abs(rot_val) - human_abs_full[part, t:min(max_t, t + min(40, bl))])))
                    if cost < min_cost:
                        min_cost = cost
                        best_t_part = t
                if best_t_part is not None:
                    relative_aha = best_t_part - bl
                    aha_per_rot.setdefault(rot_mag, []).append(relative_aha)
                                           
        if not aha_per_rot:
            print(f"No valid Aha! indices for {datasetName}.")
            hist_dfs[datasetName] = pd.DataFrame()
            continue
        data_list = []
        for rot_mag, indices in aha_per_rot.items():
            for val in indices:
                data_list.append({'relative_aha': val, 'rotation': f"{rot_mag}°"})
        df_dataset = pd.DataFrame(data_list)
                                                                                  
        bin_edges = np.arange(-0.5, max_bin + 1.5, 1)
                                                                                  
        max_per_rot = df_dataset.groupby('rotation')['relative_aha'].apply(
            lambda x: np.histogram(x, bins=bin_edges)[0].max()
        )
        max_count = max_per_rot.max()
        ylim_top = max_count + max(5, int(max_count * 0.1))
                               
        present_rots = sorted(df_dataset['rotation'].unique(), key=lambda x: int(x[:-1]), reverse=True)
        palette = {}
        for r in present_rots:
            mag = int(r[:-1])
            if mag in color_map:
                palette[r] = color_map[mag]
            else:
                palette[r] = '#999999'
                print(f"Warning: rotation {mag}° not in standard color_map, using gray.")
        hue_order = present_rots
        fig, ax = plt.subplots(figsize=(12, 6))
                                                                        
        for rot in hue_order:
            sub_df = df_dataset[df_dataset['rotation'] == rot]
            if sub_df.empty:
                continue
            counts, _ = np.histogram(sub_df['relative_aha'], bins=bin_edges)
            ax.stairs(counts, bin_edges, color=palette[rot], linewidth=3.5, label=rot)
        ax.set_xlim(-1, max_bin)
        ax.set_ylim(0, 25)
        ax.set_xlabel('')
        ax.set_ylabel('Count')
        total_n = len(df_dataset)
        ax.set_title(f'Relative Aha! Trial Indices — {datasetName} (N={total_n})')
        ax.axvline(x=0, color='gray', linestyle='--', linewidth=2.5, alpha=0.7)
        ax.legend(title='Rotation Size')
                                              
                                                                                             
                             
                                        
        ax.yaxis.set_major_locator(MultipleLocator(10))
        ax.yaxis.set_minor_locator(AutoMinorLocator(2))
        ax.xaxis.set_major_locator(MultipleLocator(20))
        ax.xaxis.set_minor_locator(AutoMinorLocator(2))
        sns.despine()
        plt.tight_layout()
                              
        if save_filename:
            base, ext = os.path.splitext(save_filename)
            individual_save = f"{base}_{datasetName}{ext}"
            plt.savefig(individual_save, dpi=300, bbox_inches='tight')
            print(f"Saved {individual_save}")
                                                        
        all_aha = df_dataset['relative_aha'].tolist()
        bins = np.arange(0, max_bin + bin_width + 1, bin_width)
        hist, bin_edges_pooled = np.histogram(all_aha, bins=bins)
        summary_data = []
        for i in range(len(hist)):
            summary_data.append({
                'bin_start': bin_edges_pooled[i],
                'bin_end': bin_edges_pooled[i + 1],
                'count': hist[i]
            })
        hist_dfs[datasetName] = pd.DataFrame(summary_data)
    return hist_dfs
df_hists = plot_aha_indices_stairs(explicit_only=True, save_filename=targetFolder + 'ahaIndicesHistogram.svg', bin_width=1, max_bin=80, font_scale=1.5)

In [ ]:

def aha_anova(explicit_only=False, included_datasets=None):
    """
    Perform a two-way ANOVA on the detected relative Aha! trial indices 
    (t - baseline), with factors 'experiment' (dataset) and 'rotation_magnitude' (absolute rotation size in degrees).
    Uses the same Aha! detection process as the original function.
    Only includes participants where Aha! trial t >= baselineLength + 1 and valid.
    
    Parameters:
    - explicit_only: If True, use only human_explicits for aims in detection.
    - included_datasets: List of dataset names to include (e.g., ['CGImp8', 'BondTaylor']).
      If None (default), uses only ['CGImp8', 'BondTaylor'].
    
    Prints:
    - Descriptive statistics (count, mean, std) by experiment and rotation magnitude
    - Overall summary statistics
    - Two-way ANOVA table (Type II sums of squares)
    
    Returns:
    - df_aha: DataFrame with columns ['relative_aha', 'experiment', 'rotation_magnitude'].
    """
    if included_datasets is None:
        included_datasets = ['CGImp8', 'BondTaylor']
    
    data_list = []
    
    datasets = [
        [ssmsAll8CG, qLearnsAll8CG, hmmsAll8CG],                  
        [ssmsAll8CGIMP, qLearnsAll8CGIMP, hmmsAll8CGIMP],         
        [ssmsAllBrudner, qLearnsAllBrudner, hmmsAllBrudner],           
        [ssmsAll8BT, qLearnsAll8BT, hmmsAll8BT]                    
    ]
    datasetLabels = ['CGVanilla8', 'CGImp8', 'Brudner', 'BondTaylor']
    
    for dsIdx, datasetName in enumerate(datasetLabels):
        if datasetName not in included_datasets:
            continue
            
        ssm_list = datasets[dsIdx][0]
        hmm_list = datasets[dsIdx][2]
        rotations = []
        
        if datasetName == 'Brudner':
            bl = 64
            washoutLength = 64
        else:
            bl = 40
            washoutLength = 40
    
        for ssm in ssm_list:
            if hasattr(ssm, 'rotations'):
                rot_array = np.asarray(ssm.rotations)
                rotations.append(rot_array)
            else:
                rotations.append(None)
    
        for rotIdx, rot_array in enumerate(rotations):
            if rot_array is None or rot_array.size == 0 or rotIdx >= len(hmm_list) or rotIdx >= len(ssm_list):
                continue
            ssm_fit = ssm_list[rotIdx]
            hmm_fit = hmm_list[rotIdx]
            if not hasattr(hmm_fit, 'human_explicits'):
                continue
            human_explicits_full = np.array(hmm_fit.human_explicits)
            if explicit_only:
                human_abs_full = np.abs(human_explicits_full)
            else:
                if hasattr(hmm_fit, 'human_implicits'):
                    human_implicits_full = np.array(hmm_fit.human_implicits)
                    human_full = human_explicits_full + human_implicits_full
                    human_abs_full = np.abs(human_full)
                else:
                    human_abs_full = np.abs(human_explicits_full)
            n_parts, n_trials = human_abs_full.shape
            min_t = bl + 1
            max_t = n_trials - (washoutLength + min(40,bl))
            if min_t > max_t:
                continue
            
                                                                                                               
            part = 0
            if rot_array.ndim == 1:
                part_rot_array_temp = rot_array[part]
            else:
                part_rot_array_temp = rot_array[part, :]
            
                                                           
            if np.isscalar(part_rot_array_temp) or len(part_rot_array_temp) == 0:
                rot_val = part_rot_array_temp if not np.isscalar(part_rot_array_temp) or not np.isnan(part_rot_array_temp) else np.nan
            elif min_t < len(part_rot_array_temp):
                rot_val = part_rot_array_temp[min_t]
            else:
                rot_val = part_rot_array_temp[-1] if len(part_rot_array_temp) > 0 else np.nan
            
            rotation_mag = np.abs(rot_val) if not np.isnan(rot_val) else np.nan
            
                                           
            if np.isnan(rotation_mag):
                flat_rots = np.asarray(part_rot_array_temp).flatten()
                valid_rots = flat_rots[~np.isnan(flat_rots)]
                if len(valid_rots) > 0:
                    rotation_mag = np.abs(valid_rots[0])
            
            if np.isnan(rotation_mag) or rotation_mag == 0:
                                                                
                continue
            
                                                           
            for part in range(n_parts):
                aim = human_abs_full[part, :]
      
                                                    
                if rot_array.ndim == 1:
                    part_rot_array = rot_array[part]
                else:
                    part_rot_array = rot_array[part, :]
      
                                                            
                min_cost = np.inf
                best_t_part = None
                for t in range(min_t, max_t + 1):
                    if t >= n_trials:
                        continue
                                            
                    if len(part_rot_array) > 1 and t < len(part_rot_array):
                        rot_val = part_rot_array[t]
                    else:
                        rot_val = part_rot_array[0] if len(part_rot_array) > 0 else 0.0
                    if np.isnan(rot_val):
                        valid_rots = [r for r in part_rot_array if not np.isnan(r)]
                        if len(valid_rots) == 0:
                            continue
                        rot_val = valid_rots[min(t % len(valid_rots), len(valid_rots)-1)]
                                             
                    cost = np.nansum(np.maximum(np.abs(human_abs_full[part, bl:t]) - np.nanmean(human_abs_full[part, :bl]), 0)) + (20) * (np.nanmean(np.abs(np.abs(rot_val) - human_abs_full[part, t:min(max_t, t + min(40, bl))])))
                    if cost < min_cost:
                        min_cost = cost
                        best_t_part = t
      
                if best_t_part is not None:
                    relative_aha = best_t_part - bl
                    data_list.append({
                        'relative_aha': relative_aha,
                        'experiment': datasetName,
                        'rotation_magnitude': rotation_mag
                    })
                    
    if not data_list:
        print("No valid Aha! indices collected.")
        return pd.DataFrame()
        
    df_aha = pd.DataFrame(data_list)
    
                            
    print("Descriptive statistics by experiment and rotation magnitude:")
    desc = df_aha.groupby(['experiment', 'rotation_magnitude'])['relative_aha'].agg(['count', 'mean', 'std'])
    desc = desc.round(1)
    desc = desc.sort_index()                      
    print(desc)
    
    print(f"\nTotal valid Aha! trials: {len(df_aha)}")
    print(f"Overall mean relative Aha! index: {df_aha['relative_aha'].mean():.1f}")
    print(f"Overall median relative Aha! index: {df_aha['relative_aha'].median():.1f}")
    
                                                 
    model = ols('relative_aha ~ C(experiment) * C(rotation_magnitude)', data=df_aha).fit()
    anova_table = anova_lm(model, typ=2)
    
    print("\nTwo-way ANOVA results (Type II sums of squares):")
    print(anova_table.round(4))
    
    return df_aha


               
                                                                                                             

df_aha = aha_anova(included_datasets=['CGImp8', 'BondTaylor'], explicit_only=True)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import matplotlib.colors as mcolors
from scipy.stats import norm
from matplotlib.colors import LogNorm
from scipy import signal
import matplotlib.cm as cm
baselineLength = 40
washoutLength = 40
def darken_color(hex_color, factor=0.7):
    rgb = mcolors.hex2color(hex_color)
    darkened = tuple(c * factor for c in rgb)
    return mcolors.to_hex(darkened)
def compute_expected_mag_slice(policies_slice):
    """
    Vectorized helper function to process multiple 361-length policy vectors:
    - Recenters to -180 to +180 (shift so original index 180 is 0° at new index 180).
    - Folds: p(mag=m) = p(-m) + p(+m) for m=1 to 179, p(180) = p(±180) (once), p(0)=p(0).
    - Computes expected magnitude: dot(magnitudes, folded_probs) for each policy.
    Args:
    - policies_slice: np.array of shape (slice_len, 361) with probs for 0° to 360°.
    Returns:
    - expected_mag_slice: np.array of shape (slice_len,), policy-weighted average aim magnitude (0-180°).
    """
    slice_len = policies_slice.shape[0]
                                                                                           
    relative = np.arange(-180, 181)
    orig_idx = (180 + relative) % 360
    recentered = policies_slice[:, orig_idx]
                                                                   
    folded_probs = np.zeros((slice_len, 181))
    folded_probs[:, 0] = recentered[:, 180]       
    for m in range(1, 181):
        neg_idx = 180 - m               
        pos_idx = 180 + m               
        if m == 180:
                                                         
            folded_probs[:, m] = recentered[:, neg_idx]                                        
        else:
            folded_probs[:, m] = recentered[:, neg_idx] + recentered[:, pos_idx]
                                  
    magnitudes = np.arange(181)
                                        
    expected_mag_slice = np.dot(folded_probs, magnitudes)
    return expected_mag_slice
def plot_human_aim_baseline_aha_next(ylim=None, save_filename=None, explicit_only=False):
    """
    Generate plots of mean absolute human aim (with 95% CI error bands across humans) for:
    - Baseline as a horizontal dashed line.
    - The trial series spanning t-20 to t+19 relative to the detected Aha! trial t (averaged across participants).
    Separate subplots for each dataset, with lines for each rotation.
    Also plots SSM and HMM model explicit aims (dashed lines with 95% CI bands).
    Only includes participants/groups where the Aha! trial t >= baselineLength
    (with valid t-20 to t+19 after baseline where possible, handling out-of-bounds as NaN), and t+19 before the washout phase (last 40 trials).
    The Aha! trial is detected separately for each participant: the post-baseline t (<=200) that minimizes
    the individual cost of sum_{k=-20 to -1} |aim[part,t+k]| + sum_{k=0 to 19} ||rot[part,t]| - |aim[part,t+k]||,
    preferring larger |aim[part,t]| in case of cost ties.
    Requires sufficient valid data around t for cost computation.
    Uses np.nanmean, np.nanstd, etc., to handle NaNs by omitting them.
    Parameters:
    - ylim: Tuple (bottom, top) for y-axis limits; if None, auto-scales to data.
    - save_filename: String for saving the plot; if None, just shows it.
    - explicit_only: If True, use only human_explicits (no +implicits) for aims.
    Returns:
    - df: The DataFrame of collected summary data (means, CIs, n per group).
    """
                                                                       
    summary_points = []
    datasets = [
        [ssmsAll8CG, qLearnsAll8CG, hmmsAll8CG],            
        [ssmsAll8CGIMP, qLearnsAll8CGIMP, hmmsAll8CGIMP],        
        [ssmsAllBrudner, qLearnsAllBrudner, hmmsAllBrudner],                  
        [ssmsAll8BT, qLearnsAll8BT, hmmsAll8BT]             
    ]
    datasetLabels = ['CGVanilla8', 'CGImp8', 'Brudner', 'BondTaylor']
                               
    rotation_colors_dict = {
        -90.0: '#44AA99',
        -60.0: '#88CCEE',
        -45.0: '#FF9825',
        -30.0: '#CC6677',
        -15.0: '#AA4499'
    }
                                               
    ssm_color = 'cyan'
    hmm_color = 'green'
    ssm_border_color = darken_color(ssm_color, factor=0.5)
    hmm_border_color = darken_color(hmm_color, factor=0.5)
    model_linewidth = 1.5
    stroke_linewidth = 2.5
    rel_positions = list(range(-30, 31))                   
    plot_rel_positions = list(range(-20, 20))                    
    for dsIdx, datasetName in enumerate(datasetLabels):
        ssm_list = datasets[dsIdx][0]
        hmm_list = datasets[dsIdx][2]
        rotations = []
        rotation_sizes = []
        if datasetName == 'Brudner':
            bl = 64
            washoutLength = 64
        elif '1' in datasetName:
            bl = 15
            washoutLength = 15
        else:
            bl = 40
            washoutLength = 40
                                                    
                                                       
                                                       
        for ssm in ssm_list:
            if hasattr(ssm, 'rotations'):
                rot_array = np.asarray(ssm.rotations)
                rotations.append(rot_array)
                non_zero_rots = rot_array[rot_array != 0]
                if len(non_zero_rots) > 0:
                    rot_size = float(np.unique(non_zero_rots)[0])
                else:
                    rot_size = 0.0
                rotation_sizes.append(rot_size)
            else:
                rotations.append(None)
                rotation_sizes.append(0.0)
                                                                                                                      
        for rotIdx, (rot_array, rotation_size) in enumerate(zip(rotations, rotation_sizes)):
            if rot_array is None or rotIdx >= len(hmm_list) or rotIdx >= len(ssm_list):
                continue
            if datasetName in ['CGVanilla8', 'CGImp8', 'BondTaylor'] and rotation_size not in [-15.0, -45.0, -90.0]:
                continue
            ssm_fit = ssm_list[rotIdx]
            hmm_fit = hmm_list[rotIdx]
            if not hasattr(hmm_fit, 'human_explicits'):
                                                                                        
                continue
            human_explicits_full = np.array(hmm_fit.human_explicits)
            if explicit_only:
                human_abs_full = np.abs(human_explicits_full)
            else:
                if hasattr(hmm_fit, 'human_implicits'):
                    human_implicits_full = np.array(hmm_fit.human_implicits)
                    human_full = human_explicits_full + human_implicits_full
                    human_abs_full = np.abs(human_full)
                else:
                    human_abs_full = np.abs(human_explicits_full)
            n_parts, n_trials = human_abs_full.shape
                                                                                            
                                   
            baseline_human_abs_list = []
            human_series_list = []
            series_list = []
            n_valid = np.zeros(len(rel_positions))
            average_pmf0 = np.zeros((len(rel_positions), 181))
            average_pmf1 = np.zeros((len(rel_positions), 181))
            valid_parts = 0
            baseline_trial = bl - 1
            if baseline_trial < 0 or baseline_trial >= n_trials:
                                                                                           
                continue
            min_t = bl+1
            max_t = n_trials - (washoutLength + min(40,bl))
            if min_t > max_t:
                                                                                                                
                continue
                                                           
            angles = np.arange(-180, 181)
            orig_idx = (180 + angles) % 360
            magnitudes = np.arange(181)
            for part in range(n_parts):
                aim = human_abs_full[part, :]
                if np.isnan(aim[baseline_trial]):
                    continue
                                                    
                if rot_array.ndim == 1:
                    part_rot_array = rot_array
                else:
                    part_rot_array = rot_array[part, :]
                                                            
                min_cost = np.inf
                best_t_part = None
                best_aim_t = -np.inf
                for t in range(min_t, max_t + 1):
                    pre_aha_trial = t - 1
                    next_trial = t + 1
                    pre_20 = t - min(20,bl)
                    post_19 = t + min(19,max_t-(bl-1))
    
                                                                   
                    if pre_20 < 0 or post_19 >= n_trials:
                        continue                                     
                                            
                    if len(part_rot_array) > 1 and t < len(part_rot_array):
                        rot_val = part_rot_array[t]
                    else:
                        rot_val = part_rot_array[0] if len(part_rot_array) > 0 else 0.0
                    if np.isnan(rot_val):
                        valid_rots = [r for r in part_rot_array if not np.isnan(r)]
                        if len(valid_rots) == 0:
                            continue
                        rot_val = valid_rots[min(t, len(valid_rots)-1)]           
                        if np.isnan(rot_val):
                            continue
                    target_mag = np.abs(rot_val)
           
                    cost = np.nansum(np.maximum(np.abs(human_abs_full[part, bl:t]) - np.nanmean(human_abs_full[part, :bl]), 0)) + (20) * (np.nanmean(np.abs(np.abs(rot_val) - human_abs_full[part, t:min(max_t, t + min(40, bl))])))
                                                             
                    if cost < min_cost or (np.isclose(cost, min_cost) and np.abs(aim[t]) > best_aim_t):
                        min_cost = cost
                        best_t_part = t
                        best_aim_t = np.abs(aim[t])
                if best_t_part is None:
                                                                                                             
                    continue
                                                                        
                human_series_part = np.full(len(rel_positions), np.nan, dtype=float)
                for ii, rel in enumerate(rel_positions):
                    trial_idx = best_t_part + rel
                    if 0 <= trial_idx < n_trials:
                        human_series_part[ii] = aim[trial_idx]
                human_series_list.append(human_series_part)
                                                                 
                hmm_expected_part = np.full(len(rel_positions), np.nan, dtype=float)
                if hasattr(hmm_fit, 'model_predictive_policies') and hasattr(hmm_fit, 'pi_preds'):
                    policies_full = np.array(hmm_fit.model_predictive_policies)
                    pi_preds = np.array(hmm_fit.pi_preds)
                    policies_part = policies_full[part, :, :]
                    state1_probs = pi_preds[part, :, 1]
                    state0_probs = pi_preds[part, :, 0]
                    sigma = 0.0
                    if datasetName == 'BondTaylor':
                        sigma = 0.0
                    elif hasattr(hmm_fit, 'xs') and part < len(hmm_fit.xs):
                        try:
                            sigma = float(hmm_fit.xs[part][0])
                            if np.isnan(sigma):
                                sigma = 0.0
                        except:
                            sigma = 0.0
                    delta = np.zeros(361)
                    delta[180] = 1.0
                                         
                    sigma = max(sigma,0.1)
                                 
                    support_size = max(int(6 * sigma) + 1, 3)
                    if support_size % 2 == 0:
                        support_size += 1
                    half_support = support_size // 2
                    support = np.arange(-half_support, half_support + 1)
                    kernel = norm.pdf(support, 0, sigma)
                    kernel /= kernel.sum()
                    convolved0 = ndimage.convolve1d(delta, kernel, mode='wrap')
                    for ii, rel in enumerate(rel_positions):
                        trial_idx = best_t_part + rel
                        if 0 <= trial_idx < n_trials:
                            policy_single = policies_part[trial_idx, :].reshape(1, -1)
                            state1_p = state1_probs[trial_idx]
                            state0_p = state0_probs[trial_idx]
                            recentered = policy_single[0, orig_idx]
                            if sigma > 0.001:
                                convolved1 = ndimage.convolve1d(recentered, kernel, mode='wrap')
                            else:
                                convolved1 = recentered.copy()
                            convolved0 = np.maximum(convolved0, 0)
                            convolved1 = np.maximum(convolved1, 0)
                            marginal = state0_p * convolved0 + state1_p * convolved1
                            marginal_sum = np.sum(marginal)
                            if marginal_sum > 0:
                                marginal /= marginal_sum
                                                
                            expected_mag = np.dot(marginal, np.abs(angles))
                            hmm_expected_part[ii] = expected_mag
                                                                         
                            state0_comp = (state0_p * convolved0) / marginal_sum if marginal_sum > 0 else np.zeros_like(convolved0)
                            state1_comp = (state1_p * convolved1) / marginal_sum if marginal_sum > 0 else np.zeros_like(convolved1)
                            full_pmf0 = np.zeros(181)
                            full_pmf1 = np.zeros(181)
                            for j in range(361):
                                mag = int(np.abs(angles[j]))
                                full_pmf0[mag] += state0_comp[j]
                                full_pmf1[mag] += state1_comp[j]
                            average_pmf0[ii] += full_pmf0
                            average_pmf1[ii] += full_pmf1
                            n_valid[ii] += 1
                series_list.append(hmm_expected_part)
                baseline_human_abs = aim[baseline_trial]
                baseline_human_abs_list.append(baseline_human_abs)
                valid_parts += 1
                                                                                                                    
                                                                                                                 
                                                                       
            n_series = len(series_list)
            if n_series > 0:
                if np.any(n_valid > 0):
                    average_pmf0 = np.divide(average_pmf0, n_valid[:, None], where=(n_valid[:, None] > 0), out=np.zeros_like(average_pmf0))
                    average_pmf1 = np.divide(average_pmf1, n_valid[:, None], where=(n_valid[:, None] > 0), out=np.zeros_like(average_pmf1))
                                                                                            
                idx_min_plot = rel_positions.index(-20)
                idx_max_plot = rel_positions.index(20)
                plot_range = slice(idx_min_plot, idx_max_plot + 1)
            
                                       
                last_valid_pmf = None
                last_valid_n = 0
                for ii in range(idx_min_plot, idx_max_plot + 1):
                    if n_valid[ii] > 0:
                        last_valid_pmf = average_pmf0[ii]
                        last_valid_n = n_valid[ii]
                    elif last_valid_pmf is not None:
                        average_pmf0[ii] = last_valid_pmf
                        n_valid[ii] = last_valid_n                             
                                        
                last_valid_pmf = None
                last_valid_n = 0
                for ii in range(idx_max_plot, idx_min_plot - 1, -1):
                    if n_valid[ii] > 0:
                        last_valid_pmf = average_pmf0[ii]
                        last_valid_n = n_valid[ii]
                    elif last_valid_pmf is not None:
                        average_pmf0[ii] = last_valid_pmf
                        n_valid[ii] = last_valid_n
                                                            
                for ii in range(len(rel_positions)):
                    if n_valid[ii] == 0:
                        if rel_positions[ii] < -20:
                            average_pmf0[ii] = average_pmf0[idx_min_plot]
                            n_valid[ii] = n_valid[idx_min_plot]
                        elif rel_positions[ii] > 20:
                            average_pmf0[ii] = average_pmf0[idx_max_plot]
                            n_valid[ii] = n_valid[idx_max_plot]
                                 
                last_valid_pmf = None
                last_valid_n = 0
                for ii in range(idx_min_plot, idx_max_plot + 1):
                    if n_valid[ii] > 0:
                        last_valid_pmf = average_pmf1[ii]
                        last_valid_n = n_valid[ii]
                    elif last_valid_pmf is not None:
                        average_pmf1[ii] = last_valid_pmf
                        n_valid[ii] = last_valid_n
                last_valid_pmf = None
                last_valid_n = 0
                for ii in range(idx_max_plot, idx_min_plot - 1, -1):
                    if n_valid[ii] > 0:
                        last_valid_pmf = average_pmf1[ii]
                        last_valid_n = n_valid[ii]
                    elif last_valid_pmf is not None:
                        average_pmf1[ii] = last_valid_pmf
                        n_valid[ii] = last_valid_n
                for ii in range(len(rel_positions)):
                    if n_valid[ii] == 0:
                        if rel_positions[ii] < -20:
                            average_pmf1[ii] = average_pmf1[idx_min_plot]
                            n_valid[ii] = n_valid[idx_min_plot]
                        elif rel_positions[ii] > 20:
                            average_pmf1[ii] = average_pmf1[idx_max_plot]
                            n_valid[ii] = n_valid[idx_max_plot]
                series_array = np.stack(series_list)
                mean_series = np.nanmean(series_array, axis=0)
                std_series = np.nanstd(series_array, axis=0, ddof=1)
                n_per_pos = np.sum(~np.isnan(series_array), axis=0)
                sem_series = np.divide(std_series, np.sqrt(n_per_pos), out=np.full_like(std_series, np.nan), where=(n_per_pos > 1))
                ci_series = 1.96 * sem_series
 
                               
                human_series_array = np.stack(human_series_list)
                mean_human_series = np.nanmean(human_series_array, axis=0)
                std_human_series = np.nanstd(human_series_array, axis=0, ddof=1)
                n_per_pos_human = np.sum(~np.isnan(human_series_array), axis=0)
                sem_human_series = np.divide(std_human_series, np.sqrt(n_per_pos_human), out=np.full_like(std_human_series, np.nan), where=(n_per_pos_human > 1))
                ci_human_series = 1.96 * sem_human_series
 
                baseline_array = np.array(baseline_human_abs_list)
                n_base = np.sum(~np.isnan(baseline_array))
                if n_base > 0:
                    mean_base = np.nanmean(baseline_array)
                    std_base = np.nanstd(baseline_array, ddof=1)
                    sem_base = std_base / np.sqrt(n_base)
                    ci_base = 1.96 * sem_base
                else:
                    mean_base = np.nan
                    ci_base = np.nan
 
                summary_points.append({
                    'rotation': rotation_size,
                    'dataset': datasetName,
                    'mean_baseline': mean_base,
                    'ci_baseline': ci_base,
                    'n_base': n_base,
                    'mean_series': mean_series.tolist(),
                    'ci_series': ci_series.tolist(),
                    'n_series': n_series,
                    'mean_human_series': mean_human_series.tolist(),
                    'ci_human_series': ci_human_series.tolist(),
                    'n_human_series': n_series,
                    'individual_series': [ser.tolist() for ser in human_series_list],
                    'rel_pos': rel_positions,
                    'average_pmf0': average_pmf0.tolist(),
                    'average_pmf1': average_pmf1.tolist(),
                    'n_per_pos': n_valid.tolist()
                })
            else:
                pass                                                                                  
                          
    df = pd.DataFrame(summary_points)
    if df.empty:
        pass                                                             
        return df
                                                        
    global_z_max = 0
    for _, row in df.iterrows():
        for pmf_type in ['0', '1']:
            key = f'average_pmf{pmf_type}'
            if key in row:
                average_pmf = np.array(row[key])
                Z = average_pmf.T
                if Z.size > 0:
                    global_z_max = max(global_z_max, np.max(Z))
    if global_z_max == 0:
        global_z_max = 1
                                           
    fig, axs = plt.subplots(2, 2, figsize=(20, 12))
    axs = axs.flatten()
    for dsIdx, datasetName in enumerate(datasetLabels):
        ax = axs[dsIdx]
        subset = df[df['dataset'] == datasetName]
        if subset.empty:
            ax.set_visible(False)
            continue
                                               
        subset_rots = sorted(subset['rotation'].unique())
                            
        ax.axhline(y=0, color='grey', linestyle='--', linewidth=1, alpha=0.3)
                                                  
        for rot in subset_rots:
            rot_color = rotation_colors_dict.get(rot, 'black')
            ax.axhline(y=-rot, color=rot_color, alpha=0.5, linestyle='--', linewidth=1)
                                              
        yticks = [-i for i in sorted([0] + list(subset_rots))] 
        yticks.append(180)
        yticklabels = [f'{int(abs(t))}°' for t in yticks]
        ax.set_yticks(yticks)
        ax.set_yticklabels(yticklabels)
        for _, row in subset.iterrows():
            rot = row['rotation']
            rot_color = rotation_colors_dict.get(rot, 'black')
            mean_ser = np.array(row['mean_series'])
            ci_ser = np.array(row['ci_series'])
            rel_pos = np.array(row['rel_pos'])                                              
                                                                    
            mean_ser = np.maximum(mean_ser, 0)
                                          
            lower_ci = np.maximum(mean_ser - ci_ser, 0)
            upper_ci = mean_ser + ci_ser
                               
                                                                                                
                                                  
                                                                                     
                                                                        
            contour_norm = LogNorm(vmin=1e-3, vmax=0.05)                                                             
            levels = np.logspace(np.log10(1e-3), np.log10(0.05), 30)                                 
                                    
            rgb = mcolors.hex2color(rot_color)
            cdict = {
                'red': [(0.0, 1.0, 1.0), (1.0, rgb[0], rgb[0])],
                'green': [(0.0, 1.0, 1.0), (1.0, rgb[1], rgb[1])],
                'blue': [(0.0, 1.0, 1.0), (1.0, rgb[2], rgb[2])],
                'alpha': [(0.0, 0.0, 0.0), (1.0, 0.8, 0.8)]
            }
            custom_cmap = LinearSegmentedColormap(f'custom_{rot}', cdict)
                                                         
            rel_pos_array = np.array(rel_pos)
            for pmf_type in ['0', '1']:
                if f'average_pmf{pmf_type}' in row:
                    average_pmf = np.array(row[f'average_pmf{pmf_type}'])
                    Z = average_pmf.T                      
                    Z_smooth = Z                                  
                                          
                    if global_z_max > 0:
                        ax.contourf(rel_pos_array, magnitudes, Z_smooth, levels=levels, cmap=custom_cmap, norm=contour_norm, zorder=3, extend='max')
                                                
                                                                                                                                                                                                    
                                                        
            mean_human_ser = np.maximum(np.array(row['mean_human_series']), 0)
            ci_human_ser = np.array(row['ci_human_series'])
            lower_ci_human = np.maximum(mean_human_ser - ci_human_ser, 0)
            upper_ci_human = mean_human_ser + ci_human_ser
                                  
            ax.scatter(rel_pos[10:51], mean_human_ser[10:51], color=rot_color,edgecolor='black', linewidth=1, s=75, zorder=6, label=f'{rot}°')                     
                                                       
            for ind_series in row['individual_series']:
                ind_series = np.array(ind_series)
                valid_mask = ~np.isnan(ind_series)
                if np.any(valid_mask):
                    valid_x = rel_pos_array[valid_mask]
                    valid_y = ind_series[valid_mask]
                    jitter = np.random.uniform(-0.15, 0.15, size=len(valid_x))
                    x_jittered = valid_x + jitter
                    ax.scatter(x_jittered, valid_y, color=rot_color, alpha=0.15,edgecolor='black', linewidth=0.16, s=10, zorder=5)
         
                           
        ax.set_title(datasetName)
        ax.set_xlabel('Relative Trial (0 = Aha!)',fontsize=14)
        ax.set_ylabel('Mean |Human Aim| (°)',fontsize=14)
        ax.axvline(x=0, color='black', linestyle='-', alpha=0.3, linewidth=1)                  
        ax.legend(fontsize=6)
        if ylim is not None:
            ax.set_ylim(ylim)
        ax.set_xlim(-15.5, 15.5) 
    gray_cdict = {
    'red': [(0.0, 1.0, 1.0), (1.0, 0.2, 0.2)],
    'green': [(0.0, 1.0, 1.0), (1.0, 0.2, 0.2)],
    'blue': [(0.0, 1.0, 1.0), (1.0, 0.2, 0.2)],
    'alpha': [(0.0, 0.0, 0.0), (1.0, 0.8, 0.8)]
}
    gray_cmap = LinearSegmentedColormap('gray_custom', gray_cdict)
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    mappable = cm.ScalarMappable(norm=contour_norm, cmap=gray_cmap)
    cbar = fig.colorbar(mappable, cax=cbar_ax, extend='max')
    cbar.set_label('Prob Density', fontsize=14)
    cbar.ax.tick_params(labelsize=14)
    sns.despine()
    fig.suptitle('Mean |Aim| Around Aha! Trial with 95% CI Bands (Separate Detection per Participant)', fontsize=14)
    plt.tight_layout()
    if save_filename:
        plt.savefig(save_filename, dpi=300, bbox_inches='tight')
               
                   
                                                                                             
                                                  
                                 
                                                                                     
    return df
df_human = plot_human_aim_baseline_aha_next(ylim=(-5,180), explicit_only=True, save_filename=targetFolder+'ahaAlignedAimMagnitudeHMMKernelDIRECTSEPLimited.svg')

In [ ]:

baselineLength = 40
washoutLength = 40

def darken_color(color, factor=0.5):
    """
    Darken a color by multiplying RGB values by the factor.
    """
    try:
        rgb = to_rgb(color)
        darkened = tuple(max(0, c * factor) for c in rgb)
        return darkened
    except:
        return color            

def plot_human_aim_baseline_aha_next(ylim=None, save_filename=None, explicit_only=False):
    """
    Generate plots of mean signed human aim (with 95% CI error bands across humans) for:
    - Baseline as a horizontal dashed line.
    - The trial series spanning t-10 to t+9 relative to the detected Aha! trial t (averaged across participants).
    Separate subplots for each dataset, with lines for each rotation.
    Only includes participants/groups where the Aha! trial t >= baselineLength
    (with valid t-10 to t+9 after baseline where possible, handling out-of-bounds as NaN), and t+9 before the washout phase (last 40 trials).
    The Aha! trial is detected separately for each participant: the post-baseline t (<=200) that minimizes
    the individual cost of |aim[part,t-1]| + max(0, |rot[part,t]| - |aim[part,t]|),
    preferring larger |aim[part,t]| in case of average cost ties.
    Requires sufficient valid data around t for cost computation (up to t-5 to t+4).
    Uses np.nanmean, np.nanstd, etc., to handle NaNs by omitting them.
    Additionally plots the mean of positive aims (>0) and mean of negative aims (<0) as dashed and dotted lines, respectively.
    Parameters:
    - ylim: Tuple (bottom, top) for y-axis limits; if None, auto-scales to data.
    - save_filename: String for saving the plot; if None, just shows it.
    - explicit_only: If True, use only human_explicits (no +implicits) for aims.
    Returns:
    - df: The DataFrame of collected summary data (means, CIs, n per group).
    """
                                                                       
    summary_points = []
    datasets = [
        [ssmsAll1CG, qLearnsAll1CG, hmmsAll1CG],            
        [ssmsAll1ImpCG, qLearnsAll1ImpCG, hmmsAll1ImpCG],        
        [ssmsAll8CG, qLearnsAll8CG, hmmsAll8CG],            
        [ssmsAll8CGIMP, qLearnsAll8CGIMP, hmmsAll8CGIMP],        
        [ssmsAllBrudner, qLearnsAllBrudner, hmmsAllBrudner],                  
        [ssmsAll8BT, qLearnsAll8BT, hmmsAll8BT]             
    ]
    datasetLabels = ['CGVanilla1', 'CGImp1', 'CGVanilla8', 'CGImp8', 'Brudner', 'BondTaylor']
                                                                                             
    rotation_colors = list(reversed(['#44AA99', '#88CCEE', '#FF9825', '#CC6677', '#AA4499']))
    rel_positions = list(range(-20, 20))
    for dsIdx, datasetName in enumerate(datasetLabels):
        ssm_list = datasets[dsIdx][0]
        hmm_list = datasets[dsIdx][2]
        rotations = []
        rotation_sizes = []
        if datasetName == 'Brudner':
            bl = 64
            washoutLength = 64
        elif '1' in datasetName:
            bl = 15
            washoutLength = 15
        else:
            bl = 40
            washoutLength = 40
                                                    
                                                       
                                                       
        for ssm in ssm_list:
            if hasattr(ssm, 'rotations'):
                rot_array = np.asarray(ssm.rotations)
                rotations.append(rot_array)
                non_zero_rots = rot_array[rot_array != 0]
                if len(non_zero_rots) > 0:
                    rot_size = float(np.unique(non_zero_rots)[0])
                else:
                    rot_size = 0.0
                rotation_sizes.append(rot_size)
            else:
                rotations.append(None)
                rotation_sizes.append(0.0)
                                                                                                                      
        for rotIdx, (rot_array, rotation_size) in enumerate(zip(rotations, rotation_sizes)):
            if rot_array is None or rotIdx >= len(hmm_list) or rotIdx >= len(ssm_list):
                continue
            ssm_fit = ssm_list[rotIdx]
            hmm_fit = hmm_list[rotIdx]
            if not hasattr(hmm_fit, 'human_explicits'):
                                                                                        
                continue
            human_explicits_full = np.array(hmm_fit.human_explicits)
            if explicit_only:
                human_aim_full = human_explicits_full
            else:
                if hasattr(hmm_fit, 'human_implicits'):
                    human_implicits_full = np.array(hmm_fit.human_implicits)
                    human_full = human_explicits_full + human_implicits_full
                    human_aim_full = human_full
                else:
                    human_aim_full = human_explicits_full
            n_parts, n_trials = human_aim_full.shape
                                                                                            
                                   
            baseline_human_aim_list = []
            series_list = []
            valid_parts = 0
            baseline_trial = bl - 1
            if baseline_trial < 0 or baseline_trial >= n_trials:
                                                                                           
                continue
            min_t = bl+1 
            max_t = n_trials - (washoutLength + min(40,bl))
            if min_t > max_t:
                                                                                                                
                continue
            human_abs_full = np.abs(human_aim_full)
                                                           
            for part in range(n_parts):
                aim = human_aim_full[part, :]
                if np.isnan(aim[baseline_trial]):
                    continue
    
                                                    
                if rot_array.ndim == 1:
                    part_rot_array = rot_array[part]
                else:
                    part_rot_array = rot_array[part, :]
    
                                                                                                 
                min_cost = np.inf
                best_t_part = None
                best_aim_t = -np.inf
                for t in range(min_t, max_t + 1):
                    pre_aha_trial = t - 1
                    pre_prev_trial = t - 2
                    next_trial = t + 1
                    pre_5 = t - 5
                    post_4 = t + 4
        
                                                                                         
        
                                            
                    if len(part_rot_array) > 1 and t < len(part_rot_array):
                        rot_val = part_rot_array[t]
                    else:
                        rot_val = part_rot_array[0] if len(part_rot_array) > 0 else 0.0
                    if np.isnan(rot_val):
                        valid_rots = [r for r in part_rot_array if not np.isnan(r)]
                        if len(valid_rots) == 0:
                            continue
                        rot_val = valid_rots[t+1]
                        if np.isnan(rot_val):
                            continue
                    target_mag = np.abs(rot_val)
                                                                                                           
                    
                    cost = np.nansum(np.maximum(np.abs(human_abs_full[part, bl:t]) - np.nanmean(human_abs_full[part, :bl]), 0)) + (20) * (np.nanmean(np.abs(np.abs(rot_val) - human_abs_full[part, t:min(max_t, t + min(40, bl))])))
                                                           
                    if cost < min_cost:
                        min_cost = cost
                        best_t_part = t
                        best_aim_t = np.abs(aim[t])
        
                if best_t_part is None:
                                                                                                             
                    continue
        
                                                                           
                series_part = np.full(len(rel_positions), np.nan, dtype=float)
                for ii, rel in enumerate(rel_positions):
                    trial_idx = best_t_part + rel
                    if 0 <= trial_idx < n_trials:
                        series_part[ii] = aim[trial_idx]
                baseline_human_aim = aim[baseline_trial]
        
                baseline_human_aim_list.append(baseline_human_aim)
                series_list.append(series_part)
                valid_parts += 1
        
                                                                                                                    
                                                                                                                 
                                                                       
            n_series = len(series_list)
            if n_series > 0:
                series_array = np.stack(series_list)
                mean_series = np.nanmean(series_array, axis=0)
                std_series = np.nanstd(series_array, axis=0, ddof=1)
                n_per_pos = np.sum(~np.isnan(series_array), axis=0)
                sem_series = np.divide(std_series, np.sqrt(n_per_pos), out=np.full_like(std_series, np.nan), where=(n_per_pos > 1))
                ci_series = 1.96 * sem_series
     
                                                           
                mean_pos_series = np.full(len(rel_positions), np.nan)
                mean_neg_series = np.full(len(rel_positions), np.nan)
                for j in range(len(rel_positions)):
                    col = series_array[:, j]
                    valid = ~np.isnan(col)
                    pos_mask = valid & (col > 0)
                    neg_mask = valid & (col < 0)
                    pos_vals = col[pos_mask]
                    neg_vals = col[neg_mask]
                    if len(pos_vals) > 0:
                        mean_pos_series[j] = np.mean(pos_vals)
                    if len(neg_vals) > 0:
                        mean_neg_series[j] = np.mean(neg_vals)
     
                baseline_array = np.array(baseline_human_aim_list)
                n_base = np.sum(~np.isnan(baseline_array))
                if n_base > 0:
                    mean_base = np.nanmean(baseline_array)
                    std_base = np.nanstd(baseline_array, ddof=1)
                    sem_base = std_base / np.sqrt(n_base)
                    ci_base = 1.96 * sem_base
                else:
                    mean_base = np.nan
                    ci_base = np.nan
     
                summary_points.append({
                    'rotation': rotation_size,
                    'dataset': datasetName,
                    'mean_baseline': mean_base,
                    'ci_baseline': ci_base,
                    'n_base': n_base,
                    'mean_series': mean_series.tolist(),
                    'ci_series': ci_series.tolist(),
                    'mean_pos_series': mean_pos_series.tolist(),
                    'mean_neg_series': mean_neg_series.tolist(),
                    'n_series': n_series,
                    'individual_series': series_list,
                    'rel_pos': rel_positions
                })
            else:
                pass                                                                                  
                          
    df = pd.DataFrame(summary_points)
    if df.empty:
                                                                      
        return df
                                                        
                                                  
    unique_rots = sorted(df['rotation'].unique(), reverse=True)
    rot_to_color_idx = {rot: i for i, rot in enumerate(unique_rots)}
                                           
    fig, axs = plt.subplots(3, 2, figsize=(15, 16))
    axs = axs.flatten()
    for dsIdx, datasetName in enumerate(datasetLabels):
        ax = axs[dsIdx]
        subset = df[df['dataset'] == datasetName]
        if subset.empty:
            ax.set_visible(False)
            continue
                                               
        subset_rots = sorted(subset['rotation'].unique())
                            
        ax.axhline(y=0, color='grey', linestyle='--', linewidth=1, alpha=0.3)
                                                  
        for rot in subset_rots:
            color_idx = rot_to_color_idx[rot]
            rot_color = rotation_colors[color_idx]
            ax.axhline(y=-rot, color=rot_color, alpha=0.5, linestyle='--', linewidth=1)
            ax.axhline(y=rot, color=rot_color, alpha=0.5, linestyle='--', linewidth=1)
                                              
        yticks = [-i for i in sorted([0] + list(subset_rots))]
        yticklabels = [f'{int(t)}°' for t in yticks]
        ax.set_yticks(yticks)
        ax.set_yticklabels(yticklabels)
        for _, row in subset.iterrows():
            rot = row['rotation']
            color_idx = rot_to_color_idx[rot]
            rot_color = rotation_colors[color_idx]
            darker_color = darken_color(rot_color, factor=0.5)
            mean_ser = np.array(row['mean_series'])
            ci_ser = np.array(row['ci_series'])
            mean_pos_ser = np.array(row['mean_pos_series'])
            mean_neg_ser = np.array(row['mean_neg_series'])
            rel_pos = np.array(row['rel_pos'])                                              
                                         
            lower_ci = mean_ser - ci_ser
            upper_ci = mean_ser + ci_ser
                                                     
            ax.scatter(rel_pos, mean_ser, color=rot_color, s=50, zorder=5, label=f'{rot}°',
                       edgecolor=darker_color, linewidth=1.0)
                                    
            ax.fill_between(rel_pos, lower_ci, upper_ci, color=rot_color, alpha=0.2)
                                                       
            for ind_series in row['individual_series']:
                valid_mask = ~np.isnan(ind_series)
                if np.any(valid_mask):
                    valid_x = rel_pos[valid_mask]
                    valid_y = ind_series[valid_mask]
                    jitter = np.random.uniform(-0.3, 0.3, size=len(valid_x))
                    x_jittered = valid_x + jitter
                    ax.scatter(x_jittered, valid_y, color=rot_color, alpha=0.3, s=2, zorder=6)
                                                                       
            valid_pos = ~np.isnan(mean_pos_ser)
            if np.any(valid_pos):
                ax.plot(rel_pos[valid_pos], mean_pos_ser[valid_pos], color=rot_color, linestyle='--', linewidth=2, alpha=0.8,
                        path_effects=[patheffects.Stroke(linewidth=4, foreground=darker_color), patheffects.Normal()])
                                                                       
            valid_neg = ~np.isnan(mean_neg_ser)
            if np.any(valid_neg):
                ax.plot(rel_pos[valid_neg], mean_neg_ser[valid_neg], color=rot_color, linestyle='--', linewidth=2, alpha=0.8,
                        path_effects=[patheffects.Stroke(linewidth=4, foreground=darker_color), patheffects.Normal()])
       
                           
        ax.set_title(datasetName)
        ax.set_xlabel('Relative Trial (0 = Aha!)')
        ax.set_ylabel('Mean Human Aim (°)')
        ax.axvline(x=0, color='black', linestyle='-', alpha=0.3, linewidth=1)                  
                    
        if ylim is not None:
            ax.set_ylim(ylim)
    sns.despine()
    fig.suptitle('Mean Aim Around Aha! Trial with 95% CI Bands (Separate Detection per Participant)', fontsize=14)
    plt.tight_layout()
    if save_filename:
        plt.savefig(save_filename, dpi=300, bbox_inches='tight')
               
                    
                                                                                              
                                                   
                                 
                                                                                      
    return df
df_human = plot_human_aim_baseline_aha_next(ylim=(-150,150), explicit_only=True,save_filename=targetFolder+'ahaAlignedAim.svg')


In [ ]:

                                                                
datasetLabels = ['CGVanilla', 'CGImp', 'Brudner', 'BondTaylor']
datasets = [
    [ssmsAll8CG, qLearnsAll8CG, hmmsAll8CG],            
    [ssmsAll8CGIMP, qLearnsAll8CGIMP, hmmsAll8CGIMP],        
    [ssmsAllBrudner, qLearnsAllBrudner, hmmsAllBrudner],                  
    [ssmsAll8BT, qLearnsAll8BT, hmmsAll8BT]             
]
def plot_confidence_series_by_rotation(ylim=None, save_filename=None):
    """
    Generate line plots of trial-wise model sign confidence (p(negative direction)) for each participant,
    pooled across datasets by rotation size. Includes individual faint lines and thick mean lines per rotation group.
    
    Assumes model_predictive_dirs is an attribute of the HMM fit with shape (n_parts, n_trials, n_dirs).
    [0] corresponds to the probability of the negative (correct) rotation direction.
    
    Plots the FULL series across all trials (no truncation). Assumes single block per fit (full length, e.g., 400 trials).
    Excludes 'BondTaylor', 'CGImp', and 'Brudner' datasets (uses only CGVanilla).
    
    Parameters:
    - ylim: Tuple (bottom, top) for y-axis limits; if None, auto-scales to data.
    - save_filename: String for saving the plot; if None, just shows it.
    
    Returns:
    - df: Long-form DataFrame of collected data (for further analysis if needed).
    """
                                                                                             
    data_points = []
    
    for dsIdx, datasetName in enumerate(datasetLabels):
        if datasetName in ['BondTaylor', 'CGImp', 'Brudner']:
            continue                                                   
        
                                 
        hmm_list = datasets[dsIdx][2]
        
                                                                 
        ssm_list = datasets[dsIdx][0]
        rotations = []
        rotation_sizes = []
        for ssm in ssm_list:
            if hasattr(ssm, 'rotations'):
                rot_array = np.asarray(ssm.rotations)
                rotations.append(rot_array)                                    
                non_zero_rots = rot_array[rot_array != 0]
                if len(non_zero_rots) > 0:
                    rot_size = float(np.unique(non_zero_rots)[0])
                else:
                    rot_size = 0.0
                rotation_sizes.append(rot_size)
            else:
                rotations.append(None)
                rotation_sizes.append(0.0)
 
        for rotIdx, (rot_array, rotation_size) in enumerate(zip(rotations, rotation_sizes)):
            if rot_array is None or rotIdx >= len(hmm_list):
                continue
     
            hmm_fit = hmm_list[rotIdx]
            if not hasattr(hmm_fit, 'model_predictive_dirs'):
                                                                                                            
                continue
          
            predictive_dirs = np.array(hmm_fit.model_predictive_dirs)                                      
     
            n_parts, n_trials, _ = predictive_dirs.shape
     
            for part in range(n_parts):
                p_neg = predictive_dirs[part, :, 0]                                                   
                trial_indices = np.arange(n_trials)
                
                                                                
                part_id = f"{datasetName}_P{part}_R{int(rotation_size)}"
                
                                                                  
                for trial, conf in zip(trial_indices, p_neg):
                    data_points.append({
                        'rotation': rotation_size,
                        'dataset': datasetName,
                        'participant': part_id,
                        'trial': trial,
                        'confidence': conf
                    })
  
                          
    df = pd.DataFrame(data_points)
    if df.empty:
                                                                      
        return df
    
                                                         
    unique_rots = sorted(df['rotation'].unique(), reverse=True)
    rot_to_color_idx = {rot: i for i, rot in enumerate(unique_rots)}
    
                                                                         
    fig, ax = plt.subplots(figsize=(12, 8))
    
                                              
    alpha_individual = 0.3
    linewidth_individual = 0.8
    for rot in unique_rots:
        subset = df[df['rotation'] == rot]
        if subset.empty:
            continue
        color = rotation_colors[rot_to_color_idx[rot]]
        
                                                 
        for part_id, group in subset.groupby('participant'):
            ax.plot(group['trial'], group['confidence'], color=color, alpha=alpha_individual,
                    linewidth=linewidth_individual, label=None)                           
    
                                    
    linewidth_mean = 5.0
    for rot in unique_rots:
        subset = df[df['rotation'] == rot]
        if subset.empty:
            continue
        color = rotation_colors[rot_to_color_idx[rot]]
        
                                                                  
        mean_conf = subset.groupby('trial')['confidence'].mean()
        ax.plot(mean_conf.index, mean_conf.values, color=color, linewidth=linewidth_mean,
                label=f'Mean {int(rot)}°', alpha=1.0)
    
                    
    ax.set_xlabel('Trial Number')
    ax.set_ylabel('Model Sign Confidence (p(Negative Direction))')
    ax.set_title('Full Trial-Wise Confidence Series Pooled by Rotation Size\n(CGVanilla Dataset Only, Individuals + Means)')
    if ylim is not None:
        ax.set_ylim(ylim)
                
                              
    plt.tight_layout()
    if save_filename:
        plt.savefig(save_filename, dpi=300, bbox_inches='tight')
               
    
                                                               
    summary_lengths = df.groupby('rotation')['trial'].max().reset_index()
    summary_lengths.columns = ['rotation', 'max_trials']
                                            
                           
    
    return df

                                                                        
df_series = plot_confidence_series_by_rotation(ylim=(0.45, 1.05),
                                               save_filename=targetFolder + 'full_confidence_series_by_rotation_CGVanilla_only.svg')

                                 
                                           
                                                                     
                                                           
                        
                          

In [ ]:

                                                                
datasetLabels = ['CGVanilla', 'CGImp', 'Brudner', 'BondTaylor']
def plot_confidence_series_by_rotation(ylim=None, save_filename=None):
    """
    Generate line plots of trial-wise model sign confidence (p(negative direction)) for each participant,
    pooled across datasets by rotation size. Includes individual faint lines and thick mean lines per rotation group.
   
    Assumes model_predictive_dirs is an attribute of the HMM fit with shape (n_parts, n_trials, n_dirs).
    [0] corresponds to the probability of the negative (correct) rotation direction.
   
    Plots the FULL series across all trials (no truncation). Assumes single block per fit (full length, e.g., 400 trials).
    Excludes 'CGVanilla', 'Brudner', and 'BondTaylor' datasets (uses only CGImp).
   
    Parameters:
    - ylim: Tuple (bottom, top) for y-axis limits; if None, auto-scales to data.
    - save_filename: String for saving the plot; if None, just shows it.
   
    Returns:
    - df: Long-form DataFrame of collected data (for further analysis if needed).
    """
                                                                                             
    data_points = []
   
    for dsIdx, datasetName in enumerate(datasetLabels):
        if datasetName in ['CGVanilla', 'Brudner', 'BondTaylor']:
            continue                                                      
       
                                 
        hmm_list = datasets[dsIdx][2]
       
                                                                 
        ssm_list = datasets[dsIdx][0]
        rotations = []
        rotation_sizes = []
        for ssm in ssm_list:
            if hasattr(ssm, 'rotations'):
                rot_array = np.asarray(ssm.rotations)
                rotations.append(rot_array)                                   
                non_zero_rots = rot_array[rot_array != 0]
                if len(non_zero_rots) > 0:
                    rot_size = float(np.unique(non_zero_rots)[0])
                else:
                    rot_size = 0.0
                rotation_sizes.append(rot_size)
            else:
                rotations.append(None)
                rotation_sizes.append(0.0)
        for rotIdx, (rot_array, rotation_size) in enumerate(zip(rotations, rotation_sizes)):
            if rot_array is None or rotIdx >= len(hmm_list):
                continue
    
            hmm_fit = hmm_list[rotIdx]
            if not hasattr(hmm_fit, 'model_predictive_dirs'):
                                                                                                            
                continue
         
            predictive_dirs = np.array(hmm_fit.model_predictive_dirs)                                     
    
            n_parts, n_trials, _ = predictive_dirs.shape
    
            for part in range(n_parts):
                p_neg = predictive_dirs[part, :, 0]                                                  
                trial_indices = np.arange(n_trials)
               
                                                                
                part_id = f"{datasetName}_P{part}_R{int(rotation_size)}"
               
                                                                  
                for trial, conf in zip(trial_indices, p_neg):
                    data_points.append({
                        'rotation': rotation_size,
                        'dataset': datasetName,
                        'participant': part_id,
                        'trial': trial,
                        'confidence': conf
                    })
 
                          
    df = pd.DataFrame(data_points)
    if df.empty:
                                                                      
        return df
   
                                                         
    unique_rots = sorted(df['rotation'].unique(), reverse=True)
    rot_to_color_idx = {rot: i for i, rot in enumerate(unique_rots)}
   
                                                                         
    fig, ax = plt.subplots(figsize=(12, 8))
   
                                              
    alpha_individual = 0.3
    linewidth_individual = 0.8
    for rot in unique_rots:
        subset = df[df['rotation'] == rot]
        if subset.empty:
            continue
        color = rotation_colors[rot_to_color_idx[rot]]
       
                                                 
        for part_id, group in subset.groupby('participant'):
            ax.plot(group['trial'], group['confidence'], color=color, alpha=alpha_individual,
                    linewidth=linewidth_individual, label=None)                          
   
                                    
    linewidth_mean = 5.0
    for rot in unique_rots:
        subset = df[df['rotation'] == rot]
        if subset.empty:
            continue
        color = rotation_colors[rot_to_color_idx[rot]]
       
                                                                  
        mean_conf = subset.groupby('trial')['confidence'].mean()
        ax.plot(mean_conf.index, mean_conf.values, color=color, linewidth=linewidth_mean,
                label=f'Mean {int(rot)}°', alpha=1.0)
   
                    
    ax.set_xlabel('Trial Number')
    ax.set_ylabel('Model Sign Confidence (p(Negative Direction))')
    ax.set_title('Full Trial-Wise Confidence Series Pooled by Rotation Size\n(CGImp Dataset Only, Individuals + Means)')
    if ylim is not None:
        ax.set_ylim(ylim)
                
                              
    plt.tight_layout()
    if save_filename:
        plt.savefig(save_filename, dpi=300, bbox_inches='tight')
               
   
                                                               
    summary_lengths = df.groupby('rotation')['trial'].max().reset_index()
    summary_lengths.columns = ['rotation', 'max_trials']
                                            
                           
   
    return df
                                                                            
df_series = plot_confidence_series_by_rotation(ylim=(-0.05, 1.05),
                                               save_filename=targetFolder + 'full_confidence_series_by_rotation_CGImp_only.svg')
                                 
                                           
                                                                     
                                                           
                        
                          

In [ ]:

                                   
                            
def process_policy_to_expected_mag(policy_vec):
    """
    Helper function to process a single 361-length policy vector:
    - Recenters to -180 to +180 (shift so original index 180 is 0° at new index 180).
    - Folds: p(mag=m) = p(-m) + p(+m) for m=1 to 179, p(180) = p(±180) (once), p(0)=p(0).
    - Computes expected magnitude: dot(magnitudes, folded_probs).
    Args:
    - policy_vec: np.array of shape (361,) with probs for 0° to 360°.
    Returns:
    - expected_mag: float, policy-weighted average aim magnitude (0-180°).
    """
                                                                                           
    relative = np.arange(-180, 181)
    orig_idx = (180 + relative) % 360
    orig_idx = orig_idx.astype(int)
    recentered = policy_vec[orig_idx]
                                           
    folded_probs = np.zeros(181)
    folded_probs[0] = recentered[180]       
    for m in range(1, 181):
        neg_idx = 180 - m               
        pos_idx = 180 + m               
        if m == 180:
                                                         
            folded_probs[m] = recentered[neg_idx]                                     
        else:
            folded_probs[m] = recentered[neg_idx] + recentered[pos_idx]
                                  
    magnitudes = np.arange(181)
                        
    expected_mag = np.dot(magnitudes, folded_probs)
    return expected_mag
def compute_rmse(y_true, y_pred):
    """Standard RMSE for magnitudes."""
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    if mask.sum() < 2:
        return np.nan
    yt, yp = y_true[mask], y_pred[mask]
    return np.sqrt(np.mean((yt - yp)**2))
def compute_r2(y_true, y_pred):
    """Standard R² for magnitudes: 1 - (SS_res / SS_tot)."""
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    if mask.sum() < 2:
        return np.nan
    yt, yp = y_true[mask], y_pred[mask]
                             
    ss_res = np.nansum((yt - yp) ** 2)
                                                
    mu = np.nanmean(yt)
    ss_tot = np.nansum((yt - mu) ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 1 else np.nan
rots = list(reversed([15, 30, 45, 60, 90]))                       
datasets = [
    [ssmsAll8CG, qLearnsAll8CG, hmmsAll8CG],            
    [ssmsAll8CGIMP, qLearnsAll8CGIMP, hmmsAll8CGIMP],        
    [ssmsAllBrudner, qLearnsAllBrudner, hmmsAllBrudner],                  
    [ssmsAll8BT, qLearnsAll8BT, hmmsAll8BT]             
]
modelLabels = ['ssm', 'hmm']
datasetLabels = ['CGVanilla', 'CGImp', 'Brudner', 'BondTaylor']
hasImp = [False, True, False, True]
                              
rotationMap = [rots, rots, [30], rots]                      
rows = []
for dsIdx, datasetName in enumerate(datasetLabels):
    fullModelLists = datasets[dsIdx]                                    
    modelLists = [fullModelLists[0], fullModelLists[2]]              
    rotations = rotationMap[dsIdx]
    impFlag = hasImp[dsIdx]
    for rotIdx, rotation in enumerate(rotations):
                                                                         
        fitShellsThisRot = []
        for modelIdx in range(2):
            modelFitList = modelLists[modelIdx]
            if rotIdx < len(modelFitList):
                fitShellsThisRot.append(modelFitList[rotIdx])
            else:
                fitShellsThisRot.append(None)
    
                                                                     
        nParticipants = None
        for fs in fitShellsThisRot:
            if fs is not None:
                                                                              
                if hasattr(fs, 'allAims'):
                    nParticipants = len(fs.allAims)
                elif hasattr(fs, 'human_explicits'):
                    nParticipants = len(fs.human_explicits)
                elif hasattr(fs, 'rmses'):                                 
                    nParticipants = len(fs.rmses)
                break
        if nParticipants is None:
            continue
    
                                                                              
        for pId in range(nParticipants):
            for modelIdx in range(2):
                modelName = modelLabels[modelIdx]
                entry = fitShellsThisRot[modelIdx]
                                                                        
                if entry is None:
                    continue
             
                                                                         
                y_true = np.array([])
             
                                                              
                h_exp = np.array([])
                try:
                    if hasattr(entry, 'allAims'):
                        h_exp = np.array(entry.allAims[pId]).ravel()
                    elif hasattr(entry, 'human_explicits'):
                        h_exp = np.array(entry.human_explicits[pId]).ravel()
                except (IndexError, AttributeError, TypeError):
                    h_exp = np.array([])
             
                y_true = np.abs(h_exp)
             
                if modelName == 'ssm':
                                           
                    m_exp = np.array([])
                    try:
                        if hasattr(entry, 'mOut1'):
                            m_exp = np.array(entry.mOut1[pId]).ravel()
                        elif hasattr(entry, 'mStates'):
                            m_exp = np.array(entry.mStates[pId]).ravel()
                    except (IndexError, AttributeError, TypeError):
                        m_exp = np.array([])
                 
                    y_pred = np.abs(m_exp)
                                        
                elif modelName == 'hmm':
                                                                   
                    y_pred = np.array([])
                    try:
                        if hasattr(entry, 'model_predictive_policies'):
                            policies = np.array(entry.model_predictive_policies)
                            if len(policies.shape) == 3 and policies.shape[0] > pId:
                                pols_this_part = policies[pId]
                               
                                                                           
                                expected_mags = np.array([process_policy_to_expected_mag(pol) for pol in pols_this_part])
                               
                                                                                 
                                state1_probs = np.ones(len(pols_this_part))           
                                if hasattr(entry, 'pi_preds') and pId < len(entry.pi_preds):
                                    pis = np.array(entry.pi_preds[pId])
                                    if pis.shape[1] == 2:
                                        state1_probs = pis[:, 1]
                               
                                y_pred = expected_mags * state1_probs
                                                    
                              
                    except (IndexError, AttributeError, TypeError, ValueError):
                        y_pred = np.array([])
             
                rmseVal = compute_rmse(y_true, y_pred)
                rSqVal = compute_r2(y_true, y_pred)
             
                rows.append({
                    'dataset' : datasetName,
                    'rotation' : rotation,
                    'participantId': pId,
                    'model' : modelName,
                    'hasImp' : impFlag,
                    'rmse' : rmseVal,
                    'rSquared' : rSqVal
                })
                  
df = pd.DataFrame(rows)
df = df[['dataset', 'rotation', 'participantId', 'model', 'hasImp', 'rmse', 'rSquared']]
df = df.sort_values(['dataset', 'rotation', 'participantId', 'model']).reset_index(drop=True)
df_clean = df.dropna()
                                                                  
                                                                                 
                                                                  
print("\n" + "="*80)
print("SUMMARY STATISTICS (mean ± std) for RMSE and R²")
print("="*80)
summary = (df.groupby(['dataset', 'rotation', 'model'])
             .agg(
                 rmse_mean = ('rmse', 'mean'),
                 rmse_std = ('rmse', 'std'),
                 rSquared_mean = ('rSquared', 'mean'),
                 rSquared_std = ('rSquared', 'std'),
                 n_participants = ('participantId', 'nunique')
             )
             .round(4))
                 
def format_mean_std(row, col_mean, col_std):
    return f"{row[col_mean]:.4f} ± {row[col_std]:.4f}"
summary['RMSE'] = summary.apply(lambda row: format_mean_std(row, 'rmse_mean', 'rmse_std'), axis=1)
summary['R²'] = summary.apply(lambda row: format_mean_std(row, 'rSquared_mean', 'rSquared_std'), axis=1)
summary['N'] = summary['n_participants']
                                                
pretty_summary = summary[['RMSE', 'R²', 'N']].reset_index()
print(pretty_summary.to_string(index=False))
                                                                  
                                   
                                                                  
print("\nFirst 12 rows of the full dataframe:")                        
print(df.head(12))
print(f"\nTotal rows: {len(df)}")
print(f"Shape: {df.shape}")
print("\nExample — Participant 0 in CGVanilla, rotation 90:")
print(df.query("dataset == 'CGVanilla' and rotation == 90 and participantId == 0"))
                                                                
                                                                               
                                                                
                                                                
                       
                                                                
summary_stats = []
detailed_stats = []
for rot in [90, 60, 45, 30, 15]:
    subset = df_clean[df_clean['rotation'] == rot]
    if len(subset) == 0:
        continue
               
    for model in ['ssm', 'hmm']:
        r2_mean = subset[subset['model'] == model]['rSquared'].mean()
        r2_std = subset[subset['model'] == model]['rSquared'].std()
        rmse_mean = subset[subset['model'] == model]['rmse'].mean()
        rmse_std = subset[subset['model'] == model]['rmse'].std()
        n = len(subset[subset['model'] == model])
    
        summary_stats.append({
            'Rotation': rot,
            'Model': model.upper(),
            'N': n,
            'R²': f"{r2_mean:.3f} ± {r2_std:.3f}",
            'RMSE': f"{rmse_mean:.3f} ± {rmse_std:.3f}"
        })
                                      
    pivot = subset.pivot_table(index='participantId', columns='model', values=['rSquared','rmse']).dropna()
    if len(pivot) < 3:
        continue
    for metric, values_col, better_higher in [('R²', 'rSquared', True), ('RMSE', 'rmse', False)]:
        a, b = 'ssm', 'hmm'
        x = pivot[values_col][a]
        y = pivot[values_col][b]
        t, p = stats.ttest_rel(x, y)
        diff = x - y if better_higher else y - x
        d = diff.mean() / diff.std() if diff.std() != 0 else 0
        winner = a.upper() if d > 0 else b.upper()
    
        detailed_stats.append({
            'Rotation': rot,
            'Metric': metric,
            'Comparison': f"{a.upper()} vs {b.upper()}",
            't': round(t, 3),
            'df': len(x)-1,
            'p': p,
            'd': round(d, 3),
            'Winner': winner
        })
            
summary_df = pd.DataFrame(summary_stats)
stats_df = pd.DataFrame(detailed_stats)
                                                
def holm_correct(g):
    if len(g) <= 1:
        g['p_corr'] = g['p']
        return g
    reject, p_corr, _, _ = multipletests(g['p'], method='holm')
    g = g.copy()
    g['p_corr'] = p_corr
    return g
stats_df = stats_df.groupby(['Rotation','Metric']).apply(holm_correct).reset_index(drop=True)
stats_df['sig'] = stats_df['p_corr'].apply(lambda p: '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '')
                                                                
                  
                                                                
print("="*100)
print("MEAN ± SD (ALL DATASETS POOLED)")
print("="*100)
print(summary_df.sort_values(['Rotation','Model']).to_string(index=False))
print("\n" + "="*100)
print("PAIRED t-TESTS (Holm-corrected) — Positive d = first model better (R²) or lower error (RMSE)")
print("="*100)
print(stats_df[['Rotation','Metric','Comparison','t','df','p_corr','d','Winner','sig']].round(4))
print("="*100)
                                                                
                               
                                                                
sns.set_style("white")
sns.set_context("paper", font_scale=1.6)
palette = {'ssm': '#2E86AB', 'hmm': '#F18F01'}                 
rot_order = list(reversed([90, 60, 45, 30, 15]))
fig_r2, ax_r2 = plt.subplots(figsize=(14, 6))
sns.boxplot(data=df_clean, x="rotation", y="rSquared", hue="model",
            order=rot_order, hue_order=['ssm', 'hmm'],
            palette=palette, width=0.6, linewidth=1.8,
            showfliers=False,
            boxprops=dict(edgecolor='black', alpha=0.3),
            medianprops=dict(color='white', linewidth=3),
            ax=ax_r2)
sns.stripplot(data=df_clean, x="rotation", y="rSquared", hue="model",
              order=rot_order, hue_order=['ssm', 'hmm'],
              palette=palette, size=4.5, dodge=True,
              jitter=0.25, alpha=0.75, linewidth=0.8, edgecolor="black",
              ax=ax_r2)
           
max_r2 = df_clean['rSquared'].max()
y_star_r2 = max_r2 + 0.08
for i, rot in enumerate(rot_order):
    comp = 'SSM vs HMM'
    star_row = stats_df[(stats_df['Rotation']==rot) &
                       (stats_df['Metric']=='R²') &
                       (stats_df['Comparison']==comp)]
    if not star_row.empty and star_row['sig'].iloc[0]:
        ax_r2.text(i, y_star_r2, star_row['sig'].iloc[0],
                   ha='center', va='bottom', fontsize=17, fontweight='bold', color='black')
ax_r2.set_xticklabels([f"{r}°" for r in rot_order])
ax_r2.set_xlabel("")
ax_r2.set_ylabel("Model Fit (R²)")
ax_r2.set_ylim((-3, 1.32))
ax_r2.legend(title="Model", loc="upper center", ncol=2,
             bbox_to_anchor=(0.5, 0.94), frameon=False)
fig_r2.subplots_adjust(top=0.86)
fig_r2.suptitle("Best-Fitting Model by Rotation Size (R²)\n(All Datasets Pooled)", fontsize=21, y=0.98, fontweight='bold')
fig_r2.savefig(targetFolder+"MAGNITUDE_R2_with_stars_and_stats.svg", dpi=600, bbox_inches='tight')
           
                                                                
                                 
                                                                
fig_rmse, ax_rmse = plt.subplots(figsize=(14, 6))
sns.boxplot(data=df_clean, x="rotation", y="rmse", hue="model",
            order=rot_order, hue_order=['ssm', 'hmm'],
            palette=palette, width=0.6, linewidth=1.8,
            showfliers=False,
            boxprops=dict(edgecolor='black', alpha=0.3),
            medianprops=dict(color='white', linewidth=3),
            ax=ax_rmse)
sns.stripplot(data=df_clean, x="rotation", y="rmse", hue="model",
              order=rot_order, hue_order=['ssm', 'hmm'],
              palette=palette, size=4.5, dodge=True,
              jitter=0.25, alpha=0.75, linewidth=0.8, edgecolor="black",
              ax=ax_rmse)
           
max_rmse = df_clean['rmse'].max()
y_star_rmse = max_rmse * 1.15
for i, rot in enumerate(rot_order):
    comp = 'SSM vs HMM'
    star_row = stats_df[(stats_df['Rotation']==rot) &
                       (stats_df['Metric']=='RMSE') &
                       (stats_df['Comparison']==comp)]
    if not star_row.empty and star_row['sig'].iloc[0]:
        ax_rmse.text(i, y_star_rmse, star_row['sig'].iloc[0],
                     ha='center', va='bottom', fontsize=17, fontweight='bold', color='black')
ax_rmse.set_xticklabels([f"{r}°" for r in rot_order])
ax_rmse.set_xlabel("")
ax_rmse.set_ylabel("Prediction Error (RMSE)\n(lower = better)")
                                       
ax_rmse.legend(title="Model", loc="upper center", ncol=2,
               bbox_to_anchor=(0.5, 0.94), frameon=False)
fig_rmse.subplots_adjust(top=0.86)
fig_rmse.suptitle("Model Magnitude Prediction Error by Rotation Size (RMSE)\n(All Datasets Pooled)", fontsize=21, y=0.98, fontweight='bold')
fig_rmse.savefig(targetFolder+"MAGNITUDE_RMSE_with_stars_and_stats.svg", dpi=600, bbox_inches='tight')
           

In [ ]:

                                               
                                               
                                                                                       
                
datasetLabels = ['CGVanilla', 'CGImp', 'Brudner', 'BondTaylor']
                                   
hasImp = [False, True, False, True]
                                                                                                 
rots = [90, 60, 45, 30, 15]                             
rotationMap = [rots, rots, [30], rots]                  
                                                                                          
datasets = [
[ssmsAll8CG, qLearnsAll8CG, hmmsAll8CG],            
[ssmsAll8CGIMP, qLearnsAll8CGIMP, hmmsAll8CGIMP],        
[ssmsAllBrudner, qLearnsAllBrudner, hmmsAllBrudner],          
[ssmsAll8BT, qLearnsAll8BT, hmmsAll8BT]             
]
                                               
                                                
plot_dir = 'indiPlots'
if not os.path.exists(plot_dir):
    os.makedirs(plot_dir)
print(f"Saving plots to: {plot_dir}")
def normalize_angle(angle):
    """
    Normalize angle(s) to [-180, 180).
    - Works on scalars or arrays.
    - Handles positives/negatives/multi-turns.
    - Preserves NaN.
    """
    if np.isscalar(angle):
        if np.isnan(angle):
            return np.nan
                                                      
        angle = angle - 360 * np.round(angle / 360)
        return ((angle + 180) % 360) - 180
    else:
        normalized = np.full_like(angle, np.nan)
        mask = ~np.isnan(angle)
                       
        unwrapped = angle[mask] - 360 * np.round(angle[mask] / 360)
        normalized[mask] = ((unwrapped + 180) % 360) - 180
        return normalized
def generate_per_participant_plots(save_dir=plot_dir, debug=False, max_participants=10, num_samples=200):
    """
    Generate per-participant plots for each dataset/rotation (first {max_participants} only).
    - SINGLE panel: Binned HMM policy heatmap (mixture of state0 and state1, convolved, sampled and binned) as background (trials x bin_width angle bins -180° to +180°).
    Saves each as {datasetName}_rot{rotation}_p{participantId}.svg.
    """
                                  
    if 'datasets' not in globals():
        raise ValueError("datasets global not defined! Load your model lists first.")
    for dsIdx, datasetName in enumerate(datasetLabels):
        modelLists = datasets[dsIdx]                                    
        rotations = rotationMap[dsIdx]
        impFlag = hasImp[dsIdx]
        for rotIdx, rotation in enumerate(rotations):
                                        
            ssm_fit = modelLists[0][rotIdx] if rotIdx < len(modelLists[0]) else None
            hmm_fit = modelLists[2][rotIdx] if rotIdx < len(modelLists[2]) else None
  
            if ssm_fit is None or hmm_fit is None:
                if debug: print(f"Skipping {datasetName} rot{rotation}: Missing SSM or HMM fit.")
                continue
  
                                                                      
            n_participants = None
            if hasattr(ssm_fit, 'allAims'):
                n_participants = len(ssm_fit.allAims)
            elif hasattr(ssm_fit, 'human_explicits'):
                n_participants = len(ssm_fit.human_explicits)
            elif hasattr(hmm_fit, 'human_explicits'):
                n_participants = len(hmm_fit.human_explicits)
            elif hasattr(hmm_fit, 'allAims'):
                n_participants = len(hmm_fit.allAims)
            if n_participants is None:
                if debug: print(f"Skipping {datasetName} rot{rotation}: No participant data.")
                continue
  
                                                          
            n_participants = min(max_participants, n_participants)
            human_abs_full = np.abs(hmm_fit.human_explicits)
            for pId in range(n_participants):
                if debug: print(f"Generating plot for {datasetName}, rot{rotation}, p{pId}...")
      
                                                                                      
                human_aims = np.array([])
                try:
                    if hasattr(ssm_fit, 'allAims'):
                        human_aims = np.array(ssm_fit.allAims[pId]).ravel()
                    elif hasattr(ssm_fit, 'human_explicits'):
                        human_aims = np.array(ssm_fit.human_explicits[pId]).ravel()
                    elif hasattr(hmm_fit, 'human_explicits'):
                        human_aims = np.array(hmm_fit.human_explicits[pId]).ravel()
                    elif hasattr(hmm_fit, 'allAims'):
                        human_aims = np.array(hmm_fit.allAims[pId]).ravel()
                    human_aims = human_aims % 360.0                        
                except (IndexError, AttributeError, TypeError):
                    human_aims = np.array([])
      
                                                                        
                n_trials = len(human_aims)
                if n_trials == 0:
                    if debug: print(f" No human aims data for p{pId}; skipping.")
                    continue
      
                                                                              
                pi_preds = np.array([])
                try:
                    pi_preds_full = np.array(hmm_fit.pi_preds)
                    pi_preds = pi_preds_full[pId, :n_trials, :]                    
                except (IndexError, AttributeError, TypeError, ValueError):
                    pi_preds = np.zeros((n_trials, 2))
      
                                            
                if pi_preds.shape != (n_trials, 2):
                    if len(pi_preds) == 0:
                        pi_preds = np.zeros((n_trials, 2))
                    else:
                                         
                        if len(pi_preds) > n_trials:
                            pi_preds = pi_preds[:n_trials]
                        else:
                            pi_preds = np.pad(pi_preds, ((0, n_trials - len(pi_preds)), (0,0)), mode='constant')
      
                state0_probs = pi_preds[:, 0]
                state1_probs = pi_preds[:, 1]
      
                                                             
                human_imps = np.array([])
                if impFlag:
                    try:
                        if hasattr(ssm_fit, 'allImps'):
                            human_imps = np.array(ssm_fit.allImps[pId]).ravel()
                        elif hasattr(ssm_fit, 'human_implicits'):
                            human_imps = np.array(ssm_fit.human_implicits[pId]).ravel()
                        elif hasattr(hmm_fit, 'human_implicits'):
                            human_imps = np.array(hmm_fit.human_implicits[pId]).ravel()
                        elif hasattr(hmm_fit, 'allImps'):
                            human_imps = np.array(hmm_fit.allImps[pId]).ravel()
                        human_imps = human_imps % 360.0
                        human_imps = human_imps[:n_trials]                               
                                         
                        human_imps[human_imps == 0] = np.nan
                    except (IndexError, AttributeError, TypeError):
                        human_imps = np.array([])
                else:
                    human_imps = np.array([])
      
                                                                      
                hmm_imps = np.array([])
                if hasattr(hmm_fit, 'model_implicits'):
                    try:
                        hmm_imps = np.array(hmm_fit.model_implicits[pId]).ravel()
                        hmm_imps = hmm_imps % 360.0
                        hmm_imps = hmm_imps[:n_trials]
                    except (IndexError, AttributeError, TypeError):
                        hmm_imps = np.array([])
      
                                                            
                ssm_aims = np.array([])
                try:
                    if hasattr(ssm_fit, 'mOut1'):
                        ssm_aims = np.array(ssm_fit.mOut1[pId]).ravel()
                    elif hasattr(ssm_fit, 'mStates'):
                        ssm_aims = np.array(ssm_fit.mStates[pId]).ravel()
                    ssm_aims = ssm_aims % 360.0
                    ssm_aims = ssm_aims[:n_trials]
                except (IndexError, AttributeError, TypeError):
                    ssm_aims = np.array([])
      
                                                                 
                ssm_imps = np.array([])
                if impFlag:
                    try:
                        if hasattr(ssm_fit, 'mOut2'):
                            ssm_imps = np.array(ssm_fit.mOut2[pId]).ravel()
                            ssm_imps = ssm_imps % 360.0
                            ssm_imps = ssm_imps[:n_trials]
                    except (IndexError, AttributeError, TypeError):
                        ssm_imps = np.array([])
      
                                                                                   
                policies_part = np.zeros((n_trials, 361))                           
                try:
                    policies_full = np.array(hmm_fit.model_predictive_policies)
                                                                                     
                    policies_part = policies_full[pId, :n_trials]
                except (IndexError, AttributeError, TypeError, ValueError):
                    policies_part = np.zeros((n_trials, 361))                 
      
                if debug:
                    print(f" human_aims len: {len(human_aims)}, human_imps len: {len(human_imps)}, hmm_imps len: {len(hmm_imps)}")
                    print(f" ssm_aims len: {len(ssm_aims)}, ssm_imps len: {len(ssm_imps)}, policies shape: {policies_part.shape}")
                    print(f" state0_probs mean: {np.mean(state0_probs):.3f}, state1_probs mean: {np.mean(state1_probs):.3f}")
      
                                                                  
                sigma = 0.0
                if hasattr(hmm_fit, 'xs') and pId < len(hmm_fit.xs):
                    try:
                        sigma = float(hmm_fit.xs[pId][0])
                        if np.isnan(sigma):
                            sigma = 0.0
                    except (ValueError, TypeError, IndexError):
                        sigma = 0.0
          
                                                                            
                use_noise = sigma > 0 and datasetName != 'BondTaylor'
          
                                                                                               
                fine_angles = np.arange(-180, 181)
                orig_idx = ((180 + fine_angles) % 360).astype(int)
                kernel = None
                if use_noise:
                    support_size = int(6 * sigma) + 1                        
                    if support_size % 2 == 0:
                        support_size += 1                                 
                    half_support = support_size // 2
                    support = np.arange(-half_support, half_support + 1, dtype=float)
                    kernel = norm_dist.pdf(support, 0, sigma)
                    kernel /= kernel.sum()                        
                                                                
                zero_idx = np.argwhere(fine_angles == 0)[0, 0] if np.any(fine_angles == 0) else len(fine_angles) // 2
                delta_probs = np.zeros(len(fine_angles))
                delta_probs[zero_idx] = 1.0
                if use_noise:
                    convolved0 = ndimage.convolve1d(delta_probs, kernel, mode='wrap')
                else:
                    convolved0 = delta_probs
                                                           
                human_aims_signed = normalize_angle(human_aims)
                human_imps_signed = normalize_angle(human_imps) if len(human_imps) > 0 else np.array([])
                hmm_imps_signed = normalize_angle(hmm_imps) if len(hmm_imps) > 0 else np.array([])
                ssm_aims_signed = normalize_angle(ssm_aims) if len(ssm_aims) > 0 else np.array([])
                ssm_imps_signed = normalize_angle(ssm_imps) if len(ssm_imps) > 0 else np.array([])
                                                    
                aha_trial = None
                baseline_length = 64 if datasetName == 'Brudner' else 40
                washout_length = baseline_length
                bl = baseline_length
                baseline_trial = bl - 1
                min_t = bl + 1
                max_t = n_trials - (washout_length + min(40,bl))
                aim = np.abs(human_aims_signed)
                rot_part = None
                
                try:
                    rot_full = np.asarray(ssm_fit.rotations)
                    if rot_full.ndim == 2 and rot_full.shape[0] == n_participants:
                        rot_part = rot_full[pId, :n_trials]
                    else:
                        rot_part = np.asarray(ssm_fit.rotations[pId])[:n_trials]
                    rot_part = normalize_angle(rot_part)
                except (AttributeError, IndexError, ValueError):
                    rot_part = np.full(n_trials, rotation)
                if min_t <= max_t:
                    min_cost = np.inf
                    best_t_part = None
                    best_aim_t = -np.inf
                    for t in range(min_t, max_t + 1):
                                                
                        if len(rot_part) > t:
                            rot_val = rot_part[t]
                        else:
                            rot_val = rotation           
                        target_mag = np.abs(rot_val)
                        pre_aims = aim[t-bl:t]
                        post_aims = aim[t:t+bl]
                        part = pId
                        cost = np.nansum(np.maximum(np.abs(human_abs_full[part, bl:t]) - np.nanmean(human_abs_full[part, :bl]), 0)) + (20) * (np.nanmean(np.abs(np.abs(rot_val) - human_abs_full[part, t:min(max_t, t + min(40, bl))])))
                        if cost < min_cost:
                            min_cost = cost
                            best_t_part = t
                            best_aim_t = np.abs(human_aims_signed[t])
                    aha_trial = best_t_part
                if debug and aha_trial is not None:
                    print(f" Human-based Aha! trial: {aha_trial - baseline_length}")
      
                                           
                bin_width = abs(rotation) / 15.0
                num_coarse_bins = int(360 / bin_width) + 1
                bin_edges = np.linspace(-180 - bin_width / 2.0, 180 + bin_width / 2.0, num_coarse_bins + 1)
                coarse_centers = np.linspace(-180, 180, num_coarse_bins)
      
                                                          
                sample_counts_matrix = np.zeros((n_trials, num_coarse_bins))
                for t in range(n_trials):
                    state0_prob = state0_probs[t]
                                                                          
                    recentered_probs = policies_part[t][orig_idx]
      
                                                                                    
                    if use_noise:
                        convolved1 = ndimage.convolve1d(recentered_probs, kernel, mode='wrap')
                    else:
                        convolved1 = recentered_probs
                    
                    convolved1 = np.maximum(convolved1, 0)
                    convolved0 = np.maximum(convolved0, 0)
                                                                                                
                    num_state0 = np.random.binomial(num_samples, state0_prob)
                    num_state1 = num_samples - num_state0
      
                    counts_t = np.zeros(num_coarse_bins)
      
                                                           
                    if num_state0 > 0:
                        if convolved0.sum() > 0:
                            samples0 = np.random.choice(fine_angles, size=num_state0, p=convolved0 / convolved0.sum())
                        else:
                            samples0 = np.random.uniform(-180, 180, num_state0)
                        bin_indices0 = np.digitize(samples0, bin_edges) - 1
                        bin_indices0 = np.clip(bin_indices0, 0, num_coarse_bins - 1)
                        counts_t += np.bincount(bin_indices0, minlength=num_coarse_bins)
      
                                                           
                    if num_state1 > 0:
                        if convolved1.sum() > 0:
                            samples1 = np.random.choice(fine_angles, size=num_state1, p=convolved1 / convolved1.sum())
                        else:
                            samples1 = np.random.uniform(-180, 180, num_state1)
                        bin_indices1 = np.digitize(samples1, bin_edges) - 1
                        bin_indices1 = np.clip(bin_indices1, 0, num_coarse_bins - 1)
                        counts_t += np.bincount(bin_indices1, minlength=num_coarse_bins)
      
                    sample_counts_matrix[t, :] = counts_t
      
                                                                                 
                fig, ax = plt.subplots(1, 1, figsize=(14, 8), constrained_layout=True)
                ax.set_title(f'{datasetName} | Rot {rotation}° | Participant {pId}', fontsize=14)
                ax.set_facecolor('white')                               
      
                                           
                sample_counts_masked = np.ma.masked_where(sample_counts_matrix == 0, sample_counts_matrix)
      
                                                                  
                all_counts = sample_counts_masked.compressed()
                vmax = np.percentile(all_counts, 95) if len(all_counts) > 0 else num_samples
                if vmax < 200:
                    vmax = 400
      
                                                                                           
                proportion_matrix = sample_counts_matrix / num_samples
                proportion_masked = np.ma.masked_where(sample_counts_matrix == 0, proportion_matrix)
      
                                                                                                    
                im = ax.imshow(proportion_masked.T, extent=[0, n_trials, -180, 180], origin='lower',
                               aspect='auto', cmap='viridis', norm=PowerNorm(gamma=0.5, vmin=0, vmax=(vmax / 3) / num_samples),
                               alpha=1, zorder=2)
                
                                                        
                cbar = fig.colorbar(im, ax=ax, orientation='vertical')
                cbar.set_label('Proportion of samples per bin', rotation=270, labelpad=25)
                
                                                                        
                ax.axhline(y=rotation, color='grey', linestyle='--', zorder=1, alpha=0.7,linewidth=3)
                ax.axhline(y=-rotation, color='grey', linestyle='--', zorder=1, alpha=0.7,linewidth=3)
                ax.axhline(y=0, color='grey', linestyle='--', zorder=1, alpha=0.7,linewidth=3)
                baseline_length = 64 if datasetName == 'Brudner' else 40
                washout_length = baseline_length
                ax.axvline(x=baseline_length, color='grey', linestyle='--', zorder=1, alpha=0.7,linewidth=3)
                if n_trials > baseline_length + washout_length:
                    ax.axvline(x=n_trials - washout_length, color='grey', linestyle='--', zorder=1, alpha=0.7,linewidth=3)
      
                                                                           
                washout_start = n_trials - baseline_length
                if True:                                                                                   
                    aha_line = ax.axvline(x=aha_trial, color='cyan', linestyle='-', linewidth=6, zorder=3, alpha=0.8)
                                                           
                    aha_line.set_path_effects([withStroke(linewidth=10, foreground='darkcyan')])
      
                                                                   
                trials = np.arange(n_trials)
                linewidth = 4.5
                ax.scatter(trials, human_aims_signed, color='#ff2400', s=180, alpha=0.8, label='Human Aims (Explicit)', zorder=8,edgecolor='darkred', linewidth=3)
                if impFlag and len(human_imps_signed) > 0:
                    nan_mask = ~np.isnan(human_imps_signed)
                    ax.scatter(trials[nan_mask], human_imps_signed[nan_mask], color='blue', s=120, alpha=0.8, label='Human Implicits', zorder=8,edgecolor='darkblue', linewidth=5)
                if len(hmm_imps_signed) > 0:
                    hmm_line = ax.plot(trials, hmm_imps_signed, color='green', ls='--', linewidth=linewidth, alpha=0.9, label='HMM Implicits', zorder=10)[0]
                    hmm_line.set_path_effects([withStroke(linewidth=linewidth*2, foreground='#00ff00')])                        
                if len(ssm_aims_signed) > 0:
                    ssm_aims_line = ax.plot(trials, ssm_aims_signed, color='magenta', linewidth=linewidth, alpha=0.9, label='SSM Aims', zorder=7)[0]
                    ssm_aims_line.set_path_effects([withStroke(linewidth=linewidth*2, foreground='purple')])
                if impFlag and len(ssm_imps_signed) > 0:
                    ssm_imps_line = ax.plot(trials, ssm_imps_signed, color='#cc5500', ls='--', linewidth=linewidth, alpha=0.9, label='SSM Implicits', zorder=9)[0]                      
                    ssm_imps_line.set_path_effects([withStroke(linewidth=linewidth*2, foreground='#ffaa00')])                         
      
                                             
                if True:                                                                                   
                    ax.set_ylim(-2.1*abs(rotation), 2.1*abs(rotation))
                    y_top = ax.get_ylim()[1]
                    y_aha_text = y_top * 0.9
                    x_offset = 1.6
                                                                   
                    x_text = aha_trial + x_offset
                    ha = 'left'
                                                                                            
                    if x_text > n_trials - 5:                                     
                        x_text = aha_trial - x_offset
                        ha = 'right'
                    ax.text(
                        x=x_text,
                        y=y_aha_text,
                        s=f'~"Aha!" trial: {aha_trial - baseline_length}',
                        ha=ha,
                        va='bottom',
                        fontsize=80,
                        color='cyan',
                        weight='bold',
                        clip_on=False
                    )
                else:
                    ax.set_ylim(-2.1*abs(rotation), 2.1*abs(rotation))
      
                ax.set_xlim(0, n_trials)
                                                                              
                ax.set_xlabel('Trial')
                ax.set_ylabel('Aim Angle (°)')
                ax.set_yticks(np.arange(-2*abs(rotation), 2*abs(rotation) + 1, 30))
                sns.despine()
                ax.legend(loc='upper left')
      
                                      
                save_path = os.path.join(save_dir, f"{datasetName}_rot{rotation}_p{pId}.svg")
                plt.savefig(save_path, dpi=300, bbox_inches='tight')
                plt.close(fig)                       
                if debug: print(f" Saved: {save_path}")
    print("All per-participant plots generated and saved to indiPlots/")
               
generate_per_participant_plots(debug=False, max_participants=30, num_samples=500)


In [ ]:


                                               
                                               
                
datasetLabels = ['CGVanilla', 'CGImp', 'Brudner', 'BondTaylor']
                                   
hasImp = [False, True, False, True]
                                        
rots = [90, 60, 45, 30, 15]
rotationMap = [rots, rots, [30], rots]                   

              
                                              
                                                       
                                                          
                                             
   

                                               
                                      
plot_dir = 'indiPlotsSignConfidence'
if not os.path.exists(plot_dir):
    os.makedirs(plot_dir)
print(f"Saving plots to: {plot_dir}")

def normalize_angle(angle):
    """Normalize angle(s) to [-180, 180). Preserves NaN."""
    if np.isscalar(angle):
        if np.isnan(angle):
            return np.nan
        angle = angle - 360 * np.round(angle / 360)
        return ((angle + 180) % 360) - 180
    else:
        normalized = np.full_like(angle, np.nan)
        mask = ~np.isnan(angle)
        unwrapped = angle[mask] - 360 * np.round(angle[mask] / 360)
        normalized[mask] = ((unwrapped + 180) % 360) - 180
        return normalized

def generate_per_participant_plots(save_dir=plot_dir, debug=False, max_participants=10, num_samples=500):
    """
    Generate per-participant plots for each dataset/rotation (first {max_participants} only).
    - SINGLE panel: Heatmap of total predictive density from HMM policy
      (mixture of state0 delta + state1 policy, convolved with execution noise,
       sampled and binned into adaptive coarse bins).
      Uses viridis colormap + exact vmax/scaling logic from the provided heatmap code.
    Overlaid with human data, HMM implicits, and sign confidence line.
    Saves each as {datasetName}_rot{rotation}_p{participantId}.svg.
    """
    if 'datasets' not in globals():
        raise ValueError("datasets global not defined! Load your model lists first.")

    for dsIdx, datasetName in enumerate(datasetLabels):
        modelLists = datasets[dsIdx]                                     
        rotations = rotationMap[dsIdx]
        impFlag = hasImp[dsIdx]

        for rotIdx, rotation in enumerate(rotations):
            ssm_fit = modelLists[0][rotIdx] if rotIdx < len(modelLists[0]) else None
            hmm_fit = modelLists[2][rotIdx] if rotIdx < len(modelLists[2]) else None

            if ssm_fit is None or hmm_fit is None:
                if debug: print(f"Skipping {datasetName} rot{rotation}: Missing SSM or HMM fit.")
                continue

                                      
            n_participants = None
            if hasattr(ssm_fit, 'allAims'):
                n_participants = len(ssm_fit.allAims)
            elif hasattr(ssm_fit, 'human_explicits'):
                n_participants = len(ssm_fit.human_explicits)
            elif hasattr(hmm_fit, 'human_explicits'):
                n_participants = len(hmm_fit.human_explicits)
            elif hasattr(hmm_fit, 'allAims'):
                n_participants = len(hmm_fit.allAims)
            if n_participants is None:
                if debug: print(f"Skipping {datasetName} rot{rotation}: No participant data.")
                continue

            n_participants = min(max_participants, n_participants)

            for pId in range(n_participants):
                if debug: print(f"Generating plot for {datasetName}, rot{rotation}, p{pId}...")

                                                   
                human_aims = np.array([])
                try:
                    if hasattr(ssm_fit, 'allAims'):
                        human_aims = np.array(ssm_fit.allAims[pId]).ravel()
                    elif hasattr(ssm_fit, 'human_explicits'):
                        human_aims = np.array(ssm_fit.human_explicits[pId]).ravel()
                    elif hasattr(hmm_fit, 'human_explicits'):
                        human_aims = np.array(hmm_fit.human_explicits[pId]).ravel()
                    elif hasattr(hmm_fit, 'allAims'):
                        human_aims = np.array(hmm_fit.allAims[pId]).ravel()
                except (IndexError, AttributeError, TypeError):
                    human_aims = np.array([])

                n_trials = len(human_aims)
                if n_trials == 0:
                    if debug: print(f"No human aims data for p{pId}; skipping.")
                    continue

                                     
                pi_preds = np.zeros((n_trials, 2))
                try:
                    pi_preds_full = np.array(hmm_fit.pi_preds)
                    pi_preds = pi_preds_full[pId, :n_trials, :]
                except (IndexError, AttributeError, TypeError, ValueError):
                    pass
                state0_probs = pi_preds[:, 0]

                                                           
                human_imps = np.array([])
                if impFlag:
                    try:
                        if hasattr(ssm_fit, 'allImps'):
                            human_imps = np.array(ssm_fit.allImps[pId]).ravel()
                        elif hasattr(ssm_fit, 'human_implicits'):
                            human_imps = np.array(ssm_fit.human_implicits[pId]).ravel()
                        elif hasattr(hmm_fit, 'human_implicits'):
                            human_imps = np.array(hmm_fit.human_implicits[pId]).ravel()
                        elif hasattr(hmm_fit, 'allImps'):
                            human_imps = np.array(hmm_fit.allImps[pId]).ravel()
                        human_imps = human_imps[:n_trials]
                        human_imps[human_imps == 0] = np.nan
                    except (IndexError, AttributeError, TypeError):
                        human_imps = np.array([])

                                                         
                hmm_imps = np.array([])
                if hasattr(hmm_fit, 'model_implicits'):
                    try:
                        hmm_imps = np.array(hmm_fit.model_implicits[pId]).ravel()
                        hmm_imps = hmm_imps[:n_trials]
                    except (IndexError, AttributeError, TypeError):
                        hmm_imps = np.array([])

                                       
                p_neg = np.zeros(n_trials)
                if hasattr(hmm_fit, 'model_predictive_dirs'):
                    try:
                        predictive_dirs = np.array(hmm_fit.model_predictive_dirs)
                        p_neg = predictive_dirs[pId, :n_trials, 0]
                    except (IndexError, AttributeError, TypeError, ValueError):
                        p_neg = np.zeros(n_trials)

                              
                policies_part = np.zeros((n_trials, 361))
                try:
                    policies_full = np.array(hmm_fit.model_predictive_policies)
                    policies_part = policies_full[pId, :n_trials]
                except (IndexError, AttributeError, TypeError, ValueError):
                    pass

                                       
                sigma = 0.0
                if hasattr(hmm_fit, 'xs') and pId < len(hmm_fit.xs):
                    try:
                        sigma = float(hmm_fit.xs[pId][0])
                        if np.isnan(sigma):
                            sigma = 0.0
                    except:
                        sigma = 0.0
                use_noise = sigma > 0 and datasetName != 'BondTaylor'

                                                       
                fine_angles = np.arange(-180, 181)
                orig_idx = ((180 + fine_angles) % 360).astype(int)

                kernel = None
                if use_noise:
                    support_size = int(6 * sigma) + 1
                    if support_size % 2 == 0:
                        support_size += 1
                    half_support = support_size // 2
                    support = np.arange(-half_support, half_support + 1, dtype=float)
                    kernel = norm_dist.pdf(support, 0, sigma)
                    kernel /= kernel.sum()

                zero_idx = np.argwhere(fine_angles == 0)[0, 0]
                delta_probs = np.zeros(len(fine_angles))
                delta_probs[zero_idx] = 1.0
                convolved0 = convolve(delta_probs, kernel, mode='same') if use_noise else delta_probs

                                                                
                bin_width = abs(rotation) / 15.0
                num_coarse_bins = int(360 / bin_width) + 1
                bin_edges = np.linspace(-180 - bin_width / 2.0, 180 + bin_width / 2.0, num_coarse_bins + 1)

                                                           
                sample_counts_matrix = np.zeros((n_trials, num_coarse_bins))

                for t in range(n_trials):
                    state0_prob = state0_probs[t]
                    recentered_probs = policies_part[t][orig_idx]

                    if use_noise:
                        convolved1 = convolve(recentered_probs, kernel, mode='same')
                    else:
                        convolved1 = recentered_probs
                    convolved1 = np.maximum(convolved1, 0)
                    convolved0 = np.maximum(convolved0, 0)
                    num_state0 = np.random.binomial(num_samples, state0_prob)
                    num_state1 = num_samples - num_state0

                    counts_t = np.zeros(num_coarse_bins)

                    if num_state0 > 0:
                        p = convolved0 / convolved0.sum() if convolved0.sum() > 0 else None
                        
                        samples0 = np.random.choice(fine_angles, size=num_state0, p=p) if p is not None else np.random.uniform(-180, 180, num_state0)
                        bin_idx = np.digitize(samples0, bin_edges) - 1
                        bin_idx = np.clip(bin_idx, 0, num_coarse_bins - 1)
                        counts_t += np.bincount(bin_idx, minlength=num_coarse_bins)

                    if num_state1 > 0:
                        p = convolved1 / convolved1.sum() if convolved1.sum() > 0 else None
                        samples1 = np.random.choice(fine_angles, size=num_state1, p=p) if p is not None else np.random.uniform(-180, 180, num_state1)
                        bin_idx = np.digitize(samples1, bin_edges) - 1
                        bin_idx = np.clip(bin_idx, 0, num_coarse_bins - 1)
                        counts_t += np.bincount(bin_idx, minlength=num_coarse_bins)

                    sample_counts_matrix[t, :] = counts_t

                                                       
                human_aims_signed = normalize_angle(human_aims)
                human_imps_signed = normalize_angle(human_imps) if len(human_imps) > 0 else np.array([])
                hmm_imps_signed = normalize_angle(hmm_imps) if len(hmm_imps) > 0 else np.array([])

                                  
                fig, ax = plt.subplots(1, 1, figsize=(14, 8), constrained_layout=True)
                ax.set_title(f'{datasetName} | Rot {rotation}° | Participant {pId}', fontsize=14)

                                                       
                sample_counts_masked = np.ma.masked_where(sample_counts_matrix == 0, sample_counts_matrix)

                                                                 
                all_counts = sample_counts_masked.compressed()
                vmax = np.percentile(all_counts, 95) if len(all_counts) > 0 else num_samples
                if vmax < 200:
                    vmax = 400

                                      
                proportion_matrix = sample_counts_matrix / num_samples
                proportion_masked = np.ma.masked_where(sample_counts_matrix == 0, proportion_matrix)

                                                         
                im = ax.imshow(proportion_masked.T,
                               extent=[0, n_trials, -180, 180],
                               origin='lower',
                               aspect='auto',
                               cmap='viridis',
                               norm=PowerNorm(gamma=0.5, vmin=0, vmax=(vmax / 3) / num_samples),
                               alpha=1.0,
                               zorder=2)

                                             
                cbar = fig.colorbar(im, ax=ax, orientation='vertical')
                cbar.set_label('Proportion of samples per bin', rotation=270, labelpad=25)

                                        
                ax.axhline(y=rotation, color='grey', linestyle='--', zorder=1, alpha=0.7)
                ax.axhline(y=-rotation, color='grey', linestyle='--', zorder=1, alpha=0.7)
                ax.axhline(y=0, color='grey', linestyle='--', zorder=1, alpha=0.7)
                ax.axhline(y=-2 * rotation, color='grey', linestyle='--', zorder=1, alpha=0.7)
                ax.axhline(y=2 * rotation, color='grey', linestyle='--', zorder=1, alpha=0.7)

                baseline_length = 64 if datasetName == 'Brudner' else 40
                washout_length = baseline_length
                ax.axvline(x=baseline_length, color='grey', linestyle='--', zorder=1, alpha=0.7)
                if n_trials > baseline_length + washout_length:
                    ax.axvline(x=n_trials - washout_length, color='grey', linestyle='--', zorder=1, alpha=0.7)

                                                                     
                trials = np.arange(n_trials)
                linewidth = 4.5
                ax.scatter(trials, human_aims_signed, color='red', s=60, alpha=0.8,
                           label='Human Aims (Explicit)', zorder=8, edgecolor='darkred', linewidth=2)

                if impFlag and len(human_imps_signed) > 0:
                    nan_mask = ~np.isnan(human_imps_signed)
                    ax.scatter(trials[nan_mask], human_imps_signed[nan_mask], color='blue', s=60, alpha=0.8,
                               label='Human Implicits', zorder=8, edgecolor='darkblue', linewidth=2)

                if len(hmm_imps_signed) > 0:
                    hmm_line = ax.plot(trials, hmm_imps_signed, color='green', ls='--', linewidth=linewidth,
                                       alpha=0.9, label='HMM Implicits', zorder=10)[0]
                    hmm_line.set_path_effects([withStroke(linewidth=linewidth*2, foreground='#00ff00')])

                if len(p_neg) > 0 and np.any(p_neg > 0):
                    scaled_p_neg = -2 * rotation + p_neg * (4 * rotation)
                    sign_line = ax.plot(trials, scaled_p_neg, color='gray', linewidth=linewidth,
                                        alpha=0.9, label='Model Sign Confidence (p_neg, scaled)', zorder=9)[0]
                    sign_line.set_path_effects([withStroke(linewidth=linewidth*2, foreground='darkgray')])

                                     
                ax.set_xlim(0, n_trials)
                ax.set_ylim(-2.1 * rotation, 2.1 * rotation)
                ax.set_yticks(np.arange(-2 * rotation, 2 * rotation + 1, 30))
                ax.set_xlabel('Trial')
                ax.set_ylabel('Aim Angle (°)')
                sns.despine()
                ax.legend(loc='upper left')                                           

                      
                save_path = os.path.join(save_dir, f"{datasetName}_rot{rotation}_p{pId}.svg")
                plt.savefig(save_path, dpi=300, bbox_inches='tight')
                plt.close(fig)
                if debug: print(f"Saved: {save_path}")

    print("All per-participant plots generated and saved to indiPlotsSignConfidence/")

                                                       
generate_per_participant_plots(debug=False, max_participants=30, num_samples=500)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm as norm_dist
import scipy.ndimage as ndimage
import matplotlib.colors as mcolors
np.random.seed(42)

def to_signed(ang):
    return np.where(ang > 180, ang - 360, ang)

def darken_color(hex_color, factor=0.7):
    rgb = mcolors.hex2color(hex_color)
    darkened = tuple(c * factor for c in rgb)
    return mcolors.to_hex(darkened)

def watson_u2_gof(samples, support, masses, n_monte=10000, random_seed=42):
    if len(samples) < 20:
        return np.nan, np.nan
    np.random.seed(random_seed)
    masses = masses / np.sum(masses)
    cdf = np.cumsum(masses)
    z = np.interp(samples, support, cdf)
    u = np.sort(z)
    n = len(u)
    i = np.arange(1, n + 1)
    term1 = np.sum((u - (2 * i - 1) / (2 * n)) ** 2)
    term2 = n * (np.mean(u) - 0.5) ** 2
    term3 = 1 / (12 * n)
    u2 = term1 - term2 + term3
    count = 0
    for _ in range(n_monte):
        unif = np.sort(np.random.uniform(0, 1, n))
        t1 = np.sum((unif - (2 * i - 1) / (2 * n)) ** 2)
        t2 = n * (np.mean(unif) - 0.5) ** 2
        t3 = 1 / (12 * n)
        u2_sim = t1 - t2 + t3
        if u2_sim >= u2:
            count += 1
    p = (count + 1) / (n_monte + 1)
    return u2, p

def generate_rotation_phase_distributions(save_dir='phasePlots/', debug=False, n_bins=36):
    """
    Loops over all datasets and their rotations.
    Updated to use Watson's U² instead of Circular Wasserstein distance.
    Only includes trials with valid (non-NaN) human aims.
    This ensures exact matching sample sizes and avoids skips due to missing data.
    Displays the U² and p-value from Monte Carlo.
    Legend made considerably smaller + labels shortened.
    """
    orig_bins = n_bins
    os.makedirs(save_dir, exist_ok=True)
    if 'datasets' not in globals():
        raise ValueError("datasets global not defined! Load your model lists first.")
    for dsIdx, datasetName in enumerate(datasetLabels):
                                           
        if datasetName == 'Brudner':
            baseline_length = 64
            washout_length = 64
        else:
            baseline_length = 40
            washout_length = 40
        
        modelLists = datasets[dsIdx]                                    
        rotations = rotationMap[dsIdx]
        if debug:
            print(f"\n=== DATASET: {datasetName} | baseline={baseline_length} washout={washout_length} ===")
        for rotIdx, rotation in enumerate(rotations):
            n_bins = orig_bins - abs(rotation)
            if rotIdx >= len(modelLists[0]) or rotIdx >= len(modelLists[2]):
                if debug: print(f"Skipping {datasetName} rot{rotation}: rotation index out of range.")
                continue
            ssm_fit = modelLists[0][rotIdx]
            hmm_fit = modelLists[2][rotIdx]
            if ssm_fit is None or hmm_fit is None:
                if debug: print(f"Skipping {datasetName} rot{rotation}: Missing SSM or HMM fit.")
                continue
            if not hasattr(ssm_fit, 'allAims') or len(ssm_fit.allAims) == 0:
                if debug: print(f"Skipping {datasetName} rot{rotation}: No participant data.")
                continue
            n_participants = len(ssm_fit.allAims)
            n_trials = len(ssm_fit.allAims[0])
            if n_trials < baseline_length + 20 + washout_length:
                if debug: print(f"Skipping {datasetName} rot{rotation}: Insufficient trials ({n_trials}).")
                continue
            rotation_start = baseline_length
            rotation_end = n_trials - washout_length
            rot_length = rotation_end - rotation_start
            if rot_length < 30:
                if debug: print(f"Skipping {datasetName} rot{rotation}: Rotation too short ({rot_length}).")
                continue
            early_start, early_end = rotation_start, rotation_start + 8
            mid_start = early_end
            mid_end = mid_start + 40
            late_start, late_end = rotation_end - 40, rotation_end
                                           
            phase_ranges = [
                (early_start, early_end, 'Early 10'),
                (mid_start, mid_end, 'Middle 40'),
                (late_start, late_end, 'Last 40')
            ]
                                                                                              
            washStart = rotation_end + 1
            washEnd = washStart + 7
            phase_ranges.append((washStart, washEnd, 'Washout next 10'))
                                  
            numPanels = len(phase_ranges)
            fig, axs = plt.subplots(1, numPanels, figsize=(5 * numPanels, 5), sharey=True)
            fig.suptitle(f'{datasetName} | Rotation {rotation}° | Aim Angle Distributions by Phase', fontsize=14)
            if numPanels == 1:
                axs = [axs]
            try:
                policies_full = np.array(hmm_fit.model_predictive_policies)
                pi_preds_full = np.array(hmm_fit.pi_preds)
            except (AttributeError, ValueError):
                if debug: print(f"Skipping {datasetName} rot{rotation}: No predictive policies or pi_preds.")
                continue
            bin_edges = np.linspace(-180, 180, n_bins + 1)
            bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
            fine_angles = np.arange(-180, 180)                              
            orig_idx = ((180 + fine_angles) % 360).astype(int)
                                                        
            phase_results_hmm = []
            phase_results_ssm = []
            for i, (start_idx, end_idx, title_str) in enumerate(phase_ranges):
                human_aims_all = []
                model_marginals = []                                              
                ssm_marginals = []           
                num_trials_phase = end_idx - start_idx
                for pId in range(n_participants):
                    human_aims = np.array(ssm_fit.allAims[pId]).ravel() % 360.0
                    phase_human_unsigned = human_aims[start_idx:end_idx]
                    phase_human = to_signed(phase_human_unsigned)
                    phase_human = ((phase_human + 180) % 360) - 180                
                                            
                    if hasattr(ssm_fit, 'mOut1'):
                        ssm_model_aims = np.array(ssm_fit.mOut1[pId]).ravel() % 360.0
                    elif hasattr(ssm_fit, 'mStates'):
                        ssm_model_aims = np.array(ssm_fit.mStates[pId]).ravel() % 360.0
                    else:
                        if debug: print(f"Skipping {datasetName} rot{rotation}: No SSM model aims (mOut1 or mStates).")
                        continue
                    phase_ssm_unsigned = ssm_model_aims[start_idx:end_idx]
                    phase_ssm = to_signed(phase_ssm_unsigned)
                    phase_ssm = ((phase_ssm + 180) % 360) - 180                
                    try:
                        policies_part = policies_full[pId, start_idx:end_idx, :]
                        pi_part = pi_preds_full[pId, start_idx:end_idx, :]
                    except (IndexError, ValueError):
                        continue                                                         
                               
                    sigma = 0.0
                    if hasattr(hmm_fit, 'xs') and pId < len(hmm_fit.xs):
                        try:
                            sigma = float(hmm_fit.xs[pId][0])
                            if np.isnan(sigma):
                                sigma = 0.0
                        except Exception:
                            sigma = 0.0
                    if datasetName == 'BondTaylor':
                        sigma = 0.0
                    kernel = None
                    if sigma > 0:
                        support_size = int(6 * sigma) + 1
                        if support_size % 2 == 0:
                            support_size += 1
                        half_support = support_size // 2
                        support = np.arange(-half_support, half_support + 1, dtype=float)
                        kernel = norm_dist.pdf(support, 0, sigma)
                        kernel /= kernel.sum()
                               
                    sigma_ssm = 0.0
                    if hasattr(ssm_fit, 'xs') and pId < len(ssm_fit.xs):
                        try:
                            sigma_ssm = float(ssm_fit.xs[pId][-1])
                            if np.isnan(sigma_ssm):
                                sigma_ssm = 0.0
                        except Exception:
                            sigma_ssm = 0.0
                    if datasetName == 'BondTaylor':
                        sigma_ssm = 0.0
                    kernel_ssm = None
                    if sigma_ssm > 0:
                        support_size_ssm = int(6 * sigma_ssm) + 1
                        if support_size_ssm % 2 == 0:
                            support_size_ssm += 1
                        half_support_ssm = support_size_ssm // 2
                        support_ssm = np.arange(-half_support_ssm, half_support_ssm + 1, dtype=float)
                        kernel_ssm = norm_dist.pdf(support_ssm, 0, sigma_ssm)
                        kernel_ssm /= kernel_ssm.sum()
                    zero_idx = np.argwhere(fine_angles == 0)[0, 0]
                    delta_probs = np.zeros(len(fine_angles))
                    delta_probs[zero_idx] = 1.0
                    convolved0 = ndimage.convolve1d(delta_probs, kernel, mode='wrap') if sigma > 0 else delta_probs
                    convolved0 = np.maximum(convolved0, 0)
                    convolved0_norm = convolved0 / convolved0.sum() if convolved0.sum() > 0 else convolved0
                    for t in range(num_trials_phase):
                        human_aim_this = phase_human[t]
                        if np.isnan(human_aim_this):
                            continue                                       
                                      
                        state0_prob = pi_part[t, 0]
                        recentered_probs = policies_part[t][orig_idx]
                        if sigma > 0:
                            convolved1 = ndimage.convolve1d(recentered_probs, kernel, mode='wrap')
                        else:
                            convolved1 = recentered_probs
                        convolved1 = np.maximum(convolved1, 0)
                        convolved1_norm = convolved1 / convolved1.sum() if convolved1.sum() > 0 else convolved1
                        marginal = state0_prob * convolved0_norm + (1 - state0_prob) * convolved1_norm
                        model_marginals.append(marginal)
                                      
                        aim_signed = phase_ssm[t]
                        dists = np.abs(fine_angles - aim_signed)
                        ang_dists = np.minimum(dists, 360 - dists)
                        closest_idx = np.argmin(ang_dists)
                        delta_ssm = np.zeros(len(fine_angles))
                        delta_ssm[closest_idx] = 1.0
                        if sigma_ssm > 0:
                            convolved_ssm = ndimage.convolve1d(delta_ssm, kernel_ssm, mode='wrap')
                        else:
                            convolved_ssm = delta_ssm
                        convolved_ssm = np.maximum(convolved_ssm, 0)
                        convolved_ssm_norm = convolved_ssm / convolved_ssm.sum() if convolved_ssm.sum() > 0 else convolved_ssm
                        ssm_marginals.append(convolved_ssm_norm)
                        human_aims_all.append(human_aim_this)
                human_aims_all = np.array(human_aims_all)
                ax = axs[i]
                ax.set_title(title_str)
                ax.set_xlabel('Signed Angle (°)')
                if i == 0:
                    ax.set_ylabel('Density')
                
                if ssm_marginals:
                    avg_ssm_marginal = np.mean(ssm_marginals, axis=0)
                    hist_ssm, _ = np.histogram(fine_angles, bins=bin_edges, weights=avg_ssm_marginal, density=True)
                    ax.stairs(hist_ssm, bin_edges, fill=False, color='magenta', label='SSM', alpha=0.9, linewidth=4.5)
                if model_marginals:
                    avg_marginal = np.mean(model_marginals, axis=0)
                    hist_model, _ = np.histogram(fine_angles, bins=bin_edges, weights=avg_marginal, density=True)
                    ax.stairs(hist_model, bin_edges, fill=False, color='green', label='HMM', alpha=0.9, linewidth=4.5)
                if len(human_aims_all) > 0:
                    hist_human, _ = np.histogram(human_aims_all, bins=bin_edges, density=True)
                    ax.stairs(hist_human, bin_edges, fill=True, color='darkred', label='Human', alpha=0.4, linewidth=0)
                                             
                n_samples = len(human_aims_all)
                u2_ssm = np.nan
                p_ssm = np.nan
                if n_samples >= 20 and np.sum(ssm_marginals) > 0:
                    u2_ssm, p_ssm = watson_u2_gof(human_aims_all, fine_angles, avg_ssm_marginal)
                    phase_results_ssm.append((title_str, n_samples, u2_ssm, p_ssm, ax))
                else:
                    if debug:
                        print(f" {datasetName} rot{rotation} | Phase {title_str}: "
                              f"Skipped SSM Watson's U² (n={n_samples} < 20)")
                                             
                u2_hmm = np.nan
                p_hmm = np.nan
                if len(model_marginals) > 0 and n_samples >= 20:
                    u2_hmm, p_hmm = watson_u2_gof(human_aims_all, fine_angles, avg_marginal)
                    phase_results_hmm.append((title_str, n_samples, u2_hmm, p_hmm, ax))
                else:
                    if debug and n_samples < 20:
                        print(f" {datasetName} rot{rotation} | Phase {title_str}: "
                              f"Skipped HMM Watson's U² (n={n_samples} < 20)")
                ax.axvline(0, color='grey', linestyle='--', linewidth=1, alpha=0.8)
                ax.axvline(rotation, color='grey', linestyle='--', linewidth=1, alpha=0.8)
                ax.axvline(-rotation, color='grey', linestyle='--', linewidth=1, alpha=0.8)
                                                    
                xlim_min = min(-60, -2 * abs(rotation))
                xlim_max = max(60, 2 * abs(rotation))
                ax.set_xlim(xlim_min, xlim_max)
                ax.set_ylim(0, 0.1)
                                                               
                key_points = [-rotation, 0, rotation]
                ax.set_xticks(key_points)
                                                   
            for ax in axs:
                ax.legend(fontsize=8)
                                  
                                            
            for title_str, n_samples, u2_hmm, p_hmm, ax in phase_results_hmm:
                if not np.isnan(u2_hmm):
                    ax.text(0.02, 0.98, f'HMM U²={u2_hmm:.3f}\np={p_hmm:.3f}',
                            transform=ax.transAxes, fontsize=10, va='top', ha='left',
                            bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
                if debug:
                    print(f" {datasetName} rot{rotation} | Phase {title_str} HMM: "
                          f"n={n_samples} | U²={u2_hmm:.3f} p={p_hmm:.3f}")
                                             
            for title_str, n_samples, u2_ssm, p_ssm, ax in phase_results_ssm:
                if not np.isnan(u2_ssm):
                    ax.text(0.98, 0.98, f'SSM U²={u2_ssm:.3f}\np={p_ssm:.3f}',
                            transform=ax.transAxes, fontsize=10, va='top', ha='right',
                            bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
                if debug:
                    print(f" {datasetName} rot{rotation} | Phase {title_str} SSM: "
                          f"n={n_samples} | U²={u2_ssm:.3f} p={p_ssm:.3f}")
            plt.tight_layout()
            save_path = os.path.join(save_dir, f"{datasetName}_rot{rotation}_phase_dists.svg")
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close(fig)
            if debug: print(f"Saved: {save_path}")
    print("All rotation phase distribution plots generated and saved.")

generate_rotation_phase_distributions(debug=True, n_bins=150, save_dir='phasePlots/')

In [ ]:
import matplotlib.ticker as ticker
from scipy.stats import ttest_rel, t, norm as norm_dist
import os
import numpy as np
import scipy.ndimage as ndimage
np.random.seed(42)
                                                                    

def to_signed(ang):
    return np.where(ang > 180, ang - 360, ang)

def watson_u2(samples, support, masses, minSamples=10):
    """
    Watson's U² statistic for circular GoF on a uniform grid.
    Returns dimensionless U² (lower = better fit).
    """
    samples = np.asarray(samples, dtype=float)
    support = np.asarray(support, dtype=float)
    masses = np.asarray(masses, dtype=float)
    if samples.size < minSamples:
        return np.nan
    masses = masses / np.sum(masses)
    cdf = np.cumsum(masses)
    z = np.interp(samples, support, cdf)                               
    u = np.sort(z)
    n = len(u)
    i = np.arange(1, n + 1)
    term1 = np.sum((u - (2 * i - 1) / (2 * n)) ** 2)
    term2 = n * (np.mean(u) - 0.5) ** 2
    term3 = 1 / (12 * n)
    u2 = term1 - term2 + term3
    return u2

def generate_whole_experiment_ppc(save_dir='wholePPC/', debug=False, min_samples=10):
    """
    Performs whole-experiment posterior predictive checks using Watson's U² on a per-participant basis.
    Provides overall mean U² as absolute fit measure.
    """
    os.makedirs(save_dir, exist_ok=True)
    if 'datasets' not in globals():
        raise ValueError("datasets global not defined! Load your model lists first.")
    for dsIdx, datasetName in enumerate(datasetLabels):
                                           
        if datasetName == 'Brudner':
            baseline_length = 64
            washout_length = 64
        else:
            baseline_length = 40
            washout_length = 40
        modelLists = datasets[dsIdx]                                     
        rotations = rotationMap[dsIdx]
        if debug:
            print(f"\n=== DATASET: {datasetName} | baseline={baseline_length} washout={washout_length} ===")
                                    
        dataset_rot_labels = []
        dataset_hmm_part_u2s = []
        dataset_ssm_part_u2s = []
        for rotIdx, rotation in enumerate(rotations):
            if rotIdx >= len(modelLists[0]) or rotIdx >= len(modelLists[2]):
                if debug: print(f"Skipping {datasetName} rot{rotation}: rotation index out of range.")
                continue
            ssm_fit = modelLists[0][rotIdx]
            hmm_fit = modelLists[2][rotIdx]
            if ssm_fit is None or hmm_fit is None:
                if debug: print(f"Skipping {datasetName} rot{rotation}: Missing SSM or HMM fit.")
                continue
            if not hasattr(ssm_fit, 'allAims') or len(ssm_fit.allAims) == 0:
                if debug: print(f"Skipping {datasetName} rot{rotation}: No participant data.")
                continue
            n_participants = len(ssm_fit.allAims)
            n_trials = len(ssm_fit.allAims[0])
            if n_trials < baseline_length + 20 + washout_length:
                if debug: print(f"Skipping {datasetName} rot{rotation}: Insufficient trials ({n_trials}).")
                continue
            rotation_start = baseline_length
            rotation_end = n_trials - washout_length
            rot_length = rotation_end - rotation_start
            if rot_length < 10:
                if debug: print(f"Skipping {datasetName} rot{rotation}: Rotation too short ({rot_length}).")
                continue
            try:
                policies_full = np.array(hmm_fit.model_predictive_policies)
                pi_preds_full = np.array(hmm_fit.pi_preds)
            except (AttributeError, ValueError):
                if debug: print(f"Skipping {datasetName} rot{rotation}: No predictive policies or pi_preds.")
                continue
            fine_angles = np.arange(-180, 180)              
            orig_idx = ((180 + fine_angles) % 360).astype(int)
                                                                     
            all_results_hmm = []                        
            all_results_ssm = []
            for pId in range(n_participants):
                human_aims_p = []
                model_marginals_p = []
                ssm_marginals_p = []
                human_aims = np.array(ssm_fit.allAims[pId]).ravel() % 360.0
                window_human_unsigned = human_aims[:n_trials]
                window_human = to_signed(window_human_unsigned)
                window_human = ((window_human + 180) % 360) - 180                
                                        
                if hasattr(ssm_fit, 'mOut1'):
                    ssm_model_aims = np.array(ssm_fit.mOut1[pId]).ravel() % 360.0
                elif hasattr(ssm_fit, 'mStates'):
                    ssm_model_aims = np.array(ssm_fit.mStates[pId]).ravel() % 360.0
                else:
                    if debug: print(f"Skipping {datasetName} rot{rotation}: No SSM model aims (mOut1 or mStates).")
                    continue
                window_ssm_unsigned = ssm_model_aims[:n_trials]
                window_ssm = to_signed(window_ssm_unsigned)
                window_ssm = ((window_ssm + 180) % 360) - 180                
                try:
                    policies_part = policies_full[pId, :n_trials, :]
                    pi_part = pi_preds_full[pId, :n_trials, :]
                except (IndexError, ValueError):
                    if debug: print(f"Skipping {datasetName} rot{rotation} pId {pId}: No model predictions.")
                    continue
                sigma = 0.0
                if hasattr(hmm_fit, 'xs') and pId < len(hmm_fit.xs):
                    try:
                        sigma = float(hmm_fit.xs[pId][0])
                        if np.isnan(sigma):
                            sigma = 0.0
                    except Exception:
                        sigma = 0.0
                if datasetName == 'BondTaylor':
                    sigma = 0.0
                kernel = None
                if sigma > 0:
                    support_size = int(6 * sigma) + 1
                    if support_size % 2 == 0:
                        support_size += 1
                    half_support = support_size // 2
                    support = np.arange(-half_support, half_support + 1, dtype=float)
                    kernel = norm_dist.pdf(support, 0, sigma)
                    kernel /= kernel.sum()
                           
                sigma_ssm = 0.0
                if hasattr(ssm_fit, 'xs') and pId < len(ssm_fit.xs):
                    try:
                        sigma_ssm = float(ssm_fit.xs[pId][-1])
                        if np.isnan(sigma_ssm):
                            sigma_ssm = 0.0
                    except Exception:
                        sigma_ssm = 0.0
                if datasetName == 'BondTaylor':
                    sigma_ssm = 0.0
                kernel_ssm = None
                if sigma_ssm > 0:
                    support_size_ssm = int(6 * sigma_ssm) + 1
                    if support_size_ssm % 2 == 0:
                        support_size_ssm += 1
                    half_support_ssm = support_size_ssm // 2
                    support_ssm = np.arange(-half_support_ssm, half_support_ssm + 1, dtype=float)
                    kernel_ssm = norm_dist.pdf(support_ssm, 0, sigma_ssm)
                    kernel_ssm /= kernel_ssm.sum()
                zero_idx = np.argwhere(fine_angles == 0)[0, 0]
                delta_probs = np.zeros(len(fine_angles))
                delta_probs[zero_idx] = 1.0
                convolved0 = ndimage.convolve1d(delta_probs, kernel, mode='wrap') if sigma > 0 else delta_probs
                convolved0 = np.maximum(convolved0, 0)
                convolved0_norm = convolved0 / convolved0.sum() if convolved0.sum() > 0 else convolved0
                for trial in range(rotation_start, rotation_end):
                    human_aim_this = window_human[trial]
                    if np.isnan(human_aim_this):
                        continue                                       
                                  
                    state0_prob = pi_part[trial, 0]
                    recentered_probs = policies_part[trial][orig_idx]
                    if sigma > 0:
                        convolved1 = ndimage.convolve1d(recentered_probs, kernel, mode='wrap')
                    else:
                        convolved1 = recentered_probs
                    convolved1 = np.maximum(convolved1, 0)
                    convolved1_norm = convolved1 / convolved1.sum() if convolved1.sum() > 0 else convolved1
                    marginal = state0_prob * convolved0_norm + (1 - state0_prob) * convolved1_norm
                    model_marginals_p.append(marginal)
                                  
                    aim_signed = window_ssm[trial]
                    dists = np.abs(fine_angles - aim_signed)
                    ang_dists = np.minimum(dists, 360 - dists)
                    closest_idx = np.argmin(ang_dists)
                    delta_ssm = np.zeros(len(fine_angles))
                    delta_ssm[closest_idx] = 1.0
                    if sigma_ssm > 0:
                        convolved_ssm = ndimage.convolve1d(delta_ssm, kernel_ssm, mode='wrap')
                    else:
                        convolved_ssm = delta_ssm
                    convolved_ssm = np.maximum(convolved_ssm, 0)
                    convolved_ssm_norm = convolved_ssm / convolved_ssm.sum() if convolved_ssm.sum() > 0 else convolved_ssm
                    ssm_marginals_p.append(convolved_ssm_norm)
                    human_aims_p.append(human_aim_this)
                human_aims_p = np.array(human_aims_p)
                n_samples = len(human_aims_p)
                if n_samples >= min_samples:
                            
                    avg_ssm_marginal = np.mean(ssm_marginals_p, axis=0)
                    u2_ssm = watson_u2(human_aims_p, fine_angles, avg_ssm_marginal, minSamples=min_samples)
                    all_results_ssm.append((pId, n_samples, u2_ssm))
                            
                    avg_hmm_marginal = np.mean(model_marginals_p, axis=0)
                    u2_hmm = watson_u2(human_aims_p, fine_angles, avg_hmm_marginal, minSamples=min_samples)
                    all_results_hmm.append((pId, n_samples, u2_hmm))
                else:
                    if debug:
                        print(f" {datasetName} rot{rotation} pId {pId}: "
                              f"Skipped U² (n={n_samples} < {min_samples})")
            num_tests = len(all_results_hmm) + len(all_results_ssm)
            if num_tests == 0:
                if debug: print(f"Skipping {datasetName} rot{rotation}: No valid participants.")
                continue
                                                               
            if all_results_hmm:
                all_u2_hmm = [res[2] for res in all_results_hmm]
                overall_mean_u2_hmm = np.mean(all_u2_hmm)
                if debug:
                    print(f" {datasetName} rot{rotation} | HMM overall: mean U²={overall_mean_u2_hmm:.3f}")
            if all_results_ssm:
                all_u2_ssm = [res[2] for res in all_results_ssm]
                overall_mean_u2_ssm = np.mean(all_u2_ssm)
                if debug:
                    print(f" {datasetName} rot{rotation} | SSM overall: mean U²={overall_mean_u2_ssm:.3f}")
                                    
            if all_results_hmm:
                hmm_part_u2s = [res[2] for res in all_results_hmm]
                dataset_hmm_part_u2s.append(hmm_part_u2s)
            if all_results_ssm:
                ssm_part_u2s = [res[2] for res in all_results_ssm]
                dataset_ssm_part_u2s.append(ssm_part_u2s)
            dataset_rot_labels.append(rotation)
                                                
        if dataset_rot_labels:
            num_rots = len(dataset_rot_labels)
            fig_width = 2 + 2 * num_rots
            fig, ax = plt.subplots(figsize=(fig_width, 6))
            fig.suptitle(f'{datasetName} | Participant Watson\'s U² per Rotation Group', fontsize=14)
            x_positions = np.flip(np.arange(num_rots))
            width = 0.5
            box_width = 0.45
            all_data = []
            all_pos = []
            for i in range(num_rots):
                             
                ssm_u2s = dataset_ssm_part_u2s[i] if i < len(dataset_ssm_part_u2s) else []
                if ssm_u2s:
                    x_ssm = np.full(len(ssm_u2s), x_positions[i] - width/2) + np.random.uniform(-box_width/2, box_width/2, len(ssm_u2s))
                    ax.scatter(x_ssm, ssm_u2s, facecolor='magenta', edgecolor='darkmagenta', linewidth=2.5, alpha=0.6, s=120)
                             
                hmm_u2s = dataset_hmm_part_u2s[i] if i < len(dataset_hmm_part_u2s) else []
                if hmm_u2s:
                    x_hmm = np.full(len(hmm_u2s), x_positions[i] + width/2) + np.random.uniform(-box_width/2, box_width/2, len(hmm_u2s))
                    ax.scatter(x_hmm, hmm_u2s, facecolor='green', edgecolor='darkgreen', linewidth=2.5, alpha=0.6, s=120)
                             
                if ssm_u2s:
                    all_data.append(ssm_u2s)
                    all_pos.append(x_positions[i] - width/2)
                if hmm_u2s:
                    all_data.append(hmm_u2s)
                    all_pos.append(x_positions[i] + width/2)
                                                                         
                if hmm_u2s and ssm_u2s and len(hmm_u2s) == len(ssm_u2s) and len(hmm_u2s) >= 2:
                    t_stat, p_value = ttest_rel(hmm_u2s, ssm_u2s)
                    num_groups = num_rots                                            
                    corrected_p = min(p_value * num_groups, 1.0)
                    sig_stars = '***' if corrected_p < 0.001 else '**' if corrected_p < 0.01 else '*' if corrected_p < 0.05 else 'n.s.'
                                               
                    mean_hmm = np.mean(hmm_u2s)
                    std_hmm = np.std(hmm_u2s, ddof=1)
                    n_hmm = len(hmm_u2s)
                    sem_hmm = std_hmm / np.sqrt(n_hmm)
                    ci_hmm_low, ci_hmm_high = t.interval(0.95, n_hmm - 1, loc=mean_hmm, scale=sem_hmm)
                    mean_ssm = np.mean(ssm_u2s)
                    std_ssm = np.std(ssm_u2s, ddof=1)
                    n_ssm = len(ssm_u2s)
                    sem_ssm = std_ssm / np.sqrt(n_ssm)
                    ci_ssm_low, ci_ssm_high = t.interval(0.95, n_ssm - 1, loc=mean_ssm, scale=sem_ssm)
                                   
                    rotation_label = dataset_rot_labels[i]
                    print(f"Dataset: {datasetName}, Rotation: {rotation_label}° (U²)")
                    print(f"HMM Mean: {mean_hmm:.3f}, 95% CI: [{ci_hmm_low:.3f}, {ci_hmm_high:.3f}]")
                    print(f"SSM Mean: {mean_ssm:.3f}, 95% CI: [{ci_ssm_low:.3f}, {ci_ssm_high:.3f}]")
                    print(f"Paired t-test: t = {t_stat:.3f}, p = {p_value:.2e}, corrected p = {corrected_p:.2e}")
                    print("---")
                                              
                    if sig_stars != 'n.s.':
                        max_y = max(max(hmm_u2s), max(ssm_u2s))
                        star_y = max_y + 0.03
                        ax.text(x_positions[i], star_y, sig_stars, ha='center', va='bottom', fontsize=14, color='black')
                                
            if all_data:
                widthMod = 2
                props = {
                    'boxprops': {'linewidth': widthMod},
                    'whiskerprops': {'linewidth': widthMod},
                    'capprops': {'linewidth': widthMod},
                    'medianprops': {'linewidth': widthMod, 'color': 'black'}
                }
                ax.boxplot(all_data, positions=all_pos, widths=box_width, patch_artist=False, showfliers=False, **props)
            ax.set_xticks(x_positions)
            ax.set_xticklabels([f"{r}°" for r in dataset_rot_labels])
            ax.set_ylabel('Watson\'s U² (lower = better fit)')
            ax.set_ylim(0, 25)  
            ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))
            ax.set_xlabel('Rotation Group')
                    
            from matplotlib.lines import Line2D
            legend_elements = [
                Line2D([0], [0], marker='o', color='w', label='SSM', markerfacecolor='magenta', markersize=10),
                Line2D([0], [0], marker='o', color='w', label='HMM', markerfacecolor='green', markersize=10)
            ]
            ax.legend(handles=legend_elements, fontsize=10)
            plt.tight_layout()
            save_path = os.path.join(save_dir, f"{datasetName}_U2_per_rot.svg")
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close(fig)
            if debug: print(f"Saved: {save_path}")
    print("All per-participant whole-experiment PPC plots generated and saved.")

generate_whole_experiment_ppc(debug=False, min_samples=10, save_dir='wholePPC_perPart/')

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import norm as norm_dist
import scipy.ndimage as ndimage
np.random.seed(42)
datasets = [[ssmsAll8CG,qLearnsAll8CG,hmmsAll8CG],[ssmsAll8CGIMP,qLearnsAll8CGIMP,hmmsAll8CGIMP],[ssmsAllBrudner,qLearnsAllBrudner,hmmsAllBrudner],
            [ssmsAll8BT,qLearnsAll8BT,hmmsAll8BT]]
def to_signed(ang):
    return np.where(ang > 180, ang - 360, ang)
def darken_color(hex_color, factor=0.7):
    rgb = mcolors.hex2color(hex_color)
    darkened = tuple(c * factor for c in rgb)
    return mcolors.to_hex(darkened)
def watson_u2(samples, support, masses, minSamples=20):
    """
    Watson's U² statistic for circular GoF on a uniform grid.
    Returns dimensionless U² (lower = better fit).
    """
    samples = np.asarray(samples, dtype=float)
    support = np.asarray(support, dtype=float)
    masses = np.asarray(masses, dtype=float)
    if samples.size < minSamples:
        return np.nan
    masses = masses / np.sum(masses)
    cdf = np.cumsum(masses)
    z = np.interp(samples, support, cdf)                               
    u = np.sort(z)
    n = len(u)
    i = np.arange(1, n + 1)
    term1 = np.sum((u - (2 * i - 1) / (2 * n)) ** 2)
    term2 = n * (np.mean(u) - 0.5) ** 2
    term3 = 1 / (12 * n)
    u2 = term1 - term2 + term3
    return u2
def generate_individual_participant_distributions(save_dir='individualPlots/', debug=False, n_participants_to_plot=10, n_bins=90):
    """
    For each dataset and rotation, selects up to n_participants_to_plot participants and plots their rotation-phase aim distributions as staircase plots for human, HMM, and SSM in separate figures.
    Computes and displays Watson's U² for HMM and SSM on each participant's plot.
    """
    os.makedirs(save_dir, exist_ok=True)
    if 'datasets' not in globals():
        raise ValueError("datasets global not defined! Load your model lists first.")
    for dsIdx, datasetName in enumerate(datasetLabels):
                                                                           
        if datasetName == 'Brudner':
            baseline_length = 64
            washout_length = 64
        else:
            baseline_length = 40
            washout_length = 40
        modelLists = datasets[dsIdx]                                    
        rotations = rotationMap[dsIdx]
        if debug:
            print(f"\n=== DATASET: {datasetName} | baseline={baseline_length} washout={washout_length} ===")
        for rotIdx, rotation in enumerate(rotations):
            if rotIdx >= len(modelLists[0]) or rotIdx >= len(modelLists[2]):
                if debug: print(f"Skipping {datasetName} rot{rotation}: rotation index out of range.")
                continue
            ssm_fit = modelLists[0][rotIdx]
            hmm_fit = modelLists[2][rotIdx]
            if ssm_fit is None or hmm_fit is None:
                if debug: print(f"Skipping {datasetName} rot{rotation}: Missing SSM or HMM fit.")
                continue
            if not hasattr(ssm_fit, 'allAims') or len(ssm_fit.allAims) == 0:
                if debug: print(f"Skipping {datasetName} rot{rotation}: No participant data.")
                continue
            n_participants = len(ssm_fit.allAims)
            n_trials = len(ssm_fit.allAims[0])
            if n_trials < baseline_length + 20 + washout_length:
                if debug: print(f"Skipping {datasetName} rot{rotation}: Insufficient trials ({n_trials}).")
                continue
                                           
            start_trial = baseline_length
            end_trial = n_trials - washout_length
            n_rotation_trials = end_trial - start_trial
            if n_rotation_trials < 20:                                            
                if debug: print(f"Skipping {datasetName} rot{rotation}: Insufficient rotation trials ({n_rotation_trials}).")
                continue
            try:
                policies_full = np.array(hmm_fit.model_predictive_policies)
                pi_preds_full = np.array(hmm_fit.pi_preds)
            except (AttributeError, ValueError):
                if debug: print(f"Skipping {datasetName} rot{rotation}: No predictive policies or pi_preds.")
                continue
            n_bins = min(60,int(n_bins * 90/abs(rotation)))
            bin_edges = np.linspace(-180, 180, n_bins + 1)
            bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
            fine_angles = np.arange(-180, 180)
            orig_idx = ((180 + fine_angles) % 360).astype(int)
                                                                    
            participant_ids = np.arange(n_participants)
            np.random.shuffle(participant_ids)                   
            selected_pids = participant_ids[:min(n_participants_to_plot, n_participants)]
            for pId in selected_pids:
                fig, ax = plt.subplots(figsize=(6, 5))
                fig.suptitle(f'{datasetName} | Rotation {rotation}° | Participant {pId} Aim Distribution (Rotation Phase)', fontsize=14)
                ax.set_xlabel('Signed Angle (°)')
                ax.set_ylabel('Density')
                human_aims_p = []
                model_marginals_p = []
                ssm_marginals_p = []
                human_aims = np.array(ssm_fit.allAims[pId]).ravel() % 360.0
                window_human_unsigned = human_aims[start_trial:end_trial]
                window_human = to_signed(window_human_unsigned)
                                        
                if hasattr(ssm_fit, 'mOut1'):
                    ssm_model_aims = np.array(ssm_fit.mOut1[pId]).ravel() % 360.0
                elif hasattr(ssm_fit, 'mStates'):
                    ssm_model_aims = np.array(ssm_fit.mStates[pId]).ravel() % 360.0
                else:
                    if debug: print(f"Skipping {datasetName} rot{rotation}: No SSM model aims (mOut1 or mStates).")
                    continue
                window_ssm_unsigned = ssm_model_aims[start_trial:end_trial]
                window_ssm = to_signed(window_ssm_unsigned)
                try:
                    policies_part = policies_full[pId, start_trial:end_trial, :]
                    pi_part = pi_preds_full[pId, start_trial:end_trial, :]
                except (IndexError, ValueError):
                    if debug: print(f"Skipping {datasetName} rot{rotation} pId {pId}: No model predictions.")
                    continue
                sigma = 0.0
                if hasattr(hmm_fit, 'xs') and pId < len(hmm_fit.xs):
                    try:
                        sigma = float(hmm_fit.xs[pId][0])
                        if np.isnan(sigma):
                            sigma = 0.0
                    except Exception:
                        sigma = 0.0
                if datasetName == 'BondTaylor':
                    sigma = 0.0
                kernel = None
                if sigma > 0:
                    support_size = int(6 * sigma) + 1
                    if support_size % 2 == 0:
                        support_size += 1
                    half_support = support_size // 2
                    support = np.arange(-half_support, half_support + 1, dtype=float)
                    kernel = norm_dist.pdf(support, 0, sigma)
                    kernel /= kernel.sum()
                sigma_ssm = 0.0
                if hasattr(ssm_fit, 'xs') and pId < len(ssm_fit.xs):
                    try:
                        sigma_ssm = float(ssm_fit.xs[pId][-1])
                        if np.isnan(sigma_ssm):
                            sigma_ssm = 0.0
                    except Exception:
                        sigma_ssm = 0.0
                if datasetName == 'BondTaylor':
                    sigma_ssm = 0.0
                kernel_ssm = None
                if sigma_ssm > 0:
                    support_size_ssm = int(6 * sigma_ssm) + 1
                    if support_size_ssm % 2 == 0:
                        support_size_ssm += 1
                    half_support_ssm = support_size_ssm // 2
                    support_ssm = np.arange(-half_support_ssm, half_support_ssm + 1, dtype=float)
                    kernel_ssm = norm_dist.pdf(support_ssm, 0, sigma_ssm)
                    kernel_ssm /= kernel_ssm.sum()
                zero_idx = np.argwhere(fine_angles == 0)[0, 0]
                delta_probs = np.zeros(len(fine_angles))
                delta_probs[zero_idx] = 1.0
                convolved0 = ndimage.convolve1d(delta_probs, kernel, mode='wrap') if sigma > 0 else delta_probs
                convolved0 = np.maximum(convolved0, 0)
                convolved0_norm = convolved0 / convolved0.sum() if convolved0.sum() > 0 else convolved0
                for trial in range(n_rotation_trials):
                    human_aim_this = window_human[trial]
                    if np.isnan(human_aim_this):
                        continue
                    state0_prob = pi_part[trial, 0]
                    recentered_probs = policies_part[trial][orig_idx]
                    if sigma > 0:
                        convolved1 = ndimage.convolve1d(recentered_probs, kernel, mode='wrap')
                    else:
                        convolved1 = recentered_probs
                    convolved1 = np.maximum(convolved1, 0)
                    convolved1_norm = convolved1 / convolved1.sum() if convolved1.sum() > 0 else convolved1
                    marginal = state0_prob * convolved0_norm + (1 - state0_prob) * convolved1_norm
                    model_marginals_p.append(marginal)
                    aim_signed = window_ssm[trial]
                    dists = np.abs(fine_angles - aim_signed)
                    ang_dists = np.minimum(dists, 360 - dists)
                    closest_idx = np.argmin(ang_dists)
                    delta_ssm = np.zeros(len(fine_angles))
                    delta_ssm[closest_idx] = 1.0
                    if sigma_ssm > 0:
                        convolved_ssm = ndimage.convolve1d(delta_ssm, kernel_ssm, mode='wrap')
                    else:
                        convolved_ssm = delta_ssm
                    convolved_ssm = np.maximum(convolved_ssm, 0)
                    convolved_ssm_norm = convolved_ssm / convolved_ssm.sum() if convolved_ssm.sum() > 0 else convolved_ssm
                    ssm_marginals_p.append(convolved_ssm_norm)
                    human_aims_p.append(human_aim_this)
                human_aims_p = np.array(human_aims_p)
                n_samples = len(human_aims_p)
                if n_samples < 20:                       
                    if debug:
                        print(f"Skipping pId {pId}: Insufficient samples ({n_samples})")
                    plt.close(fig)
                    continue
                                 
                                       
                if ssm_marginals_p:
                    avg_ssm_marginal = np.mean(ssm_marginals_p, axis=0)
                    hist_ssm, _ = np.histogram(fine_angles, bins=bin_edges, weights=avg_ssm_marginal, density=True)
                    ax.stairs(hist_ssm, bin_edges, fill=False, color='magenta', label='SSM', alpha=0.9,linewidth=4.5)
                                       
                if model_marginals_p:
                    avg_marginal = np.mean(model_marginals_p, axis=0)
                    hist_model, _ = np.histogram(fine_angles, bins=bin_edges, weights=avg_marginal, density=True)
                    ax.stairs(hist_model, bin_edges, fill=False, color='green', label='HMM', alpha=0.9,linewidth=4.5)
                if len(human_aims_p) > 0:
                    hist_human, _ = np.histogram(human_aims_p, bins=bin_edges, density=True)
                    ax.stairs(hist_human, bin_edges, fill=True, color='darkred', label='Human', alpha=0.5,linewidth=0)
                                    
                u2_ssm = watson_u2(human_aims_p, fine_angles, avg_ssm_marginal)
                                    
                u2_hmm = watson_u2(human_aims_p, fine_angles, avg_marginal)
                                  
                text_str = f'HMM U²: {u2_hmm:.3f}\nSSM U²: {u2_ssm:.3f}'
                ax.text(0.02, 0.98, text_str, transform=ax.transAxes, fontsize=10, va='top', ha='left',
                        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
                ax.axvline(0, color='grey', linestyle='--', linewidth=1, alpha=0.8)
                ax.axvline(rotation, color='grey', linestyle='--', linewidth=1, alpha=0.8)
                ax.axvline(-rotation, color='grey', linestyle='--', linewidth=1, alpha=0.8)
                                                    
                xlim_min = min(-60,-2 * abs(rotation))
                xlim_max = max(60,2 * abs(rotation))
                ax.set_xlim(xlim_min, xlim_max)
                ax.set_ylim(0, 0.08)
             
                                                               
                key_points = [-rotation, 0, rotation]
                ax.set_xticks(key_points)
                ax.legend(fontsize=8)
                plt.tight_layout()
                save_path = os.path.join(save_dir, f"{datasetName}_rot{rotation}_pId{pId}_dist.svg")
                plt.savefig(save_path, dpi=300, bbox_inches='tight')
                plt.close(fig)
                if debug: print(f"Saved: {save_path}")
    print("All individual participant distribution plots generated and saved.")
generate_individual_participant_distributions(debug=True, n_participants_to_plot=30, n_bins=45, save_dir='individualModelDistPlots/')

In [ ]:
gc.collect()

In [ ]:

                  
def to_signed(ang):
    return np.where(ang > 180, ang - 360, ang)
def darken_color(hex_color, factor=0.7):
    rgb = mcolors.hex2color(hex_color)
    darkened = tuple(c * factor for c in rgb)
    return mcolors.to_hex(darkened)
def generate_trialseries_adaptation(save_dir=None, debug=False, model_linewidth=5, stroke_linewidth=9):
    """
    Loops over all datasets and generates one trial series plot per dataset,
    plotting each rotation separately on the same panel without normalization.
    If hasImp for the dataset, plots the sum of explicit/aims + implicit for humans, SSM, and HMM.
    Also computes and prints RMSE and R-squared for model means vs human means per rotation.
    """
    if save_dir is None:
        raise ValueError("save_dir must be provided.")
    if 'datasets' not in globals():
        raise ValueError("datasets global not defined! Load your model lists first.")
    rotation_colors = list(reversed(['#44AA99', '#88CCEE', '#FF9825', '#CC6677', '#AA4499']))
    ssm_color = '#FF00FF'                  
    ssm_border_color = darken_color(ssm_color, factor=0.5)
    hmm_color = '#32CD32'                    
    hmm_border_color = darken_color(hmm_color, factor=0.5)
    hasImp = [False, True, False, True]                                       
    ci_factor = stats.norm.ppf(0.975)                      
    for dsIdx, datasetName in enumerate(datasetLabels):
        impFlag = hasImp[dsIdx]
                                           
        if datasetName == 'Brudner':
            baseline_length = 64
            washout_length = 64
        else:
            baseline_length = 40
            washout_length = 40
        modelLists = datasets[dsIdx]                                    
        rotations = rotationMap[dsIdx]
        if debug:
            print(f"\n=== DATASET: {datasetName} | baseline={baseline_length} washout={washout_length} | impFlag={impFlag} ===")
                                                                                 
        human_data = defaultdict(lambda: defaultdict(list))
        ssm_data = defaultdict(lambda: defaultdict(list))
        hmm_data = defaultdict(lambda: defaultdict(list))
        valid_rots = []
        for rotIdx, rotation in enumerate(rotations):
            if rotIdx >= len(modelLists[0]) or rotIdx >= len(modelLists[2]):
                if debug: print(f"Skipping {datasetName} rot{rotation}: rotation index out of range.")
                continue
            ssm_fit = modelLists[0][rotIdx]
            hmm_fit = modelLists[2][rotIdx]
            if ssm_fit is None or hmm_fit is None:
                if debug: print(f"Skipping {datasetName} rot{rotation}: Missing SSM or HMM fit.")
                continue
            if not hasattr(ssm_fit, 'allAims') or len(ssm_fit.allAims) == 0:
                if debug: print(f"Skipping {datasetName} rot{rotation}: No participant data.")
                continue
            n_participants = len(ssm_fit.allAims)
            n_trials_this = len(ssm_fit.allAims[0])
            if n_trials_this < baseline_length + 20 + washout_length:
                if debug: print(f"Skipping {datasetName} rot{rotation}: Insufficient trials ({n_trials_this}).")
                continue
            if rotation == 0:
                if debug: print(f"Skipping {datasetName} rot{rotation}: Zero rotation.")
                continue
            valid_rots.append(rotation)
                                               
            all_imps = None
            if impFlag:
                try:
                    all_imps = ssm_fit.allImps
                except AttributeError:
                    all_imps = None
                    if debug: print(f"Warning: No allImps for {datasetName} rot{rotation}.")
    
                                       
            fine_angles = np.arange(-180, 181)
            orig_idx = ((180 + fine_angles) % 360).astype(int)
            zero_idx = np.argwhere(fine_angles == 0)[0, 0]
            delta_probs = np.zeros(len(fine_angles))
            delta_probs[zero_idx] = 1.0
    
            try:
                policies_full = np.array(hmm_fit.model_predictive_policies)
                pi_preds_full = np.array(hmm_fit.pi_preds)
                hmm_implicits_full = np.array(hmm_fit.model_implicits) if impFlag and hasattr(hmm_fit, 'model_implicits') else None
            except (AttributeError, ValueError):
                if debug: print(f"Skipping {datasetName} rot{rotation}: No predictive policies or pi_preds.")
                continue
    
                                                      
            for t in range(n_trials_this):
                            
                for pId in range(n_participants):
                    aim = ssm_fit.allAims[pId][t]
                    if np.isnan(aim):
                        continue
                    unsigned_aim = aim % 360.0
                    signed_aim = to_signed(unsigned_aim)
                    total = signed_aim
                    if impFlag and all_imps is not None and t < len(all_imps[pId]):
                        imp = all_imps[pId][t]
                        if not np.isnan(imp):
                            unsigned_imp = imp % 360.0
                            signed_imp = to_signed(unsigned_imp)
                            total = signed_aim + signed_imp
                    human_data[rotation][t].append(total)
        
                                            
                for pId in range(n_participants):
                    aim = np.nan
                    if hasattr(ssm_fit, 'mOut1') and pId < len(ssm_fit.mOut1) and t < len(ssm_fit.mOut1[pId]):
                        aim = ssm_fit.mOut1[pId][t]
                    if np.isnan(aim) and hasattr(ssm_fit, 'mStates') and pId < len(ssm_fit.mStates) and t < len(ssm_fit.mStates[pId]):
                        aim = ssm_fit.mStates[pId][t]
                    if np.isnan(aim):
                        continue
                    unsigned_aim = aim % 360.0
                    signed_aim = to_signed(unsigned_aim)
                    total = signed_aim
                    if impFlag and hasattr(ssm_fit, 'mOut2') and pId < len(ssm_fit.mOut2) and t < len(ssm_fit.mOut2[pId]):
                        imp = ssm_fit.mOut2[pId][t]
                        if not np.isnan(imp):
                            unsigned_imp = imp % 360.0
                            signed_imp = to_signed(unsigned_imp)
                            total = signed_aim + signed_imp
                    ssm_data[rotation][t].append(total)
        
                                            
                for pId in range(n_participants):
                    if t >= len(policies_full[pId]) or t >= len(pi_preds_full[pId]):
                        continue
            
                                     
                    sigma_p = 0.0
                    if hasattr(hmm_fit, 'xs') and pId < len(hmm_fit.xs):
                        try:
                            sigma_p = float(hmm_fit.xs[pId][0])
                            if np.isnan(sigma_p):
                                sigma_p = 0.0
                        except:
                            sigma_p = 0.0
            
                                                              
                    if sigma_p > 0:
                        support_size = int(6 * sigma_p) + 1
                        if support_size % 2 == 0:
                            support_size += 1
                        half_support = support_size // 2
                        support = np.arange(-half_support, half_support + 1, dtype=float)
                        kernel = norm.pdf(support, 0, sigma_p)
                        kernel /= kernel.sum()
                        convolved0 = ndimage.convolve1d(delta_probs, kernel, mode='wrap')
                        convolved0_norm = convolved0 / convolved0.sum() if convolved0.sum() > 0 else convolved0
                    else:
                        convolved0_norm = delta_probs
            
                    policies_part_t = policies_full[pId, t, :]
                    pi_part_t = pi_preds_full[pId, t, :]
                    state0_prob = pi_part_t[0]
            
                    recentered_probs = policies_part_t[orig_idx]
                    recentered_probs /= recentered_probs.sum() if recentered_probs.sum() > 0 else recentered_probs
            
                    if sigma_p > 0:
                        convolved1 = ndimage.convolve1d(recentered_probs, kernel, mode='wrap')
                        convolved1_norm = convolved1 / convolved1.sum() if convolved1.sum() > 0 else convolved1
                    else:
                        convolved1_norm = recentered_probs
                    
                    convolved1 = np.maximum(convolved1, 0)
                    convolved0 = np.maximum(convolved0, 0)
                    marginal = state0_prob * convolved0_norm + (1 - state0_prob) * convolved1_norm
                    pred_aim = np.sum(marginal * fine_angles)
                    total = pred_aim
                    if impFlag and hmm_implicits_full is not None and t < hmm_implicits_full.shape[1]:
                        imp = hmm_implicits_full[pId, t]
                        if not np.isnan(imp):
                            unsigned_imp = imp % 360.0
                            signed_imp = to_signed(unsigned_imp)
                            total = pred_aim + signed_imp
                    hmm_data[rotation][t].append(total)
        if not human_data:
            if debug: print(f"No valid data for {datasetName}.")
            continue
                                                         
        n_trials = min(len(human_data[rot]) for rot in human_data)
        trials = np.arange(n_trials)
                                                
        valid_rotations = sorted(human_data.keys())
              
        fig, ax = plt.subplots(figsize=(12, 8))
        ssm_label = f'SSM {"(Aims + Implicits)" if impFlag else "Aims"}'
        hmm_label = f'HMM {"(Aims + Implicits)" if impFlag else "Aims"}'
        ssm_metrics = {}
        hmm_metrics = {}
        for i, rotation in enumerate(valid_rotations):
          
            ax.axhline(rotation, color='gray', linestyle='--', linewidth=1, alpha=0.8)
            color = rotation_colors[i % len(rotation_colors)]
            dark_color = darken_color(color)
    
                                            
            human_vals = [human_data[rotation].get(t, []) for t in trials]
            human_mean = np.array([np.mean(vals) if vals else np.nan for vals in human_vals])
            human_sem = np.array([sem(vals) if vals else np.nan for vals in human_vals])
            human_ci = ci_factor * human_sem
            human_label = f'Human {"(Explicit + Implicit)" if impFlag else "Explicit"} {rotation}°'
            ax.plot(trials, human_mean, marker='o', markersize=6, color=color, linestyle='None',
                    markeredgecolor=dark_color, markeredgewidth=1.5,
                    label=human_label)
            ax.fill_between(trials, human_mean - human_ci, human_mean + human_ci,
                            color=color, alpha=0.2)
    
                                      
            ssm_vals = [ssm_data[rotation].get(t, []) for t in trials]
            ssm_has_data = any(len(vals) > 0 for vals in ssm_vals)
            ssm_mean = np.full(n_trials, np.nan)
            ssm_sem = np.full(n_trials, np.nan)
            ssm_ci = np.full(n_trials, np.nan)
            if ssm_has_data:
                ssm_mean = np.array([np.mean(vals) if vals else np.nan for vals in ssm_vals])
                ssm_sem = np.array([sem(vals) if vals else np.nan for vals in ssm_vals])
                ssm_ci = ci_factor * ssm_sem
                ssm_line, = ax.plot(trials, ssm_mean, '-', color=ssm_color, linewidth=model_linewidth, linestyle='--',
                                    label=ssm_label if i == 0 else None)
                ssm_line.set_path_effects([withStroke(linewidth=stroke_linewidth, foreground=ssm_border_color)])
                ax.fill_between(trials, ssm_mean - ssm_ci, ssm_mean + ssm_ci,
                                color=ssm_color, alpha=0.05)
            elif debug:
                print(f"No SSM data for {datasetName} rot{rotation}")
    
                                      
            hmm_vals = [hmm_data[rotation].get(t, []) for t in trials]
            hmm_has_data = any(len(vals) > 0 for vals in hmm_vals)
            hmm_mean = np.full(n_trials, np.nan)
            hmm_sem = np.full(n_trials, np.nan)
            hmm_ci = np.full(n_trials, np.nan)
            if hmm_has_data:
                hmm_mean = np.array([np.mean(vals) if vals else np.nan for vals in hmm_vals])
                hmm_sem = np.array([sem(vals) if vals else np.nan for vals in hmm_vals])
                hmm_ci = ci_factor * hmm_sem
                hmm_line, = ax.plot(trials, hmm_mean, '-', color=hmm_color, linewidth=model_linewidth, linestyle='--',
                                    label=hmm_label if i == 0 else None)
                hmm_line.set_path_effects([withStroke(linewidth=stroke_linewidth, foreground=hmm_border_color)])
                ax.fill_between(trials, hmm_mean - hmm_ci, hmm_mean + hmm_ci,
                                color=hmm_color, alpha=0.2)
            elif debug:
                print(f"No HMM data for {datasetName} rot{rotation}")
                                                         
            rmse_ssm = np.nan
            r2_ssm = np.nan
            if ssm_has_data:
                valid_idx_ssm = ~np.isnan(human_mean) & ~np.isnan(ssm_mean)
                if np.sum(valid_idx_ssm) > 1:
                    diff_ssm = human_mean[valid_idx_ssm] - ssm_mean[valid_idx_ssm]
                    rmse_ssm = np.sqrt(np.mean(diff_ssm ** 2))
                    ss_res_ssm = np.sum(diff_ssm ** 2)
                    human_valid_ssm = human_mean[valid_idx_ssm]
                    ss_tot_ssm = np.sum((human_valid_ssm - np.mean(human_valid_ssm)) ** 2)
                    r2_ssm = 1 - (ss_res_ssm / ss_tot_ssm) if ss_tot_ssm > 0 else np.nan
                                                         
            rmse_hmm = np.nan
            r2_hmm = np.nan
            if hmm_has_data:
                valid_idx_hmm = ~np.isnan(human_mean) & ~np.isnan(hmm_mean)
                if np.sum(valid_idx_hmm) > 1:
                    diff_hmm = human_mean[valid_idx_hmm] - hmm_mean[valid_idx_hmm]
                    rmse_hmm = np.sqrt(np.mean(diff_hmm ** 2))
                    ss_res_hmm = np.sum(diff_hmm ** 2)
                    human_valid_hmm = human_mean[valid_idx_hmm]
                    ss_tot_hmm = np.sum((human_valid_hmm - np.mean(human_valid_hmm)) ** 2)
                    r2_hmm = 1 - (ss_res_hmm / ss_tot_hmm) if ss_tot_hmm > 0 else np.nan
                                                               
            print(f"\nRotation {rotation}° (color: {color}):")
            print(f" SSM: RMSE = {rmse_ssm:.2f}, R² = {r2_ssm:.3f}")
            print(f" HMM: RMSE = {rmse_hmm:.2f}, R² = {r2_hmm:.3f}")
            ssm_metrics[rotation] = (rmse_ssm, r2_ssm)
            hmm_metrics[rotation] = (rmse_hmm, r2_hmm)
                                                                      
        x_text = n_trials - 25
        line_height = 2.5                              
        value_offset_x = 23                                         
        for rotation in valid_rotations:
            rmse_s, r2_s = ssm_metrics[rotation]
            rmse_h, r2_h = hmm_metrics[rotation]
            if np.isnan(r2_s) or np.isnan(r2_h):
                continue
            y_base = rotation - 5
            x_base = x_text
                      
            r2_label = ax.text(x_base, y_base, 'R² =', color='black', fontsize=7, ha='left', va='top')
            r2_label.set_path_effects([withStroke(linewidth=0.5, foreground='white')])
                                  
            ssm_r2_value = ax.text(x_base + value_offset_x, y_base + 0.5 * line_height, f'{r2_s:.3f}',
                    color=ssm_color, fontsize=7, ha='left', va='top')
            ssm_r2_value.set_path_effects([withStroke(linewidth=0.5, foreground='black')])
                                      
            hmm_r2_value = ax.text(x_base + value_offset_x, y_base - 0.5 * line_height, f'{r2_h:.3f}',
                    color=hmm_color, fontsize=7, ha='left', va='top')
            hmm_r2_value.set_path_effects([withStroke(linewidth=0.5, foreground='black')])
                                   
            y_rmse_base = y_base - 3 * line_height
            rmse_label = ax.text(x_base, y_rmse_base, 'RMSE =', color='black', fontsize=7, ha='left', va='top')
            rmse_label.set_path_effects([withStroke(linewidth=0.5, foreground='white')])
                                    
            ssm_rmse_value = ax.text(x_base + value_offset_x, y_rmse_base + 0.5 * line_height, f'{rmse_s:.2f}',
                    color=ssm_color, fontsize=7, ha='left', va='top')
            ssm_rmse_value.set_path_effects([withStroke(linewidth=0.5, foreground='black')])
                                        
            hmm_rmse_value = ax.text(x_base + value_offset_x, y_rmse_base - 0.5 * line_height, f'{rmse_h:.2f}',
                    color=hmm_color, fontsize=7, ha='left', va='top')
            hmm_rmse_value.set_path_effects([withStroke(linewidth=0.5, foreground='black')])
        rotation_start = baseline_length
        washout_start = n_trials - washout_length
        ax.axvline(rotation_start, color='gray', linestyle='--', linewidth=1, alpha=0.8, label='Rotation Onset')
        ax.axvline(washout_start, color='gray', linestyle='--', linewidth=1, alpha=0.8, label='Washout Onset')
        ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.8)
        ax.set_xlabel('Trial Number')
        ax.set_ylabel('Signed Adaptation (°)')
        ax.set_title(f'{datasetName} | Trial Series by Rotation')
        ax.legend(loc='best')
                                 
        plt.tight_layout()
        save_path = os.path.join(save_dir, f"{datasetName}_trialseries_by_rotation.svg")
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        if debug: print(f"Saved: {save_path}")
    print("All trial series by rotation plots generated and saved.")
                       
generate_trialseries_adaptation(save_dir=targetFolder, debug=True)

In [ ]:

                  
def to_signed(ang):
    return np.where(ang > 180, ang - 360, ang)
def darken_color(hex_color, factor=0.7):
    rgb = mcolors.hex2color(hex_color)
    darkened = tuple(c * factor for c in rgb)
    return mcolors.to_hex(darkened)
def generate_trialseries_adaptation(save_dir=None, debug=False, model_linewidth=5, stroke_linewidth=9):
    """
    Modified to only process BondTaylor dataset, plot each rotation in a separate panel,
    and plot explicit/aims and implicits as separate series on the same panel.
    Model data plotted on top with twice-darkened colors.
    Implicits always blue, explicits/aims use original color convention, all markers 'o'.
    Added SSM plotting: explicit in magenta, implicit in darker magenta.
    HMM explicit in limegreen, implicit in darker green.
    Model dashed lines have borders using path_effects for perfect alignment.
    Updated to compute and print R² and RMSE separately for aims and implicits vs. human.
    Added annotations in each panel for the metrics, styled like the combined plot, positioned to avoid overlap.
    """
    if save_dir is None:
        raise ValueError("save_dir must be provided.")
    if 'datasets' not in globals():
        raise ValueError("datasets global not defined! Load your model lists first.")
    rotation_colors = list(reversed(['#44AA99', '#88CCEE', '#FF9825', '#CC6677', '#AA4499']))
    imp_color = '#0072B2'                           
    hmm_aim_color = '#32CD32'                             
    hmm_imp_color = darken_color(hmm_aim_color)                                
    ssm_aim_color = '#FF00FF'                           
    ssm_imp_color = darken_color(ssm_aim_color)                                  
    hasImp = [False, True, False, True]                                       
    ci_factor = stats.norm.ppf(0.975)                      
    for dsIdx, datasetName in enumerate(datasetLabels):
        if datasetName != 'BondTaylor':
            continue
        impFlag = hasImp[dsIdx]
                                           
        if datasetName == 'Brudner':
            baseline_length = 64
            washout_length = 64
        else:
            baseline_length = 40
            washout_length = 40
        modelLists = datasets[dsIdx]                                    
        rotations = rotationMap[dsIdx]
        if debug:
            print(f"\n=== DATASET: {datasetName} | baseline={baseline_length} washout={washout_length} | impFlag={impFlag} ===")
  
                                                                                 
        human_aims = defaultdict(lambda: defaultdict(list))
        human_imps = defaultdict(lambda: defaultdict(list))
        ssm_aims = defaultdict(lambda: defaultdict(list))
        ssm_imps = defaultdict(lambda: defaultdict(list))
        hmm_aims = defaultdict(lambda: defaultdict(list))
        hmm_imps = defaultdict(lambda: defaultdict(list))
  
        valid_rots = []
        for rotIdx, rotation in enumerate(rotations):
            if rotIdx >= len(modelLists[0]) or rotIdx >= len(modelLists[2]):
                if debug: print(f"Skipping {datasetName} rot{rotation}: rotation index out of range.")
                continue
            ssm_fit = modelLists[0][rotIdx]
            hmm_fit = modelLists[2][rotIdx]
            if ssm_fit is None or hmm_fit is None:
                if debug: print(f"Skipping {datasetName} rot{rotation}: Missing SSM or HMM fit.")
                continue
            if not hasattr(ssm_fit, 'allAims') or len(ssm_fit.allAims) == 0:
                if debug: print(f"Skipping {datasetName} rot{rotation}: No participant data.")
                continue
            n_participants = len(ssm_fit.allAims)
            n_trials_this = len(ssm_fit.allAims[0])
            if n_trials_this < baseline_length + 20 + washout_length:
                if debug: print(f"Skipping {datasetName} rot{rotation}: Insufficient trials ({n_trials_this}).")
                continue
            if rotation == 0:
                if debug: print(f"Skipping {datasetName} rot{rotation}: Zero rotation.")
                continue
            valid_rots.append(rotation)
                                               
            all_imps = None
            if impFlag:
                try:
                    all_imps = ssm_fit.allImps
                except AttributeError:
                    all_imps = None
                    if debug: print(f"Warning: No allImps for {datasetName} rot{rotation}.")
      
                                               
            fine_angles = np.arange(-180, 181)
            orig_idx = ((180 + fine_angles) % 360).astype(int)
            zero_idx = np.argwhere(fine_angles == 0)[0, 0]
            delta_probs = np.zeros(len(fine_angles))
            delta_probs[zero_idx] = 1.0
      
            try:
                policies_full = np.array(hmm_fit.model_predictive_policies)
                pi_preds_full = np.array(hmm_fit.pi_preds)
                model_implicits_full = np.array(hmm_fit.model_implicits) if impFlag and hasattr(hmm_fit, 'model_implicits') else None
            except (AttributeError, ValueError):
                if debug: print(f"Skipping {datasetName} rot{rotation}: No predictive policies or pi_preds.")
                continue
      
                                                      
            for t in range(n_trials_this):
                            
                for pId in range(n_participants):
                    aim = ssm_fit.allAims[pId][t]
                    if np.isnan(aim):
                        continue
                    unsigned_aim = aim % 360.0
                    signed_aim = to_signed(unsigned_aim)
                    human_aims[rotation][t].append(signed_aim)
                    if impFlag and all_imps is not None and t < len(all_imps[pId]):
                        imp = all_imps[pId][t]
                        if not np.isnan(imp):
                            unsigned_imp = imp % 360.0
                            signed_imp = to_signed(unsigned_imp)
                            human_imps[rotation][t].append(signed_imp)
          
                                            
                for pId in range(n_participants):
                                      
                    if hasattr(ssm_fit, 'mOut1') and t < len(ssm_fit.mOut1[pId]):
                        exp = ssm_fit.mOut1[pId][t]
                        if not np.isnan(exp):
                            unsigned_exp = exp % 360.0
                            signed_exp = to_signed(unsigned_exp)
                            ssm_aims[rotation][t].append(signed_exp)
                                                    
                    if impFlag and hasattr(ssm_fit, 'mOut2') and t < len(ssm_fit.mOut2[pId]):
                        imp = ssm_fit.mOut2[pId][t]
                        if not np.isnan(imp):
                            unsigned_imp = imp % 360.0
                            signed_imp = to_signed(unsigned_imp)
                            ssm_imps[rotation][t].append(signed_imp)
          
                                            
                for pId in range(n_participants):
                    if t >= len(policies_full[pId]) or t >= len(pi_preds_full[pId]):
                        continue
              
                                     
                    sigma_p = 0.0
                    if hasattr(hmm_fit, 'xs') and pId < len(hmm_fit.xs):
                        try:
                            sigma_p = float(hmm_fit.xs[pId][0])
                            if np.isnan(sigma_p):
                                sigma_p = 0.0
                        except:
                            sigma_p = 0.0
              
                                                              
                    if sigma_p > 0:
                        support_size = int(6 * sigma_p) + 1
                        if support_size % 2 == 0:
                            support_size += 1
                        half_support = support_size // 2
                        support = np.arange(-half_support, half_support + 1, dtype=float)
                        kernel = norm.pdf(support, 0, sigma_p)
                        kernel /= kernel.sum()
                        convolved0 = ndimage.convolve1d(delta_probs, kernel, mode='wrap')
                        convolved0_norm = convolved0 / convolved0.sum() if convolved0.sum() > 0 else convolved0
                    else:
                        convolved0_norm = delta_probs
              
                    policies_part_t = policies_full[pId, t, :]
                    pi_part_t = pi_preds_full[pId, t, :]
                    state0_prob = pi_part_t[0]
              
                    recentered_probs = policies_part_t[orig_idx]
                    recentered_probs /= recentered_probs.sum() if recentered_probs.sum() > 0 else recentered_probs
              
                    if sigma_p > 0:
                        convolved1 = ndimage.convolve1d(recentered_probs, kernel, mode='wrap')
                        convolved1_norm = convolved1 / convolved1.sum() if convolved1.sum() > 0 else convolved1
                    else:
                        convolved1_norm = recentered_probs
                    
                    convolved1 = np.maximum(convolved1, 0)
                    convolved0 = np.maximum(convolved0, 0)
                    marginal = state0_prob * convolved0_norm + (1 - state0_prob) * convolved1_norm
                    pred_aim = np.sum(marginal * fine_angles)
                    hmm_aims[rotation][t].append(pred_aim)
                    if impFlag and model_implicits_full is not None and t < model_implicits_full.shape[1]:
                        imp = model_implicits_full[pId, t]
                        if not np.isnan(imp):
                            unsigned_imp = imp % 360.0
                            signed_imp = to_signed(unsigned_imp)
                            hmm_imps[rotation][t].append(signed_imp)
  
        if len(human_aims) == 0:
            if debug: print(f"No valid data for {datasetName}.")
            continue
  
                                                         
        n_trials = min(len(human_aims[rot]) for rot in human_aims)
        trials = np.arange(n_trials)
  
                                                
        valid_rotations = sorted(human_aims.keys())
        num_rots = len(valid_rotations)
  
                                                
        nrows = 2
        ncols = 3
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 6 * nrows), sharex=True, sharey=True)
        rotation_start = baseline_length
        washout_start = n_trials - washout_length
        for i, rotation in enumerate(valid_rotations):
            row = i // ncols
            col = i % ncols
            ax = axes[row, col]
            color = rotation_colors[i % len(rotation_colors)]
            dark_color = darken_color(color)
      
                                                   
            human_aim_vals = [human_aims[rotation].get(t, []) for t in trials]
            human_aim_mean = np.array([np.mean(vals) if vals else np.nan for vals in human_aim_vals])
            human_aim_sem = np.array([sem(vals) if vals else np.nan for vals in human_aim_vals])
            human_aim_ci = ci_factor * human_aim_sem
            ax.plot(trials, human_aim_mean, marker='o', markersize=6, color=color, linestyle='None',
                    markeredgecolor=dark_color, markeredgewidth=1.5,
                    label='Human Explicit')
            ax.fill_between(trials, human_aim_mean - human_aim_ci, human_aim_mean + human_aim_ci,
                            color=color, alpha=0.2)
      
                                                                   
            human_imp_mean = None
            human_imp_sem = None
            human_imp_ci = None
            has_human_imp = False
            if impFlag:
                human_imp_vals = [human_imps[rotation].get(t, []) for t in trials]
                if any(len(vals) > 0 for vals in human_imp_vals):
                    human_imp_mean = np.array([np.mean(vals) if vals else np.nan for vals in human_imp_vals])
                    human_imp_sem = np.array([sem(vals) if vals else np.nan for vals in human_imp_vals])
                    human_imp_ci = ci_factor * human_imp_sem
                    ax.plot(trials, human_imp_mean, marker='o', markersize=6, color=imp_color, linestyle='None',
                            markeredgecolor=darken_color(imp_color), markeredgewidth=1.5,
                            label='Human Implicit')
                    ax.fill_between(trials, human_imp_mean - human_imp_ci, human_imp_mean + human_imp_ci,
                                    color=imp_color, alpha=0.2)
                    has_human_imp = True
      
                                             
            ssm_aim_mean = None
            ssm_aim_sem = None
            ssm_aim_ci = None
            has_ssm_aim = False
            ssm_aim_vals = [ssm_aims[rotation].get(t, []) for t in trials]
            if any(len(vals) > 0 for vals in ssm_aim_vals):
                ssm_aim_mean = np.array([np.mean(vals) if vals else np.nan for vals in ssm_aim_vals])
                ssm_aim_sem = np.array([sem(vals) if vals else np.nan for vals in ssm_aim_vals])
                ssm_aim_ci = ci_factor * ssm_aim_sem
                has_ssm_aim = True
                ssm_aim_border_color = darken_color(ssm_aim_color, factor=0.5)
                ssm_aim_line, = ax.plot(trials, ssm_aim_mean, '-', color=ssm_aim_color, linewidth=model_linewidth, linestyle='--', label='SSM Aims')
                ssm_aim_line.set_path_effects([withStroke(linewidth=stroke_linewidth, foreground=ssm_aim_border_color)])
                ax.fill_between(trials, ssm_aim_mean - ssm_aim_ci, ssm_aim_mean + ssm_aim_ci,
                                color=ssm_aim_color, alpha=0.05)
      
                                                    
            ssm_imp_mean = None
            ssm_imp_sem = None
            ssm_imp_ci = None
            has_ssm_imp = False
            if impFlag:
                ssm_imp_vals = [ssm_imps[rotation].get(t, []) for t in trials]
                if any(len(vals) > 0 for vals in ssm_imp_vals):
                    ssm_imp_mean = np.array([np.mean(vals) if vals else np.nan for vals in ssm_imp_vals])
                    ssm_imp_sem = np.array([sem(vals) if vals else np.nan for vals in ssm_imp_vals])
                    ssm_imp_ci = ci_factor * ssm_imp_sem
                    has_ssm_imp = True
                    ssm_imp_border_color = darken_color(ssm_imp_color, factor=0.5)
                    ssm_imp_line, = ax.plot(trials, ssm_imp_mean, '-', color=ssm_imp_color, linewidth=model_linewidth, linestyle='--', label='SSM Implicits')
                    ssm_imp_line.set_path_effects([withStroke(linewidth=stroke_linewidth, foreground=ssm_imp_border_color)])
                    ax.fill_between(trials, ssm_imp_mean - ssm_imp_ci, ssm_imp_mean + ssm_imp_ci,
                                    color=ssm_imp_color, alpha=0.05)
      
                                                  
            hmm_aim_vals = [hmm_aims[rotation].get(t, []) for t in trials]
            hmm_aim_mean = np.array([np.mean(vals) if vals else np.nan for vals in hmm_aim_vals])
            hmm_aim_sem = np.array([sem(vals) if vals else np.nan for vals in hmm_aim_vals])
            hmm_aim_ci = ci_factor * hmm_aim_sem
            hmm_aim_border_color = darken_color(hmm_aim_color, factor=0.5)
            hmm_aim_line, = ax.plot(trials, hmm_aim_mean, '-', color=hmm_aim_color, linewidth=model_linewidth, linestyle='--', label='HMM Aims')
            hmm_aim_line.set_path_effects([withStroke(linewidth=stroke_linewidth, foreground=hmm_aim_border_color)])
            ax.fill_between(trials, hmm_aim_mean - hmm_aim_ci, hmm_aim_mean + hmm_aim_ci,
                            color=hmm_aim_color, alpha=0.2)
      
                                                  
            hmm_imp_mean = None
            hmm_imp_sem = None
            hmm_imp_ci = None
            has_hmm_imp = False
            if impFlag:
                hmm_imp_vals = [hmm_imps[rotation].get(t, []) for t in trials]
                if any(len(vals) > 0 for vals in hmm_imp_vals):
                    hmm_imp_mean = np.array([np.mean(vals) if vals else np.nan for vals in hmm_imp_vals])
                    hmm_imp_sem = np.array([sem(vals) if vals else np.nan for vals in hmm_imp_vals])
                    hmm_imp_ci = ci_factor * hmm_imp_sem
                    has_hmm_imp = True
                    hmm_imp_border_color = darken_color(hmm_imp_color, factor=0.5)
                    hmm_imp_line, = ax.plot(trials, hmm_imp_mean, '-', color=hmm_imp_color, linewidth=model_linewidth, linestyle='--', label='HMM Implicits')
                    hmm_imp_line.set_path_effects([withStroke(linewidth=stroke_linewidth, foreground=hmm_imp_border_color)])
                    ax.fill_between(trials, hmm_imp_mean - hmm_imp_ci, hmm_imp_mean + hmm_imp_ci,
                                    color=hmm_imp_color, alpha=0.2)
      
                                                
            rmse_ssm_aim = np.nan
            r2_ssm_aim = np.nan
            if has_ssm_aim:
                valid_idx_ssm_aim = ~np.isnan(human_aim_mean) & ~np.isnan(ssm_aim_mean)
                if np.sum(valid_idx_ssm_aim) > 1:
                    diff_ssm_aim = human_aim_mean[valid_idx_ssm_aim] - ssm_aim_mean[valid_idx_ssm_aim]
                    rmse_ssm_aim = np.sqrt(np.mean(diff_ssm_aim ** 2))
                    ss_res_ssm_aim = np.sum(diff_ssm_aim ** 2)
                    human_valid_ssm_aim = human_aim_mean[valid_idx_ssm_aim]
                    ss_tot_ssm_aim = np.sum((human_valid_ssm_aim - np.mean(human_valid_ssm_aim)) ** 2)
                    r2_ssm_aim = 1 - (ss_res_ssm_aim / ss_tot_ssm_aim) if ss_tot_ssm_aim > 0 else np.nan
           
            rmse_hmm_aim = np.nan
            r2_hmm_aim = np.nan
            valid_idx_hmm_aim = ~np.isnan(human_aim_mean) & ~np.isnan(hmm_aim_mean)
            if np.sum(valid_idx_hmm_aim) > 1:
                diff_hmm_aim = human_aim_mean[valid_idx_hmm_aim] - hmm_aim_mean[valid_idx_hmm_aim]
                rmse_hmm_aim = np.sqrt(np.mean(diff_hmm_aim ** 2))
                ss_res_hmm_aim = np.sum(diff_hmm_aim ** 2)
                human_valid_hmm_aim = human_aim_mean[valid_idx_hmm_aim]
                ss_tot_hmm_aim = np.sum((human_valid_hmm_aim - np.mean(human_valid_hmm_aim)) ** 2)
                r2_hmm_aim = 1 - (ss_res_hmm_aim / ss_tot_hmm_aim) if ss_tot_hmm_aim > 0 else np.nan
           
                                                                   
            rmse_ssm_imp = np.nan
            r2_ssm_imp = np.nan
            rmse_hmm_imp = np.nan
            r2_hmm_imp = np.nan
            has_imp_metrics = False
            if impFlag and has_human_imp:
                         
                if has_ssm_imp:
                    valid_idx_ssm_imp = ~np.isnan(human_imp_mean) & ~np.isnan(ssm_imp_mean)
                    if np.sum(valid_idx_ssm_imp) > 1:
                        diff_ssm_imp = human_imp_mean[valid_idx_ssm_imp] - ssm_imp_mean[valid_idx_ssm_imp]
                        rmse_ssm_imp = np.sqrt(np.mean(diff_ssm_imp ** 2))
                        ss_res_ssm_imp = np.sum(diff_ssm_imp ** 2)
                        human_valid_ssm_imp = human_imp_mean[valid_idx_ssm_imp]
                        ss_tot_ssm_imp = np.sum((human_valid_ssm_imp - np.mean(human_valid_ssm_imp)) ** 2)
                        r2_ssm_imp = 1 - (ss_res_ssm_imp / ss_tot_ssm_imp) if ss_tot_ssm_imp > 0 else np.nan
                         
                if has_hmm_imp:
                    valid_idx_hmm_imp = ~np.isnan(human_imp_mean) & ~np.isnan(hmm_imp_mean)
                    if np.sum(valid_idx_hmm_imp) > 1:
                        diff_hmm_imp = human_imp_mean[valid_idx_hmm_imp] - hmm_imp_mean[valid_idx_hmm_imp]
                        rmse_hmm_imp = np.sqrt(np.mean(diff_hmm_imp ** 2))
                        ss_res_hmm_imp = np.sum(diff_hmm_imp ** 2)
                        human_valid_hmm_imp = human_imp_mean[valid_idx_hmm_imp]
                        ss_tot_hmm_imp = np.sum((human_valid_hmm_imp - np.mean(human_valid_hmm_imp)) ** 2)
                        r2_hmm_imp = 1 - (ss_res_hmm_imp / ss_tot_hmm_imp) if ss_tot_hmm_imp > 0 else np.nan
                has_imp_metrics = True
           
                              
            print(f"\nRotation {rotation}°:")
            print(f" Aim SSM: RMSE = {rmse_ssm_aim:.2f}, R² = {r2_ssm_aim:.3f}")
            print(f" Aim HMM: RMSE = {rmse_hmm_aim:.2f}, R² = {r2_hmm_aim:.3f}")
            if has_imp_metrics:
                print(f" Imp SSM: RMSE = {rmse_ssm_imp:.2f}, R² = {r2_ssm_imp:.3f}")
                print(f" Imp HMM: RMSE = {rmse_hmm_imp:.2f}, R² = {r2_hmm_imp:.3f}")
      
            ax.axvline(rotation_start, color='gray', linestyle='--', linewidth=1, alpha=0.8,
                       label='Rotation Onset' if i == 0 else "")
            ax.axvline(washout_start, color='gray', linestyle='--', linewidth=1, alpha=0.8,
                       label='Washout Onset' if i == 0 else "")
            ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.8)
            ax.axhline(rotation, color='gray', linestyle='--', linewidth=1, alpha=0.8)
      
                                                                                                     
            x_text = n_trials - 80
            value_offset_x = 60
            line_height = 5                              
            y_base_aim = rotation + 40                               
            current_y = y_base_aim
      
                    
            r2_label = ax.text(x_text, current_y, 'Aim R² =', color='black', fontsize=7, ha='left', va='top')
            r2_label.set_path_effects([withStroke(linewidth=0.5, foreground='white')])
            ssm_r2_value = ax.text(x_text + value_offset_x, current_y + 0.5 * line_height, f'{r2_ssm_aim:.3f}',
                                   color=ssm_aim_color, fontsize=7, ha='left', va='top')
            ssm_r2_value.set_path_effects([withStroke(linewidth=0.5, foreground='black')])
            hmm_r2_value = ax.text(x_text + value_offset_x, current_y - 0.5 * line_height, f'{r2_hmm_aim:.3f}',
                                   color=hmm_aim_color, fontsize=7, ha='left', va='top')
            hmm_r2_value.set_path_effects([withStroke(linewidth=0.5, foreground='black')])
            current_y -= 2 * line_height
      
                      
            rmse_label = ax.text(x_text, current_y, 'Aim RMSE =', color='black', fontsize=7, ha='left', va='top')
            rmse_label.set_path_effects([withStroke(linewidth=0.5, foreground='white')])
            ssm_rmse_value = ax.text(x_text + value_offset_x, current_y + 0.5 * line_height, f'{rmse_ssm_aim:.2f}',
                                     color=ssm_aim_color, fontsize=7, ha='left', va='top')
            ssm_rmse_value.set_path_effects([withStroke(linewidth=0.5, foreground='black')])
            hmm_rmse_value = ax.text(x_text + value_offset_x, current_y - 0.5 * line_height, f'{rmse_hmm_aim:.2f}',
                                     color=hmm_aim_color, fontsize=7, ha='left', va='top')
            hmm_rmse_value.set_path_effects([withStroke(linewidth=0.5, foreground='black')])
            current_y -= 2 * line_height
      
                                       
            if has_imp_metrics:
                        
                imp_r2_label = ax.text(x_text, current_y, 'Imp R² =', color='black', fontsize=7, ha='left', va='top')
                imp_r2_label.set_path_effects([withStroke(linewidth=0.5, foreground='white')])
                ssm_imp_r2_value = ax.text(x_text + value_offset_x, current_y + 0.5 * line_height, f'{r2_ssm_imp:.3f}',
                                           color=ssm_imp_color, fontsize=7, ha='left', va='top')
                ssm_imp_r2_value.set_path_effects([withStroke(linewidth=0.5, foreground='black')])
                hmm_imp_r2_value = ax.text(x_text + value_offset_x, current_y - 0.5 * line_height, f'{r2_hmm_imp:.3f}',
                                           color=hmm_imp_color, fontsize=7, ha='left', va='top')
                hmm_imp_r2_value.set_path_effects([withStroke(linewidth=0.5, foreground='black')])
                current_y -= 2 * line_height
      
                          
                imp_rmse_label = ax.text(x_text, current_y, 'Imp RMSE =', color='black', fontsize=7, ha='left', va='top')
                imp_rmse_label.set_path_effects([withStroke(linewidth=0.5, foreground='white')])
                ssm_imp_rmse_value = ax.text(x_text + value_offset_x, current_y + 0.5 * line_height, f'{rmse_ssm_imp:.2f}',
                                             color=ssm_imp_color, fontsize=7, ha='left', va='top')
                ssm_imp_rmse_value.set_path_effects([withStroke(linewidth=0.5, foreground='black')])
                hmm_imp_rmse_value = ax.text(x_text + value_offset_x, current_y - 0.5 * line_height, f'{rmse_hmm_imp:.2f}',
                                             color=hmm_imp_color, fontsize=7, ha='left', va='top')
                hmm_imp_rmse_value.set_path_effects([withStroke(linewidth=0.5, foreground='black')])
      
            ax.set_title(f'{rotation}°')
            ax.legend(loc='best')
            sns.despine()
  
                          
        for i in range(num_rots, nrows * ncols):
            row = i // ncols
            col = i % ncols
            axes[row, col].axis('off')
  
        axes[0, 0].set_ylabel('Signed Adaptation (°)')
        for ax in axes[-1, :]:
            ax.set_xlabel('Trial Number')
        fig.suptitle(f'{datasetName} | Trial Series by Rotation (Separate Explicit/Aims and Implicits)')
        plt.tight_layout()
        save_path = os.path.join(save_dir, f"{datasetName}_trialseries_by_rotation_separate.svg")
        
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        if debug: print(f"Saved: {save_path}")
    print("BondTaylor trial series plots (separate panels and series) generated and saved.")
                       
generate_trialseries_adaptation(save_dir=targetFolder, debug=True)

In [ ]:


                                                   
def to_signed(ang):
    return np.where(ang > 180, ang - 360, ang)

                                                               
                                                           
def simulate_implicit_only(targets, allAims_part, rotation,
                          gen_type, lr2, retain2, sigmaGen=30):
    targets = np.asarray(targets)
    allAims_part = np.asarray(allAims_part)
    n = len(targets)
    rots = np.full(n, float(rotation))
   
    implicit = np.zeros(n)
    angles = np.arange(0.0, 360.0)
    x2Vec = np.zeros(360)
   
    signed_aims = to_signed(allAims_part % 360.0)
    out2 = 0
    pe = 0
    g2Vec = 0
    for i in range(n):
        if gen_type == 'target':
            raw_center = targets[i]
        elif gen_type == 'aim':
            raw_center = targets[i] + signed_aims[i]
        else:
            raise ValueError("gen_type must be 'target' or 'aim'")
        if not np.isnan(raw_center):
            idx = int(np.round(raw_center)) % 360
            center = raw_center % 360.0
           
            out2 = x2Vec[idx]
            implicit[i] = out2
           
            pe = rots[i] + out2
           
            angular_diff = np.minimum(np.abs(angles - center), 360.0 - np.abs(angles - center))
            g2Vec = np.exp(-(angular_diff**2) / (2 * sigmaGen**2))
           
            x2Vec = retain2 * x2Vec - lr2 * pe * g2Vec
        else:
            implicit[i] = out2
            x2Vec = retain2 * x2Vec - lr2 * pe * g2Vec
    return implicit

                                
data_rows = []

                                                                      
target_rotations = [15, 30, 45, 60, 90]

                                                     
                                                                             
rotation_params = {
    15: (0.0385, 0.9985),                 
    30: (0.0152, 0.9956),
    45: (0.0152, 0.9952),
    60: (0.0118, 0.9875),
    90: (0.0044, 0.9918)
}

participant_counter = 0

if 'datasets' not in globals() or 'datasetLabels' not in globals() or 'rotationMap' not in globals() or 'hasImp' not in globals():
    raise ValueError("Required globals (datasets, datasetLabels, rotationMap, hasImp) not found.")

for dsIdx, datasetName in enumerate(datasetLabels):
    print(f"Checking dataset: {datasetName}")
   
    if 'CGImp' not in datasetName:
        continue
   
    impFlag = hasImp[dsIdx]
    if not impFlag:
        print(f" Skipping {datasetName}: no implicit component")
        continue
   
    modelLists = datasets[dsIdx]
    rotations = rotationMap[dsIdx]
   
    for rotIdx, rotation in enumerate(rotations):
        if rotation not in target_rotations:
            continue
       
        print(f" Found {rotation}° rotation in {datasetName}")
       
                                              
        if rotation not in rotation_params:
            print(f" No parameters defined for {rotation}°, skipping")
            continue
        lr2, retain2 = rotation_params[rotation]
       
        ssm_fit = modelLists[0][rotIdx]
       
        if ssm_fit is None or not hasattr(ssm_fit, 'allAims') or len(ssm_fit.allAims) == 0:
            print(" No participant data, skipping")
            continue
       
        n_participants = len(ssm_fit.allAims)
        if n_participants == 0:
            continue
       
        if not hasattr(ssm_fit, 'targetPositions'):
            raise AttributeError(f"{datasetName} rot{rotation}: ssm_fit has no 'targetPositions' attribute.")
       
        targets_all = np.asarray(ssm_fit.targetPositions)
        print(f" targets_all shape: {targets_all.shape}")
       
        if targets_all.ndim != 2 or targets_all.shape[0] != n_participants:
            print(" Unexpected targetPositions structure, skipping rotation")
            continue
       
        n_trials = targets_all.shape[1]
       
        for pId in range(3,4):                                       
            targets_part = targets_all[pId]
            allAims_part = np.asarray(ssm_fit.allAims[pId])
           
            if len(allAims_part) != n_trials:
                print(f" Participant {pId} allAims length mismatch, skipping")
                continue
           
            participant_id = f"{datasetName}_p{pId:02d}_rot{rotation}"
            participant_counter += 1
           
            for gen_type in ['target', 'aim']:
                implicit_sim = simulate_implicit_only(
                    targets=targets_part[40:360],
                    allAims_part=allAims_part[40:360],
                    rotation=rotation,
                    gen_type=gen_type,
                    lr2=lr2,
                    retain2=retain2,
                    sigmaGen=30
                )
               
                for t, imp in enumerate(implicit_sim):
                    data_rows.append({
                        'participant': participant_id,
                        'trial': t + 1,
                        'implicit_compensation': imp,
                        'gen_type': gen_type,
                        'dataset': datasetName,
                        'rotation': rotation
                    })

                          
if not data_rows:
    print("No data collected. Check if any of the target rotations exist in the dataset.")
else:
    implicit_df = pd.DataFrame(data_rows)
    implicit_df = implicit_df.sort_values(['rotation', 'participant', 'gen_type', 'trial']).reset_index(drop=True)
    
                                       
    for rot in sorted(implicit_df['rotation'].unique()):
        sub_df = implicit_df[implicit_df['rotation'] == rot]
        csv_name = f'implicit_simulations_CGImp_{int(rot)}deg.csv'
        sub_df.to_csv(csv_name, index=False)
        print(f"Saved {csv_name} ({len(sub_df)} rows)")
    
                        
    implicit_df.to_csv('implicit_simulations_CGImp_variable_params_all.csv', index=False)
    
    print(f"\nDone! Processed rotations: {sorted(implicit_df['rotation'].unique())}")
    print(f"Total rows: {len(implicit_df)}")
    print("\nFirst few rows:")
    print(implicit_df.head(20))

In [ ]:


                               
                                             
                               
if 'implicit_df' not in globals() or implicit_df.empty:
    raise ValueError("implicit_df not found or empty. Run the simulation code first.")

unique_rots = sorted(implicit_df['rotation'].unique())
print(f"Unique rotations found: {unique_rots}")

metrics = {}
max_trial = 0

for rotation in unique_rots:
    sub_df = implicit_df[implicit_df['rotation'] == rotation]
    if sub_df.empty:
        continue
    
    stats_df = (sub_df.groupby(['trial', 'gen_type'])['implicit_compensation']
                .agg(['mean', 'sem'])
                .reset_index())
    stats_df = stats_df.sort_values(['gen_type', 'trial']).reset_index(drop=True)
    
    means_df = stats_df.pivot(index='trial', columns='gen_type', values='mean')
    
    if 'target' not in means_df.columns or 'aim' not in means_df.columns:
        print(f"Missing gen_type for {rotation}°, skipping metrics")
        r_squared = rmse = np.nan
    else:
        target_means = means_df['target'].values
        aim_means = means_df['aim'].values
        if len(target_means) < 2:
            r_squared = np.nan
        else:
            corr, _ = pearsonr(target_means, aim_means)
            r_squared = corr ** 2
        rmse = np.sqrt(np.mean((target_means - aim_means)**2))
    
    print(f"{rotation}°: R² = {r_squared:.3f}, RMSE = {rmse:.3f}°")
    metrics[rotation] = (r_squared, rmse)
    
    current_max = stats_df['trial'].max()
    if current_max > max_trial:
        max_trial = current_max

                               
                                           
                               
                             
selected_rots = [45, 90]
selected_rots = [r for r in selected_rots if r in metrics]

if not selected_rots:
    raise ValueError("None of the requested rotations (15, 45, 90) were found in the data.")

print(f"Plotting selected rotations on one panel: {selected_rots}")

fig, ax = plt.subplots(1, 1, figsize=(10, 6), constrained_layout=True)

for rotation in selected_rots:
    sub_df = implicit_df[implicit_df['rotation'] == rotation]
    stats_df = (sub_df.groupby(['trial', 'gen_type'])['implicit_compensation']
                .agg(['mean', 'sem'])
                .reset_index())
    stats_df = stats_df.sort_values(['gen_type', 'trial']).reset_index(drop=True)
    
                                               
    for gen_type, color in [('aim', 'magenta'), ('target', 'limegreen')]:
        sub = stats_df[stats_df['gen_type'] == gen_type]
        ax.plot(sub['trial'], -sub['mean'], color=color, linewidth=2, alpha=0.7,ls='-')
        """#all same so no fill
        ax.fill_between(sub['trial'],
                        -sub['mean'] - sub['sem'],
                        -sub['mean'] + sub['sem'],
                        color=color, alpha=0.25)
        """
        
                                                                                                             
x_text = max_trial * 1.05
for rotation in selected_rots:
    r_squared, rmse = metrics[rotation]
    
                                           
    sub_df = implicit_df[implicit_df['rotation'] == rotation]
    stats_df = (sub_df.groupby(['trial', 'gen_type'])['implicit_compensation']
                .agg(['mean', 'sem'])
                .reset_index())
    stats_df = stats_df.sort_values(['gen_type', 'trial']).reset_index(drop=True)
    means_df = stats_df.pivot(index='trial', columns='gen_type', values='mean')
    
    last_trial = means_df.index[-1]
    target_y = -means_df.loc[last_trial, 'target']
    aim_y = -means_df.loc[last_trial, 'aim']
    y_pos = (target_y + aim_y) / 2
    
    textstr = f'{int(rotation)}°\nR² = {r_squared:.3f}\nRMSE = {rmse:.3f}°'
    ax.text(x_text, y_pos, textstr, fontsize=20,
            ha='left', va='center',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray'))

                       
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, max_trial * 1.15)                              
ax.set_xlabel('Trial', fontsize=20)
ax.set_ylabel('Implicit Compensation (degrees)', fontsize=20)
ax.set_title('Average Implicit Compensation over Trials\n'
             'Selected Rotations (15°, 45°, 90°); Target vs Aim (magenta)\n'
             '(rotation-specific learning rate & retention; σ=30 fixed)',
             fontsize=20)

                                    
legend_elements = [
    Line2D([0], [0], color='limegreen', lw=2, label='Target'),
    Line2D([0], [0], color='magenta', lw=2, label='Aim')
]
ax.legend(handles=legend_elements, title='Generalisation Type',
          loc='upper left', fontsize=20)

plt.savefig(targetFolder+'implicitGenTypeExamples.svg',dpi=300)
           

In [ ]:

                                                   
def to_signed(ang):
    return np.where(ang > 180, ang - 360, ang)

                                                               
                                                           
def simulate_implicit_only(targets, allAims_part, rotation,
                          gen_type, lr2, retain2, sigmaGen=30):
    targets = np.asarray(targets)
    allAims_part = np.asarray(allAims_part)
    n = len(targets)
    rots = np.full(n, float(rotation))
   
    implicit = np.zeros(n)
    angles = np.arange(0.0, 360.0)
    x2Vec = np.zeros(360)
   
    signed_aims = to_signed(allAims_part % 360.0)
    out2 = 0
    pe = 0
    g2Vec = 0
    for i in range(n):
        if gen_type == 'target':
            raw_center = targets[i]
        elif gen_type == 'aim':
            raw_center = targets[i] + signed_aims[i]
        else:
            raise ValueError("gen_type must be 'target' or 'aim'")
        if not np.isnan(raw_center):
            idx = int(np.round(raw_center)) % 360
            center = raw_center % 360.0
           
            out2 = x2Vec[idx]
            implicit[i] = out2
           
            pe = rots[i] + out2
           
            angular_diff = np.minimum(np.abs(angles - center), 360.0 - np.abs(angles - center))
            g2Vec = np.exp(-(angular_diff**2) / (2 * sigmaGen**2))
           
            x2Vec = retain2 * x2Vec - lr2 * pe * g2Vec
        else:
            implicit[i] = out2
            x2Vec = retain2 * x2Vec - lr2 * pe * g2Vec
    return implicit

                                
data_rows = []

                                                                      
target_rotations = [15, 30, 45, 60, 90]

                                                     
                                                                             
rotation_params = {
    15: (0.0385, 0.9985),                 
    30: (0.0152, 0.9956),
    45: (0.0152, 0.9952),
    60: (0.0118, 0.9875),
    90: (0.0044, 0.9918)
}

participant_counter = 0

if 'datasets' not in globals() or 'datasetLabels' not in globals() or 'rotationMap' not in globals() or 'hasImp' not in globals():
    raise ValueError("Required globals (datasets, datasetLabels, rotationMap, hasImp) not found.")

for dsIdx, datasetName in enumerate(datasetLabels):
    print(f"Checking dataset: {datasetName}")
   
    if 'CGImp' not in datasetName:
        continue
   
    impFlag = hasImp[dsIdx]
    if not impFlag:
        print(f" Skipping {datasetName}: no implicit component")
        continue
   
    modelLists = datasets[dsIdx]
    rotations = rotationMap[dsIdx]
   
    for rotIdx, rotation in enumerate(rotations):
        if rotation not in target_rotations:
            continue
       
        print(f" Found {rotation}° rotation in {datasetName}")
       
                                              
        if rotation not in rotation_params:
            print(f" No parameters defined for {rotation}°, skipping")
            continue
        lr2, retain2 = rotation_params[rotation]
       
        ssm_fit = modelLists[0][rotIdx]
       
        if ssm_fit is None or not hasattr(ssm_fit, 'allAims') or len(ssm_fit.allAims) == 0:
            print(" No participant data, skipping")
            continue
       
        n_participants = len(ssm_fit.allAims)
        if n_participants == 0:
            continue
       
        if not hasattr(ssm_fit, 'targetPositions'):
            raise AttributeError(f"{datasetName} rot{rotation}: ssm_fit has no 'targetPositions' attribute.")
       
        targets_all = np.asarray(ssm_fit.targetPositions)
        print(f" targets_all shape: {targets_all.shape}")
       
        if targets_all.ndim != 2 or targets_all.shape[0] != n_participants:
            print(" Unexpected targetPositions structure, skipping rotation")
            continue
       
        n_trials = targets_all.shape[1]
       
        for pId in range(n_participants):                      
            targets_part = targets_all[pId]
            allAims_part = np.asarray(ssm_fit.allAims[pId])
           
            if len(allAims_part) != n_trials:
                print(f" Participant {pId} allAims length mismatch, skipping")
                continue
           
            participant_id = f"{datasetName}_p{pId:02d}_rot{rotation}"
            participant_counter += 1
           
            for gen_type in ['target', 'aim']:
                implicit_sim = simulate_implicit_only(
                    targets=targets_part[40:360],
                    allAims_part=allAims_part[40:360],
                    rotation=rotation,
                    gen_type=gen_type,
                    lr2=lr2,
                    retain2=retain2,
                    sigmaGen=30
                )
               
                for t, imp in enumerate(implicit_sim):
                    data_rows.append({
                        'participant': participant_id,
                        'trial': t + 1,
                        'implicit_compensation': imp,
                        'gen_type': gen_type,
                        'dataset': datasetName,
                        'rotation': rotation
                    })

                          
if not data_rows:
    print("No data collected. Check if any of the target rotations exist in the dataset.")
else:
    implicit_df = pd.DataFrame(data_rows)
    implicit_df = implicit_df.sort_values(['rotation', 'participant', 'gen_type', 'trial']).reset_index(drop=True)
    
                                       
    for rot in sorted(implicit_df['rotation'].unique()):
        sub_df = implicit_df[implicit_df['rotation'] == rot]
        csv_name = f'implicit_simulations_CGImp_{int(rot)}deg.csv'
        sub_df.to_csv(csv_name, index=False)
        print(f"Saved {csv_name} ({len(sub_df)} rows)")
    
                        
    implicit_df.to_csv('implicit_simulations_CGImp_variable_params_all.csv', index=False)
    
    print(f"\nDone! Processed rotations: {sorted(implicit_df['rotation'].unique())}")
    print(f"Total rows: {len(implicit_df)}")
    print("\nFirst few rows:")
    print(implicit_df.head(20))

In [ ]:
                                                                              
print("\nComputing per-participant R² and RMSE...")

per_part_metrics = []

for participant in implicit_df['participant'].unique():
    sub_df = implicit_df[implicit_df['participant'] == participant]
    
                                                          
    rotation = sub_df['rotation'].unique()[0]
    
                                          
    pivot_df = sub_df.pivot(index='trial', columns='gen_type', values='implicit_compensation')
    
    if {'target', 'aim'}.issubset(pivot_df.columns):
        target_vals = pivot_df['target'].dropna().values
        aim_vals = pivot_df['aim'].dropna().values
        
                                                               
        if len(target_vals) == len(aim_vals) and len(target_vals) >= 2:
            corr, _ = pearsonr(target_vals, aim_vals)
            r_squared = corr ** 2
            rmse = np.sqrt(np.mean((target_vals - aim_vals) ** 2))
            
            per_part_metrics.append({
                'participant': participant,
                'rotation': rotation,
                'r_squared': r_squared,
                'rmse_deg': rmse,
                'n_trials': len(target_vals)
            })

per_part_df = pd.DataFrame(per_part_metrics)

if not per_part_df.empty:
                                    
    summary = per_part_df.groupby('rotation')[['r_squared', 'rmse_deg']].agg(
        ['mean', 'std', 'count', 'min', 'max']
    )
    print("\nPer-participant summary by rotation:")
    print(summary.round(3))
    
             
    per_part_df.to_csv('implicit_simulations_per_participant_metrics.csv', index=False)
    print(f"\nSaved per-participant metrics ({len(per_part_df)} participants)")
else:
    print("No per-participant metrics computed.")

In [ ]:

                                                                                
rotation_colors = list(reversed(['#44AA99', '#88CCEE', '#FF9825', '#CC6677', '#AA4499']))

if not per_part_df.empty:
                                                                         
    unique_rots = sorted(per_part_df['rotation'].unique())
    
                                                                  
    per_part_df['rotation_str'] = per_part_df['rotation'].apply(
        lambda x: f"{int(x)}°" 
    )
    
                                                  
    hue_order = [f"{int(r)}°" for r in unique_rots]
    
                                                                               
    palette = dict(zip(hue_order, rotation_colors[:len(unique_rots)]))
    
                                                                        
                                                      
                                                                        
    fig, axs = plt.subplots(1, 2, figsize=(10, 6))
    
                                      
    ax = axs[0]
                                                                                                  
    sns.boxplot(data=per_part_df, y='r_squared', ax=ax,
                color='white', linewidth=2, fliersize=0,
                boxprops=dict(edgecolor='black'),
                whiskerprops=dict(color='black'),
                capprops=dict(color='black'),
                medianprops=dict(color='black'))
    
                                       
    sns.swarmplot(data=per_part_df, y='r_squared', hue='rotation_str',
                  palette=palette, hue_order=hue_order,
                  ax=ax, size=6, alpha=0.85, edgecolor='white', linewidth=0.5)
    
    ax.set_title('Per-participant R²', fontsize=20)
    ax.set_ylabel('R²', fontsize=20)
    ax.set_xlabel('')
    ax.set_xticks([])                                           
    ax.legend_.remove()                                  
    
                                        
    ax = axs[1]
    sns.boxplot(data=per_part_df, y='rmse_deg', ax=ax,
                color='white', linewidth=2, fliersize=0,
                boxprops=dict(edgecolor='black'),
                whiskerprops=dict(color='black'),
                capprops=dict(color='black'),
                medianprops=dict(color='black'))
    
    sns.swarmplot(data=per_part_df, y='rmse_deg', hue='rotation_str',
                  palette=palette, hue_order=hue_order,
                  ax=ax, size=6, alpha=0.85, edgecolor='white', linewidth=0.5)
    
    ax.set_title('Per-participant RMSE', fontsize=20)
    ax.set_ylabel('RMSE (degrees)', fontsize=20)
    ax.set_xlabel('')
    ax.set_xticks([])
    
                                                                     
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, title='Rotation size',
              bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True)
    
                            
    fig.suptitle('Per-participant similarity metrics '
                 '(implicit target vs. aim compensation)\n'
                 f'n = {len(per_part_df)} participants', fontsize=20)
    sns.despine()
    plt.tight_layout(rect=[0, 0, 0.85, 0.95])                        
    
                   
    save_path = 'implicit_simulations_per_participant_metrics_box_swarm.svg'
    plt.savefig(targetFolder+save_path, dpi=300, bbox_inches='tight')
    print(f"\nSaved box + swarmplot figure to {save_path}")
               

In [ ]:

gc.collect()

In [ ]:

                                                                                                                 
print("\nComputing per-trial target vs aim pairs across all participants...")

trial_pairs = []

for participant in implicit_df['participant'].unique():
    sub_df = implicit_df[implicit_df['participant'] == participant].copy()
    
                                       
    rotation = sub_df['rotation'].unique()[0]
    
                                                                 
    pivot = sub_df.pivot(index='trial', columns='gen_type', values='implicit_compensation').reset_index()
    
    if {'target', 'aim'}.issubset(pivot.columns):
        for _, row in pivot.iterrows():
            trial_pairs.append({
                'participant': participant,
                'rotation': int(rotation),
                'trial': int(row['trial']),
                'target_imp': row['target'],
                'aim_imp': row['aim']
            })

trial_df = pd.DataFrame(trial_pairs)

if trial_df.empty:
    raise ValueError("No per-trial pairs computed – check data structure.")

                                                         
trial_df['target_neg'] = -trial_df['target_imp']
trial_df['aim_neg'] = -trial_df['aim_imp']

                                                                                    
target_all = trial_df['target_imp'].values
aim_all = trial_df['aim_imp'].values
corr, _ = pearsonr(target_all, aim_all)
r_squared = corr ** 2
rmse = np.sqrt(np.mean((target_all - aim_all) ** 2))

print(f"\nPer-trial overall similarity (all {len(trial_df)} trial points):")
print(f"  R² = {r_squared:.3f}, RMSE = {rmse:.3f}°")

                                
trial_df.to_csv('implicit_simulations_per_trial_target_vs_aim_pairs.csv', index=False)
print(f"Saved per-trial pairs ({len(trial_df)} rows)")

                                                                                                              
fig, ax = plt.subplots(1, 1, figsize=(12, 10), constrained_layout=True)

                                 
plot_x = trial_df['target_neg']
plot_y = trial_df['aim_neg']

                                                                                       
sns.scatterplot(
    x=plot_x,
    y=plot_y,
    color='none',                                              
    s=15,                                 
    alpha=0.2,                                                                            
    edgecolor='black',                                       
    lw = 0.2,
    ax=ax
)

                                        
all_vals = np.concatenate([plot_x, plot_y])
margin = (all_vals.max() - all_vals.min()) * 0.05
overall_min = all_vals.min() - margin
overall_max = all_vals.max() + margin
ax.plot([overall_min, overall_max], [overall_min, overall_max],
        color='limegreen', linestyle='--', linewidth=3)

              
textstr = f'R² = {r_squared:.3f}\nRMSE = {rmse:.3f}°\nN = {len(trial_df)} trial points\n(all participants & rotations pooled)'
ax.text(0.05, 0.95, textstr,
        transform=ax.transAxes,
        va='top', ha='left',
        fontsize=13,
        bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray'))

                                                
ax.set_xlabel('Target Generalization\n(-Implicit Compensation °)', fontsize=13)
ax.set_ylabel('Aim Generalization\n(-Implicit Compensation °)', fontsize=13)
ax.set_title('Trial-by-Trial Similarity between Target and Aim Generalization\n'
             '(Negated Implicit Compensation; Every single trial, all participants & rotations pooled)',
             fontsize=16)

                                  
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.axvline(0, color='gray', linestyle=':', linewidth=1)

           

In [ ]:

                                                                                                                 
print("\nComputing per-trial target vs aim pairs across all participants...")
trial_pairs = []
for participant in implicit_df['participant'].unique():
    sub_df = implicit_df[implicit_df['participant'] == participant].copy()
   
                                       
    rotation = sub_df['rotation'].unique()[0]
   
                                                                 
    pivot = sub_df.pivot(index='trial', columns='gen_type', values='implicit_compensation').reset_index()
   
    if {'target', 'aim'}.issubset(pivot.columns):
        for _, row in pivot.iterrows():
            trial_pairs.append({
                'participant': participant,
                'rotation': int(rotation),
                'trial': int(row['trial']),
                'target_imp': row['target'],
                'aim_imp': row['aim']
            })
trial_df = pd.DataFrame(trial_pairs)
if trial_df.empty:
    raise ValueError("No per-trial pairs computed – check data structure.")
                                                         
trial_df['target_neg'] = -trial_df['target_imp']
trial_df['aim_neg'] = -trial_df['aim_imp']
                                                                                    
target_all = trial_df['target_imp'].values
aim_all = trial_df['aim_imp'].values
corr, _ = pearsonr(target_all, aim_all)
r_squared = corr ** 2
rmse = np.sqrt(np.mean((target_all - aim_all) ** 2))
print(f"\nPer-trial overall similarity (all {len(trial_df)} trial points):")
print(f" R² = {r_squared:.3f}, RMSE = {rmse:.3f}°")
                                
trial_df.to_csv('implicit_simulations_per_trial_target_vs_aim_pairs.csv', index=False)
print(f"Saved per-trial pairs ({len(trial_df)} rows)")

                                                                                            
fig, ax = plt.subplots(1, 1, figsize=(14, 10), constrained_layout=True)

                                 
plot_x = trial_df['target_neg']
plot_y = trial_df['aim_neg']

                   
all_vals = np.concatenate([plot_x, plot_y])
margin = (all_vals.max() - all_vals.min()) * 0.05
overall_min = all_vals.min() - margin
overall_max = all_vals.max() + margin

                                                                                     
hb = ax.hexbin(plot_x, plot_y, 
               gridsize=60,                                                              
               cmap='Blues',                                                                            
               bins='log',                                                                 
               mincnt=1,                                                    
               extent=[overall_min, overall_max, overall_min, overall_max])

ax.set_xlim(overall_min, overall_max)
ax.set_ylim(overall_min, overall_max)

               
ax.plot([overall_min, overall_max], [overall_min, overall_max],
        color='black', linestyle='--', linewidth=2)
ax.xaxis.set_major_locator(MultipleLocator(5))
ax.yaxis.set_major_locator(MultipleLocator(5))
ax.xaxis.set_minor_locator(AutoMinorLocator(2))
ax.yaxis.set_minor_locator(AutoMinorLocator(2))
ax.set_aspect('equal', adjustable='box')
          
cb = fig.colorbar(hb, ax=ax, shrink=0.8)
cb.set_label('log$_{10}$(Number of trials per bin)', fontsize=20)

              
textstr = f'R² = {r_squared:.3f}\nRMSE = {rmse:.3f}°\nN = {len(trial_df)} trial points\n(all participants & rotations pooled)'
ax.text(0.05, 0.95, textstr,
        transform=ax.transAxes,
        va='top', ha='left',
        fontsize=20,
        bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray'))

                                           
ax.set_xlabel('Target Generalization\n(-Implicit Compensation °)', fontsize=20)
ax.set_ylabel('Aim Generalization\n(-Implicit Compensation °)', fontsize=20)
ax.set_title('Trial-by-Trial Similarity between Target and Aim Generalization\n'
             '(Negated Implicit Compensation; Every single trial, all participants & rotations pooled)',
             fontsize=16)
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.axvline(0, color='gray', linestyle=':', linewidth=1)

                                                                
diff = plot_y - plot_x
mean_diff = np.mean(diff)
std_diff = np.std(diff)

inset_ax = ax.inset_axes([0.565, 0.07, 0.42, 0.42])
inset_ax.hist(diff, bins=120, color='darkblue', alpha=0.8, edgecolor='black', linewidth=0.5)
inset_ax.axvline(0, color='red', linewidth=2, label='Zero difference')
inset_ax.axvline(mean_diff, color='black', linestyle='--', linewidth=1.5)

inset_ax.set_xlabel('Aim − Target (°)', fontsize=20)
inset_ax.set_ylabel('Count', fontsize=20)
inset_ax.set_title('Residuals Distribution', fontsize=20)
inset_ax.tick_params(axis='both', which='major', labelsize=20)

                
stats_text = f'Mean: {mean_diff:.2f}°\nSD: {std_diff:.2f}°\nRMSE: {rmse:.2f}°'
inset_ax.text(0.95, 0.95, stats_text, transform=inset_ax.transAxes,
              va='top', ha='right', fontsize=20,
              bbox=dict(facecolor='white', alpha=0.85, edgecolor='none'))
inset_ax.patch.set_visible(False)
plt.savefig(targetFolder+'implicitGenTypeHexbin.svg',dpi=300)
           

In [ ]:
importlib.reload(DM)
importlib.reload(hmm)
importlib.reload(hmmImp)
importlib.reload(hmmKI)

                                                                                                                              
                                                                              
                                                                                                                                                      
hmmsAll8BT        
hmmsAll8CG     
hmmsAll8CGIMP
hmmsAllBrudner     

In [ ]:
for rot in hmmsAll8BT:
    xs = rot.xs
    humanExps = rot.human_explicits
    humanImps = rot.human_implicits
    for pp in range(len(xs)):
        indiParams = xs[pp]
        indiExps = humanExps[pp]
        indiImps = humanImps[pp]
        modelTotal, modelExp, ModelImp,humanTotal, _, _, rmse, r2 = generateData(
            params=indiParams,
            rots=rots,
            trials=np.arange(n_trials),
            aims=participant_aims,
            imps=participant_imps,
            targetPositions=target_pos,
            use_implicit=True
        )

In [ ]:
import NonGenSSMWithNoise as NonGenSSM
import HMM as hmm
import QLearning as qLearn
import NonGenDualProcessSSMWithNoise as SSMImp
import HMMImplicit as hmmImp
import QLearningImplicit as qLearnImp
import DualProcessKnownImplicit as SSMImpKI
import HMMKnownImplicit as hmmKI
import QLearningKnownImplicit as qLearnKI

In [ ]:
hmmsAll8BT

In [ ]:
                        
np.shape(hmmsAllBrudner[0].model_explicit_samples)       